<a href="https://colab.research.google.com/github/rodricanaglia/reporte-gerencial-grapes/blob/main/reporte_grapes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Montar sesion

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [2]:
!pip install fuzzywuzzy python-Levenshtein
!pip install python-pptx matplotlib
!pip install python-docx

#INICIALIZACION

In [3]:
import os
import re
import numpy as np
import pandas as pd

# =========================
# CONFIG
# =========================
root_path = "/content/gdrive/My Drive/Grapes/reporte_grapes/inicializacion/"
MASTER_PARQUET = os.path.join(root_path, "master_articulo_venta.parquet")

HIST_FILE    = os.path.join(root_path, "reporte_articulo_venta_31dic25.xlsx")
HIST_PERIODO = "2025-12"

CAB_RE = re.compile(r"^(fc|nc|nd)\s*[ab]\b", re.IGNORECASE)

# =========================
# MAPEO P/N → EAN-13
# =========================
_EAN_MAP_FILE = os.path.join(root_path, "ean_codigos.xlsx")

def _cargar_pn_a_ean(path: str) -> dict:
    if not os.path.exists(path):
        print(f"⚠️ No se encontró {path} — sin mapeo P/N→EAN.")
        return {}
    df = pd.read_excel(path, dtype=str)
    df.columns = [c.strip() for c in df.columns]
    df = df.dropna(subset=["P/N", "EAN-13"])
    df["P/N"]    = df["P/N"].str.strip().str.upper()
    df["EAN-13"] = df["EAN-13"].str.strip()
    m = dict(zip(df["P/N"], df["EAN-13"]))
    print(f"✅ Mapeo P/N→EAN cargado: {len(m)} entradas.")
    return m

PN_A_EAN = _cargar_pn_a_ean(_EAN_MAP_FILE)

def aplicar_ean(serie: pd.Series) -> pd.Series:
    """Reemplaza P/N por EAN-13 donde exista mapeo."""
    return serie.map(lambda x: PN_A_EAN.get(str(x).strip().upper(), x) if pd.notna(x) else x)

# =========================
# PARSER NUM (AR/US) — unificado
# =========================
def parse_num_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan})
    s = s.str.replace(r"[\$\s]", "", regex=True)

    has_comma = s.str.contains(",", na=False)
    has_dot   = s.str.contains(r"\.", na=False)

    both = has_comma & has_dot
    if both.any():
        sb = s[both]
        ar = sb.str.rfind(",") > sb.str.rfind(".")
        s.loc[sb.index[ar]]  = sb[ar].str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
        s.loc[sb.index[~ar]] = sb[~ar].str.replace(",", "", regex=False)

    only_comma = has_comma & ~has_dot
    if only_comma.any():
        s.loc[only_comma] = s.loc[only_comma].str.replace(",", ".", regex=False)

    return pd.to_numeric(s, errors="coerce")

# =========================
# PARSER EXCEL → DF LIMPIO
# =========================
def parse_reporte_articulo_venta_xlsx(input_file: str) -> pd.DataFrame:
    df_raw = pd.read_excel(
        input_file,
        header=None,
        dtype=str,
        usecols=list(range(9)),
        engine="openpyxl"
    )

    df = df_raw.drop([0, 1, 2], errors="ignore").reset_index(drop=True)

    df.columns = [
        "Comprobante",
        "Total Neto",
        "Total Final",
        "Costo Total",
        "Cantidad",
        "Interés S/Imp",
        "Impuestos",
        "$Profit",
        "$ProfitII"
    ]

    df = df.dropna(how="all").reset_index(drop=True)

    txt       = df["Comprobante"].astype(str).str.strip()
    is_header = txt.str.match(CAB_RE)

    df["comprobante_articulo_venta"] = np.where(is_header, txt, np.nan)
    df["comprobante_articulo_venta"] = df["comprobante_articulo_venta"].ffill()

    is_valid_txt = txt.notna() & (txt.str.lower() != "nan") & (txt != "")
    is_item      = (~is_header) & is_valid_txt & df["comprobante_articulo_venta"].notna()

    out = df.loc[is_item, [
        "comprobante_articulo_venta",
        "Comprobante",
        "Cantidad",
        "Total Neto",
        "Total Final",
        "Costo Total",
        "Interés S/Imp",
        "Impuestos",
        "$Profit",
        "$ProfitII"
    ]].copy()

    out = out.rename(columns={
        "Comprobante":    "codigo_articulo_venta",
        "Cantidad":       "cantidad_articulo_venta",
        "Total Neto":     "total_neto_articulo_venta",
        "Total Final":    "total_final_articulo_venta",
        "Costo Total":    "costo_total_articulo_venta",
        "Interés S/Imp":  "interes_sin_imp_articulo_venta",
        "Impuestos":      "impuestos_articulo_venta",
        "$Profit":        "profit_articulo_venta",
        "$ProfitII":      "profit_ii_articulo_venta",
    })

    # Quitar subtotales
    mask_total = out["codigo_articulo_venta"].astype(str).str.contains("total", case=False, na=False)
    out = out[~mask_total].reset_index(drop=True)

    # Numéricos
    num_cols = [
        "cantidad_articulo_venta",
        "total_neto_articulo_venta",
        "total_final_articulo_venta",
        "costo_total_articulo_venta",
        "interes_sin_imp_articulo_venta",
        "impuestos_articulo_venta",
        "profit_articulo_venta",
        "profit_ii_articulo_venta",
    ]
    for c in num_cols:
        out[c] = parse_num_series(out[c])

    # Normalizar strings
    out["comprobante_articulo_venta"] = (
        out["comprobante_articulo_venta"].astype(str).str.strip().str.lower()
    )
    out["codigo_articulo_venta"] = (
        out["codigo_articulo_venta"].astype(str).str.strip().str.upper()
    )

    # >>> MAPEO P/N → EAN (datos nuevos del Excel)
    out["codigo_articulo_venta"] = aplicar_ean(out["codigo_articulo_venta"])

    return out

# =========================
# ROW KEY
# =========================
def build_row_key(df: pd.DataFrame) -> pd.Series:
    return (
        df["comprobante_articulo_venta"].astype(str) + "||" +
        df["codigo_articulo_venta"].astype(str) + "||" +
        df["cantidad_articulo_venta"].fillna(0).round(6).astype(str) + "||" +
        df["total_neto_articulo_venta"].fillna(0).round(6).astype(str) + "||" +
        df["costo_total_articulo_venta"].fillna(0).round(6).astype(str)
    )

def load_master() -> pd.DataFrame:
    if os.path.exists(MASTER_PARQUET):
        df = pd.read_parquet(MASTER_PARQUET)
        # >>> MAPEO P/N → EAN (datos históricos del parquet)
        if "codigo_articulo_venta" in df.columns:
            df["codigo_articulo_venta"] = aplicar_ean(df["codigo_articulo_venta"])
        return df
    return pd.DataFrame()

def save_master(df_master: pd.DataFrame) -> None:
    df_master.to_parquet(MASTER_PARQUET, index=False)

def update_master_with_month(input_file: str, periodo: str, replace_period: bool = True) -> pd.DataFrame:
    if not os.path.exists(input_file):
        raise FileNotFoundError(f"No existe: {input_file}")

    new_df = parse_reporte_articulo_venta_xlsx(input_file)
    new_df["_periodo"]      = str(periodo)
    new_df["_row_key"]      = build_row_key(new_df)
    new_df["_source_file"]  = os.path.basename(input_file)
    new_df["_source_mtime"] = int(os.path.getmtime(input_file))

    master = load_master()

    if not master.empty:
        if "_row_key" not in master.columns:
            master["_row_key"] = build_row_key(master)
        if "_periodo" not in master.columns:
            master["_periodo"] = ""
        if replace_period:
            master = master[master["_periodo"].astype(str) != str(periodo)].copy()

    combined = pd.concat([master, new_df], ignore_index=True)
    combined = combined.drop_duplicates(subset=["_row_key"], keep="last").reset_index(drop=True)

    save_master(combined)
    return combined

def master_to_df_reporte(master: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "comprobante_articulo_venta",
        "codigo_articulo_venta",
        "cantidad_articulo_venta",
        "total_neto_articulo_venta",
        "total_final_articulo_venta",
        "costo_total_articulo_venta",
        "interes_sin_imp_articulo_venta",
        "impuestos_articulo_venta",
        "profit_articulo_venta",
        "profit_ii_articulo_venta",
    ]
    if master.empty:
        return pd.DataFrame(columns=cols)
    return master[cols].copy()

# =========================
# BOOTSTRAP
# =========================
df_master_art_venta = load_master()

if df_master_art_venta.empty:
    df_master_art_venta = update_master_with_month(HIST_FILE, HIST_PERIODO, replace_period=True)

df_reporte_articulo_venta = master_to_df_reporte(df_master_art_venta)

print("✅ Master filas:", len(df_master_art_venta))
print("✅ df_reporte_articulo_venta filas:", len(df_reporte_articulo_venta))
print("✅ Periodos:", sorted(df_master_art_venta["_periodo"].dropna().unique().tolist()) if not df_master_art_venta.empty else [])

✅ Mapeo P/N→EAN cargado: 49 entradas.
✅ Master filas: 10096
✅ df_reporte_articulo_venta filas: 10096
✅ Periodos: ['2025-12', '2026-01', '2026-02', '2026-03']


In [4]:
import pandas as pd
import re
import numpy as np

file_path = '/content/gdrive/My Drive/Grapes/reporte_grapes/inicializacion/costos_grapes2026.xlsx'
df_costos = pd.read_excel(file_path)
df_costos.columns = df_costos.columns.str.strip()

df_costos = df_costos.rename(columns={
    "Codigo":      "Codigo_costo",
    "Descr¡pcion": "Articulo_costo",
    "Costo 1":     "costo_neto_unitario",
    "Moneda 1":    "Moneda_costo",
})

df_costos["Moneda_costo"] = df_costos["Moneda_costo"].astype(str).str.strip().str.upper()
df_costos = df_costos[df_costos["Moneda_costo"].isin(["USD", "US$"])].copy()

df_costos["costo_neto_unitario"] = pd.to_numeric(
    df_costos["costo_neto_unitario"], errors="coerce").fillna(0)

def normalizar_codigo_excel(x):
    if pd.isna(x): return ""
    if isinstance(x, (float, np.floating)) and float(x).is_integer():
        s = str(int(x))
    elif isinstance(x, (int, np.integer)):
        s = str(int(x))
    else:
        s = str(x)
    s = s.strip().upper()
    s = re.sub(r"[‐\-‒–—]", "-", s)
    s = re.sub(r"\s*-\s*", "-", s)
    s = re.sub(r"\s+", " ", s)
    return s

df_costos["Codigo_costo_raw"]   = df_costos["Codigo_costo"]
df_costos["Codigo_costo"]       = df_costos["Codigo_costo_raw"].astype(str).str.strip().str.upper()
df_costos["Codigo_key_strict"]  = df_costos["Codigo_costo_raw"].apply(normalizar_codigo_excel)
df_costos["Codigo_key_relaxed"] = df_costos["Codigo_key_strict"].apply(
    lambda s: re.sub(r"[^A-Z0-9]", "", s))

# Aplicar mapeo P/N → EAN (igual que df_merged_usd)
df_costos["Codigo_costo"]      = aplicar_ean(df_costos["Codigo_costo"])
df_costos["Codigo_key_strict"] = aplicar_ean(
    pd.Series(df_costos["Codigo_key_strict"].tolist())).values

# ── Dedup: si el mismo EAN aparece dos veces (P/N viejo + EAN nuevo),
#    nos quedamos con la fila que tenga el costo más alto (no la que tenga 0)
df_costos = (df_costos
    .sort_values("costo_neto_unitario", ascending=True)   # el mayor queda último
    .drop_duplicates("Codigo_costo", keep="last")          # keep="last" = el mayor
    .reset_index(drop=True))

col_strict = df_costos["Codigo_key_strict"].duplicated(keep=False).sum()
print(f"Filas tras dedup: {len(df_costos)}")
print(f"Colisiones key_strict: {col_strict}")
print(f"Nulos costo: {(df_costos['costo_neto_unitario']==0).sum()}")

if "Articulo_costo" in df_costos.columns:
    df_costos["Articulo_costo"] = (
        df_costos["Articulo_costo"].astype(str).str.strip()
        .str.replace(r"\s+", " ", regex=True))

print("✅ df_costos listo")
print(df_costos[["Codigo_costo","Articulo_costo","costo_neto_unitario"]].head())

Filas tras dedup: 593
Colisiones key_strict: 0
Nulos costo: 6
✅ df_costos listo
           Codigo_costo                                     Articulo_costo  \
0      AXB32-192-40-JUN  Modulo SFP+ BiDi 10G 40KM 10GBase-BX-D SMF Sin...   
1      AXB23-192-40-JUN  Modulo SFP+ BiDi 10G 40KM 10GBase-BX-U SMF Sin...   
2      AXBXX-192-40-JUN  PAR de Modulos SFP+ BiDi 10G 40KM 10GBase-BX S...   
3  AXS85-192-M3-JUNIPER  Modulo SFP+ 10G 300Mts 850nm 10GBase-SR MMF Du...   
4         6920075741865  Modulo Interruptor Inteligente WI-FI Sonoff Ba...   

   costo_neto_unitario  
0                  0.0  
1                  0.0  
2                  0.0  
3                  0.0  
4                  0.0  


In [5]:
import pandas as pd
import re
import numpy as np

# --- Configuración de rutas ---
input_path  = "/content/gdrive/My Drive/Grapes/reporte_grapes/inicializacion/planes_tarjeta_web.xlsx"

# 1) Cargar el Excel en df_tarjetas
df_tarjetas = pd.read_excel(input_path)

# Limpieza de headers
df_tarjetas.columns = (
    df_tarjetas.columns.astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print("Columnas originales:", df_tarjetas.columns.tolist())

# 2) Eliminar las tres primeras columnas (auditable)
cols_drop = df_tarjetas.columns[:3].tolist()
df_tarjetas = df_tarjetas.drop(columns=cols_drop)
print("Columnas eliminadas (primeras 3):", cols_drop)

# 3) Renombrar columnas
mapeo = {
    "Tarjeta de Credito": "tarjeta_credito",
    "Codigo": "codigo_tarjeta",
    "Descripcion": "descripcion_tarjeta",
    "Nota": "nota",
    "Deshabilitado": "deshabilitado",
    "Cantidad de Cuotas": "cantidad_cuotas",
    "Porcentaje": "porcentaje_interes_cuotas",
    "Porcentaje Adicional": "porcentaje_adicional",
    "Grupo de Sucursales": "grupo_sucursales",
    "Porcentaje en Cobranzas para Interes por pago de Cuotas": "porcentaje_interes_cobranzas",
    "Forzar Condicion de Venta para Sitio Web": "condicion_venta_web",
    "Usuario de Creacion": "usuario_creacion",
    "Usuario de ultima Modificacion": "usuario_ultima_modificacion",
    "Ultima Fecha de Modificacion": "ultima_fecha_modificacion",
    "ID": "id"
}
df_tarjetas = df_tarjetas.rename(columns=mapeo)

# Validación: qué columnas esperadas NO aparecieron
esperadas = set(mapeo.values())
presentes = set(df_tarjetas.columns)
faltan = sorted(list(esperadas - presentes))
if faltan:
    print("⚠️ Columnas esperadas que NO quedaron tras rename:", faltan)

# 4) Reordenar columnas (solo las existentes)
orden = [
    "tarjeta_credito", "codigo_tarjeta", "descripcion_tarjeta",
    "nota", "deshabilitado", "cantidad_cuotas",
    "porcentaje_interes_cuotas", "porcentaje_adicional",
    "grupo_sucursales", "porcentaje_interes_cobranzas",
    "condicion_venta_web", "usuario_creacion",
    "usuario_ultima_modificacion", "ultima_fecha_modificacion", "id"
]
df_tarjetas = df_tarjetas[[c for c in orden if c in df_tarjetas.columns]]

# 5) Convertir porcentajes enteros a decimales (incluye adicional)
for c in ["porcentaje_interes_cuotas", "porcentaje_interes_cobranzas", "porcentaje_adicional"]:
    if c in df_tarjetas.columns:
        df_tarjetas[c] = pd.to_numeric(df_tarjetas[c], errors="coerce") / 100

# 6) Tipos recomendados
if "cantidad_cuotas" in df_tarjetas.columns:
    df_tarjetas["cantidad_cuotas"] = pd.to_numeric(df_tarjetas["cantidad_cuotas"], errors="coerce").astype("Int64")

if "deshabilitado" in df_tarjetas.columns:
    # Normalizo a booleano sin mezclar tipos
    s = df_tarjetas["deshabilitado"].astype(str).str.strip().str.lower()
    df_tarjetas["deshabilitado"] = s.map({"1": True, "0": False, "true": True, "false": False})

if "ultima_fecha_modificacion" in df_tarjetas.columns:
    df_tarjetas["ultima_fecha_modificacion"] = pd.to_datetime(
        df_tarjetas["ultima_fecha_modificacion"],
        errors="coerce",
        dayfirst=True
    )

print("✅ DataFrame limpio:")
print(df_tarjetas.head(10))
print("\nDtypes:\n", df_tarjetas.dtypes)

print("Cuotas únicas:", sorted(df_tarjetas["cantidad_cuotas"].dropna().unique().tolist()))

for c in ["porcentaje_interes_cuotas","porcentaje_adicional","porcentaje_interes_cobranzas"]:
    if c in df_tarjetas.columns:
        print(c, "min/max:", df_tarjetas[c].min(), df_tarjetas[c].max())

print("\nDeshabilitados:")
print(df_tarjetas["deshabilitado"].value_counts(dropna=False))

print("\nCondición web (top):")
print(df_tarjetas["condicion_venta_web"].value_counts(dropna=False).head(10))



Columnas originales: ['Column1', '1', '2', 'Tarjeta de Credito', 'Codigo', 'Descripcion', 'Nota', 'Deshabilitado', 'Cantidad de Cuotas', 'Porcentaje', 'Porcentaje Adicional', 'Grupo de Sucursales', 'Porcentaje en Cobranzas para Interes por pago de Cuotas', 'Forzar Condicion de Venta para Sitio Web', 'Usuario de Creacion', 'Usuario de ultima Modificacion', 'Ultima Fecha de Modificacion', 'ID']
Columnas eliminadas (primeras 3): ['Column1', '1', '2']
✅ DataFrame limpio:
                       tarjeta_credito  codigo_tarjeta descripcion_tarjeta  \
0   01.American Express , Pesos  (ARS)     1CUOTA-Amex             1 CUOTA   
1   01.American Express , Pesos  (ARS)    AHORA12-Amex            AHORA 12   
2   01.American Express , Pesos  (ARS)    AHORA18-Amex            AHORA 18   
3          02.Matercard , Pesos  (ARS)   1CUOTA-Master             1 CUOTA   
4          02.Matercard , Pesos  (ARS)  AHORA12-Master            AHORA 12   
5          02.Matercard , Pesos  (ARS)  AHORA18-Master      

In [6]:
# ================================
# Cargar FULL + Cerrito y normalizar/renombrar columnas de valorización
# ================================
import pandas as pd
import os, re

base_path    = "/content/gdrive/My Drive/Grapes/reporte_grapes/inicializacion/"
file_full    = os.path.join(base_path, "stock_valorizado_mar26_full.xlsx")
file_cerrito = os.path.join(base_path, "stock_valorizado_mar26_cerrito.xlsx")

# 0) Leer headers para forzar como texto SOLO si existe "Codigo / EAN"
def _read_safe(path):
    cols = pd.read_excel(path, nrows=0).columns
    conv = {"Codigo / EAN": str} if "Codigo / EAN" in cols else None
    return pd.read_excel(path, converters=conv) if conv else pd.read_excel(path)

# 1) Leer ambos archivos
df_full    = _read_safe(file_full)
df_cerrito = _read_safe(file_cerrito)

# 2) Etiqueta de depósito
df_full["Deposito"]    = "FULL"
df_cerrito["Deposito"] = "Cerrito"

# 3) Concatenar
df_valorizacion_stock = pd.concat([df_full, df_cerrito], ignore_index=True)

# 4) Normalizar nombres de columnas (colapsar espacios múltiples)
def _norm_cols(cols):
    out = []
    for c in cols:
        c = str(c)
        c = re.sub(r"\s+", " ", c).strip()
        out.append(c)
    return out

df_valorizacion_stock.columns = _norm_cols(df_valorizacion_stock.columns)

# 5) Mapeo de columnas → esquema estándar
rename_map = {
    "Codigo / EAN":            "Codigo",
    "Articulo":                "Articulo",
    "Cantidad":                "Cantidad",
    "Costo":                   "Costo_usd",
    "Costo Valorizado":        "valor_stock_final",
    "Precio Valorizado":       "Precio_Valorizado",
    "Costo PPP Valorizado":    "Costo_PPP_Valorizado",
    "Impuestos del Articulo":  "Impuestos_Articulo",
    "Impuestos   del Articulo":"Impuestos_Articulo",
    "Costo PPP":               "Costo_PPP",
    "Moneda":                  "Moneda",
    "Deposito":                "Deposito",
    "Categoria":               "Categoria",
    "Sub Categoria":           "Sub_Categoria",
    "coslis_curr_desc":        "coslis_curr_desc",
    "Permitir Cantidad con Decimales":   "Permite_Cant_Decimales",
    "Permitir Cantidad   con Decimales": "Permite_Cant_Decimales"
}

rename_map = {k: v for k, v in rename_map.items() if k in df_valorizacion_stock.columns}
df_valorizacion_stock = df_valorizacion_stock.rename(columns=rename_map)

# 5.1) Codigo como texto + key conservadora (para merges/consistencia)
if "Codigo" in df_valorizacion_stock.columns:
    df_valorizacion_stock["Codigo"] = df_valorizacion_stock["Codigo"].astype(str).str.strip()

    # key: mayúsculas + normaliza espacios múltiples + normaliza guiones raros a "-"
    df_valorizacion_stock["Codigo_key"] = (
        df_valorizacion_stock["Codigo"]
        .str.upper()
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(r"[‐-‒–—]", "-", regex=True)
        .str.replace(r"\s*-\s*", "-", regex=True)
        .str.strip()
    )

# 6) Convertir a numérico las columnas que deben serlo (si existen)
numeric_candidates = [
    "Cantidad","Costo_usd","valor_stock_final","Precio_Valorizado",
    "Costo_PPP_Valorizado","Impuestos_Articulo","Costo_PPP","Precio"
]
for c in numeric_candidates:
    if c in df_valorizacion_stock.columns:
        df_valorizacion_stock[c] = pd.to_numeric(df_valorizacion_stock[c], errors="coerce")

# 7) Selección de columnas principales ya estandarizadas (si existen)
main_cols = ["Codigo","Articulo","Cantidad","Costo_usd","valor_stock_final","Moneda","Deposito"]
present   = [c for c in main_cols if c in df_valorizacion_stock.columns]
df_valorizacion_stock = df_valorizacion_stock[present + [c for c in df_valorizacion_stock.columns if c not in present]]

# 8) Mostrar para verificar
print("✅ DataFrame combinado y renombrado:")
print(df_valorizacion_stock.head())
if "Deposito" in df_valorizacion_stock.columns:
    print(df_valorizacion_stock["Deposito"].value_counts(dropna=False))

# Extra mínimo de control (clave para FULL/ML)
if "Codigo" in df_valorizacion_stock.columns:
    print("Ejemplos Codigo:", df_valorizacion_stock["Codigo"].head(10).tolist())


✅ DataFrame combinado y renombrado:
          Codigo                                           Articulo  Cantidad  \
0  6935364032630  Kit Extensor Powerline TP-LINK TL-WPA4220KIT V...         0   
1   850065783635                UDC PARTS BLADES KIT X2 46IN 405380         0   
2   885022017386  Adaptador 2.5" SATA a Dual M2 2280 SATA QNAP Q...         0   
3   885022022724  Adaptador 2.5" U2 a M2 2280 NVME PCIe4 QNAP QD...         0   
4   885022017591      Adaptador 2.5" U2 a M2 2280 NVME QNAP QDA-UMP         0   

     Costo_usd  valor_stock_final      Moneda Deposito  \
0     40.80374                0.0  USD            FULL   
1  95000.00000                0.0  USD            FULL   
2     67.21100                0.0  USD            FULL   
3     50.93000                0.0  USD            FULL   
4     51.24100                0.0  USD            FULL   

                   Categoria Sub_Categoria  Precio  Precio_Valorizado  \
0          Accesorios Varios   Adaptadores   40.80     

In [7]:
df_valorizacion_stock.columns

Index(['Codigo', 'Articulo', 'Cantidad', 'Costo_usd', 'valor_stock_final',
       'Moneda', 'Deposito', 'Categoria', 'Sub_Categoria', 'Precio',
       'Precio_Valorizado', 'coslis_curr_desc', 'Costo_PPP_Valorizado',
       'Impuestos_Articulo', 'Costo_PPP', 'Permite_Cant_Decimales',
       'Codigo_key'],
      dtype='object')

In [8]:
import os
import pandas as pd

# Base del proyecto
root_path = "/content/gdrive/My Drive/Grapes/reporte_grapes/inicializacion/"

# Carpeta donde guardás los excels mensuales
mensuales_path = os.path.join(root_path, "mensuales/")

# Carpeta donde guardás outputs / master
output_path_base = os.path.join(root_path, "outputs/")
os.makedirs(mensuales_path, exist_ok=True)
os.makedirs(output_path_base, exist_ok=True)

# Master (recomendado parquet)
master_path = os.path.join(root_path, "master_articulo_venta.parquet")

# Si ya existe master, lo cargás; si no, arrancás vacío
if os.path.exists(master_path):
    df_reporte_articulo_venta_master = pd.read_parquet(master_path)
    print("✅ Master cargado:", master_path, "| filas:", len(df_reporte_articulo_venta_master))
else:
    df_reporte_articulo_venta_master = pd.DataFrame()
    print("ℹ️ No existe master todavía. Queda vacío y se crea cuando agregues meses.")

✅ Master cargado: /content/gdrive/My Drive/Grapes/reporte_grapes/inicializacion/master_articulo_venta.parquet | filas: 10096


In [9]:
# =========================
# MESES NUEVOS (cargá los que quieras)
# =========================
NEW_FILES = [
    (os.path.join(mensuales_path, "reporte_articulo_venta_enero26.xlsx"), "2026-01"),
    (os.path.join(mensuales_path, "reporte_articulo_venta_febrero26.xlsx"), "2026-02"),
    (os.path.join(mensuales_path, "reporte_articulo_venta_marzo26.xlsx"), "2026-03"),

]

for fpath, per in NEW_FILES:
    df_master_art_venta = update_master_with_month(
        input_file=fpath,
        periodo=per,
        replace_period=True  # ✅ clave: si lo corrés otra vez con enero NO duplica
    )

# Re-armar el DF final "como antes" (sin columnas internas)
df_reporte_articulo_venta = master_to_df_reporte(df_master_art_venta)

print("✅ Master filas:", len(df_master_art_venta))
print("✅ df_reporte_articulo_venta filas:", len(df_reporte_articulo_venta))
print("✅ Periodos en master:", sorted(df_master_art_venta["_periodo"].dropna().unique().tolist()) if not df_master_art_venta.empty else [])

✅ Master filas: 10096
✅ df_reporte_articulo_venta filas: 10096
✅ Periodos en master: ['2025-12', '2026-01', '2026-02', '2026-03']


In [10]:
# =========================
# VERIFICACIÓN: exportar master completo a Excel
# =========================
verify_path = os.path.join(output_path_base, "verificacion_master_completo.xlsx")

df_master_art_venta.to_excel(verify_path, index=False)

print("✅ Exportado a:", verify_path)
print("✅ Total filas:", len(df_master_art_venta))
print("✅ Periodos:", sorted(df_master_art_venta["_periodo"].dropna().unique().tolist()))
print()
print("📊 Filas por periodo:")
print(df_master_art_venta.groupby("_periodo").size().to_string())
print()
print("📋 Primeras 5 filas:")
display(df_master_art_venta.head())
print()
print("📋 Últimas 5 filas:")
display(df_master_art_venta.tail())
print()
print("🔍 Columnas:", df_master_art_venta.columns.tolist())
print("🔍 Nulos por columna:")
print(df_master_art_venta.isnull().sum().to_string())

✅ Exportado a: /content/gdrive/My Drive/Grapes/reporte_grapes/inicializacion/outputs/verificacion_master_completo.xlsx
✅ Total filas: 10096
✅ Periodos: ['2025-12', '2026-01', '2026-02', '2026-03']

📊 Filas por periodo:
_periodo
2025-12    8908
2026-01     278
2026-02     426
2026-03     484

📋 Primeras 5 filas:


,comprobante_articulo_venta,codigo_articulo_venta,cantidad_articulo_venta,total_neto_articulo_venta,total_final_articulo_venta,costo_total_articulo_venta,interes_sin_imp_articulo_venta,impuestos_articulo_venta,profit_articulo_venta,profit_ii_articulo_venta,_periodo,_row_key,_source_file,_source_mtime
0,fc b 00006-00004435,885022023349,1.0,1.012725e+06,1.119061e+06,742339.3765,0.0,106336.1131,270385.510400,270385.510400,2025-12,fc b 00006-00004435||885022023349||1||1012724....,reporte_articulo_venta_31dic25.xlsx,1767365135
1,fc a 00006-00008091,AXBXX-192-20-CISCO,1.0,0.000000e+00,0.000000e+00,0.0000,0.0,0.0000,0.000000,0.000000,2025-12,fc a 00006-00008091||AXBXX-192-20-CISCO||1||0....,reporte_articulo_venta_31dic25.xlsx,1767365135
2,fc a 00006-00008091,AXB23-192-20-CISCO,1.0,7.333019e+04,8.102986e+04,32723.9596,0.0,7699.6701,40606.232335,40606.232335,2025-12,fc a 00006-00008091||AXB23-192-20-CISCO||1||73...,reporte_articulo_venta_31dic25.xlsx,1767365135
3,fc a 00006-00008091,AXB32-192-20-CISCO,1.0,7.332863e+04,8.102813e+04,32723.9120,0.0,7699.5058,40604.714721,40604.714721,2025-12,fc a 00006-00008091||AXB32-192-20-CISCO||1||73...,reporte_articulo_venta_31dic25.xlsx,1767365135
4,fc a 00006-00008090,ABLXX-24-20-D-CISCO,3.0,0.000000e+00,0.000000e+00,0.0000,0.0,0.0000,0.000000,0.000000,2025-12,fc a 00006-00008090||ABLXX-24-20-D-CISCO||3||0...,reporte_articulo_venta_31dic25.xlsx,1767365135



📋 Últimas 5 filas:


,comprobante_articulo_venta,codigo_articulo_venta,cantidad_articulo_venta,total_neto_articulo_venta,total_final_articulo_venta,costo_total_articulo_venta,interes_sin_imp_articulo_venta,impuestos_articulo_venta,profit_articulo_venta,profit_ii_articulo_venta,_periodo,_row_key,_source_file,_source_mtime
10091,fc b 00006-00004567,093053014117,1.0,321080.543000,354794.000000,237957.3884,0.0,33713.4570,83123.154600,83123.154600,2026-03,fc b 00006-00004567||093053014117||1||321080.5...,reporte_articulo_venta_marzo26.xlsx,1775483117
10092,fc a 00006-00008369,ABLXX-24-20-D-CISCO,1.0,0.000000,0.000000,0.0000,0.0,0.0000,0.000000,0.000000,2026-03,fc a 00006-00008369||ABLXX-24-20-D-CISCO||1||0...,reporte_articulo_venta_marzo26.xlsx,1775483117
10093,fc a 00006-00008369,6976499413208,1.0,23741.628959,26234.499959,13654.3761,0.0,2492.8710,10087.252859,10087.252859,2026-03,fc a 00006-00008369||6976499413208||1||23741.6...,reporte_articulo_venta_marzo26.xlsx,1775483117
10094,fc a 00006-00008369,6976499415202,1.0,23741.628959,26234.499959,13654.3761,0.0,2492.8710,10087.252859,10087.252859,2026-03,fc a 00006-00008369||6976499415202||1||23741.6...,reporte_articulo_venta_marzo26.xlsx,1775483117
10095,fc a 00006-00008368,6976499411327,2.0,49949.321200,55193.999900,19980.0000,0.0,5244.6787,29969.321200,29969.321200,2026-03,fc a 00006-00008368||6976499411327||2||49949.3...,reporte_articulo_venta_marzo26.xlsx,1775483117



🔍 Columnas: ['comprobante_articulo_venta', 'codigo_articulo_venta', 'cantidad_articulo_venta', 'total_neto_articulo_venta', 'total_final_articulo_venta', 'costo_total_articulo_venta', 'interes_sin_imp_articulo_venta', 'impuestos_articulo_venta', 'profit_articulo_venta', 'profit_ii_articulo_venta', '_periodo', '_row_key', '_source_file', '_source_mtime']
🔍 Nulos por columna:
comprobante_articulo_venta        0
codigo_articulo_venta             0
cantidad_articulo_venta           1
total_neto_articulo_venta         1
total_final_articulo_venta        1
costo_total_articulo_venta        1
interes_sin_imp_articulo_venta    1
impuestos_articulo_venta          1
profit_articulo_venta             1
profit_ii_articulo_venta          1
_periodo                          0
_row_key                          0
_source_file                      0
_source_mtime                     0


In [11]:
 df_reporte_articulo_venta

,comprobante_articulo_venta,codigo_articulo_venta,cantidad_articulo_venta,total_neto_articulo_venta,total_final_articulo_venta,costo_total_articulo_venta,interes_sin_imp_articulo_venta,impuestos_articulo_venta,profit_articulo_venta,profit_ii_articulo_venta
0,fc b 00006-00004435,885022023349,1.0,1.012725e+06,1.119061e+06,742339.3765,0.0,106336.1131,270385.510400,270385.510400
1,fc a 00006-00008091,AXBXX-192-20-CISCO,1.0,0.000000e+00,0.000000e+00,0.0000,0.0,0.0000,0.000000,0.000000
2,fc a 00006-00008091,AXB23-192-20-CISCO,1.0,7.333019e+04,8.102986e+04,32723.9596,0.0,7699.6701,40606.232335,40606.232335
3,fc a 00006-00008091,AXB32-192-20-CISCO,1.0,7.332863e+04,8.102813e+04,32723.9120,0.0,7699.5058,40604.714721,40604.714721
4,fc a 00006-00008090,ABLXX-24-20-D-CISCO,3.0,0.000000e+00,0.000000e+00,0.0000,0.0,0.0000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
10091,fc b 00006-00004567,093053014117,1.0,3.210805e+05,3.547940e+05,237957.3884,0.0,33713.4570,83123.154600,83123.154600
10092,fc a 00006-00008369,ABLXX-24-20-D-CISCO,1.0,0.000000e+00,0.000000e+00,0.0000,0.0,0.0000,0.000000,0.000000
10093,fc a 00006-00008369,6976499413208,1.0,2.374163e+04,2.623450e+04,13654.3761,0.0,2492.8710,10087.252859,10087.252859
10094,fc a 00006-00008369,6976499415202,1.0,2.374163e+04,2.623450e+04,13654.3761,0.0,2492.8710,10087.252859,10087.252859


In [12]:
df_reporte_articulo_venta.columns

Index(['comprobante_articulo_venta', 'codigo_articulo_venta',
       'cantidad_articulo_venta', 'total_neto_articulo_venta',
       'total_final_articulo_venta', 'costo_total_articulo_venta',
       'interes_sin_imp_articulo_venta', 'impuestos_articulo_venta',
       'profit_articulo_venta', 'profit_ii_articulo_venta'],
      dtype='object')

In [13]:
totales = df_reporte_articulo_venta[[
    "cantidad_articulo_venta",
    "total_neto_articulo_venta",
    "total_final_articulo_venta",
    "costo_total_articulo_venta",
    "interes_sin_imp_articulo_venta",
    "impuestos_articulo_venta",
    "profit_articulo_venta",
    "profit_ii_articulo_venta"
]].sum()

print("\n📊 Totales generales:")
for col, val in totales.items():
    print(f"{col}: {val:,.2f}")


📊 Totales generales:
cantidad_articulo_venta: 320,638.00
total_neto_articulo_venta: 1,990,292,225.42
total_final_articulo_venta: 2,198,169,098.33
costo_total_articulo_venta: 1,235,788,700.09
interes_sin_imp_articulo_venta: 6,053,163.72
impuestos_articulo_venta: 207,876,872.91
profit_articulo_venta: 754,503,525.33
profit_ii_articulo_venta: 748,450,361.61


In [14]:
import pandas as pd

input_file = root_path + "clientes_detalle.xlsx"

# 1) Cargar el archivo Excel
df_clientes_detalle = pd.read_excel(input_file)

# 2) Eliminar las primeras 6 columnas
df_clientes_detalle = df_clientes_detalle.iloc[:, 6:].copy()

# 2.1) Normalizar headers (IMPORTANTE para que el rename matchee aunque haya espacios dobles)
df_clientes_detalle.columns = (
    df_clientes_detalle.columns.astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# 3) Limpieza de texto vectorizada (segura)
texto_cols = df_clientes_detalle.select_dtypes(include=["object"]).columns
df_clientes_detalle[texto_cols] = (
    df_clientes_detalle[texto_cols]
    .apply(lambda s: s.astype(str).str.replace(r'^[.\-\s]+|[.\-\s]+$', '', regex=True))
)

# 4) Renombrar columnas
new_column_names = {
    "Moneda de la Cuenta Corriente": "moneda_cc",
    "Pais": "pais",
    "Provincia / Estado / Region": "provincia_estado_region",
    "Clase Fiscal": "clase_fiscal",
    "Tipos de Documentos": "tipos_de_documentos",
    "Numero de Documento": "numero_de_documento",
    "RAZoN SOCIAL / APELLIDO": "razon_social_apellido",
    "NOMBRE": "nombre",
    "Ciudad / Localidad / Comuna": "ciudad_localidad_comuna",
    "Domicilio": "domicilio",
    "Vendedor": "vendedor",
    "Fecha de alta": "fecha_de_alta",
    "Inactivo - CUENTA BLOQUEADA": "inactivo_cuenta_bloqueada",
    "Foto del Cliente": "foto_del_cliente",
    "Usuario de Creacion": "usuario_de_creacion",
    "Usuario de ultima Modificacion": "usuario_de_ultima_modificacion",
    "Ultima Fecha de Modificacion": "ultima_fecha_de_modificacion",
    "ID": "id_detalle_cliente"
}

missing = set(new_column_names) - set(df_clientes_detalle.columns)
if missing:
    print("¡Atención! Faltan columnas a renombrar:", missing)

df_clientes_detalle.rename(columns=new_column_names, inplace=True)

# 5) Fechas
df_clientes_detalle["fecha_de_alta"] = pd.to_datetime(
    df_clientes_detalle["fecha_de_alta"], dayfirst=True, errors="coerce"
)
df_clientes_detalle["ultima_fecha_de_modificacion"] = pd.to_datetime(
    df_clientes_detalle["ultima_fecha_de_modificacion"], dayfirst=True, errors="coerce"
)

# 6) Preview
print(df_clientes_detalle.head())


      moneda_cc       pais          provincia_estado_region  \
0  Pesos  (ARS)  Argentina  Ciudad Autónoma de Buenos Aires   
1  Pesos  (ARS)  Argentina                         Santa Fe   
2  Pesos  (ARS)  Argentina  Ciudad Autónoma de Buenos Aires   
3  Pesos  (ARS)  Argentina  Ciudad Autónoma de Buenos Aires   
4  Pesos  (ARS)  Argentina                          Córdoba   

                clase_fiscal tipos_de_documentos numero_de_documento  \
0           Consumidor Final                 DNI            28768325   
1  IVA Responsable Inscripto                CUIT       30-71220776-7   
2  IVA Responsable Inscripto                CUIT       30-71665604-3   
3           Consumidor Final                CUIT       30-71657600-7   
4  IVA Responsable Inscripto                CUIT       30-71163816-0   

     razon_social_apellido nombre ciudad_localidad_comuna  \
0                      DAC    nan            villa crespo   
1  0341 PRODUCCIONES S.R.L    nan                 Rosario   
2    

In [15]:
import pandas as pd
import re
import os
import numpy as np

# =========================
# CONFIG FC CLIENTES
# =========================
MASTER_FC_CLIENTES_PARQUET = os.path.join(root_path, "master_fc_clientes.parquet")
HIST_FC_CLIENTES_FILE      = os.path.join(root_path, "reporte_fc_clientes_31dic2025.xlsx")
HIST_FC_CLIENTES_PERIODO   = "2025-12"

pat_comp = re.compile(r'^(?:FC|NC|ND)\s+[AB]\s*\d{5}-\d{8}$', re.IGNORECASE)

# =========================
# PARSER NUMÉRICO
# =========================
def parse_num_fc(x):
    if x is None:
        return None
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return None
    s = s.replace("$", "").replace(" ", "")
    if "," in s and "." in s:
        if s.rfind(",") > s.rfind("."):
            s = s.replace(".", "").replace(",", ".")
        else:
            s = s.replace(",", "")
        return pd.to_numeric(s, errors="coerce")
    if "," in s and "." not in s:
        s = s.replace(".", "").replace(",", ".")
        return pd.to_numeric(s, errors="coerce")
    return pd.to_numeric(s, errors="coerce")

# =========================
# PARSER EXCEL -> DF LIMPIO
# =========================
def parse_reporte_fc_clientes_xlsx(input_file: str) -> pd.DataFrame:
    df_raw = pd.read_excel(input_file, header=None, dtype=str, engine="openpyxl")
    df = df_raw.drop([0, 1, 2], errors="ignore").reset_index(drop=True)
    df.columns = ["Comprobante", "Importe Final", "Importe Neto", "Impuestos", "Costo", "$Profit"]
    df = df.dropna(how="all").reset_index(drop=True)

    records = []
    i = 0
    n = len(df)

    while i < n:
        txt = str(df.at[i, "Comprobante"]).strip()
        if pat_comp.match(txt):
            importe_final = parse_num_fc(df.at[i, "Importe Final"])
            importe_neto  = parse_num_fc(df.at[i, "Importe Neto"])
            impuestos     = parse_num_fc(df.at[i, "Impuestos"])
            costo         = parse_num_fc(df.at[i, "Costo"])
            profit        = parse_num_fc(df.at[i, "$Profit"])

            cliente = ""
            if i + 1 < n:
                cliente = str(df.at[i + 1, "Comprobante"]).strip()
                if cliente.lower() == "nan":
                    cliente = ""

            records.append({
                "comprobante_clientes":   txt,
                "cliente":                cliente,
                "importe_final_clientes": importe_final,
                "importe_neto_clientes":  importe_neto,
                "impuestos_clientes":     impuestos,
                "costo_clientes":         costo,
                "profit_clientes":        profit
            })
            i += 2
        else:
            i += 1

    out = pd.DataFrame(records)

    if "cliente" in out.columns:
        out["cliente"] = (
            out["cliente"]
            .astype(str)
            .str.replace(r'^[\.\-\s]+|[\.\-\s]+$', '', regex=True)
            .str.strip()
        )
        out.loc[out["cliente"].str.lower() == "nan", "cliente"] = ""

    return out

# =========================
# MASTER HELPERS
# =========================
def build_row_key_fc(df: pd.DataFrame) -> pd.Series:
    return (
        df["comprobante_clientes"].astype(str) + "||" +
        df["cliente"].astype(str) + "||" +
        df["importe_final_clientes"].fillna(0).round(6).astype(str) + "||" +
        df["costo_clientes"].fillna(0).round(6).astype(str)
    )

def load_master_fc() -> pd.DataFrame:
    if os.path.exists(MASTER_FC_CLIENTES_PARQUET):
        return pd.read_parquet(MASTER_FC_CLIENTES_PARQUET)
    return pd.DataFrame()

def save_master_fc(df: pd.DataFrame) -> None:
    df.to_parquet(MASTER_FC_CLIENTES_PARQUET, index=False)

def update_master_fc_clientes(input_file: str, periodo: str, replace_period: bool = True) -> pd.DataFrame:
    if not os.path.exists(input_file):
        raise FileNotFoundError(f"No existe: {input_file}")

    new_df = parse_reporte_fc_clientes_xlsx(input_file)
    new_df["_periodo"]      = str(periodo)
    new_df["_row_key"]      = build_row_key_fc(new_df)
    new_df["_source_file"]  = os.path.basename(input_file)
    new_df["_source_mtime"] = int(os.path.getmtime(input_file))

    master = load_master_fc()

    if not master.empty:
        if "_row_key" not in master.columns:
            master["_row_key"] = build_row_key_fc(master)
        if "_periodo" not in master.columns:
            master["_periodo"] = ""
        if replace_period:
            master = master[master["_periodo"].astype(str) != str(periodo)].copy()

    combined = pd.concat([master, new_df], ignore_index=True)
    combined = combined.drop_duplicates(subset=["_row_key"], keep="last").reset_index(drop=True)

    save_master_fc(combined)
    return combined

def master_to_df_fc_clientes(master: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "comprobante_clientes",
        "cliente",
        "importe_final_clientes",
        "importe_neto_clientes",
        "impuestos_clientes",
        "costo_clientes",
        "profit_clientes",
    ]
    if master.empty:
        return pd.DataFrame(columns=cols)
    return master[cols].copy()

# =========================
# BOOTSTRAP
# =========================
df_master_fc_clientes = load_master_fc()

if df_master_fc_clientes.empty:
    df_master_fc_clientes = update_master_fc_clientes(HIST_FC_CLIENTES_FILE, HIST_FC_CLIENTES_PERIODO, replace_period=True)

df_reporte_fc_clientes = master_to_df_fc_clientes(df_master_fc_clientes)

print("✅ Master FC Clientes filas:", len(df_master_fc_clientes))
print("✅ df_reporte_fc_clientes filas:", len(df_reporte_fc_clientes))
print("✅ Periodos:", sorted(df_master_fc_clientes["_periodo"].dropna().unique().tolist()) if not df_master_fc_clientes.empty else [])

✅ Master FC Clientes filas: 6124
✅ df_reporte_fc_clientes filas: 6124
✅ Periodos: ['2025-12', '2026-01', '2026-02', '2026-03']


In [16]:
# =========================
# MESES NUEVOS FC CLIENTES
# =========================
NEW_FILES_FC = [
    (os.path.join(mensuales_path, "reporte_fc_clientes_enero26.xlsx"), "2026-01"),
    (os.path.join(mensuales_path, "reporte_fc_clientes_febrero26.xlsx"), "2026-02"),
    (os.path.join(mensuales_path, "reporte_fc_clientes_marzo26.xlsx"), "2026-03"),
]

for fpath, per in NEW_FILES_FC:
    df_master_fc_clientes = update_master_fc_clientes(
        input_file=fpath,
        periodo=per,
        replace_period=True
    )

df_reporte_fc_clientes = master_to_df_fc_clientes(df_master_fc_clientes)

print("✅ Master FC Clientes filas:", len(df_master_fc_clientes))
print("✅ df_reporte_fc_clientes filas:", len(df_reporte_fc_clientes))
print("✅ Periodos:", sorted(df_master_fc_clientes["_periodo"].dropna().unique().tolist()))
print()
print("📊 Filas por periodo:")
print(df_master_fc_clientes.groupby("_periodo").size().to_string())

✅ Master FC Clientes filas: 6124
✅ df_reporte_fc_clientes filas: 6124
✅ Periodos: ['2025-12', '2026-01', '2026-02', '2026-03']

📊 Filas por periodo:
_periodo
2025-12    5388
2026-01     159
2026-02     261
2026-03     316


In [17]:
# 10) Sumar totales generales
cols_tot = [
    "importe_final_clientes",
    "importe_neto_clientes",
    "impuestos_clientes",
    "costo_clientes",
    "profit_clientes"
]
totales = df_reporte_fc_clientes[cols_tot].sum()
print("\n📊 Totales generales:")
for nombre, valor in totales.items():
    print(f"{nombre}: {valor:,.2f}")


📊 Totales generales:
importe_final_clientes: 2,228,537,563.00
importe_neto_clientes: 2,017,214,173.09
impuestos_clientes: 211,323,389.33
costo_clientes: 1,235,902,806.34
profit_clientes: 781,311,366.75


In [18]:
""" NO TIENE SENTIDO HACER EL MERGE
import pandas as pd

# --- Suponiendo que ya tienes en memoria ---
# df_reporte_articulo_venta_31may25  (varias líneas por factura)
# df_reporte_fc_clientes             (una línea por factura)

# 1) Agrupo los artículos por comprobante y sumo sus métricas
cols_art = [
    "cantidad_articulo_venta",
    "total_neto_articulo_venta",
    "total_final_articulo_venta",
    "costo_total_articulo_venta",
    "interes_sin_imp_articulo_venta",
    "impuestos_articulo_venta",
    "profit_articulo_venta",
    "profit_ii_articulo_venta"
]
df_items_agg = (
    df_reporte_articulo_venta_31may25
    .groupby("comprobante_articulo_venta", as_index=False)[cols_art]
    .sum()
    .rename(columns={"comprobante_articulo_venta":"comprobante"})
)

# 2) Preparo la tabla de clientes para merge: renombro la llave
df_fc = (
    df_reporte_fc_clientes
    .rename(columns={"comprobante_clientes":"comprobante"})
    # y aseguro que la columna clave exista limpia
    .assign(comprobante=lambda d: d["comprobante"].str.strip().str.lower())
)

# 3) También limpio y bajo a minúscula en el agreg agregado
df_items_agg = df_items_agg.assign(
    comprobante=lambda d: d["comprobante"].str.strip().str.lower()
)

# 4) Hago el merge 1-a-1
df_reporte_summary = pd.merge(
    df_items_agg,
    df_fc,
    on="comprobante",
    how="left"
)

# 5) Visto rápido
print(df_reporte_summary.head(10))

"""

' NO TIENE SENTIDO HACER EL MERGE\nimport pandas as pd\n\n# --- Suponiendo que ya tienes en memoria ---\n# df_reporte_articulo_venta_31may25  (varias líneas por factura)\n# df_reporte_fc_clientes             (una línea por factura)\n\n# 1) Agrupo los artículos por comprobante y sumo sus métricas\ncols_art = [\n    "cantidad_articulo_venta",\n    "total_neto_articulo_venta",\n    "total_final_articulo_venta",\n    "costo_total_articulo_venta",\n    "interes_sin_imp_articulo_venta",\n    "impuestos_articulo_venta",\n    "profit_articulo_venta",\n    "profit_ii_articulo_venta"\n]\ndf_items_agg = (\n    df_reporte_articulo_venta_31may25\n    .groupby("comprobante_articulo_venta", as_index=False)[cols_art]\n    .sum()\n    .rename(columns={"comprobante_articulo_venta":"comprobante"})\n)\n\n# 2) Preparo la tabla de clientes para merge: renombro la llave\ndf_fc = (\n    df_reporte_fc_clientes\n    .rename(columns={"comprobante_clientes":"comprobante"})\n    # y aseguro que la columna clave ex

In [19]:
"""df_reporte_clean.columns"""

'df_reporte_clean.columns'

In [20]:
"""
# Celda: Sumar todas las columnas numéricas de df_reporte_clean con punto de miles y coma decimal

# 1) Lista de todas las columnas numéricas a sumar
cols_a_sumar = [
    "cantidad_articulo_venta",
    "total_neto_articulo_venta",
    "total_final_articulo_venta",
    "costo_total_articulo_venta",
    "interes_sin_imp_articulo_venta",
    "impuestos_articulo_venta",
    "profit_articulo_venta",
    "profit_ii_articulo_venta",
    "importe_final_clientes",
    "importe_neto_clientes",
    "impuestos_clientes",
    "costo_clientes",
    "profit_clientes"
]

# 2) Asegurarnos de que son numéricas
for col in cols_a_sumar:
    df_reporte_clean[col] = pd.to_numeric(df_reporte_clean[col], errors="coerce")

# 3) Calcular sumas
totales = df_reporte_clean[cols_a_sumar].sum()

# 4) Mostrar resultados formateados
print("\n📊 Sumas de todas las columnas:")
for col, val in totales.items():
    # formatear: punto como separador de miles y coma como decimal
    formatted = f"{val:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    print(f"{col}: {formatted}")
"""

'\n# Celda: Sumar todas las columnas numéricas de df_reporte_clean con punto de miles y coma decimal\n\n# 1) Lista de todas las columnas numéricas a sumar\ncols_a_sumar = [\n    "cantidad_articulo_venta",\n    "total_neto_articulo_venta",\n    "total_final_articulo_venta",\n    "costo_total_articulo_venta",\n    "interes_sin_imp_articulo_venta",\n    "impuestos_articulo_venta",\n    "profit_articulo_venta",\n    "profit_ii_articulo_venta",\n    "importe_final_clientes",\n    "importe_neto_clientes",\n    "impuestos_clientes",\n    "costo_clientes",\n    "profit_clientes"\n]\n\n# 2) Asegurarnos de que son numéricas\nfor col in cols_a_sumar:\n    df_reporte_clean[col] = pd.to_numeric(df_reporte_clean[col], errors="coerce")\n\n# 3) Calcular sumas\ntotales = df_reporte_clean[cols_a_sumar].sum()\n\n# 4) Mostrar resultados formateados\nprint("\n📊 Sumas de todas las columnas:")\nfor col, val in totales.items():\n    # formatear: punto como separador de miles y coma como decimal\n    formatted

In [21]:
import numpy as np
import pandas as pd
import re
import os

# =========================
# CONFIG MOV STOCK
# =========================
MASTER_MOV_STOCK_PARQUET = os.path.join(root_path, "master_mov_stock.parquet")
HIST_MOV_STOCK_FILE      = os.path.join(root_path, "mov_stock_ago23_31dic25.xlsx")
HIST_MOV_STOCK_PERIODO   = "2025-12"

# =========================
# PARSER NUMÉRICO
# =========================
def parse_num_stock(x):
    if x is None:
        return np.nan
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return np.nan
    s = s.replace("$", "").replace(" ", "")
    if "," in s and "." in s:
        if s.rfind(",") > s.rfind("."):
            s = s.replace(".", "").replace(",", ".")
        else:
            s = s.replace(",", "")
        return pd.to_numeric(s, errors="coerce")
    if "," in s and "." not in s:
        s = s.replace(".", "").replace(",", ".")
        return pd.to_numeric(s, errors="coerce")
    return pd.to_numeric(s, errors="coerce")

def separar_cliente_proveedor(cp):
    m = re.match(r"^(.*?)(?:\s*\((\d+)\))?$", str(cp))
    if m:
        nombre = m.group(1).strip().rstrip("-")
        numero = m.group(2)
        return nombre, numero
    return cp, None

# =========================
# PARSER EXCEL -> DF LIMPIO
# =========================
def parse_mov_stock_xlsx(input_file: str) -> pd.DataFrame:
    df_raw = pd.read_excel(input_file)

    df_raw.columns = (
        df_raw.columns.astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    relevant_columns = [
        "Fecha de Carga", "Deposito", "Comprobante", "Cantidad",
        "Stock Anterior", "Stock Actual", "Codigo", "Articulo", "Marca",
        "Condicion de Venta", "Fecha", "Cliente / Proveedor", "Precio",
        "Total", "Moneda", "Tipo de Cambio", "Costo PPP",
        "Costo de ultima Compra", "Costo Adicional", "Costo Adicional PPP",
        "Costo Adicional de ultima Compra", "Precio Original",
        "Comprobante Asociado", "Tipo de Cambio del Comprobante",
        "ID", "Tipo ID", "ID Articulo", "Clase"
    ]

    missing_cols = [c for c in relevant_columns if c not in df_raw.columns]
    if missing_cols:
        print(f"⚠️ Faltan columnas en {os.path.basename(input_file)}: {missing_cols}")

    df = df_raw[[c for c in relevant_columns if c in df_raw.columns]].copy()

    rename_map = {
        "Fecha de Carga":                    "Fecha_de_Carga",
        "Stock Anterior":                    "Stock_Anterior",
        "Stock Actual":                      "Stock_Actual",
        "Precio":                            "Precio_sinIVA",
        "Total":                             "Total_sinIVA",
        "Condicion de Venta":                "Condicion_de_Venta",
        "Cliente / Proveedor":               "Cliente_Proveedor",
        "Tipo de Cambio":                    "Tipo_de_Cambio",
        "Costo PPP":                         "Costo_PPP_comisiones",
        "Costo de ultima Compra":            "Costo_Ultima_Compra",
        "Costo Adicional":                   "Costo_Adicional",
        "Costo Adicional PPP":               "Costo_Adicional_PPP",
        "Costo Adicional de ultima Compra":  "Costo_Adicional_Ultima_Compra",
        "Precio Original":                   "Costo_masmarkup_usd",
        "Comprobante Asociado":              "Compr_Asoc",
        "Tipo de Cambio del Comprobante":    "Tipo_Cambio_Compr",
        "ID Articulo":                       "ID_Articulo",
    }
    df = df.rename(columns=rename_map)

    columns_to_format = [
        "Tipo_de_Cambio", "Costo_PPP_comisiones", "Costo_Ultima_Compra",
        "Costo_Adicional", "Costo_Adicional_PPP", "Costo_Adicional_Ultima_Compra",
        "Costo_masmarkup_usd", "Precio_sinIVA", "Total_sinIVA"
    ]
    for col in columns_to_format:
        if col not in df.columns:
            continue
        s = df[col].apply(parse_num_stock)
        s = s.mask(s.eq(1) | s.eq(1.0)).ffill()
        df[col] = s

    if "Cliente_Proveedor" in df.columns:
        df[["Cliente_Proveedor", "Nro_cliente_proveedor"]] = \
            df["Cliente_Proveedor"].apply(lambda x: pd.Series(separar_cliente_proveedor(x)))
        df["Nro_cliente_proveedor"] = pd.to_numeric(
            df["Nro_cliente_proveedor"], errors="coerce"
        )

    # >>> MAPEO P/N → EAN (datos nuevos del Excel)
    if "Codigo" in df.columns:
        df["Codigo"] = df["Codigo"].astype(str).str.strip().str.upper()
        df["Codigo"] = aplicar_ean(df["Codigo"])

    return df

# =========================
# MASTER HELPERS
# =========================
def build_row_key_stock(df: pd.DataFrame) -> pd.Series:
    return (
        df["Comprobante"].astype(str) + "||" +
        df["Codigo"].astype(str) + "||" +
        df["Cantidad"].fillna(0).astype(str) + "||" +
        df["Fecha_de_Carga"].astype(str) + "||" +
        df["Total_sinIVA"].fillna(0).round(4).astype(str)
    )

def load_master_stock() -> pd.DataFrame:
    if os.path.exists(MASTER_MOV_STOCK_PARQUET):
        df = pd.read_parquet(MASTER_MOV_STOCK_PARQUET)
        # >>> MAPEO P/N → EAN (datos históricos del parquet)
        if "Codigo" in df.columns:
            df["Codigo"] = df["Codigo"].astype(str).str.strip().str.upper()
            df["Codigo"] = aplicar_ean(df["Codigo"])
        return df
    return pd.DataFrame()

def save_master_stock(df: pd.DataFrame) -> None:
    df.to_parquet(MASTER_MOV_STOCK_PARQUET, index=False)

def update_master_mov_stock(input_file: str, periodo: str, replace_period: bool = True) -> pd.DataFrame:
    if not os.path.exists(input_file):
        raise FileNotFoundError(f"No existe: {input_file}")

    new_df = parse_mov_stock_xlsx(input_file)
    new_df["_periodo"]      = str(periodo)
    new_df["_row_key"]      = build_row_key_stock(new_df)
    new_df["_source_file"]  = os.path.basename(input_file)
    new_df["_source_mtime"] = int(os.path.getmtime(input_file))

    master = load_master_stock()

    if not master.empty:
        if "_row_key" not in master.columns:
            master["_row_key"] = build_row_key_stock(master)
        if "_periodo" not in master.columns:
            master["_periodo"] = ""
        if replace_period:
            master = master[master["_periodo"].astype(str) != str(periodo)].copy()

    combined = pd.concat([master, new_df], ignore_index=True)
    combined = combined.drop_duplicates(subset=["_row_key"], keep="last").reset_index(drop=True)

    save_master_stock(combined)
    return combined

def master_to_df_stock(master: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "Fecha_de_Carga", "Deposito", "Comprobante", "Cantidad",
        "Stock_Anterior", "Stock_Actual", "Codigo", "Articulo", "Marca",
        "Condicion_de_Venta", "Fecha", "Cliente_Proveedor", "Nro_cliente_proveedor",
        "Precio_sinIVA", "Total_sinIVA", "Moneda", "Tipo_de_Cambio",
        "Costo_PPP_comisiones", "Costo_Ultima_Compra", "Costo_Adicional",
        "Costo_Adicional_PPP", "Costo_Adicional_Ultima_Compra",
        "Costo_masmarkup_usd", "Compr_Asoc", "Tipo_Cambio_Compr",
        "ID", "Tipo ID", "ID_Articulo", "Clase"
    ]
    if master.empty:
        return pd.DataFrame(columns=cols)
    return master[[c for c in cols if c in master.columns]].copy()

# =========================
# BOOTSTRAP
# =========================
df_master_mov_stock = load_master_stock()

if df_master_mov_stock.empty:
    print("ℹ️ No existe master mov_stock. Creando desde histórico, puede tardar unos segundos...")
    df_master_mov_stock = update_master_mov_stock(HIST_MOV_STOCK_FILE, HIST_MOV_STOCK_PERIODO, replace_period=True)

df_selected = master_to_df_stock(df_master_mov_stock)

print("✅ Master mov_stock filas:", len(df_master_mov_stock))
print("✅ df_selected filas:", len(df_selected))
print("✅ Periodos:", sorted(df_master_mov_stock["_periodo"].dropna().unique().tolist()) if not df_master_mov_stock.empty else [])

✅ Master mov_stock filas: 10678
✅ df_selected filas: 10678
✅ Periodos: ['2025-12', '2026-01', '2026-02', '2026-03']


In [22]:
import os

# 1) Borrar parquet
f = "/content/gdrive/My Drive/Grapes/reporte_grapes/inicializacion/master_mov_stock.parquet"
if os.path.exists(f):
    os.remove(f)
    print("Eliminado")

# 2) Reconstruir histórico
print("Reconstruyendo desde histórico...")
df_master_mov_stock = update_master_mov_stock(
    HIST_MOV_STOCK_FILE, HIST_MOV_STOCK_PERIODO, replace_period=True
)
df_selected = master_to_df_stock(df_master_mov_stock)

# 3) ORDENAR POR FECHA ASC — crítico para stock y TC
df_selected["Fecha_de_Carga"] = pd.to_datetime(
    df_selected["Fecha_de_Carga"], errors="coerce", dayfirst=True
)
df_selected = df_selected.sort_values("Fecha_de_Carga", ascending=True).reset_index(drop=True)

print("Master mov_stock filas:", len(df_master_mov_stock))
print("df_selected filas:", len(df_selected))
print("Rango fechas:", df_selected["Fecha_de_Carga"].min().date(),
      "→", df_selected["Fecha_de_Carga"].max().date())
print("Periodos:", sorted(df_master_mov_stock["_periodo"].dropna().unique().tolist()))

Eliminado
Reconstruyendo desde histórico...
Master mov_stock filas: 9438
df_selected filas: 9438
Rango fechas: 2023-08-01 → 2025-12-31
Periodos: ['2025-12']


In [23]:
# =========================
# MESES NUEVOS MOV STOCK
# =========================
NEW_FILES_STOCK = [
    (os.path.join(mensuales_path, "mov_stock_enero26.xlsx"), "2026-01"),
    (os.path.join(mensuales_path, "mov_stock_febrero26.xlsx"), "2026-02"),
    (os.path.join(mensuales_path, "mov_stock_marzo26.xlsx"), "2026-03"),
]

for fpath, per in NEW_FILES_STOCK:
    df_master_mov_stock = update_master_mov_stock(
        input_file=fpath,
        periodo=per,
        replace_period=True
    )

df_selected = master_to_df_stock(df_master_mov_stock)

print("✅ Master mov_stock filas:", len(df_master_mov_stock))
print("✅ df_selected filas:", len(df_selected))
print("✅ Periodos:", sorted(df_master_mov_stock["_periodo"].dropna().unique().tolist()))
print()
print("📊 Filas por periodo:")
print(df_master_mov_stock.groupby("_periodo").size().to_string())

✅ Master mov_stock filas: 10678
✅ df_selected filas: 10678
✅ Periodos: ['2025-12', '2026-01', '2026-02', '2026-03']

📊 Filas por periodo:
_periodo
2025-12    9438
2026-01     333
2026-02     424
2026-03     483


In [24]:
# Invertir signos en 'Cantidad'
df_selected['Cantidad'] *= -1

# Invertir signos en 'Total_sinIVA'
df_selected['Total_sinIVA'] *= -1

In [25]:
# Eliminar las columnas no deseadas
columns_to_drop = [
    'Costo_PPP_comisiones', 'Costo_Adicional', 'Costo_Adicional_PPP',
    'Costo_Adicional_Ultima_Compra', 'Costo_masmarkup_usd'
]

df_selected.drop(columns=columns_to_drop, inplace=True)

In [26]:
df_selected.columns

Index(['Fecha_de_Carga', 'Deposito', 'Comprobante', 'Cantidad',
       'Stock_Anterior', 'Stock_Actual', 'Codigo', 'Articulo', 'Marca',
       'Condicion_de_Venta', 'Fecha', 'Cliente_Proveedor',
       'Nro_cliente_proveedor', 'Precio_sinIVA', 'Total_sinIVA', 'Moneda',
       'Tipo_de_Cambio', 'Costo_Ultima_Compra', 'Compr_Asoc',
       'Tipo_Cambio_Compr', 'ID', 'Tipo ID', 'ID_Articulo', 'Clase'],
      dtype='object')

In [27]:
import pandas as pd
import re

# Eliminar filas basado en palabras clave en 'Articulo'
palabras_clave_articulo = ["EOL", "thule", "case logic", "lowerpro"]
pat_art = "|".join(map(re.escape, palabras_clave_articulo))
df_cleaned = df_selected[
    ~df_selected["Articulo"].astype(str).str.contains(pat_art, case=False, na=False, regex=True)
]

# Eliminar filas donde 'Comprobante' contenga 'Pres x'
df_cleaned = df_cleaned[
    ~df_cleaned["Comprobante"].astype(str).str.contains("Pres x", case=False, na=False)
]

# Eliminar filas donde 'Clase' contenga palabras/frases a excluir
palabras_clave_clase = [
    "presupuesto",
    "RMA",
    "Remito / Guía de Despacho de ENTRADA (Opuesto a REMITO)",
    "Anulación de Remito x Ventas"
]
pat_clase = "|".join(map(re.escape, palabras_clave_clase))
df_cleaned = df_cleaned[
    ~df_cleaned["Clase"].astype(str).str.contains(pat_clase, case=False, na=False, regex=True)
]

# Convertir 'Fecha_de_Carga' a formato de fecha (evita SettingWithCopyWarning)
df_cleaned = df_cleaned.copy()
df_cleaned["Fecha_de_Carga"] = pd.to_datetime(df_cleaned["Fecha_de_Carga"], errors="coerce", dayfirst=True)

# Muestra las primeras filas del dataframe después de la limpieza
print(df_cleaned.head())


           Fecha_de_Carga             Deposito          Comprobante  Cantidad  \
0 2023-08-01 09:18:18.940  01-Deposito Cerrito  Fc A 00006-00004091         7   
1 2023-08-01 09:18:18.940  01-Deposito Cerrito  Fc A 00006-00004091         7   
2 2023-08-01 09:18:18.940  01-Deposito Cerrito  Fc A 00006-00004091         2   
3 2023-08-01 09:18:18.940  01-Deposito Cerrito  Fc A 00006-00004091         2   
4 2023-08-01 09:21:33.390  01-Deposito Cerrito  Fc A 00006-00004092         1   

   Stock_Anterior  Stock_Actual              Codigo  \
0              37            72  AXB23-192-20-CISCO   
1              37            72  AXB32-192-20-CISCO   
2               2            76  AXB32-192-40-CISCO   
3               2            76  AXB23-192-40-CISCO   
4              14             0  AXB23-192-20-ARUBA   

                                            Articulo   Marca  \
0  Modulo SFP+ BiDi 10G 20KM 10GBase-BX-U SMF Sin...  10GteK   
1  Modulo SFP+ BiDi 10G 20KM 10GBase-BX-D SMF Sin...  

In [28]:
import pandas as pd

# =========================
# Redondeo + TOTALES (AR) + EXPORT a Excel (resultados/)
# =========================

root_path        = "/content/gdrive/My Drive/Grapes/reporte_grapes/inicializacion/"
output_path_base = "/content/gdrive/My Drive/Grapes/reporte_grapes/resultados/"

def fmt_ar(x):
    if pd.isna(x):
        return ""
    s = f"{float(x):,.2f}"  # 1,234,567.89
    return s.replace(",", "_").replace(".", ",").replace("_", ".")  # 1.234.567,89

# 1) df_cleaned: columnas de precio y total (solo redondeo)
cols_cleaned_round = ["Precio_sinIVA", "Total_sinIVA"]
for col in cols_cleaned_round:
    if col in df_cleaned.columns:
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors="coerce").round(2)

# 2) df_reporte_articulo_venta: métricas de artículo (redondeo + asegurar numérico)
cols_reporte_round = [
    "cantidad_articulo_venta",
    "total_neto_articulo_venta",
    "total_final_articulo_venta",
    "costo_total_articulo_venta",
    "interes_sin_imp_articulo_venta",
    "impuestos_articulo_venta",
    "profit_articulo_venta",
    "profit_ii_articulo_venta",
]
cols_reporte_round = [c for c in cols_reporte_round if c in df_reporte_articulo_venta.columns]

for col in cols_reporte_round:
    df_reporte_articulo_venta[col] = pd.to_numeric(df_reporte_articulo_venta[col], errors="coerce").round(2)

# 3) IMPRIMIR TOTALES (para comparar contra sistema)
print("\n📊 Totales df_reporte_articulo_venta POST-REDONDEO (formato AR):")
tot = df_reporte_articulo_venta[cols_reporte_round].sum()
for col in cols_reporte_round:
    print(f"{col}: {fmt_ar(tot[col])}")

# 4) EXPORTAR EXCEL (para que me lo puedas pasar)
output_file = output_path_base + "df_reporte_articulo_venta_post_redondeo.xlsx"
df_reporte_articulo_venta.to_excel(output_file, index=False)
print(f"\n✅ Archivo guardado en: {output_file}")



📊 Totales df_reporte_articulo_venta POST-REDONDEO (formato AR):
cantidad_articulo_venta: 320.638,00
total_neto_articulo_venta: 1.990.292.225,92
total_final_articulo_venta: 2.198.169.098,67
costo_total_articulo_venta: 1.235.788.698,73
interes_sin_imp_articulo_venta: 6.053.163,72
impuestos_articulo_venta: 207.876.872,28
profit_articulo_venta: 754.503.526,01
profit_ii_articulo_venta: 748.450.362,31

✅ Archivo guardado en: /content/gdrive/My Drive/Grapes/reporte_grapes/resultados/df_reporte_articulo_venta_post_redondeo.xlsx


In [29]:
import pandas as pd

# --- Rutas estándar del proyecto ---
root_path        = "/content/gdrive/My Drive/Grapes/reporte_grapes/inicializacion/"
output_path_base = "/content/gdrive/My Drive/Grapes/reporte_grapes/resultados/"

# =========================
# GROUPBY df_cleaned (sum numéricas, first resto) + GUARDAR EXCEL
# =========================

# 1) Definir columnas numéricas que se suman
columnas_numericas = [
    "Cantidad", "Stock_Anterior", "Stock_Actual",
    "Precio_sinIVA", "Total_sinIVA", "Costo_Ultima_Compra"
]
columnas_numericas = [c for c in columnas_numericas if c in df_cleaned.columns]

# 2) Asegurar numérico en las que se suman (y redondeo a 2 decimales)
for c in columnas_numericas:
    df_cleaned[c] = pd.to_numeric(df_cleaned[c], errors="coerce").fillna(0).round(2)

# 3) Groupby: sum en numéricas, first en el resto
group_keys = ["Comprobante", "Codigo", "Cliente_Proveedor"]
df_cleaned = (
    df_cleaned
    .groupby(group_keys, as_index=False)
    .agg({col: ("sum" if col in columnas_numericas else "first") for col in df_cleaned.columns})
)

# 4) Guardar en Excel en carpeta "resultados"
output_file = output_path_base + "df_cleaned_groupby.xlsx"
df_cleaned.to_excel(output_file, index=False)
print(f"✅ Guardado: {output_file}")

# 5) (Opcional) ver preview
df_cleaned.head(10)


✅ Guardado: /content/gdrive/My Drive/Grapes/reporte_grapes/resultados/df_cleaned_groupby.xlsx


,Fecha_de_Carga,Deposito,Comprobante,Cantidad,Stock_Anterior,Stock_Actual,Codigo,Articulo,Marca,Condicion_de_Venta,...,Total_sinIVA,Moneda,Tipo_de_Cambio,Costo_Ultima_Compra,Compr_Asoc,Tipo_Cambio_Compr,ID,Tipo ID,ID_Articulo,Clase
0,2024-07-22 09:42:42.307,01-Deposito Cerrito,Aj.Stock 000000000000114,-3,0,0,AD-120W,Fuente Switching 12V 10A UpBRIGHT apto QNAP NA...,UpBright,None,...,0.0,Pesos (ARS),1350.0,0.0,None,NaN,20548,30,590,STOCK - Ajustes de Stock
1,2024-07-25 19:05:35.750,01-Deposito Cerrito,Aj.Stock 000000000000115,0,58,80,2GB-SO-DDR3-UNK,RAM 2GB DDR3/L 1600-1866 Mhz 204-Pin TRANSCEND,Generico,None,...,0.0,Pesos (ARS),1350.0,0.0,None,NaN,20610,30,121,STOCK - Ajustes de Stock
2,2024-11-20 10:54:34.240,01-Deposito Cerrito,Aj.Stock 000000000000117,-50,0,0,GLC-SX-MMD,Modulo SFP 1.25G 550Mts 850nm 1000Base-SX MMF ...,Cisco,None,...,0.0,Pesos (ARS),1260.0,0.0,None,NaN,22071,30,625,STOCK - Ajustes de Stock
3,2024-11-20 11:01:44.720,01-Deposito Cerrito,Aj.Stock 000000000000118,-15,3,0,888462859158,Adaptador Apple Thunderbolt 3 USB-C a Thunderb...,Apple,None,...,0.0,Pesos (ARS),1260.0,0.0,None,NaN,22072,30,584,STOCK - Ajustes de Stock
4,2025-01-29 16:42:47.050,01-Deposito Cerrito,Aj.Stock 000000000000119,1,4,3,3660619406784,STHT8000800 Disco LaCie Rugged Raid Shuttle 8T...,LaCie,None,...,0.0,Pesos (ARS),1200.0,0.0,None,NaN,22885,30,600,STOCK - Ajustes de Stock
5,2025-03-20 13:46:30.177,01-Deposito Cerrito,Aj.Stock 000000000000120,-1,1,0,ST16000NM000J,HDD 16TB Seagate EXOS X18 NE000 512MB 7200RPM,Seagate,None,...,0.0,Pesos (ARS),1320.0,0.0,None,NaN,23527,30,572,STOCK - Ajustes de Stock
6,2025-07-07 14:11:22.503,01-Deposito Cerrito,Aj.Stock 000000000000121,10,258,0,6976499412928,Modulo SFP+ 10G 20KM 10GBase-LR SMF Dual-LC OE...,10GteK,None,...,0.0,Pesos (ARS),1290.0,0.0,None,NaN,25517,30,378,STOCK - Ajustes de Stock
7,2025-07-07 14:11:22.503,01-Deposito Cerrito,Aj.Stock 000000000000121,-10,0,18,AXS13-192-20-DELL,Modulo SFP+ 10G 20KM 10GBase-LR SMF Dual-LC DELL,10GteK,None,...,0.0,Pesos (ARS),1290.0,0.0,None,NaN,25518,30,412,STOCK - Ajustes de Stock
8,2025-11-18 10:36:58.703,01-Deposito Cerrito,Aj.Stock 000000000000123,-2,4,6,850065783635,UDC PARTS BLADES KIT X2 46IN 405380,Generico,None,...,0.0,Pesos (ARS),1480.0,0.0,None,NaN,27929,30,696,STOCK - Ajustes de Stock
9,2023-11-03 11:38:18.187,01-Deposito Cerrito,Aj.Stock 000000000000396,0,1,0,ST18000NM000J,HDD 18TB Seagate EXOS X18 NE000 256MB 7200RPM,Seagate,None,...,0.0,Pesos (ARS),1060.0,0.0,None,366.5004,17989,30,550,STOCK - Ajustes de Stock


In [30]:
df_cleaned.columns

Index(['Fecha_de_Carga', 'Deposito', 'Comprobante', 'Cantidad',
       'Stock_Anterior', 'Stock_Actual', 'Codigo', 'Articulo', 'Marca',
       'Condicion_de_Venta', 'Fecha', 'Cliente_Proveedor',
       'Nro_cliente_proveedor', 'Precio_sinIVA', 'Total_sinIVA', 'Moneda',
       'Tipo_de_Cambio', 'Costo_Ultima_Compra', 'Compr_Asoc',
       'Tipo_Cambio_Compr', 'ID', 'Tipo ID', 'ID_Articulo', 'Clase'],
      dtype='object')

In [31]:
df_reporte_articulo_venta.columns

Index(['comprobante_articulo_venta', 'codigo_articulo_venta',
       'cantidad_articulo_venta', 'total_neto_articulo_venta',
       'total_final_articulo_venta', 'costo_total_articulo_venta',
       'interes_sin_imp_articulo_venta', 'impuestos_articulo_venta',
       'profit_articulo_venta', 'profit_ii_articulo_venta'],
      dtype='object')

In [32]:
# =========================
# CELDA ÚNICA — Match mov_stock vs ventas + df_merged (con export)
# =========================
import pandas as pd
import numpy as np
import re, os

OUTPUT_PATH_BASE = "/content/gdrive/My Drive/Grapes/reporte_grapes/resultados/"
REPORTE_POST_REDONDEO_XLSX = os.path.join(OUTPUT_PATH_BASE, "df_reporte_articulo_venta_post_redondeo.xlsx")

USE_REPORTE_POST_REDONDEO_FILE = True
USE_PROXY_REMITO = True
ATOL_TOTAL = 1.0
ATOL_QTY   = 0.0
ONLY_YEAR  = 2025

OUT_XLSX = os.path.join(OUTPUT_PATH_BASE, f"diagnostico_match_mov_vs_ventas_{ONLY_YEAR}.xlsx")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)

def fmt_ar(x):
    if pd.isna(x): return ""
    s = f"{float(x):,.2f}"
    return s.replace(",", "_").replace(".", ",").replace("_", ".")

def _to_num(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def norm_code(s: pd.Series) -> pd.Series:
    return (s.astype(str).str.lower().str.strip()
             .str.replace(r"\s+", " ", regex=True))

def norm_comp(s: pd.Series) -> pd.Series:
    s0 = (s.astype(str).str.lower().str.strip()
           .str.replace(r"\s+", " ", regex=True))
    s1 = (s0.str.replace(r"[^\w\s\-\/]", " ", regex=True)
            .str.replace(r"\s+", " ", regex=True).str.strip())
    tipo  = s1.str.extract(r"^\s*(fc|nc|nd)\b", expand=False)
    letra = s1.str.extract(r"^\s*(?:fc|nc|nd)\s*([ab])\b", expand=False)
    nums  = s1.str.replace(r"^\s*(?:fc|nc|nd)\s*[ab]\b", "", regex=True).str.findall(r"\d+")

    def build(row):
        t, l, arr, raw = row
        if pd.isna(t) or pd.isna(l) or not isinstance(arr, list) or len(arr) == 0:
            return raw
        try:
            pto = int(arr[0]); nro = int(arr[-1])
            return f"{t} {l} {pto:05d}-{nro:08d}"
        except:
            return raw

    canon = pd.DataFrame({"tipo":tipo,"letra":letra,"nums":nums,"raw":s1}).apply(
        lambda r: build((r["tipo"],r["letra"],r["nums"],r["raw"])), axis=1)
    return canon.astype(str).str.lower().str.strip().str.replace(r"\s+"," ",regex=True)

if "df_cleaned" not in globals():
    raise NameError("df_cleaned no existe en memoria.")
if "df_reporte_articulo_venta" not in globals():
    raise NameError("df_reporte_articulo_venta no existe en memoria.")

# ---------- 1) MOV ----------
mov = df_cleaned.copy()
for c in ["Comprobante","Codigo"]:
    if c not in mov.columns:
        raise KeyError(f"df_cleaned debe tener columna '{c}'")

mov["k_comp"] = norm_comp(mov["Comprobante"])
mov["k_code"] = norm_code(mov["Codigo"])

if "Fecha_de_Carga" in mov.columns:
    mov["Fecha_de_Carga"] = pd.to_datetime(mov["Fecha_de_Carga"], errors="coerce", dayfirst=True)
else:
    mov["Fecha_de_Carga"] = pd.NaT
    print("⚠️ df_cleaned NO tiene Fecha_de_Carga.")

mov = _to_num(mov, ["Cantidad","Precio_sinIVA","Total_sinIVA","Tipo_de_Cambio"])

mov["es_proxy_remito"]           = False
mov["k_comp_origen_remito"]      = np.nan
mov["Comprobante_origen_remito"] = np.nan

mov_comp_dates = (mov.groupby("k_comp", as_index=False)
                     .agg(Fecha_comp_min=("Fecha_de_Carga","min"),
                          Fecha_comp_max=("Fecha_de_Carga","max")))

# ---------- 2) VENTAS ----------
if USE_REPORTE_POST_REDONDEO_FILE and os.path.exists(REPORTE_POST_REDONDEO_XLSX):
    df_reporte_articulo_venta = pd.read_excel(REPORTE_POST_REDONDEO_XLSX, engine="openpyxl")
    print(f"✅ Usando reporte POST-REDONDEO desde: {REPORTE_POST_REDONDEO_XLSX}")
else:
    print("⚠️ Usando df_reporte_articulo_venta en memoria.")

ven = df_reporte_articulo_venta.copy()
for c in ["comprobante_articulo_venta","codigo_articulo_venta"]:
    if c not in ven.columns:
        raise KeyError(f"Falta columna: '{c}'")

ven["k_comp"] = norm_comp(ven["comprobante_articulo_venta"])
ven["k_code"] = norm_code(ven["codigo_articulo_venta"])

cols_ven = [
    "cantidad_articulo_venta","total_neto_articulo_venta",
    "total_final_articulo_venta","costo_total_articulo_venta",
    "interes_sin_imp_articulo_venta","impuestos_articulo_venta",
    "profit_articulo_venta","profit_ii_articulo_venta",
]
cols_ven = [c for c in cols_ven if c in ven.columns]
ven = _to_num(ven, cols_ven)
ven = ven.merge(mov_comp_dates, on="k_comp", how="left")

date_candidates = [c for c in ven.columns
                   if c.lower() in ["fecha","fecha_comprobante","fecha_articulo_venta","fecha_venta"]]
ven["Fecha_reporte"] = pd.NaT
if date_candidates:
    ven["Fecha_reporte"] = pd.to_datetime(ven[date_candidates[0]], errors="coerce", dayfirst=True)

ven["anio_venta"] = ven["Fecha_reporte"].dt.year
ven.loc[ven["anio_venta"].isna(), "anio_venta"] = (
    pd.to_datetime(ven["Fecha_comp_min"], errors="coerce").dt.year)

# ---------- 3) MATCH ----------
mov_key_index = pd.MultiIndex.from_frame(mov[["k_comp","k_code"]].drop_duplicates())
ven_key_index = pd.MultiIndex.from_frame(ven[["k_comp","k_code"]].drop_duplicates())
ven["match_en_mov"] = pd.MultiIndex.from_frame(ven[["k_comp","k_code"]]).isin(mov_key_index)
mov["match_en_ven"] = pd.MultiIndex.from_frame(mov[["k_comp","k_code"]]).isin(ven_key_index)

# ---------- 4) PROXY remito->factura ----------
if USE_PROXY_REMITO:
    comp_l  = mov["Comprobante"].astype(str).str.lower()
    clase_l = mov["Clase"].astype(str).str.lower() if "Clase" in mov.columns else pd.Series("", index=mov.index)
    is_remito = (
        comp_l.str.contains(r"\brem\b|\bremito\b|\brto\b", regex=True, na=False) |
        clase_l.str.contains("remit", na=False)
    )
    mov_rem = mov.loc[is_remito].copy()
    mov_rem["Cantidad"]     = pd.to_numeric(mov_rem.get("Cantidad",    0), errors="coerce").fillna(0)
    mov_rem["Total_sinIVA"] = pd.to_numeric(mov_rem.get("Total_sinIVA",0), errors="coerce").fillna(0)

    if "cantidad_articulo_venta" in ven.columns:
        ven["cantidad_articulo_venta"]     = pd.to_numeric(ven["cantidad_articulo_venta"],     errors="coerce").fillna(0)
    if "total_neto_articulo_venta" in ven.columns:
        ven["total_neto_articulo_venta"]   = pd.to_numeric(ven["total_neto_articulo_venta"],   errors="coerce").fillna(0)

    ven_sin_mov  = ven.loc[~ven["match_en_mov"], ["k_comp","k_code"]].drop_duplicates()
    rem_by_code  = {k: g for k, g in mov_rem.groupby("k_code", sort=False)}
    proxies, ambiguous, no_match = [], 0, 0

    for _, rkey in ven_sin_mov.iterrows():
        k_comp_v = rkey["k_comp"]; k_code_v = rkey["k_code"]
        sub = ven.loc[(ven["k_comp"]==k_comp_v) & (ven["k_code"]==k_code_v)]
        if sub.empty: no_match += 1; continue
        v_qty = float(sub.get("cantidad_articulo_venta",    0).fillna(0).sum())
        v_tot = float(sub.get("total_neto_articulo_venta",  0).fillna(0).sum())
        cand  = rem_by_code.get(k_code_v)
        if cand is None or cand.empty: no_match += 1; continue
        c = cand.copy()
        ok_qty = (c["Cantidad"].abs()    - abs(v_qty)).abs() <= ATOL_QTY
        ok_tot = (c["Total_sinIVA"].abs()- abs(v_tot)).abs() <= ATOL_TOTAL
        c = c.loc[ok_qty & ok_tot]
        if len(c) == 1:
            rr = c.iloc[0].copy()
            rr["k_comp_origen_remito"]      = rr["k_comp"]
            rr["Comprobante_origen_remito"] = rr["Comprobante"]
            rr["k_comp"]      = k_comp_v
            rr["Comprobante"] = sub["comprobante_articulo_venta"].iloc[0]
            rr["es_proxy_remito"] = True
            proxies.append(rr)
        elif len(c) == 0: no_match += 1
        else: ambiguous += 1

    print(f"\n=========================")
    print(f"PROXY remito->factura")
    print(f"=========================")
    print("Proxies creados (match único):", len(proxies))
    print("Ambiguos:", ambiguous)
    print("Sin match:", no_match)
    mov_proxy = pd.concat([mov, pd.DataFrame(proxies)], ignore_index=True) if proxies else mov.copy()
else:
    mov_proxy = mov.copy()

mov_proxy["k_comp"] = norm_comp(mov_proxy["Comprobante"])
mov_proxy["k_code"] = norm_code(mov_proxy["Codigo"])

mov_key_index2 = pd.MultiIndex.from_frame(mov_proxy[["k_comp","k_code"]].drop_duplicates())
ven["match_en_mov"] = pd.MultiIndex.from_frame(ven[["k_comp","k_code"]]).isin(mov_key_index2)

ven_key_index2 = pd.MultiIndex.from_frame(ven[["k_comp","k_code"]].drop_duplicates())
mov_proxy["match_en_ven"] = pd.MultiIndex.from_frame(mov_proxy[["k_comp","k_code"]]).isin(ven_key_index2)

# ---------- 5) df_merged ----------
ven_agg = (ven.groupby(["k_comp","k_code"], as_index=False)[cols_ven].sum()
           if cols_ven else ven[["k_comp","k_code"]].drop_duplicates())
df_merged = mov_proxy.merge(ven_agg, on=["k_comp","k_code"], how="left")

# ── ORDENAR POR FECHA ASC ─────────────────────────────────
df_merged["Fecha_de_Carga"] = pd.to_datetime(
    df_merged["Fecha_de_Carga"], errors="coerce", dayfirst=True)
df_merged = df_merged.sort_values("Fecha_de_Carga", ascending=True).reset_index(drop=True)

print(f"\ndf_merged: {len(df_merged):,} filas ordenadas por fecha ASC")
print(f"Rango: {df_merged['Fecha_de_Carga'].min().date()} → {df_merged['Fecha_de_Carga'].max().date()}")

# ---------- 6) Diagnóstico NO-MATCH ----------
ventas_sin_mov_full      = ven.loc[~ven["match_en_mov"]].copy()
ventas_sin_mov_2025_full = ventas_sin_mov_full.loc[ventas_sin_mov_full["anio_venta"]==ONLY_YEAR].copy()
ventas_sin_mov_sin_fecha = ventas_sin_mov_full.loc[ventas_sin_mov_full["anio_venta"].isna()].copy()

mov_proxy["anio_mov"]    = pd.to_datetime(mov_proxy["Fecha_de_Carga"], errors="coerce").dt.year
mov_sin_ventas_full      = mov_proxy.loc[~mov_proxy["match_en_ven"]].copy()
mov_sin_ventas_2025_full = mov_sin_ventas_full.loc[mov_sin_ventas_full["anio_mov"]==ONLY_YEAR].copy()

if "total_neto_articulo_venta" in ven.columns:
    ven["total_neto_articulo_venta"] = pd.to_numeric(
        ven["total_neto_articulo_venta"], errors="coerce").fillna(0)
    tot_ventas_2025 = ven.loc[ven["anio_venta"]==ONLY_YEAR, "total_neto_articulo_venta"].sum()
    tot_match_2025  = ven.loc[(ven["anio_venta"]==ONLY_YEAR)&ven["match_en_mov"], "total_neto_articulo_venta"].sum()
    diff_2025       = tot_ventas_2025 - tot_match_2025
else:
    tot_ventas_2025 = tot_match_2025 = diff_2025 = np.nan

print(f"\n=========================")
print(f"RESUMEN NO-MATCH — {ONLY_YEAR}")
print(f"=========================")
print("Ventas sin mov (filas) 2025:", len(ventas_sin_mov_2025_full))
print("Mov sin ventas (filas) 2025:", len(mov_sin_ventas_2025_full))
print("Ventas sin mov (sin año imputable):", len(ventas_sin_mov_sin_fecha))
print(f"\n=========================")
print(f"TOTALES VENTAS {ONLY_YEAR}")
print(f"=========================")
print("Ventas TOTAL:",      fmt_ar(tot_ventas_2025))
print("Ventas MATCHEADAS:", fmt_ar(tot_match_2025))
print("DIF:",               fmt_ar(diff_2025))

resumen_diff_2025 = pd.DataFrame()
if len(ventas_sin_mov_2025_full) > 0 and "total_neto_articulo_venta" in ventas_sin_mov_2025_full.columns:
    tmp = ventas_sin_mov_2025_full.copy()
    tmp["total_neto_articulo_venta"] = pd.to_numeric(
        tmp["total_neto_articulo_venta"], errors="coerce").fillna(0)
    resumen_diff_2025 = (
        tmp.groupby(["comprobante_articulo_venta","Fecha_reporte","Fecha_comp_min"], dropna=False)
           .agg(lineas=("codigo_articulo_venta","count"),
                neto_sin_mov=("total_neto_articulo_venta","sum"))
           .reset_index()
           .sort_values("neto_sin_mov", ascending=False))
    show = resumen_diff_2025.head(30).copy()
    show["neto_sin_mov"] = show["neto_sin_mov"].map(fmt_ar)
    print("\n--- TOP 30 comprobantes que explican DIF ---")
    print(show.to_string(index=False))
else:
    print("\n(OK) No hay ventas 2025 sin mov.")

if "total_neto_articulo_venta" in ven.columns:
    pct_zero = (pd.to_numeric(ven["total_neto_articulo_venta"], errors="coerce")
                .fillna(0).eq(0).mean() * 100)
    print(f"\nChequeo: % líneas con neto=0: {pct_zero:.2f}%")
    if pct_zero > 50:
        print("⚠️ ATENCIÓN: demasiadas ventas con neto=0. Verificar post-redondeo.")

# ---------- 7) Export ----------
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    df_merged.to_excel(writer, sheet_name="df_merged", index=False)
    ventas_sin_mov_2025_full.to_excel(writer, sheet_name="ventas_sin_mov_2025", index=False)
    mov_sin_ventas_2025_full.to_excel(writer, sheet_name="mov_sin_ventas_2025", index=False)
    ventas_sin_mov_sin_fecha.to_excel(writer, sheet_name="ventas_sin_mov_sin_fecha", index=False)
    if len(resumen_diff_2025) > 0:
        resumen_diff_2025.to_excel(writer, sheet_name="resumen_diff_2025", index=False)

print(f"\n✅ Exportado diagnóstico: {OUT_XLSX}")
print("✅ df_merged quedó en memoria (mismo nombre para tus celdas siguientes).")

✅ Usando reporte POST-REDONDEO desde: /content/gdrive/My Drive/Grapes/reporte_grapes/resultados/df_reporte_articulo_venta_post_redondeo.xlsx

PROXY remito->factura
Proxies creados (match único): 132
Ambiguos: 21
Sin match: 1445

df_merged: 9,299 filas ordenadas por fecha ASC
Rango: 2023-08-01 → 2026-03-31

RESUMEN NO-MATCH — 2025
Ventas sin mov (filas) 2025: 554
Mov sin ventas (filas) 2025: 282
Ventas sin mov (sin año imputable): 56

TOTALES VENTAS 2025
Ventas TOTAL: 1.041.029.602,18
Ventas MATCHEADAS: 1.039.991.431,85
DIF: 1.038.170,33

--- TOP 30 comprobantes que explican DIF ---
comprobante_articulo_venta Fecha_reporte          Fecha_comp_min  lineas neto_sin_mov
       fc b 00006-00004422           NaT 2025-12-19 15:30:53.127       1   168.975,21
       fc b 00006-00004426           NaT 2025-12-24 10:40:57.097       1   168.975,21
       fc a 00006-00007277           NaT 2025-08-08 15:52:50.080       1   148.795,04
       fc a 00006-00007193           NaT 2025-07-30 13:43:23.987   

In [33]:
df_tarjetas.columns

Index(['tarjeta_credito', 'codigo_tarjeta', 'descripcion_tarjeta', 'nota', 'deshabilitado', 'cantidad_cuotas', 'porcentaje_interes_cuotas', 'porcentaje_adicional', 'grupo_sucursales', 'porcentaje_interes_cobranzas', 'condicion_venta_web',
       'usuario_creacion', 'usuario_ultima_modificacion', 'ultima_fecha_modificacion', 'id'],
      dtype='object')

In [34]:
"""import pandas as pd

# Carga el archivo Excel desde el path especificado
root_path = "/content/gdrive/My Drive/Colab Notebooks/Grapes/"
df_rellenar = pd.read_excel(root_path + "rellenar_filas.xlsx")

# Asegurarse de que las columnas para el merge no tengan problemas de formato
df_rellenar['Codigo_relleno'] = df_rellenar['Codigo_relleno'].str.strip().str.lower()
df_rellenar['Cliente_relleno'] = df_rellenar['Cliente_relleno'].str.strip().str.lower()

# Identificar filas donde 'Total_neto_reporte' es nulo en df_merged
mask_vacias = df_merged['Total_neto_reporte'].isna()

# Filas que necesitan actualización
rows_to_update = df_merged[mask_vacias]

# Contador para ver cuántas filas necesitan actualización
print(f"Total rows to update from rellenar_filas: {len(rows_to_update)}")

# Realizar la búsqueda adicional y actualizar las filas correspondientes
for index, row in rows_to_update.iterrows():
    match = df_rellenar[(df_rellenar['Codigo_relleno'] == row['Codigo']) &
                        (df_rellenar['Cantidad_relleno'] == row['Cantidad']) &
                        (df_rellenar['Cliente_relleno'] == row['Cliente_Proveedor'])]

    if not match.empty:
        df_merged.at[index, 'Total_neto_reporte'] = match.iloc[0]['Total_neto_relleno']
        df_merged.at[index, 'Costo_total_reporte'] = match.iloc[0]['Costo_total_relleno']
        df_merged.at[index, 'Interes_sin_imp_reporte'] = match.iloc[0]['Interes_sin_imp_relleno']
        df_merged.at[index, 'Profit_ii_reporte'] = match.iloc[0]['Profit_ii_relleno']
    else:
        print(f"No match found for index {index}: Codigo={row['Codigo']}, Cantidad={row['Cantidad']}, Cliente={row['Cliente_Proveedor']}")

# Completar filas vacías en 'Interes_sin_imp_reporte' con 0
df_merged['Interes_sin_imp_reporte'].fillna(0, inplace=True)

# Eliminar las filas que no tienen coincidencia
df_merged = df_merged.dropna(subset=['Total_neto_reporte'])


# Ver las primeras filas del DataFrame resultante
print(df_merged.head())"""


'import pandas as pd\n\n# Carga el archivo Excel desde el path especificado\nroot_path = "/content/gdrive/My Drive/Colab Notebooks/Grapes/"\ndf_rellenar = pd.read_excel(root_path + "rellenar_filas.xlsx")\n\n# Asegurarse de que las columnas para el merge no tengan problemas de formato\ndf_rellenar[\'Codigo_relleno\'] = df_rellenar[\'Codigo_relleno\'].str.strip().str.lower()\ndf_rellenar[\'Cliente_relleno\'] = df_rellenar[\'Cliente_relleno\'].str.strip().str.lower()\n\n# Identificar filas donde \'Total_neto_reporte\' es nulo en df_merged\nmask_vacias = df_merged[\'Total_neto_reporte\'].isna()\n\n# Filas que necesitan actualización\nrows_to_update = df_merged[mask_vacias]\n\n# Contador para ver cuántas filas necesitan actualización\nprint(f"Total rows to update from rellenar_filas: {len(rows_to_update)}")\n\n# Realizar la búsqueda adicional y actualizar las filas correspondientes\nfor index, row in rows_to_update.iterrows():\n    match = df_rellenar[(df_rellenar[\'Codigo_relleno\'] == row

In [35]:
import pandas as pd
import numpy as np
import re

# Fuzzy: intento fuzzywuzzy, si no está, caigo a difflib
try:
    from fuzzywuzzy import process
    _FUZZY_BACKEND = "fuzzywuzzy"
except Exception:
    import difflib
    _FUZZY_BACKEND = "difflib"

# ----------------------------
# PRECONDICIONES
#   df_merged
#   df_clientes_detalle (con 'razon_social_apellido')
# ----------------------------
if "df_merged" not in globals():
    raise NameError("df_merged no existe. Ejecutá antes la celda que lo construye.")
if "df_clientes_detalle" not in globals():
    raise NameError("df_clientes_detalle no existe. Cargalo antes de esta celda.")
if "razon_social_apellido" not in df_clientes_detalle.columns:
    raise KeyError("df_clientes_detalle no tiene columna 'razon_social_apellido'.")

# 1) Copia
df_merged_usd = df_merged.copy()

# 2) Asegurar numéricos base (ventas)
base_cols = [
    "total_neto_articulo_venta",
    "costo_total_articulo_venta",
    "interes_sin_imp_articulo_venta",
    "profit_ii_articulo_venta",
]
for c in base_cols:
    if c in df_merged_usd.columns:
        df_merged_usd[c] = pd.to_numeric(df_merged_usd[c], errors="coerce").fillna(0.0)
    else:
        df_merged_usd[c] = 0.0

# 3) Tipo de cambio: limpiar Tipo_de_Cambio del comprobante
if "Tipo_de_Cambio" not in df_merged_usd.columns:
    raise KeyError("No existe 'Tipo_de_Cambio' en df_merged.")

df_merged_usd["Tipo_de_Cambio"] = pd.to_numeric(df_merged_usd["Tipo_de_Cambio"], errors="coerce")
df_merged_usd.loc[df_merged_usd["Tipo_de_Cambio"] == 0, "Tipo_de_Cambio"] = np.nan

tc_nulos = df_merged_usd["Tipo_de_Cambio"].isna().sum()
if tc_nulos > 0:
    print(f"⚠️ {tc_nulos} filas sin Tipo_de_Cambio — los _usd de esas filas quedarán NaN.")
else:
    print("✅ Tipo_de_Cambio OK — sin valores nulos.")

# 4) Calcular USD usando el TC del propio comprobante
df_merged_usd["total_neto_articulo_venta_usd"]      = (df_merged_usd["total_neto_articulo_venta"]      / df_merged_usd["Tipo_de_Cambio"]).round(2)
df_merged_usd["costo_total_articulo_venta_usd"]     = (df_merged_usd["costo_total_articulo_venta"]     / df_merged_usd["Tipo_de_Cambio"]).round(2)
df_merged_usd["interes_sin_imp_articulo_venta_usd"] = (df_merged_usd["interes_sin_imp_articulo_venta"] / df_merged_usd["Tipo_de_Cambio"]).round(2)

df_merged_usd["profit_ii_articulo_venta_usd"] = (
    df_merged_usd["total_neto_articulo_venta_usd"]
    - df_merged_usd["costo_total_articulo_venta_usd"]
    - df_merged_usd["interes_sin_imp_articulo_venta_usd"]
).round(2)

# 5) Reordenar para que las _usd queden al final
usd_cols = [
    "total_neto_articulo_venta_usd",
    "costo_total_articulo_venta_usd",
    "interes_sin_imp_articulo_venta_usd",
    "profit_ii_articulo_venta_usd",
]
base_cols_all = [c for c in df_merged_usd.columns if c not in usd_cols]
df_merged_usd = df_merged_usd[base_cols_all + usd_cols]

# 6) Fuzzy matching (cacheado)
def _norm_text(x):
    if x is None:
        return ""
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return ""
    s = re.sub(r"\s+", " ", s)
    return s.lower()

clientes = df_clientes_detalle.copy()
clientes["razon_key"] = clientes["razon_social_apellido"].apply(_norm_text)

# 6.b) CLAVE: deduplicar clientes por razon_key (evita inflar filas de ventas)
clientes["razon_key"] = clientes["razon_key"].astype(str).str.strip().str.lower()

dup_keys = clientes["razon_key"].duplicated(keep=False)
print("Duplicados en df_clientes_detalle por razon_key:", int(dup_keys.sum()))
if dup_keys.any():
    if "id_detalle_cliente" in clientes.columns:
        clientes = (clientes
                    .sort_values(["razon_key", "id_detalle_cliente"])
                    .drop_duplicates(subset=["razon_key"], keep="last"))
    else:
        clientes = clientes.drop_duplicates(subset=["razon_key"], keep="last")

choices = (clientes["razon_key"]
           .dropna()
           .loc[lambda s: s != ""]
           .drop_duplicates()
           .tolist())

def best_match(xk, cutoff=80):
    if not xk:
        return None
    if _FUZZY_BACKEND == "fuzzywuzzy":
        m = process.extractOne(xk, choices)
        return m[0] if m and m[1] >= cutoff else None
    else:
        m = difflib.get_close_matches(xk, choices, n=1, cutoff=cutoff/100)
        return m[0] if m else None

uniq_cp = df_merged_usd["Cliente_Proveedor"].astype(str).fillna("").unique() if "Cliente_Proveedor" in df_merged_usd.columns else []
cp_map = {}
for v in uniq_cp:
    k = _norm_text(v)
    cp_map[v] = best_match(k, cutoff=80)

df_merged_usd["Mejor_Coincidencia"] = df_merged_usd["Cliente_Proveedor"].map(cp_map) if "Cliente_Proveedor" in df_merged_usd.columns else None

# 7) Merge con detalle de clientes (PROTEGIDO)
len_before = len(df_merged_usd)

df_merged_usd = df_merged_usd.merge(
    clientes[[
        "razon_key",
        "provincia_estado_region",
        "clase_fiscal",
        "tipos_de_documentos",
        "numero_de_documento",
        "id_detalle_cliente"
    ]],
    left_on="Mejor_Coincidencia",
    right_on="razon_key",
    how="left",
    validate="m:1"
)

len_after = len(df_merged_usd)
print(f"Filas antes merge clientes: {len_before} | después: {len_after} | Δ: {len_after - len_before}")

# 8) Limpiar auxiliares
df_merged_usd.drop(columns=[c for c in ["razon_key", "Mejor_Coincidencia"] if c in df_merged_usd.columns], inplace=True)

# 9) Reordenar para insertar datos cliente después de Cliente_Proveedor
cols = df_merged_usd.columns.tolist()
if "Cliente_Proveedor" in cols:
    i = cols.index("Cliente_Proveedor") + 1
    insert = [
        "provincia_estado_region",
        "clase_fiscal",
        "tipos_de_documentos",
        "numero_de_documento",
        "id_detalle_cliente"
    ]
    cols = cols[:i] + [c for c in insert if c in cols] + [c for c in cols[i:] if c not in insert]
    df_merged_usd = df_merged_usd[cols]

# 9.b) Chequeo extra
if all(c in df_merged_usd.columns for c in ["k_comp", "k_code"]):
    dup_post = df_merged_usd.duplicated(["k_comp", "k_code"]).sum()
    print("Duplicados por (k_comp,k_code) luego del merge clientes:", int(dup_post))

# 10) Preview
cols_preview = [c for c in ["Fecha_de_Carga","Comprobante","Codigo","Articulo","Tipo_de_Cambio","profit_ii_articulo_venta_usd"] if c in df_merged_usd.columns]
print("=== Primeras filas df_merged_usd (USD) ===\n")
display(df_merged_usd[cols_preview].head(10))

⚠️ 18 filas sin Tipo_de_Cambio — los _usd de esas filas quedarán NaN.
Duplicados en df_clientes_detalle por razon_key: 6
Filas antes merge clientes: 9299 | después: 9299 | Δ: 0
Duplicados por (k_comp,k_code) luego del merge clientes: 0
=== Primeras filas df_merged_usd (USD) ===



,Fecha_de_Carga,Comprobante,Codigo,Articulo,Tipo_de_Cambio,profit_ii_articulo_venta_usd
0,2023-08-01 09:18:18.940,Fc A 00006-00004091,AXB23-192-20-CISCO,Modulo SFP+ BiDi 10G 20KM 10GBase-BX-U SMF Sin...,287.5,278.54
1,2023-08-01 09:18:18.940,Fc A 00006-00004091,AXB32-192-40-CISCO,Modulo SFP+ BiDi 10G 40KM 10GBase-BX-D SMF Sin...,287.5,92.51
2,2023-08-01 09:18:18.940,Fc A 00006-00004091,AXB32-192-20-CISCO,Modulo SFP+ BiDi 10G 20KM 10GBase-BX-D SMF Sin...,287.5,278.54
3,2023-08-01 09:18:18.940,Fc A 00006-00004091,AXB23-192-40-CISCO,Modulo SFP+ BiDi 10G 40KM 10GBase-BX-U SMF Sin...,287.5,144.26
4,2023-08-01 09:21:33.390,Fc A 00006-00004092,ENVIOS-PACK,ANDREANI / OCA a Domicilio,287.5,7.43
5,2023-08-01 09:21:33.390,Fc A 00006-00004092,AXB32-192-20-ARUBA,Modulo SFP+ BiDi 10G 20KM 10GBase-BX-D SMF Sin...,287.5,125.62
6,2023-08-01 09:21:33.390,Fc A 00006-00004092,AXB23-192-20-ARUBA,Modulo SFP+ BiDi 10G 20KM 10GBase-BX-U SMF Sin...,287.5,125.62
7,2023-08-01 09:24:06.163,Fc B 00006-00003237,ENVIOS-PACK,ANDREANI / OCA a Domicilio,287.5,7.71
8,2023-08-01 09:24:06.163,Fc B 00006-00003237,82571-GE-2T-X1,Placa NIC 2x RJ45 1G - Intel 82571 - PCIe v1.0...,287.5,106.83
9,2023-08-01 11:17:48.360,Fc B 00006-00003238,763649111697,STFR4000800 Disco LaCie Rugged 4TB - USB-C,287.5,77.97


In [36]:
import pandas as pd
import numpy as np

# ==========================
# MERGE COSTOS + TOTALES (ALL y 2025 SIN DUPLICAR VENTAS) + EXPORT df_merged_usd.xlsx
# ==========================

# --- Precondiciones ---
if "df_merged_usd" not in globals():
    raise NameError("df_merged_usd no existe. Ejecutá antes la celda que arma df_merged_usd.")
if "df_costos" not in globals():
    raise NameError("df_costos no existe. Cargá df_costos antes de esta celda.")
if "Codigo" not in df_merged_usd.columns:
    raise KeyError("df_merged_usd no tiene columna 'Codigo'.")
if "Codigo_costo" not in df_costos.columns or "costo_neto_unitario" not in df_costos.columns:
    raise KeyError("df_costos debe tener columnas 'Codigo_costo' y 'costo_neto_unitario'.")

# 0) Robustez a rerun: si ya mergeaste antes, limpiamos columnas del merge previo
cols_drop_rerun = [c for c in ["costo_neto_unitario", "Codigo_costo"] if c in df_merged_usd.columns]
if cols_drop_rerun:
    df_merged_usd = df_merged_usd.drop(columns=cols_drop_rerun)

# 1) Normalizar keys (seguro contra NaN)  (NO rompe D+OEM: no tocás '+')
df_merged_usd["Codigo"] = (
    df_merged_usd["Codigo"]
    .astype(str)
    .replace({"nan": np.nan, "None": np.nan})
    .str.upper()
    .str.strip()
)

df_costos = df_costos.copy()
df_costos["Codigo_costo"] = (
    df_costos["Codigo_costo"]
    .astype(str)
    .replace({"nan": np.nan, "None": np.nan})
    .str.upper()
    .str.strip()
)

# 2) (Recomendado) deduplicar costos por código: me quedo con el último
#    + diagnóstico: cuántos códigos repetidos hay en costos
dups_costos = df_costos.duplicated(["Codigo_costo"]).sum()
print(f"Duplicados en df_costos por Codigo_costo (antes dedup): {int(dups_costos)}")

df_costos_dedup = (
    df_costos.sort_values(by=["Codigo_costo"])

             .drop_duplicates(subset=["Codigo_costo"], keep="last")
)

# 3) Merge costos (PROTEGIDO: validate m:1 + check filas)
len_before = len(df_merged_usd)

df_merged_usd = df_merged_usd.merge(
    df_costos_dedup[["Codigo_costo", "costo_neto_unitario"]],
    left_on="Codigo",
    right_on="Codigo_costo",
    how="left",
    validate="m:1"   # <- SI el right tiene duplicados reales, falla acá (mejor así)
)

len_after = len(df_merged_usd)
print(f"Filas antes del merge: {len_before}, filas después del merge: {len_after}")

# Si esto cambia, el merge te infló filas y hay que corregir antes de seguir.
if len_after != len_before:
    raise ValueError(f"❌ El merge de costos duplicó filas: Δ = {len_after - len_before}. Revisar df_costos_dedup.")

# 4) Match rate
match_rate = df_merged_usd["costo_neto_unitario"].notna().mean() * 100
print(f"✅ Match rate costo_neto_unitario: {match_rate:.2f}%")

# 5) Totales a comparar (ALL) - columnas que existan
cols_tot = [
    "total_neto_articulo_venta",
    "total_final_articulo_venta",
    "costo_total_articulo_venta",
    "interes_sin_imp_articulo_venta",
    "impuestos_articulo_venta",
    "profit_articulo_venta",
    "profit_ii_articulo_venta",
    "total_neto_articulo_venta_usd",
    "costo_total_articulo_venta_usd",
    "interes_sin_imp_articulo_venta_usd",
    "profit_ii_articulo_venta_usd",
]
cols_tot = [c for c in cols_tot if c in df_merged_usd.columns]

df_merged_usd[cols_tot] = df_merged_usd[cols_tot].apply(pd.to_numeric, errors="coerce")

def fmt_ar(x):
    if pd.isna(x):
        return ""
    s = f"{float(x):,.2f}"
    return s.replace(",", "_").replace(".", ",").replace("_", ".")

# ---- Diagnóstico duplicados por key (esto NO lo genera el merge de costos; ya venía de antes) ----
if all(c in df_merged_usd.columns for c in ["k_comp", "k_code"]):
    dup_count = df_merged_usd.duplicated(["k_comp", "k_code"]).sum()
    print("\n=========================")
    print("DIAGNÓSTICO DUPLICADOS")
    print("=========================")
    print("Filas df_merged_usd:", len(df_merged_usd))
    print("Keys únicas (k_comp,k_code):", df_merged_usd[["k_comp","k_code"]].drop_duplicates().shape[0])
    print("Filas duplicadas por key:", int(dup_count))
else:
    print("\n⚠️ No existen k_comp/k_code en df_merged_usd -> no puedo deduplicar ventas por key.")

# ---- TOTALES ALL (suma sobre filas = “detalle”, puede incluir duplicados de origen) ----
tot_all = df_merged_usd[cols_tot].sum(numeric_only=True)

print("\n=========================")
print("📊 TOTALES df_merged_usd - TODO EL PERIODO (SUMA SOBRE FILAS)")
print("=========================")
for col, val in tot_all.items():
    print(f"{col}: {fmt_ar(val)}")

# 6) TOTALES 2025 SIN DUPLICAR VENTAS
key_cols = [c for c in ["k_comp", "k_code"] if c in df_merged_usd.columns]

if "Fecha_de_Carga" in df_merged_usd.columns:
    df_merged_usd["Fecha_de_Carga"] = pd.to_datetime(df_merged_usd["Fecha_de_Carga"], errors="coerce", dayfirst=True)

# A) Construyo Fecha_comp_min si no existe
if "Fecha_comp_min" not in df_merged_usd.columns:
    if "Fecha_de_Carga" in df_merged_usd.columns and "k_comp" in df_merged_usd.columns:
        comp_dates = (
            df_merged_usd.groupby("k_comp", as_index=False)["Fecha_de_Carga"]
                        .min()
                        .rename(columns={"Fecha_de_Carga": "Fecha_comp_min"})
        )
        df_merged_usd = df_merged_usd.merge(comp_dates, on="k_comp", how="left", validate="m:1")
    else:
        print("\n⚠️ No puedo construir Fecha_comp_min (falta Fecha_de_Carga o k_comp).")

# B) Corte 2025: último día real (NO hardcode 11-30)
if key_cols and cols_tot and "Fecha_comp_min" in df_merged_usd.columns:
    df_sales_unique = df_merged_usd.drop_duplicates(subset=key_cols).copy()
    df_sales_unique["Fecha_comp_min"] = pd.to_datetime(df_sales_unique["Fecha_comp_min"], errors="coerce")

    start = pd.Timestamp("2025-01-01")
    end = df_sales_unique.loc[df_sales_unique["Fecha_comp_min"].dt.year == 2025, "Fecha_comp_min"].max()
    if pd.isna(end):
        end = pd.Timestamp("2025-12-31")
    end = end.normalize()

    mask_range = df_sales_unique["Fecha_comp_min"].between(start, end + pd.Timedelta(days=1) - pd.Timedelta(seconds=1))
    tot_2025 = df_sales_unique.loc[mask_range, cols_tot].sum(numeric_only=True)

    print("\n=========================")
    print(f"📊 TOTALES 2025 (DEDUP por k_comp,k_code) - {start.date()} a {end.date()}")
    print("=========================")
    for col, val in tot_2025.items():
        print(f"{col}: {fmt_ar(val)}")

    print(f"\n📌 Corte detectado en 2025 (Fecha_comp_min): {end.date()}")
else:
    print("\n⚠️ No puedo calcular 2025 SIN DUPLICAR: faltan k_comp/k_code o Fecha_comp_min o cols_tot.")

# 7) Preview mínimo
print("\n👀 Preview (Codigo, costo_neto_unitario):")
cols_prev = [c for c in ["Codigo", "costo_neto_unitario", "Codigo_costo"] if c in df_merged_usd.columns]
print(df_merged_usd[cols_prev].head(10))

# 8) Exportar a Excel (detalle). Si querés, te agrego 2 sheets: detalle + dedup.
output_path = "/content/gdrive/My Drive/Grapes/reporte_grapes/resultados/df_merged_usd.xlsx"
df_merged_usd.to_excel(output_path, index=False)
print(f"\n✅ Guardado: {output_path}")


Duplicados en df_costos por Codigo_costo (antes dedup): 0
Filas antes del merge: 9299, filas después del merge: 9299
✅ Match rate costo_neto_unitario: 93.34%

DIAGNÓSTICO DUPLICADOS
Filas df_merged_usd: 9299
Keys únicas (k_comp,k_code): 9299
Filas duplicadas por key: 0

📊 TOTALES df_merged_usd - TODO EL PERIODO (SUMA SOBRE FILAS)
total_neto_articulo_venta: 1.958.532.413,29
total_final_articulo_venta: 2.162.915.544,73
costo_total_articulo_venta: 1.217.952.310,38
interes_sin_imp_articulo_venta: 6.053.163,72
impuestos_articulo_venta: 204.383.130,99
profit_articulo_venta: 740.580.101,78
profit_ii_articulo_venta: 734.526.938,08
total_neto_articulo_venta_usd: 1.557.756,49
costo_total_articulo_venta_usd: 950.806,68
interes_sin_imp_articulo_venta_usd: 4.610,72
profit_ii_articulo_venta_usd: 602.339,09

📊 TOTALES 2025 (DEDUP por k_comp,k_code) - 2025-01-01 a 2025-12-31
total_neto_articulo_venta: 1.039.991.431,85
total_final_articulo_venta: 1.147.028.314,61
costo_total_articulo_venta: 696.382.106

In [37]:
df_merged_usd.columns

Index(['Fecha_de_Carga', 'Deposito', 'Comprobante', 'Cantidad', 'Stock_Anterior', 'Stock_Actual', 'Codigo', 'Articulo', 'Marca', 'Condicion_de_Venta', 'Fecha', 'Cliente_Proveedor', 'provincia_estado_region', 'clase_fiscal',
       'tipos_de_documentos', 'numero_de_documento', 'id_detalle_cliente', 'Nro_cliente_proveedor', 'Precio_sinIVA', 'Total_sinIVA', 'Moneda', 'Tipo_de_Cambio', 'Costo_Ultima_Compra', 'Compr_Asoc', 'Tipo_Cambio_Compr', 'ID', 'Tipo ID',
       'ID_Articulo', 'Clase', 'k_comp', 'k_code', 'es_proxy_remito', 'k_comp_origen_remito', 'Comprobante_origen_remito', 'match_en_ven', 'cantidad_articulo_venta', 'total_neto_articulo_venta', 'total_final_articulo_venta',
       'costo_total_articulo_venta', 'interes_sin_imp_articulo_venta', 'impuestos_articulo_venta', 'profit_articulo_venta', 'profit_ii_articulo_venta', 'total_neto_articulo_venta_usd', 'costo_total_articulo_venta_usd',
       'interes_sin_imp_articulo_venta_usd', 'profit_ii_articulo_venta_usd', 'Codigo_costo', 'co

In [38]:
import pandas as pd

# =========================
# GROUPBY POR COMPROBANTE (robusto)
# =========================

# 0) Base
df_base = df_merged_usd.copy()

# 1) Columnas numéricas (solo las que existan)
numeric_columns = [
    "total_neto_articulo_venta",
    "total_neto_articulo_venta_usd",
    "total_final_articulo_venta",
    "costo_total_articulo_venta",
    "costo_total_articulo_venta_usd",
    "interes_sin_imp_articulo_venta",
    "interes_sin_imp_articulo_venta_usd",
    "impuestos_articulo_venta",
    "profit_articulo_venta",
    "profit_ii_articulo_venta",
    "profit_ii_articulo_venta_usd",
]
numeric_columns = [c for c in numeric_columns if c in df_base.columns]

if numeric_columns:
    df_base[numeric_columns] = (
        df_base[numeric_columns]
        .apply(pd.to_numeric, errors="coerce")
        .round(2)
    )

# 2) Definir agregaciones: sum para métricas + first para “campos”
agg_map = {}

# métricas en pesos (sum)
for c in [
    "total_neto_articulo_venta",
    "total_final_articulo_venta",
    "costo_total_articulo_venta",
    "interes_sin_imp_articulo_venta",
    "impuestos_articulo_venta",
    "profit_articulo_venta",
    "profit_ii_articulo_venta",
]:
    if c in df_base.columns:
        agg_map[c] = "sum"

# métricas en USD (sum)
for c in [
    "total_neto_articulo_venta_usd",
    "costo_total_articulo_venta_usd",
    "interes_sin_imp_articulo_venta_usd",
    "profit_ii_articulo_venta_usd",
]:
    if c in df_base.columns:
        agg_map[c] = "sum"

# campos adicionales (first)
first_cols = [
    "Fecha_de_Carga",
    "Cantidad",
    "Stock_Anterior",
    "Stock_Actual",
    "Codigo",
    "Articulo",
    "Marca",
    "Condicion_de_Venta",
    "Fecha",
    "Cliente_Proveedor",
    "Precio_sinIVA",
    "Total_sinIVA",
    "Moneda",
    "Tipo_de_Cambio",
    "Costo_Ultima_Compra",
    "Compr_Asoc",
    "Tipo_Cambio_Compr",
    "ID",
    "Tipo ID",
    "ID_Articulo",
    "Clase",
    "Nro_cliente_proveedor",
]
for c in first_cols:
    if c in df_base.columns:
        agg_map[c] = "first"

# 3) Validación dura: si no hay nada para agregar, corto con mensaje claro
if not agg_map:
    raise ValueError(
        "agg_map quedó vacío: no se encontró ninguna de las columnas esperadas en df_merged_usd. "
        "Revisá df_merged_usd.columns y confirmá nombres."
    )

# 4) Groupby
df_grouped = (
    df_base.groupby("Comprobante", as_index=False)
           .agg(agg_map)
)

print("OK - df_grouped creado.")
display(df_grouped.head(10))


OK - df_grouped creado.


,Comprobante,total_neto_articulo_venta,total_final_articulo_venta,costo_total_articulo_venta,interes_sin_imp_articulo_venta,impuestos_articulo_venta,profit_articulo_venta,profit_ii_articulo_venta,total_neto_articulo_venta_usd,costo_total_articulo_venta_usd,interes_sin_imp_articulo_venta_usd,profit_ii_articulo_venta_usd,Fecha_de_Carga,Cantidad,Stock_Anterior,Stock_Actual,Codigo,Articulo,Marca,Condicion_de_Venta,Fecha,Cliente_Proveedor,Precio_sinIVA,Total_sinIVA,Moneda,Tipo_de_Cambio,Costo_Ultima_Compra,Compr_Asoc,Tipo_Cambio_Compr,ID,Tipo ID,ID_Articulo,Clase,Nro_cliente_proveedor
0,Aj.Stock 000000000000114,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2024-07-22 09:42:42.307,-3,0,0,AD-120W,Fuente Switching 12V 10A UpBRIGHT apto QNAP NA...,UpBright,None,2024-07-22,,0.0,0.0,Pesos (ARS),1350.0,0.0,None,NaN,20548,30,590,STOCK - Ajustes de Stock,0
1,Aj.Stock 000000000000115,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2024-07-25 19:05:35.750,0,58,80,2GB-SO-DDR3-UNK,RAM 2GB DDR3/L 1600-1866 Mhz 204-Pin TRANSCEND,Generico,None,2024-07-25,,0.0,0.0,Pesos (ARS),1350.0,0.0,None,NaN,20610,30,121,STOCK - Ajustes de Stock,0
2,Aj.Stock 000000000000117,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2024-11-20 10:54:34.240,-50,0,0,GLC-SX-MMD,Modulo SFP 1.25G 550Mts 850nm 1000Base-SX MMF ...,Cisco,None,2024-11-20,,0.0,0.0,Pesos (ARS),1260.0,0.0,None,NaN,22071,30,625,STOCK - Ajustes de Stock,0
3,Aj.Stock 000000000000118,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2024-11-20 11:01:44.720,-15,3,0,888462859158,Adaptador Apple Thunderbolt 3 USB-C a Thunderb...,Apple,None,2024-11-20,,0.0,0.0,Pesos (ARS),1260.0,0.0,None,NaN,22072,30,584,STOCK - Ajustes de Stock,0
4,Aj.Stock 000000000000119,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025-01-29 16:42:47.050,1,4,3,3660619406784,STHT8000800 Disco LaCie Rugged Raid Shuttle 8T...,LaCie,None,2025-01-29,,0.0,0.0,Pesos (ARS),1200.0,0.0,None,NaN,22885,30,600,STOCK - Ajustes de Stock,0
5,Aj.Stock 000000000000120,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025-03-20 13:46:30.177,-1,1,0,ST16000NM000J,HDD 16TB Seagate EXOS X18 NE000 512MB 7200RPM,Seagate,None,2025-03-20,,0.0,0.0,Pesos (ARS),1320.0,0.0,None,NaN,23527,30,572,STOCK - Ajustes de Stock,0
6,Aj.Stock 000000000000121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025-07-07 14:11:22.503,10,258,0,6976499412928,Modulo SFP+ 10G 20KM 10GBase-LR SMF Dual-LC OE...,10GteK,None,2025-07-07,,0.0,0.0,Pesos (ARS),1290.0,0.0,None,NaN,25517,30,378,STOCK - Ajustes de Stock,0
7,Aj.Stock 000000000000123,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025-11-18 10:36:58.703,-2,4,6,850065783635,UDC PARTS BLADES KIT X2 46IN 405380,Generico,None,2025-11-18,,0.0,0.0,Pesos (ARS),1480.0,0.0,None,NaN,27929,30,696,STOCK - Ajustes de Stock,0
8,Aj.Stock 000000000000396,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2023-11-03 11:38:18.187,0,1,0,ST18000NM000J,HDD 18TB Seagate EXOS X18 NE000 256MB 7200RPM,Seagate,None,2023-11-03,,0.0,0.0,Pesos (ARS),1060.0,0.0,None,366.5004,17989,30,550,STOCK - Ajustes de Stock,0
9,Aj.Stock 000000000000398,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2023-11-22 13:43:21.937,0,1,0,ST18000NM000J,HDD 18TB Seagate EXOS X18 NE000 256MB 7200RPM,Seagate,None,2023-11-22,,0.0,0.0,Pesos (ARS),1300.0,0.0,None,NaN,18291,30,550,STOCK - Ajustes de Stock,0


In [39]:
df_merged_usd.columns

Index(['Fecha_de_Carga', 'Deposito', 'Comprobante', 'Cantidad', 'Stock_Anterior', 'Stock_Actual', 'Codigo', 'Articulo', 'Marca', 'Condicion_de_Venta', 'Fecha', 'Cliente_Proveedor', 'provincia_estado_region', 'clase_fiscal',
       'tipos_de_documentos', 'numero_de_documento', 'id_detalle_cliente', 'Nro_cliente_proveedor', 'Precio_sinIVA', 'Total_sinIVA', 'Moneda', 'Tipo_de_Cambio', 'Costo_Ultima_Compra', 'Compr_Asoc', 'Tipo_Cambio_Compr', 'ID', 'Tipo ID',
       'ID_Articulo', 'Clase', 'k_comp', 'k_code', 'es_proxy_remito', 'k_comp_origen_remito', 'Comprobante_origen_remito', 'match_en_ven', 'cantidad_articulo_venta', 'total_neto_articulo_venta', 'total_final_articulo_venta',
       'costo_total_articulo_venta', 'interes_sin_imp_articulo_venta', 'impuestos_articulo_venta', 'profit_articulo_venta', 'profit_ii_articulo_venta', 'total_neto_articulo_venta_usd', 'costo_total_articulo_venta_usd',
       'interes_sin_imp_articulo_venta_usd', 'profit_ii_articulo_venta_usd', 'Codigo_costo', 'co

In [40]:
import pandas as pd

year, month = 2025, 11

# Aseguro datetime por si acaso
df_merged["Fecha_de_Carga"] = pd.to_datetime(df_merged["Fecha_de_Carga"], errors="coerce", dayfirst=True)
df_merged_usd["Fecha_de_Carga"] = pd.to_datetime(df_merged_usd["Fecha_de_Carga"], errors="coerce", dayfirst=True)

# Columnas a sumar (pesos y usd)
cols_pesos = [
    "total_neto_articulo_venta",
    "total_final_articulo_venta",
    "costo_total_articulo_venta",
    "interes_sin_imp_articulo_venta",
    "impuestos_articulo_venta",
    "profit_articulo_venta",
    "profit_ii_articulo_venta",
]

cols_usd = [
    "total_neto_articulo_venta_usd",
    "costo_total_articulo_venta_usd",
    "interes_sin_imp_articulo_venta_usd",
    "profit_ii_articulo_venta_usd",
]

# Me quedo solo con las que existen (robusto)
cols_pesos = [c for c in cols_pesos if c in df_merged.columns]
cols_usd   = [c for c in cols_usd   if c in df_merged_usd.columns]

# Filtro del mes
mask_pesos = (df_merged["Fecha_de_Carga"].dt.year == year) & (df_merged["Fecha_de_Carga"].dt.month == month)
mask_usd   = (df_merged_usd["Fecha_de_Carga"].dt.year == year) & (df_merged_usd["Fecha_de_Carga"].dt.month == month)

df_pesos_mes = df_merged.loc[mask_pesos].copy()
df_usd_mes   = df_merged_usd.loc[mask_usd].copy()

# Aseguro numéricos y sumo
df_pesos_mes[cols_pesos] = df_pesos_mes[cols_pesos].apply(pd.to_numeric, errors="coerce").fillna(0)
df_usd_mes[cols_usd]     = df_usd_mes[cols_usd].apply(pd.to_numeric, errors="coerce").fillna(0)

tot_pesos = df_pesos_mes[cols_pesos].sum()
tot_usd   = df_usd_mes[cols_usd].sum()

# Formateo AR
def fmt_ar(x):
    return f"{float(x):,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

# Imprimir estilo "col: 1.234,56"
print(f"\n=== Totales {year}-{month:02d} ===")

for c in cols_pesos:
    print(f"{c}: {fmt_ar(tot_pesos[c])}")

for c in cols_usd:
    print(f"{c}: {fmt_ar(tot_usd[c])}")



=== Totales 2025-11 ===
total_neto_articulo_venta: 92.692.516,53
total_final_articulo_venta: 102.229.995,61
costo_total_articulo_venta: 62.505.380,83
interes_sin_imp_articulo_venta: 37.172,85
impuestos_articulo_venta: 9.537.479,07
profit_articulo_venta: 30.187.135,63
profit_ii_articulo_venta: 30.149.962,78
total_neto_articulo_venta_usd: 62.283,95
costo_total_articulo_venta_usd: 41.991,63
interes_sin_imp_articulo_venta_usd: 24,82
profit_ii_articulo_venta_usd: 20.267,50


In [41]:
import pandas as pd
import numpy as np

# ==========================
# VALORIZACIÓN DE STOCK (último stock por Código) + agrupado por Marca
# ==========================

# 0) Parámetros
tipo_de_cambio = 1320
max_stock_value = 100000  # límite máximo razonable

# 1) Trabajo en copia (no piso df_merged_usd)
df = df_merged_usd.copy()

# 2) Asegurar numéricos (robusto)
for c in ["costo_neto_unitario", "Stock_Actual"]:
    if c not in df.columns:
        raise KeyError(f"Falta columna '{c}' en df_merged_usd")
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 3) Fecha a datetime (robusto)
if "Fecha_de_Carga" not in df.columns:
    raise KeyError("Falta columna 'Fecha_de_Carga' en df_merged_usd")

df["Fecha_de_Carga"] = pd.to_datetime(df["Fecha_de_Carga"], errors="coerce", dayfirst=True)

# 4) Filtros mínimos (evitar basura)
#    - Stock razonable
#    - costo_neto_unitario válido (si no hay costo, valorización no tiene sentido)
df = df[df["Stock_Actual"].between(0, max_stock_value, inclusive="both")]
df = df[df["costo_neto_unitario"].notna()]
df = df[df["Fecha_de_Carga"].notna()]

# 5) "Última entrada por Código"
#    Para evitar problemas con ties (misma fecha), ordeno y me quedo con el último.
#    Si tenés una columna más granular para desempatar (ej. ID), la agregamos al sort.
if "Codigo" not in df.columns:
    raise KeyError("Falta columna 'Codigo' en df_merged_usd")

df = df.sort_values(["Codigo", "Fecha_de_Carga"])
latest_entries = df.drop_duplicates(subset=["Codigo"], keep="last").copy()

# 6) Valorización USD
latest_entries["valorizacion_stock_usd"] = (
    latest_entries["Stock_Actual"] * latest_entries["costo_neto_unitario"]
).round(2)

# 7) Agrupar por Marca
if "Marca" not in latest_entries.columns:
    latest_entries["Marca"] = "SIN_MARCA"

grouped_by_marca = (
    latest_entries.groupby("Marca", as_index=False)
                 .agg(valorizacion_stock_usd=("valorizacion_stock_usd", "sum"))
)

# 8) Valorización ARS
grouped_by_marca["valorizacion_stock_ars"] = (grouped_by_marca["valorizacion_stock_usd"] * tipo_de_cambio).round(2)
grouped_by_marca["valorizacion_stock_usd"] = grouped_by_marca["valorizacion_stock_usd"].round(2)

# 9) Mostrar
grouped_by_marca = grouped_by_marca.sort_values("valorizacion_stock_usd", ascending=False)

print("✅ Valorización de stock agrupada por marca (top 10):\n")
print(grouped_by_marca.head(10))

print("\n📊 Totales globales:")
print(grouped_by_marca[["valorizacion_stock_usd", "valorizacion_stock_ars"]].sum())

print("\nℹ️ Control de filas:")
print("Códigos únicos (latest_entries):", latest_entries["Codigo"].nunique())
print("Filas latest_entries:", len(latest_entries))


✅ Valorización de stock agrupada por marca (top 10):

              Marca  valorizacion_stock_usd  valorizacion_stock_ars
11             QNAP                97019.79             128066122.8
0            10GteK                58679.77              77457296.4
9             LaCie                19851.17              26203544.4
8          Kingston                16273.53              21481059.6
10              OWC                13818.23              18240063.6
12          Seagate                 5709.14               7536064.8
16  Western Digital                 5205.01               6870613.2
14      Terramaster                 2573.29               3396742.8
13         Synology                  900.03               1188039.6
3              Asus                  155.37                205088.4

📊 Totales globales:
valorizacion_stock_usd    2.201985e+05
valorizacion_stock_ars    2.906621e+08
dtype: float64

ℹ️ Control de filas:
Códigos únicos (latest_entries): 230
Filas latest_entries: 230

In [42]:
df_merged_usd.columns

Index(['Fecha_de_Carga', 'Deposito', 'Comprobante', 'Cantidad', 'Stock_Anterior', 'Stock_Actual', 'Codigo', 'Articulo', 'Marca', 'Condicion_de_Venta', 'Fecha', 'Cliente_Proveedor', 'provincia_estado_region', 'clase_fiscal',
       'tipos_de_documentos', 'numero_de_documento', 'id_detalle_cliente', 'Nro_cliente_proveedor', 'Precio_sinIVA', 'Total_sinIVA', 'Moneda', 'Tipo_de_Cambio', 'Costo_Ultima_Compra', 'Compr_Asoc', 'Tipo_Cambio_Compr', 'ID', 'Tipo ID',
       'ID_Articulo', 'Clase', 'k_comp', 'k_code', 'es_proxy_remito', 'k_comp_origen_remito', 'Comprobante_origen_remito', 'match_en_ven', 'cantidad_articulo_venta', 'total_neto_articulo_venta', 'total_final_articulo_venta',
       'costo_total_articulo_venta', 'interes_sin_imp_articulo_venta', 'impuestos_articulo_venta', 'profit_articulo_venta', 'profit_ii_articulo_venta', 'total_neto_articulo_venta_usd', 'costo_total_articulo_venta_usd',
       'interes_sin_imp_articulo_venta_usd', 'profit_ii_articulo_venta_usd', 'Codigo_costo', 'co

In [43]:
import numpy as np
import pandas as pd

# --- Configuración de rutas ---
root_path        = "/content/gdrive/My Drive/Grapes/reporte_grapes/inicializacion/"
output_path_base = "/content/gdrive/My Drive/Grapes/reporte_grapes/resultados/"

# ==========================
# PRECONDICIONES
#   - df_valorizacion_stock
#   - df_merged_usd  (debe tener: Codigo, Articulo, Marca, Fecha_de_Carga, Stock_Actual, costo_neto_unitario)
# ==========================
req_cols = ["Codigo", "Fecha_de_Carga", "Stock_Actual", "costo_neto_unitario", "Marca"]
for c in req_cols:
    if c not in df_merged_usd.columns:
        raise KeyError(f"df_merged_usd no tiene columna requerida: {c}")

# Aseguro tipos base en df_merged_usd
df_merged_usd = df_merged_usd.copy()
df_merged_usd["Codigo"] = df_merged_usd["Codigo"].astype(str).str.upper().str.strip()
if "Articulo" in df_merged_usd.columns:
    df_merged_usd["Articulo"] = df_merged_usd["Articulo"].astype(str).str.strip()

df_merged_usd["Fecha_de_Carga"] = pd.to_datetime(df_merged_usd["Fecha_de_Carga"], errors="coerce", dayfirst=True)
df_merged_usd["Stock_Actual"] = pd.to_numeric(df_merged_usd["Stock_Actual"], errors="coerce")
df_merged_usd["costo_neto_unitario"] = pd.to_numeric(df_merged_usd["costo_neto_unitario"], errors="coerce")

# Paso 1: Cargar el stock final desde df_valorizacion_stock
df_stock_final = df_valorizacion_stock.rename(columns={
    "Codigo_valorizado":      "Codigo",
    "Articulo_valorizado":    "Articulo",
    "Cantidad_valorizado":    "Cantidad",
    "Costo_usd_valorizado":   "Costo_usd",
    "costo_total_valorizado": "valor_stock_final"
}).copy()

# Normalizo keys y numéricos en stock final
df_stock_final["Codigo"] = df_stock_final["Codigo"].astype(str).str.upper().str.strip()
df_stock_final["Articulo"] = df_stock_final["Articulo"].astype(str).str.strip()
df_stock_final["valor_stock_final"] = pd.to_numeric(df_stock_final["valor_stock_final"], errors="coerce").fillna(0.0)

# (Opcional recomendado) si hay duplicados en stock final por Codigo+Articulo, consolido valor final
df_stock_final = (
    df_stock_final.groupby(["Codigo", "Articulo"], as_index=False)
                  .agg(valor_stock_final=("valor_stock_final", "sum"))
)

# Paso 2: Calcular el stock inicial y su valor
mes_iniciales = pd.date_range("2023-01", "2024-04", freq="M")[::-1].to_period("M").astype(str)
stock_inicial_list = []

for _, row in df_stock_final.iterrows():
    codigo, articulo = row["Codigo"], row["Articulo"]
    encontrado = False

    for mes in mes_iniciales:
        df_ini = df_merged_usd[
            (df_merged_usd["Codigo"] == codigo) &
            (df_merged_usd["Fecha_de_Carga"].dt.to_period("M").astype(str) == mes)
        ]

        if not df_ini.empty:
            # primer movimiento del mes (Fecha mínima)
            primer = df_ini.loc[df_ini["Fecha_de_Carga"].idxmin()]

            stock_actual = pd.to_numeric(primer["Stock_Actual"], errors="coerce")
            costo_unit   = pd.to_numeric(primer["costo_neto_unitario"], errors="coerce")

            valor_ini = 0.0
            if pd.notna(stock_actual) and pd.notna(costo_unit):
                valor_ini = float(stock_actual) * float(costo_unit)

            stock_inicial_list.append({
                "Codigo": codigo,
                "Articulo": articulo,
                "Marca": primer.get("Marca", None),
                "valor_stock_inicial": valor_ini,
                "Fecha_Stock_Inicial": primer["Fecha_de_Carga"]
            })
            encontrado = True
            break

    if not encontrado:
        stock_inicial_list.append({
            "Codigo": codigo,
            "Articulo": articulo,
            "Marca": None,
            "valor_stock_inicial": 0.0,
            "Fecha_Stock_Inicial": pd.NaT
        })

df_stock_inicial = pd.DataFrame(stock_inicial_list)

# Paso 3: Combinar stock final e inicial
df_stock = pd.merge(
    df_stock_final[["Codigo", "Articulo", "valor_stock_final"]],
    df_stock_inicial[["Codigo", "Articulo", "Marca", "valor_stock_inicial", "Fecha_Stock_Inicial"]],
    on=["Codigo", "Articulo"],
    how="outer"
)

# Completo NaN
df_stock["valor_stock_final"] = pd.to_numeric(df_stock["valor_stock_final"], errors="coerce").fillna(0.0)
df_stock["valor_stock_inicial"] = pd.to_numeric(df_stock["valor_stock_inicial"], errors="coerce").fillna(0.0)

# Paso 4: Calcular rotación sin división por cero
denom = (df_stock["valor_stock_final"] + df_stock["valor_stock_inicial"]) / 2

df_stock["rotacion"] = np.where(
    denom != 0,
    (df_stock["valor_stock_final"] - df_stock["valor_stock_inicial"]) / denom,
    0.0
)

# (Opcional) Redondeo prolijo
df_stock["valor_stock_final"] = df_stock["valor_stock_final"].round(2)
df_stock["valor_stock_inicial"] = df_stock["valor_stock_inicial"].round(2)
df_stock["rotacion"] = pd.to_numeric(df_stock["rotacion"], errors="coerce").round(6)

# Paso 5: Guardar el resultado en la carpeta "resultados"
output_file = output_path_base + "rotacion_inventario.xlsx"
df_stock.to_excel(output_file, index=False)
print(f"✅ Guardado: {output_file}")

df_stock


/tmp/ipykernel_36231/3577987229.py:49: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  mes_iniciales = pd.date_range("2023-01", "2024-04", freq="M")[::-1].to_period("M").astype(str)


✅ Guardado: /content/gdrive/My Drive/Grapes/reporte_grapes/resultados/rotacion_inventario.xlsx


,Codigo,Articulo,valor_stock_final,Marca,valor_stock_inicial,Fecha_Stock_Inicial,rotacion
0,056035369841,EOL - Lowepro Ridgeline BP250AW - EOL,0.00,None,0.00,NaT,0.0
1,056035369872,EOL - Lowepro Ridgeline BP300AW - EOL,0.00,None,0.00,NaT,0.0
2,0811643015463,Adaptador USB 3.0 a Ethernet Gigabit RJ45 Newe...,0.00,None,0.00,NaT,0.0
3,085854220408,EOL - Case Logic DLBP-114 - EOL,0.00,None,0.00,NaT,0.0
4,085854226639,EOL - Case Logic CPL-106 - EOL,0.00,None,0.00,NaT,0.0
...,...,...,...,...,...,...,...
615,X002207R2V,D2-TB3 - DAS 2 Bahias Terramaster D2 - Thunder...,0.00,None,0.00,NaT,0.0
616,X520-10G-1S-X8,Placa NIC 1x SFP+ 10G - Intel 82599 X520-DA1 -...,0.00,10GteK,0.00,2024-03-08 09:22:59.363,0.0
617,X520-10G-2S-X8,Placa NIC 2x SFP+ 10G - Intel 82599 X520-DA2 -...,0.00,10GteK,0.00,2024-03-01 11:47:44.630,0.0
618,X550-10G-2T-X4,Placa NIC 2x RJ45 10G - Intel X550-AT2 - PCI-e...,0.00,10GteK,490.88,2023-12-19 16:05:26.193,-2.0


In [44]:
df_merged_usd.columns

Index(['Fecha_de_Carga', 'Deposito', 'Comprobante', 'Cantidad', 'Stock_Anterior', 'Stock_Actual', 'Codigo', 'Articulo', 'Marca', 'Condicion_de_Venta', 'Fecha', 'Cliente_Proveedor', 'provincia_estado_region', 'clase_fiscal',
       'tipos_de_documentos', 'numero_de_documento', 'id_detalle_cliente', 'Nro_cliente_proveedor', 'Precio_sinIVA', 'Total_sinIVA', 'Moneda', 'Tipo_de_Cambio', 'Costo_Ultima_Compra', 'Compr_Asoc', 'Tipo_Cambio_Compr', 'ID', 'Tipo ID',
       'ID_Articulo', 'Clase', 'k_comp', 'k_code', 'es_proxy_remito', 'k_comp_origen_remito', 'Comprobante_origen_remito', 'match_en_ven', 'cantidad_articulo_venta', 'total_neto_articulo_venta', 'total_final_articulo_venta',
       'costo_total_articulo_venta', 'interes_sin_imp_articulo_venta', 'impuestos_articulo_venta', 'profit_articulo_venta', 'profit_ii_articulo_venta', 'total_neto_articulo_venta_usd', 'costo_total_articulo_venta_usd',
       'interes_sin_imp_articulo_venta_usd', 'profit_ii_articulo_venta_usd', 'Codigo_costo', 'co

In [45]:
import pandas as pd
import numpy as np

output_path_base = "/content/gdrive/My Drive/Grapes/reporte_grapes/resultados/"

# =========================
# 0) Tipos base
# =========================
df = df_merged_usd.copy()

df["Fecha_de_Carga"] = pd.to_datetime(df["Fecha_de_Carga"], errors="coerce", dayfirst=True)

for c in ["Cantidad", "Stock_Actual"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    else:
        raise KeyError(f"Falta columna requerida: {c}")

# Normalizo keys mínimas (evita merges raros por espacios/casing)
for c in ["Codigo", "Articulo", "Marca"]:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip()
    else:
        raise KeyError(f"Falta columna requerida: {c}")

# =========================
# 1) Elegir periodo (RECOMENDADO)
#    - Si querés "últimos 30 días", lo calculo con max fecha.
# =========================
# Opción A: últimos N días
N_DIAS = 30
fecha_max = df["Fecha_de_Carga"].max()
fecha_min = fecha_max - pd.Timedelta(days=N_DIAS-1)

df_periodo = df[(df["Fecha_de_Carga"] >= fecha_min) & (df["Fecha_de_Carga"] <= fecha_max)].copy()

# Si preferís rango fijo, reemplazá por:
# fecha_min = pd.Timestamp("2025-11-01")
# fecha_max = pd.Timestamp("2025-11-30 23:59:59")
# df_periodo = df[(df["Fecha_de_Carga"] >= fecha_min) & (df["Fecha_de_Carga"] <= fecha_max)].copy()

# Días reales del periodo (mínimo 1 para no dividir por 0)
dias_periodo = (fecha_max - fecha_min).days + 1
dias_periodo = max(int(dias_periodo), 1)

print(f"Periodo usado: {fecha_min.date()} a {fecha_max.date()}  |  días_periodo = {dias_periodo}")

# =========================
# 2) Consumo total (ventas/salidas) por artículo
#    SUPUESTO: salidas vienen con Cantidad < 0
#    - Consumo_Total lo dejo positivo (abs).
# =========================
df_salidas = df_periodo[df_periodo["Cantidad"] < 0].copy()
df_salidas["Consumo"] = df_salidas["Cantidad"].abs()

ventas_totales = (
    df_salidas
    .groupby(["Codigo", "Articulo", "Marca"], as_index=False)
    .agg(Consumo_Total=("Consumo", "sum"))
)

ventas_totales["Consumo_Promedio_Diario"] = ventas_totales["Consumo_Total"] / dias_periodo

# Si tu sistema NO usa cantidades negativas para salidas:
# - En vez de df_salidas, filtrás por Clase (ej: "Factura" / "FC A" / etc.)
# - O usás alguna columna que indique "Venta" vs "Compra".
# (Lo ajustamos cuando me confirmes cómo viene Cantidad.)

# =========================
# 3) Stock actual (último valor por artículo)
# =========================
stock_actual = (
    df.sort_values("Fecha_de_Carga")
      .groupby(["Codigo", "Articulo", "Marca"], as_index=False)
      .agg(Stock_Actual=("Stock_Actual", "last"),
           Fecha_Ultimo_Stock=("Fecha_de_Carga", "last"))
)

# =========================
# 4) Días de stock = Stock / Consumo diario
#    - Evito división por cero
# =========================
df_stock_dias = ventas_totales.merge(stock_actual, on=["Codigo", "Articulo", "Marca"], how="left")

df_stock_dias["Dias_Stock_Deposito"] = np.where(
    df_stock_dias["Consumo_Promedio_Diario"] > 0,
    df_stock_dias["Stock_Actual"] / df_stock_dias["Consumo_Promedio_Diario"],
    np.nan
)

# =========================
# 5) Prolijidad + export
# =========================
df_stock_dias = df_stock_dias.round({
    "Consumo_Total": 2,
    "Consumo_Promedio_Diario": 4,
    "Stock_Actual": 2,
    "Dias_Stock_Deposito": 2
})

# Ordeno para ver críticos (pocos días) primero
df_stock_dias = df_stock_dias.sort_values("Dias_Stock_Deposito", ascending=True)

print(df_stock_dias.head(30))

output_file = output_path_base + "dias_stock_deposito.xlsx"
df_stock_dias.to_excel(output_file, index=False)
print(f"✅ Guardado: {output_file}")


Periodo usado: 2026-03-02 a 2026-03-31  |  días_periodo = 30
                Codigo                                           Articulo                  Marca  Consumo_Total  Consumo_Promedio_Diario  Stock_Actual      Fecha_Ultimo_Stock  Dias_Stock_Deposito
1                  573  HDD 300GB Lenovo 7XB7A00024-00YK013 2.5" SAS 1...               Generico              5                   0.1667             0 2026-03-26 16:07:34.937                  0.0
2                  574         HDD 2.4TB IBM 02PX539 02YC027 SAS 10k 12GB               Generico             10                   0.3333             0 2026-03-26 16:07:34.937                  0.0
8         763649111697         STFR4000800 Disco LaCie Rugged 4TB - USB-C                  LaCie              1                   0.0333             0 2026-03-26 15:58:26.227                  0.0
0         093053014117   LAC9000633 Disco LaCie Rugged MINI 4TB - USB 3.0                  LaCie              1                   0.0333             1 2026

In [46]:
import os
import pandas as pd
import itertools
import numpy as np

output_path_base = "/content/gdrive/My Drive/Grapes/reporte_grapes/resultados/"
os.makedirs(output_path_base, exist_ok=True)
out_file = os.path.join(output_path_base, "ventas_mensuales.xlsx")

# 0) Normalizar fechas y campos base
df_all = df_merged_usd.copy()
df_all["Fecha_de_Carga"] = pd.to_datetime(df_all["Fecha_de_Carga"], errors="coerce", dayfirst=True)

for col in ["Comprobante","Codigo","Articulo","Marca","Deposito","Clase"]:
    if col in df_all.columns:
        df_all[col] = df_all[col].astype(str).str.strip()
    else:
        df_all[col] = ""

df_all["Comprobante_norm"] = (
    df_all["Comprobante"].astype(str).str.lower()
        .str.replace(r"\s+", " ", regex=True).str.strip()
)

df_all["Clase_norm"] = df_all["Clase"].astype(str).str.lower().str.strip()

# 1) Filtrar desde agosto 2023 (para ventas mensuales)
df = df_all[df_all["Fecha_de_Carga"] >= "2023-08-01"].copy()

# 2) Solo documentos de venta/ajuste/remito, EXCLUYENDO presupuestos/cotizaciones
pat_ok   = r"^(?:fc\s+[ab]\b|nc\s+[abd]\b|rnt\b)"
pat_pres = r"\b(pre|pres|presup|presupuesto|cot|cotiz|cotización)\b"

mask_ok   = df["Comprobante_norm"].str.match(pat_ok, na=False)
mask_pres = df["Comprobante_norm"].str.contains(pat_pres, na=False)
mask_cls  = df["Clase_norm"].str.contains("presup", na=False)

df = df[mask_ok & ~mask_pres & ~mask_cls].copy()

if df.empty:
    raise ValueError("No quedaron filas luego de filtros (pat_ok / presup). Revisá los patrones o la data.")

# 3) Crear 'Mes' y asegurar 'Cantidad' numérica
df["Mes"] = df["Fecha_de_Carga"].dt.to_period("M")
df["Cantidad"] = pd.to_numeric(df["Cantidad"], errors="coerce").fillna(0)

# 4) Rango de meses (agosto-2023 → último mes presente en df)
start = pd.Period("2023-08", freq="M")
end   = df["Mes"].max()
months = pd.period_range(start=start, end=end, freq="M")

# 5) Malla productos × mes (todos los códigos conocidos)
#    (si Codigo es único, esto no cambia nada; solo blinda)
unique_items = (
    df_all[["Codigo","Articulo","Marca"]]
    .drop_duplicates(subset=["Codigo"], keep="first")
)

prod = list(itertools.product(unique_items["Codigo"], months))
df_monthly = pd.DataFrame(prod, columns=["Codigo","Mes"])
df_monthly = df_monthly.merge(unique_items, on="Codigo", how="left")

# 6) Sumar ventas por mes (sobre df depurado)
df_qty = df.groupby(["Codigo","Mes"], as_index=False)["Cantidad"].sum()
df_monthly = df_monthly.merge(df_qty, on=["Codigo","Mes"], how="left")

# 7) Detectar QS (quiebre de stock)
df["QS"] = (
    df["Comprobante_norm"].str.startswith("fc ", na=False) &
    (pd.to_numeric(df["Stock_Anterior"], errors="coerce") ==
     pd.to_numeric(df["Cantidad"], errors="coerce"))
)
df_qs = df[["Codigo","Mes","QS"]].drop_duplicates().sort_values(["Codigo","Mes"])

# ---- CAMBIO IMPORTANTE: first_purchase sobre TODO df_all (no solo ventas) ----
df_all_fp = df_all.copy()
df_all_fp = df_all_fp[df_all_fp["Fecha_de_Carga"].notna()].copy()

first_purchase = (
    df_all_fp[
        pd.to_numeric(df_all_fp["Stock_Actual"], errors="coerce") >
        pd.to_numeric(df_all_fp["Stock_Anterior"], errors="coerce")
    ]
    .groupby("Codigo")["Fecha_de_Carga"]
    .min()
    .dt.to_period("M")
    .to_dict()
)

# 8) Llenado de Cantidad_Final con QS / SRA
def get_monthly_value(row):
    code = row["Codigo"]
    mes  = row["Mes"]

    # 8a) Si hubo venta ese mes
    if pd.notna(row.get("Cantidad", None)):
        return row["Cantidad"]

    # 8b) Antes de la 1ª compra → SRA
    if code in first_purchase and mes < first_purchase[code]:
        return "SRA"

    # 8c) Después de la 1ª compra: ver si antes hubo QS
    prev = df_qs[(df_qs["Codigo"] == code) & (df_qs["Mes"] < mes)]
    if prev.empty:
        return 0
    return "QS" if bool(prev.iloc[-1]["QS"]) else 0

df_monthly["Cantidad_Final"] = df_monthly.apply(get_monthly_value, axis=1)

# 9) Pivot
df_pivot = df_monthly.pivot_table(
    index=["Codigo","Articulo","Marca"],
    columns="Mes",
    values="Cantidad_Final",
    aggfunc="first"
).reset_index()

# 10) Reordenar meses
month_cols = sorted([c for c in df_pivot.columns if isinstance(c, pd.Period)],
                    key=lambda x: x.to_timestamp())
df_pivot = df_pivot[["Codigo","Articulo","Marca"] + month_cols]
df_pivot.columns = ["Codigo","Articulo","Marca"] + [str(m) for m in month_cols]

# 11) Stock_Actual desde depósito Cerrito (última fecha)
mask_sa = (df_all["Deposito"] == "01-Deposito Cerrito") & df_all["Stock_Actual"].notna()
if mask_sa.any():
    idx = df_all[mask_sa].groupby("Codigo")["Fecha_de_Carga"].idxmax()
    last = df_all.loc[idx, ["Codigo","Stock_Actual"]].drop_duplicates("Codigo").set_index("Codigo")["Stock_Actual"]
else:
    last = pd.Series(dtype="float64")

# Promedio ventas (solo meses numéricos)
def _promedio_num(row, cols):
    nums = [v for v in row[cols].tolist() if isinstance(v, (int, float, np.integer, np.floating))]
    return round(sum(nums)/len(nums), 2) if nums else 0.0

num_cols = [str(m) for m in month_cols]
df_pivot["Promedio_Ventas"] = df_pivot.apply(_promedio_num, cols=num_cols, axis=1)
df_pivot["Stock_Actual"] = df_pivot["Codigo"].map(last)

# 12) Guardar
final_cols = ["Codigo","Articulo","Marca"] + num_cols + ["Promedio_Ventas","Stock_Actual"]
df_pivot[final_cols].to_excel(out_file, index=False)
print(f"✅ Archivo guardado en: {out_file}")


/tmp/ipykernel_36231/3705199185.py:35: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_pres = df["Comprobante_norm"].str.contains(pat_pres, na=False)


✅ Archivo guardado en: /content/gdrive/My Drive/Grapes/reporte_grapes/resultados/ventas_mensuales.xlsx


In [47]:
import os
import pandas as pd
import numpy as np
import itertools

# PRECONDICIONES:
#  • df_merged_usd
#  • output_path_base

out_base = output_path_base
os.makedirs(out_base, exist_ok=True)
out_file = os.path.join(out_base, "reporte_stock_completo.xlsx")

# =========================
# 0) Normalizar base + AñoMes + numéricos críticos
# =========================
df = df_merged_usd.copy()

df["Fecha_de_Carga"] = pd.to_datetime(df["Fecha_de_Carga"], errors="coerce", dayfirst=True)

for col in ["Comprobante","Codigo","Articulo","Marca","Deposito","Clase"]:
    if col not in df.columns:
        df[col] = ""
    df[col] = df[col].astype(str).str.strip()

df["AñoMes"] = df["Fecha_de_Carga"].dt.to_period("M")

# Normalizaciones robustas para clasificar
df["Comprobante_norm"] = (
    df["Comprobante"].astype(str)
      .str.lower()
      .str.normalize("NFKC")
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)

df["Clase_norm"] = (
    df["Clase"].astype(str)
      .str.lower()
      .str.normalize("NFKC")
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)

# Numéricos críticos (evita comparaciones/operaciones string)
for c in ["Cantidad","Stock_Anterior","Stock_Actual",
          "costo_total_articulo_venta_usd","costo_neto_unitario"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["Cantidad"] = df["Cantidad"].fillna(0)

# =========================
# 0b) Pares para dinero (canonización)
# =========================
pares = [
    ("abl35-24-20-d-cisco","abl53-24-20-d-cisco"),
    ("abl35-24-20-d-aruba","abl53-24-20-d-aruba"),
    ("axb23-192-20-cisco","axb32-192-20-cisco"),
    ("axb23-192-20-aruba","axb32-192-20-aruba"),
    ("axb23-192-40-cisco","axb32-192-40-cisco"),
    ("axb23-192-40-aruba","axb32-192-40-aruba"),
    ("abl45-24-80-d-cisco","abl54-24-80-d-cisco"),
]
pair_map = {}
for a, b in pares:
    pair_map[a] = a
    pair_map[b] = a

df["Codigo_lc"] = df["Codigo"].astype(str).str.lower().str.strip()
df["Codigo_Canonico"] = df["Codigo_lc"].map(pair_map).fillna(df["Codigo_lc"])

# Mapa Codigo -> Canonico (para mapear COGS canónico a cada Código único)
cod_to_canon = (
    df[["Codigo","Codigo_Canonico"]]
    .drop_duplicates(subset=["Codigo"], keep="first")
    .set_index("Codigo")["Codigo_Canonico"]
)

# =========================
# 1) Clasificar transacción (claro y determinístico)
# =========================
# Reglas:
# - COMPRA:     Clase_norm == '01.factura'
# - VENTA:      fc a/b  (y NO compra)
# - DEVOLUCION: nc a/b/d (y NO compra)
# - RNT:        lo tratamos como venta (si querés que cuente como salida)
# - OTRO:       resto

is_compra = df["Clase_norm"].eq("01.factura")

is_venta = df["Comprobante_norm"].str.startswith(("fc a","fc b"), na=False) & ~is_compra
is_dev   = df["Comprobante_norm"].str.startswith(("nc a","nc b","nc d"), na=False) & ~is_compra
is_rnt   = df["Comprobante_norm"].str.startswith("rnt", na=False) & ~is_compra

df["Tipo_Transaccion"] = np.select(
    [is_compra, is_venta | is_rnt, is_dev],
    ["compra", "venta", "devolucion"],
    default="otro"
)

# =========================
# 1b) First purchase real (SRA) desde TODO df (no solo ventas)
# =========================
# Entrada de stock: Stock_Actual > Stock_Anterior
mask_in = (df["Stock_Actual"].fillna(0) > df["Stock_Anterior"].fillna(0)) & df["Fecha_de_Carga"].notna()

first_purchase = (
    df.loc[mask_in]
      .groupby("Codigo")["Fecha_de_Carga"]
      .min()
      .dt.to_period("M")
      .to_dict()
)

# =========================
# 2) Scope ventas/devoluciones para mensual (no compras)
# =========================
df_ventas = df[df["Tipo_Transaccion"].isin(["venta","devolucion"])].copy()

df_ventas["venta_qty"] = np.where(df_ventas["Tipo_Transaccion"].eq("venta"), df_ventas["Cantidad"], 0)
df_ventas["dev_qty"]   = np.where(df_ventas["Tipo_Transaccion"].eq("devolucion"), df_ventas["Cantidad"], 0)

mensual = (
    df_ventas.groupby(["Codigo","AñoMes"], as_index=False)
             .agg(venta=("venta_qty","sum"),
                  devolucion=("dev_qty","sum"))
)

mensual["DemandaNeta"] = mensual["venta"] - mensual["devolucion"]

# QS (stockout): sobre ventas, robusto a NaN
# Si Cantidad ya viene positiva, NO usamos abs().
sa = pd.to_numeric(df_ventas["Stock_Anterior"], errors="coerce")
cq = pd.to_numeric(df_ventas["Cantidad"], errors="coerce")
df_ventas["QS"] = (
    df_ventas["Comprobante_norm"].str.startswith("fc ", na=False) &
    df_ventas["Tipo_Transaccion"].eq("venta") &
    (sa.fillna(-999999) == cq.fillna(-999998))
)

qs = df_ventas[df_ventas["QS"]][["Codigo","AñoMes"]].drop_duplicates()
qs["Stockout"] = True

mensual = mensual.merge(qs, on=["Codigo","AñoMes"], how="left")
mensual["Stockout"] = mensual["Stockout"].fillna(False).astype(bool)

# =========================
# 3) Matriz completa Codigo × AñoMes y pivot (cantidad)
# =========================
todos_meses = pd.period_range(mensual["AñoMes"].min(), mensual["AñoMes"].max(), freq="M")

# Si Codigo es único, deduplicar por Codigo (no por Articulo/Marca)
productos = (
    df[["Codigo","Articulo","Marca"]]
    .drop_duplicates(subset=["Codigo"], keep="first")
)

malla = (
    pd.DataFrame(itertools.product(productos["Codigo"], todos_meses), columns=["Codigo","AñoMes"])
      .merge(productos, on="Codigo", how="left")
)

dfm = (
    malla.merge(mensual, on=["Codigo","AñoMes"], how="left")
         .fillna({"venta":0,"devolucion":0,"DemandaNeta":0,"Stockout":False})
)

def completar(row):
    cod = row["Codigo"]
    ym  = row["AñoMes"]

    # Si hubo demanda neta ese mes, se reporta numérico
    if row["DemandaNeta"] != 0:
        return row["DemandaNeta"]

    # Antes de primera compra/entrada -> SRA
    fp = first_purchase.get(cod, None)
    if fp is not None and ym < fp:
        return "SRA"

    # Si en algún mes previo hubo stockout -> QS
    prev_qs = mensual[(mensual["Codigo"] == cod) & (mensual["AñoMes"] < ym)]["Stockout"]
    return "QS" if (not prev_qs.empty and prev_qs.any()) else 0

dfm["Cantidad_Final"] = dfm.apply(completar, axis=1)

df_pivot = (
    dfm.pivot_table(index=["Codigo","Articulo","Marca"],
                    columns="AñoMes",
                    values="Cantidad_Final",
                    aggfunc="first")
      .reset_index()
)

mes_cols = sorted([c for c in df_pivot.columns if isinstance(c, pd.Period)],
                  key=lambda x: x.to_timestamp())
df_pivot = df_pivot[["Codigo","Articulo","Marca"] + mes_cols]
df_pivot.columns = ["Codigo","Articulo","Marca"] + [str(m) for m in mes_cols]

# =========================
# 4) Stock real: Cerrito + Full (misma idea, pero numérico seguro)
# =========================
df_mov = df.copy()
df_mov["Stock_after"] = df_mov["Stock_Anterior"].fillna(0) - df_mov["Cantidad"].fillna(0)

df_cerr = df_mov[df_mov["Deposito"].eq("01-Deposito Cerrito")].copy()
stock_cerr = (
    df_cerr.sort_values("Fecha_de_Carga")
           .groupby("Codigo")["Stock_after"]
           .last()
           .rename("Stock_Cerrito")
)

df_full = df_mov[df_mov["Deposito"].eq("04-Deposito FULL")].copy()

sent_full = (
    df_full[df_full["Comprobante_norm"].str.startswith("rm r", na=False)]
      .groupby("Codigo")["Cantidad"]
      .sum()
      .mul(-1)
      .rename("Sent_Full")
)

sales_full = (
    df_full[df_full["Tipo_Transaccion"].eq("venta")]
      .sort_values("Fecha_de_Carga")
      .groupby("Codigo")["Stock_Actual"]
      .last()
      .rename("SaleStock_Full")
)

stock_full = sales_full.combine_first(sent_full).rename("Stock_Full")
stock_total = stock_cerr.add(stock_full, fill_value=0).rename("Stock_ActualUnid")

df_cerr["Mes"] = df_cerr["Fecha_de_Carga"].dt.to_period("M")
stock_prom_mes = (
    df_cerr.groupby(["Codigo","Mes"])["Stock_after"]
           .mean()
           .rename("StockPromedioMes")
           .reset_index()
)

# =========================
# 5) Métricas demanda/stock
# =========================
last = mensual["AñoMes"].max()
sixm  = last - 5
twelm = last - 11

avg6 = (mensual[mensual["AñoMes"].between(sixm,last)].groupby("Codigo")["DemandaNeta"].mean().rename("Promedio6m"))
avg12= (mensual[mensual["AñoMes"].between(twelm,last)].groupby("Codigo")["DemandaNeta"].mean().rename("Promedio12m"))
max6 = (mensual[mensual["AñoMes"].between(sixm,last)].groupby("Codigo")["DemandaNeta"].max().rename("MaxDem6m"))
max12= (mensual[mensual["AñoMes"].between(twelm,last)].groupby("Codigo")["DemandaNeta"].max().rename("MaxDem12m"))

stock_prom = (
    stock_prom_mes.groupby("Codigo")["StockPromedioMes"]
                  .mean()
                  .rename("StockPromedioUnid")
)

# =========================
# 6) Dinero: COGS canónico y mapeo a Código
# =========================
if "costo_total_articulo_venta_usd" not in df.columns:
    df["costo_total_articulo_venta_usd"] = 0.0

df["costo_total_articulo_venta_usd"] = pd.to_numeric(df["costo_total_articulo_venta_usd"], errors="coerce").fillna(0)

cogs_v = (
    df[df["Tipo_Transaccion"].eq("venta")]
      .groupby("Codigo_Canonico")["costo_total_articulo_venta_usd"]
      .sum()
)

cogs_d = (
    df[df["Tipo_Transaccion"].eq("devolucion")]
      .groupby("Codigo_Canonico")["costo_total_articulo_venta_usd"]
      .sum()
)

cogs_canon = cogs_v.subtract(cogs_d, fill_value=0).rename("COGS_USD")

# Costo unitario (último) desde Cerrito
cost_u = (
    df_cerr.sort_values("Fecha_de_Carga")
           .groupby("Codigo")["costo_neto_unitario"]
           .last()
           .fillna(0)
           .rename("CostoUnitarioUSD")
)

# Armo tabla base por Código (único)
metricas = pd.concat([avg6, avg12, max6, max12, stock_total, stock_prom, cost_u], axis=1).fillna(0)
metricas.index.name = "Codigo"
metricas = metricas.reset_index()

# Agrego Canonico y COGS mapeado
metricas["Codigo_Canonico"] = metricas["Codigo"].map(cod_to_canon)
metricas["COGS_USD"] = metricas["Codigo_Canonico"].map(cogs_canon).fillna(0)

# Agrego Articulo/Marca
metricas = metricas.merge(productos, on="Codigo", how="left")

# =========================
# 7) Reorden (igual que vos)
# =========================
LT6   = 2
LT1_5 = 1.5
HZ    = 6
mult6 = HZ + LT6
mult1 = HZ + LT1_5

metricas["CantReorden_Max6m"]        = np.clip(metricas["MaxDem6m"]*mult6 - metricas["Stock_ActualUnid"], 0, None).round()
metricas["CantReorden_Max12m"]       = np.clip(metricas["MaxDem12m"]*mult6 - metricas["Stock_ActualUnid"], 0, None).round()
metricas["CantReorden_Prom6m_LT1_5"] = np.clip(metricas["Promedio6m"]*mult1 - metricas["Stock_ActualUnid"], 0, None).round()

# =========================
# 8) Rotación, Cobertura, Stock valorizado (con guards)
# =========================
den_rot = (metricas["StockPromedioUnid"] * metricas["CostoUnitarioUSD"]).replace(0, np.nan)
metricas["Rotacion"] = (metricas["COGS_USD"] / den_rot).replace([np.inf, -np.inf, np.nan], 0).round(2)

den_cov = (metricas["MaxDem6m"] / 30).replace(0, np.nan)
metricas["Dias_Cobertura"] = (metricas["Stock_ActualUnid"] / den_cov).replace([np.inf, -np.inf, np.nan], 0).round(1)

metricas["Stock_Valorizado_USD"] = (metricas["Stock_ActualUnid"] * metricas["CostoUnitarioUSD"]).round(2)

# =========================
# 9) Exportar (3 hojas)
# =========================
with pd.ExcelWriter(out_file, engine="openpyxl") as escritor:
    df_pivot.to_excel(escritor, sheet_name="Ventas_Mensuales", index=False)
    mensual[["Codigo","AñoMes","Stockout"]].to_excel(escritor, sheet_name="Stockouts", index=False)

    cols = [
        "Articulo","Marca","Codigo_Canonico",
        "Promedio6m","Promedio12m","MaxDem6m","MaxDem12m",
        "Stock_ActualUnid","StockPromedioUnid","COGS_USD","CostoUnitarioUSD",
        "CantReorden_Max6m","CantReorden_Max12m","CantReorden_Prom6m_LT1_5",
        "Rotacion","Dias_Cobertura","Stock_Valorizado_USD"
    ]
    metricas[["Codigo"] + cols].to_excel(escritor, sheet_name="Metricas_Stock", index=False)

print("✅ Reporte completo guardado en:", out_file)


/tmp/ipykernel_36231/1011765444.py:146: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mensual["Stockout"] = mensual["Stockout"].fillna(False).astype(bool)
/tmp/ipykernel_36231/1011765444.py:166: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna({"venta":0,"devolucion":0,"DemandaNeta":0,"Stockout":False})


✅ Reporte completo guardado en: /content/gdrive/My Drive/Grapes/reporte_grapes/resultados/reporte_stock_completo.xlsx


In [48]:
# ================================================
# PRODUCTOS SIN ROTACIÓN
# - Ignora SRA y QS (no cuentan como venta)
# - Ventana configurable: últimos N meses o año calendario
# - Si se pide excluir sin stock, intenta reconstruir Stock_Actual desde df_merged_usd (Cerrito)
# ================================================
import os
import re
import pandas as pd
import numpy as np

# ---------- CONFIGURACIÓN ----------
WINDOW_MODE = "last_n_months"     # "last_n_months" | "year"
N_MONTHS    = 6                   # usado si WINDOW_MODE == "last_n_months"
YEAR        = 2025                # usado si WINDOW_MODE == "year"

EXCLUDE_NO_STOCK = True           # True = excluir sin stock (<=0 o NaN)
CERRITO_NAME     = "01-Deposito Cerrito"

out_path = output_path_base
os.makedirs(out_path, exist_ok=True)

def _window_tag(cols):
    return f"{cols[0]}_a_{cols[-1]}".replace("-", "")

# ---------- 0) Tipos consistentes ----------
df_pivot = df_pivot.copy()
df_pivot["Codigo"] = df_pivot["Codigo"].astype(str).str.strip()

# ---------- 1) Detectar columnas de meses en df_pivot ----------
month_col_pattern = re.compile(r"^\d{4}-\d{2}$")
all_month_cols = [c for c in df_pivot.columns if isinstance(c, str) and month_col_pattern.match(c)]
if not all_month_cols:
    raise ValueError("No se detectaron columnas de meses en df_pivot (formato 'YYYY-MM').")

# Orden cronológico real
all_month_periods = pd.to_datetime(all_month_cols, format="%Y-%m")
all_month_cols_sorted = [c for _, c in sorted(zip(all_month_periods, all_month_cols))]

# ---------- 2) Elegir ventana ----------
if WINDOW_MODE == "last_n_months":
    target_cols = all_month_cols_sorted[-N_MONTHS:] if len(all_month_cols_sorted) >= N_MONTHS else all_month_cols_sorted
    window_label = f"Últimos {len(target_cols)} meses: {target_cols[0]} → {target_cols[-1]}"
elif WINDOW_MODE == "year":
    target_cols = [c for c in all_month_cols_sorted if c.startswith(f"{YEAR}-")]
    if not target_cols:
        raise ValueError(f"No hay columnas para el año {YEAR} en df_pivot.")
    window_label = f"Año {YEAR} completo ({target_cols[0]} → {target_cols[-1]})"
else:
    raise ValueError("WINDOW_MODE inválido. Usa 'last_n_months' o 'year'.")

outfile = os.path.join(out_path, f"productos_sin_rotacion_{_window_tag(target_cols)}.xlsx")

# ---------- 3) Asegurar/Reconstruir Stock_Actual si se va a excluir sin stock ----------
stock_mask_series = pd.Series(True, index=df_pivot.index)

if EXCLUDE_NO_STOCK:
    if "Stock_Actual" not in df_pivot.columns:
        # Intentar reconstruir desde df_merged_usd (Cerrito)
        try:
            df_aux = df_merged_usd.copy()
            df_aux["Codigo"] = df_aux["Codigo"].astype(str).str.strip()

            # asegurar columnas necesarias
            if "Fecha_de_Carga" not in df_aux.columns:
                raise KeyError("df_merged_usd no tiene 'Fecha_de_Carga'.")
            df_aux["Fecha_de_Carga"] = pd.to_datetime(df_aux["Fecha_de_Carga"], errors="coerce", dayfirst=True)

            if "Deposito" not in df_aux.columns:
                df_aux["Deposito"] = ""

            mask_cerr = (
                df_aux["Deposito"].astype(str).str.strip().eq(CERRITO_NAME) &
                df_aux["Stock_Actual"].notna() &
                df_aux["Fecha_de_Carga"].notna()
            )

            if mask_cerr.any():
                idx_last = df_aux.loc[mask_cerr].groupby("Codigo")["Fecha_de_Carga"].idxmax()
                last_stock = (
                    df_aux.loc[idx_last, ["Codigo","Stock_Actual"]]
                          .drop_duplicates("Codigo")
                          .set_index("Codigo")["Stock_Actual"]
                )
                df_pivot["Stock_Actual"] = df_pivot["Codigo"].map(last_stock)
            else:
                print("⚠️ No se pudo reconstruir Stock_Actual (no hay filas válidas en Cerrito). No se filtrará por stock.")
                EXCLUDE_NO_STOCK = False

        except Exception as e:
            print(f"⚠️ No se pudo reconstruir Stock_Actual por error: {e}. No se filtrará por stock.")
            EXCLUDE_NO_STOCK = False

    if EXCLUDE_NO_STOCK and "Stock_Actual" in df_pivot.columns:
        stock_mask_series = pd.to_numeric(df_pivot["Stock_Actual"], errors="coerce") > 0

# ---------- 4) Matriz numérica en ventana (ignorar SRA/QS) ----------
# SRA/QS -> NaN
win = df_pivot[target_cols].apply(pd.to_numeric, errors="coerce")

# ---------- 5) Criterio de no rotación ----------
# Sin rotación = suma en ventana == 0 (todo 0 o NaN)
no_rotation_mask = (win.fillna(0).sum(axis=1) == 0)

# ---------- 6) Construir resultado ----------
select_cols = ["Codigo","Articulo","Marca"]
if "Stock_Actual" in df_pivot.columns:
    select_cols.append("Stock_Actual")
select_cols.extend(target_cols)

result_mask = no_rotation_mask & stock_mask_series
no_rotation_df = df_pivot.loc[result_mask, select_cols].copy()

# Orden: más stock primero (si existe), luego Marca/Articulo
if "Stock_Actual" in no_rotation_df.columns:
    no_rotation_df["Stock_Actual"] = pd.to_numeric(no_rotation_df["Stock_Actual"], errors="coerce")
    no_rotation_df = no_rotation_df.sort_values(["Stock_Actual","Marca","Articulo"], ascending=[False, True, True])
else:
    no_rotation_df = no_rotation_df.sort_values(["Marca","Articulo"], ascending=[True, True])

# ---------- 7) Exportar ----------
no_rotation_df.to_excel(outfile, index=False)

print("✅ Productos sin rotación exportados.")
print(f"Ventana analizada: {window_label}")
print(f"Total artículos sin ventas en la ventana{' (con stock > 0)' if EXCLUDE_NO_STOCK else ''}: {len(no_rotation_df)}")
print("Archivo:", outfile)


✅ Productos sin rotación exportados.
Ventana analizada: Últimos 6 meses: 2025-10 → 2026-03
Total artículos sin ventas en la ventana (con stock > 0): 24
Archivo: /content/gdrive/My Drive/Grapes/reporte_grapes/resultados/productos_sin_rotacion_202510_a_202603.xlsx


#ANALISIS MERCADOLIBRE FULL




In [49]:
# ======================================================
# FULL (04-Deposito FULL): ingresos positivos, ventas netas mensuales,
# meses/días en stock, stock actual, objetivo mensual 4m y
# PROMEDIO mensual desde el ingreso (incluye meses con 0 ventas)
# ======================================================

import os
import numpy as np
import pandas as pd

# -----------------------------
# 0) Normalizaciones mínimas
# -----------------------------
df_all = df_merged_usd.copy()

# columnas mínimas esperadas
for col in ["Fecha_de_Carga", "Comprobante", "Codigo", "Articulo", "Marca", "Cantidad", "Deposito"]:
    if col not in df_all.columns:
        raise KeyError(f"df_merged_usd no tiene la columna requerida: '{col}'")

df_all["Fecha_de_Carga"] = pd.to_datetime(df_all["Fecha_de_Carga"], errors="coerce", dayfirst=True)
df_all["Comprobante"]    = df_all["Comprobante"].astype(str).str.strip().str.lower()
df_all["Codigo"]         = df_all["Codigo"].astype(str).str.strip()
df_all["Articulo"]       = df_all["Articulo"].astype(str).str.strip()
df_all["Marca"]          = df_all["Marca"].astype(str).str.strip()
df_all["Cantidad"]       = pd.to_numeric(df_all["Cantidad"], errors="coerce").fillna(0)

# Última fecha del documento (para “hace cuánto”)
ultima_fecha_doc = df_all["Fecha_de_Carga"].max()
if pd.isna(ultima_fecha_doc):
    raise ValueError("No hay fechas válidas en df_merged_usd['Fecha_de_Carga'].")

ultima_fecha_doc = ultima_fecha_doc.normalize()

# Subconjunto FULL
df_full = df_all[df_all["Deposito"].astype(str).str.strip().eq("04-Deposito FULL")].copy()

# Si FULL está vacío, igual generamos archivo vacío sin romper
if df_full.empty:
    out_dir = output_path_base if "output_path_base" in globals() else "."
    os.makedirs(out_dir, exist_ok=True)
    fname = os.path.join(out_dir, "full_resumen_con_objetivo_4m.xlsx")
    pd.DataFrame(columns=[
        "Codigo","Fecha_Ingreso","Articulo","Marca",
        "Ventas_Netas_Totales","Meses_en_Stock","Dias_en_Stock",
        "Unidades_Ingresadas","Stock_Actual_Full",
        "Objetivo_Venta_Mensual_4m","Promedio_Venta_Mensual_desde_Ingreso"
    ]).to_excel(fname, index=False)
    print(f"✅ FULL vacío. Archivo guardado en: {fname}")
    print(f"ℹ️ Última fecha de documento usada: {ultima_fecha_doc.date()}")
else:
    # -----------------------------
    # 1) Ingresos a FULL (rm r...) -> SIEMPRE POSITIVOS
    # -----------------------------
    mask_ing = df_full["Comprobante"].str.startswith("rm r", na=False)
    df_ingresos = df_full[mask_ing].copy()

    fecha_ingreso = (
        df_ingresos.groupby("Codigo")["Fecha_de_Carga"]
        .min()
        .rename("Fecha_Ingreso")
    )

    unid_ingresadas = (
        df_ingresos.groupby("Codigo")["Cantidad"]
        .apply(lambda s: s.abs().sum())
        .rename("Unidades_Ingresadas")
    )

    # -----------------------------
    # 2) Ventas netas (fc a/b suman, nc a/b/d restan) SOLO dentro de FULL
    # -----------------------------
    ventas_mask = df_full["Comprobante"].str.startswith(("fc a","fc b"), na=False)
    devol_mask  = df_full["Comprobante"].str.startswith(("nc a","nc b","nc d"), na=False)

    # Cantidad_Ajustada = +Cantidad para FC, -Cantidad para NC, 0 resto
    df_full["Cantidad_Ajustada"] = np.select(
        [ventas_mask, devol_mask],
        [df_full["Cantidad"], -df_full["Cantidad"]],
        default=0
    )

    ventas_net_tot = (
        df_full.groupby("Codigo")["Cantidad_Ajustada"]
        .sum()
        .rename("Ventas_Netas_Totales")
    )

    # Ventas netas por mes
    df_full["Mes"] = df_full["Fecha_de_Carga"].dt.to_period("M").astype(str)
    ventas_mens = (
        df_full.groupby(["Codigo","Mes"])["Cantidad_Ajustada"]
        .sum()
        .unstack(fill_value=0)
    )

    month_cols_sorted = sorted(list(ventas_mens.columns))

    # -----------------------------
    # 3) Stock actual (en FULL) por Código: último Stock_Actual observado
    # -----------------------------
    if "Stock_Actual" in df_full.columns:
        df_full["Stock_Actual"] = pd.to_numeric(df_full["Stock_Actual"], errors="coerce")
        mask_stock = df_full["Stock_Actual"].notna() & df_full["Fecha_de_Carga"].notna()
        if mask_stock.any():
            idx_last = df_full.loc[mask_stock].groupby("Codigo")["Fecha_de_Carga"].idxmax()
            stock_actual_full = (
                df_full.loc[idx_last, ["Codigo","Stock_Actual"]]
                      .drop_duplicates("Codigo")
                      .set_index("Codigo")["Stock_Actual"]
                      .rename("Stock_Actual_Full")
            )
        else:
            stock_actual_full = pd.Series(dtype="float64", name="Stock_Actual_Full")
    else:
        stock_actual_full = pd.Series(dtype="float64", name="Stock_Actual_Full")

    # -----------------------------
    # 4) Base descriptiva (Articulo, Marca)
    # -----------------------------
    base = (
        df_full[["Codigo","Articulo","Marca"]]
        .drop_duplicates(subset=["Codigo"], keep="last")
        .set_index("Codigo")
    )

    # -----------------------------
    # 5) Armar DataFrame final base (SIN fillna global)
    # -----------------------------
    res_full = base.join([fecha_ingreso, unid_ingresadas, ventas_net_tot, stock_actual_full], how="outer")

    # Adjuntar matriz mensual (si existe)
    if not ventas_mens.empty:
        res_full = res_full.join(ventas_mens, how="left")

    # Asegurar columnas numéricas a rellenar
    numeric_core = ["Unidades_Ingresadas","Ventas_Netas_Totales","Stock_Actual_Full"]
    for c in numeric_core:
        if c not in res_full.columns:
            res_full[c] = 0

    # Rellenar SOLO numéricas (meses + métricas), NO Fecha_Ingreso / Articulo / Marca
    month_cols_present = [c for c in month_cols_sorted if c in res_full.columns]
    numeric_cols = numeric_core + month_cols_present
    res_full[numeric_cols] = res_full[numeric_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

    # -----------------------------
    # 6) Meses/Días en stock (desde Fecha_Ingreso)
    # -----------------------------
    fi = pd.to_datetime(res_full["Fecha_Ingreso"], errors="coerce")

    dias_en_stock = np.where(
        fi.notna(),
        (ultima_fecha_doc - fi).dt.days,
        0
    ).astype(int)

    res_full["Dias_en_Stock"]  = dias_en_stock
    res_full["Meses_en_Stock"] = (res_full["Dias_en_Stock"] / 30.44).round(1)

    # -----------------------------
    # 7) Objetivo de venta mensual para agotar a los 4 meses
    # -----------------------------
    meses_transcurridos = (res_full["Dias_en_Stock"] / 30.44)
    meses_restantes = np.clip(4 - meses_transcurridos, 0, None)

    res_full["Objetivo_Venta_Mensual_4m"] = np.where(
        res_full["Stock_Actual_Full"] <= 0,
        0,
        np.where(
            meses_restantes > 0,
            (res_full["Stock_Actual_Full"] / meses_restantes).round(2),
            res_full["Stock_Actual_Full"]
        )
    )

    # -----------------------------
    # 8) Promedio mensual desde el ingreso (incluye meses con 0)
    # -----------------------------
    # Asegurar que existan columnas de meses (si no, promedio = 0)
    for m in month_cols_sorted:
        if m not in res_full.columns:
            res_full[m] = 0.0

    def _promedio_desde_ingreso(row):
        fi = row["Fecha_Ingreso"]
        if pd.isna(fi):
            return 0.0
        start_m = pd.to_datetime(fi).to_period("M").strftime("%Y-%m")
        cols = [c for c in month_cols_sorted if c >= start_m]
        if not cols:
            return 0.0
        vals = [float(row[c]) if pd.notna(row[c]) else 0.0 for c in cols]
        return round(sum(vals) / len(cols), 2)

    res_full["Promedio_Venta_Mensual_desde_Ingreso"] = res_full.apply(_promedio_desde_ingreso, axis=1)

    # -----------------------------
    # 9) Ordenar columnas
    # -----------------------------
    res_full = res_full.reset_index()  # Codigo vuelve a columna

    cols_inicio = ["Codigo","Fecha_Ingreso","Articulo","Marca"]
    month_cols_sorted_final = sorted([c for c in res_full.columns if isinstance(c, str) and len(c) == 7 and c[:4].isdigit()])

    cols_metric = [
        "Ventas_Netas_Totales",
        "Meses_en_Stock","Dias_en_Stock",
        "Unidades_Ingresadas","Stock_Actual_Full",
        "Objetivo_Venta_Mensual_4m",
        "Promedio_Venta_Mensual_desde_Ingreso"
    ]

    for c in cols_inicio + month_cols_sorted_final + cols_metric:
        if c not in res_full.columns:
            res_full[c] = 0

    res_full = res_full[cols_inicio + month_cols_sorted_final + cols_metric]

    # -----------------------------
    # 10) Guardar a Excel
    # -----------------------------
    out_dir = output_path_base if "output_path_base" in globals() else "."
    os.makedirs(out_dir, exist_ok=True)
    fname = os.path.join(out_dir, "full_resumen_con_objetivo_4m.xlsx")
    res_full.to_excel(fname, index=False)

    print(f"✅ FULL resumido guardado en: {fname}")
    print(f"ℹ️ Última fecha de documento usada: {ultima_fecha_doc.date()}")


✅ FULL resumido guardado en: /content/gdrive/My Drive/Grapes/reporte_grapes/resultados/full_resumen_con_objetivo_4m.xlsx
ℹ️ Última fecha de documento usada: 2026-03-31


#ORDEN 10GTEK

In [50]:
# ============================================================
# PLAN DE REPOSICION 10GTeK v2
#
# Objetivo: cubrir Lead time (2m) + Cobertura (6m) = 8 meses
# Próximo pedido esperado: diciembre 2026
#
# Correcciones v2 vs v1:
#  - mu_activo = Total12m / Meses_venta (no divide por meses en 0 por QS)
#  - sigma_activo calculado solo sobre meses con ventas > 0
#  - consumo_proyectado: QS usa mu_activo × 1.10 (NO max12 que infla con picos)
#  - Q formula order-up-to: Target = μ × 8 + Z × σ × √8 (SS cubre todo el ciclo)
#  - Z_COB = 1.28 (90% NS — objetivo "justo", no sobrestock)
#
# Jerarquía consumo proyectado:
#   1. Lote confiable (>= 3m) → Consumo_implicito_lote como base
#      QS_lote: +15% | SUBE: +20% | BAJA: -15%
#   2. Sin lote confiable → base = mu_activo
#      Pico concentrado sin recompra → mu_sinpico
#      SUBE  → max(mu_activo, avg_signal) × 1.15
#      BAJA  → mu_activo × 0.85
#      QS    → mu_activo × 1.10
#      resto → mu_activo
# ============================================================

import os, itertools
import numpy as np
import pandas as pd

FECHA_STOCK       = pd.Timestamp("2026-04-06")
LEAD_TIME         = 2.0
COBERTURA_OBJ     = 6.0
MESES_TOTAL       = LEAD_TIME + COBERTURA_OBJ   # 8.0
Z_COB             = 1.28   # 90% NS para el período de cobertura
N_PROM            = 6
N_PICO_LARGO      = 12
K_SIGNAL          = 2
N_BASE_SIGNAL     = 4
THR_HIGH_BASE     = 20
THR_MID_BASE      = 5
THR_HIGH_UP       = 0.20
THR_MID_UP        = 0.35
THR_LOW_ABS       = 2
FACTOR_OVERSIZING = 2.0
LOTE_MIN_MESES    = 3.0
MARCA_FILTRO      = "10gtek"

out_dir  = output_path_base if "output_path_base" in globals() else "."
os.makedirs(out_dir, exist_ok=True)
out_file = os.path.join(out_dir, "plan_reposicion_10gtek.xlsx")

# ── 1) NORMALIZAR ──────────────────────────────────────────────────────
if "df_merged_usd" not in globals():
    raise NameError("df_merged_usd no existe.")

req  = ["Fecha_de_Carga","Marca","Comprobante","Codigo","Articulo","Cantidad","Stock_Actual","Clase"]
miss = [c for c in req if c not in df_merged_usd.columns]
if miss:
    raise KeyError(f"Faltan columnas: {miss}")

cliente_col = next(
    (c for c in ["Cliente_Proveedor","cliente_proveedor","Cliente","cliente"]
     if c in df_merged_usd.columns), None)
if cliente_col is None:
    print("AVISO: no se encontró columna cliente. Análisis de concentración desactivado.")

df0 = df_merged_usd.copy()
df0["Fecha_de_Carga"] = pd.to_datetime(df0["Fecha_de_Carga"], errors="coerce", dayfirst=True)
for col in ["Comprobante","Codigo","Articulo","Marca","Clase"]:
    df0[col] = df0[col].astype(str).str.strip()
df0["Codigo"]     = df0["Codigo"].str.replace(r"\s+","-",regex=True).str.replace(r"-{2,}","-",regex=True)
df0["Comp_norm"]  = df0["Comprobante"].str.lower().str.replace(r"\s+"," ",regex=True).str.strip()
df0["Clase_norm"] = df0["Clase"].str.lower().str.strip()
df0["Marca_norm"] = df0["Marca"].str.lower().str.strip()
df0["Cantidad"]     = pd.to_numeric(df0["Cantidad"],     errors="coerce").fillna(0)
df0["Stock_Actual"] = pd.to_numeric(df0["Stock_Actual"], errors="coerce")
if "Stock_Anterior" in df0.columns:
    df0["Stock_Anterior"] = pd.to_numeric(df0["Stock_Anterior"], errors="coerce")
if cliente_col:
    df0[cliente_col] = df0[cliente_col].astype(str).str.strip().str.lower()

# ── 2) FILTRAR 10GTeK ──────────────────────────────────────────────────
df_g = df0[df0["Marca_norm"].str.contains(MARCA_FILTRO, na=False) & df0["Fecha_de_Carga"].notna()].copy()
if df_g.empty:
    raise ValueError("No hay filas 10GTeK en df_merged_usd.")
ultima_fecha = df_g["Fecha_de_Carga"].max()
print(f"Filas 10GTeK: {len(df_g):,}  |  {df_g['Fecha_de_Carga'].min().date()} → {ultima_fecha.date()}")

# ── 3) CLASIFICAR ──────────────────────────────────────────────────────
is_compra = df_g["Clase_norm"].eq("01.factura")
pat_pres  = r"\b(?:pre|pres|presup|cot|cotiz)\b"
is_pres   = (df_g["Comp_norm"].str.contains(pat_pres, na=False, regex=True) |
             df_g["Clase_norm"].str.contains("presup", na=False))
is_venta  = df_g["Comp_norm"].str.startswith(("fc a","fc b","fc c"), na=False) & ~is_compra & ~is_pres
is_nc     = df_g["Comp_norm"].str.startswith(("nc a","nc b","nc c","nc d"), na=False) & ~is_compra

df_g["Tipo"]     = np.select([is_venta, is_nc, is_compra], ["venta","devolucion","compra"], default="otro")
df_g["Cant_neta"]= np.where(df_g["Tipo"]=="venta", df_g["Cantidad"].abs(),
                   np.where(df_g["Tipo"]=="devolucion", -df_g["Cantidad"].abs(), 0))
df_g["AnoMes"]   = df_g["Fecha_de_Carga"].dt.to_period("M")

# ── 4) QS POR MES — clásico + meses en cero con historial ─────────────
if "Stock_Anterior" in df_g.columns:
    sa = df_g["Stock_Anterior"].fillna(-999)
    cq = df_g["Cantidad"].abs().fillna(-998)
    df_g["QS_flag"] = (df_g["Tipo"]=="venta") & (sa == cq)
else:
    df_g["QS_flag"] = False

qs_clasico = df_g[df_g["QS_flag"]][["Codigo","AnoMes"]].drop_duplicates().assign(tuvo_QS=True)

codigos_con_ventas = set(df_g[df_g["Tipo"]=="venta"]["Codigo"].unique())
if "Stock_Anterior" in df_g.columns:
    stock_min_mes = (df_g.groupby(["Codigo","AnoMes"])["Stock_Anterior"]
                     .min().reset_index().rename(columns={"Stock_Anterior":"stock_min_mes"}))
    sin_stock_mes = stock_min_mes[
        (stock_min_mes["stock_min_mes"] <= 0) &
        (stock_min_mes["Codigo"].isin(codigos_con_ventas))
    ][["Codigo","AnoMes"]].assign(tuvo_QS=True)
    qs_por_mes = (pd.concat([qs_clasico, sin_stock_mes])
                  .drop_duplicates(["Codigo","AnoMes"]).reset_index(drop=True))
else:
    qs_por_mes = qs_clasico

# ── 5) ANÁLISIS ÚLTIMA COMPRA ──────────────────────────────────────────
df_compras = df_g[df_g["Tipo"]=="compra"].copy()
df_compras["Cant_compra"] = df_compras["Cantidad"].abs()

if df_compras.empty:
    print("AVISO: no se encontraron compras para 10GTeK.")
    duc = pd.DataFrame(columns=["Codigo","Ultima_compra_fecha","Ultima_compra_cant",
                                 "Ultima_compra_comp","Lote_duro_meses","Lote_termino_en_QS","Consumo_implicito_lote"])
else:
    cmp_agg = (df_compras.groupby(["Codigo","Comprobante","Fecha_de_Carga"], as_index=False)
                          .agg(Cant_compra=("Cant_compra","sum")))
    idx_ult = cmp_agg.groupby("Codigo")["Fecha_de_Carga"].idxmax()
    duc = (cmp_agg.loc[idx_ult, ["Codigo","Fecha_de_Carga","Cant_compra","Comprobante"]]
                  .copy()
                  .rename(columns={"Fecha_de_Carga":"Ultima_compra_fecha",
                                   "Cant_compra":"Ultima_compra_cant","Comprobante":"Ultima_compra_comp"}))
    primer_qs = df_g[df_g["QS_flag"]].groupby("Codigo")["Fecha_de_Carga"].min().rename("Primer_QS_fecha")
    duc = duc.merge(primer_qs.reset_index(), on="Codigo", how="left")

    def _calc_lote(row):
        fc = row["Ultima_compra_fecha"]
        qt = float(row["Ultima_compra_cant"] or 0)
        pq = row.get("Primer_QS_fecha")
        if pd.isna(fc):
            return pd.Series({"Lote_duro_meses":np.nan,"Lote_termino_en_QS":False,"Consumo_implicito_lote":np.nan})
        if pd.notna(pq) and pq > fc:
            fecha_fin = pq; termino_qs = True
        else:
            fecha_fin = FECHA_STOCK; termino_qs = False
        dur   = max((fecha_fin - fc).days / 30.44, 0.5)
        c_imp = round(qt / dur, 2) if dur > 0 else np.nan
        return pd.Series({"Lote_duro_meses":round(dur,1),"Lote_termino_en_QS":termino_qs,"Consumo_implicito_lote":c_imp})

    duc = pd.concat([duc, duc.apply(_calc_lote, axis=1)], axis=1)
    duc.drop(columns=["Primer_QS_fecha"], errors="ignore", inplace=True)

# ── 6) PIVOT VENTAS MENSUALES ──────────────────────────────────────────
df_ventas   = df_g[df_g["Cant_neta"] != 0].copy()
mensual     = df_ventas.groupby(["Codigo","AnoMes"], as_index=False).agg(Cant_mes=("Cant_neta","sum"))
todos_meses = pd.period_range(df_g["AnoMes"].min(), df_g["AnoMes"].max(), freq="M")
productos   = df_g[["Codigo","Articulo","Marca"]].drop_duplicates("Codigo", keep="last")

malla = (pd.DataFrame(itertools.product(productos["Codigo"], todos_meses), columns=["Codigo","AnoMes"])
           .merge(productos, on="Codigo", how="left"))
dfm = (malla.merge(mensual, on=["Codigo","AnoMes"], how="left")
            .merge(qs_por_mes[["Codigo","AnoMes","tuvo_QS"]], on=["Codigo","AnoMes"], how="left"))
dfm["Cant_mes"] = dfm["Cant_mes"].fillna(0)
dfm["tuvo_QS"]  = dfm["tuvo_QS"].fillna(False).infer_objects(copy=False)

fp_dict = (df_g[df_g["Stock_Actual"].fillna(0) > 0]
           .groupby("Codigo")["Fecha_de_Carga"].min().dt.to_period("M").to_dict())

def celda_mensual(row):
    cod, mes, cant, qs = row["Codigo"], row["AnoMes"], row["Cant_mes"], row["tuvo_QS"]
    if cant != 0: return cant
    fp = fp_dict.get(cod)
    if fp is not None and mes < fp: return "SRA"
    return "QS" if qs else 0

dfm["Valor_Mes"] = dfm.apply(celda_mensual, axis=1)

pivot = (dfm.pivot_table(index=["Codigo","Articulo","Marca"], columns="AnoMes",
                         values="Valor_Mes", aggfunc="first").reset_index())
mes_cols_p = sorted([c for c in pivot.columns if isinstance(c, pd.Period)], key=lambda x: x.to_timestamp())
pivot = pivot[["Codigo","Articulo","Marca"] + mes_cols_p]
pivot.columns = ["Codigo","Articulo","Marca"] + [str(m) for m in mes_cols_p]

# ── 7) MÉTRICAS DEMANDA ────────────────────────────────────────────────
ultimo_mes = df_g["AnoMes"].max()
m6m  = [m for m in sorted(todos_meses, reverse=True) if m<=ultimo_mes][:N_PROM]
m12m = [m for m in sorted(todos_meses, reverse=True) if m<=ultimo_mes][:N_PICO_LARGO]
msen = [m for m in sorted(todos_meses, reverse=True) if m<=ultimo_mes][:K_SIGNAL]
mbas = [m for m in sorted(todos_meses, reverse=True) if m<=ultimo_mes][K_SIGNAL:K_SIGNAL+N_BASE_SIGNAL]

def cs(meses):
    return [str(m) for m in meses if str(m) in pivot.columns]

c6 = cs(m6m); c12 = cs(m12m); csen = cs(msen); cbas = cs(mbas)

pn = (pivot.set_index("Codigo")
      [[c for c in pivot.columns if c not in ["Codigo","Articulo","Marca"]]]
      .apply(pd.to_numeric, errors="coerce").fillna(0))

met = productos.set_index("Codigo").copy()
met["Prom6m"]      = pn[c6].mean(axis=1).round(2)  if c6  else 0.0
met["Max6m"]       = pn[c6].max(axis=1).round(1)   if c6  else 0.0
met["Max12m"]      = pn[c12].max(axis=1).round(1)  if c12 else 0.0
met["Total12m"]    = pn[c12].sum(axis=1).round(0)  if c12 else 0.0
met["Meses_venta"] = (pn[c12] > 0).sum(axis=1)     if c12 else 0

# mu_activo: tasa real = Total12m / meses con ventas (no divide por meses en 0 por QS)
met["mu_activo"] = np.where(
    met["Meses_venta"] > 0,
    (met["Total12m"] / met["Meses_venta"]).round(2),
    0.0
)

# sigma_activo: desv.std solo en meses con ventas > 0 (excluye ceros por stockout)
def _sigma_activo(cod, cols):
    if not cols or cod not in pn.index: return 0.0
    vals    = pd.to_numeric(pn.loc[cod, cols], errors="coerce").fillna(0)
    activos = vals[vals > 0]
    if len(activos) < 2:
        return round(float(activos.iloc[0]) * 0.30, 2) if len(activos) == 1 else 0.0
    return round(float(activos.std(ddof=1)), 2)

met["sigma_activo"] = [_sigma_activo(c, c12) for c in met.index]

if csen and cbas:
    avg_s = pn[csen].mean(axis=1)
    avg_b = pn[cbas].mean(axis=1)
    met["Avg_signal_2m"] = avg_s.round(2)
    met["Avg_base_4m"]   = avg_b.round(2)
    dabs = avg_s - avg_b
    dpct = np.where(avg_b > 0, dabs / avg_b, np.nan)

    def _señal(bavg, da, dp):
        if np.isnan(dp):
            return "NUEVO" if bavg==0 and da>0 else "SIN DATOS"
        if bavg < 1:
            return "SUBE" if da>=THR_LOW_ABS else ("BAJA" if da<=-THR_LOW_ABS else "ESTABLE")
        if bavg >= THR_HIGH_BASE:
            return "SUBE" if dp>=THR_HIGH_UP else ("BAJA" if dp<=-THR_HIGH_UP else "ESTABLE")
        if bavg >= THR_MID_BASE:
            return "SUBE" if dp>=THR_MID_UP else ("BAJA" if dp<=-THR_MID_UP else "ESTABLE")
        return "SUBE" if da>=THR_LOW_ABS else ("BAJA" if da<=-THR_LOW_ABS else "ESTABLE")

    met["Señal"] = [_señal(b,d,p) for b,d,p in zip(avg_b.values, dabs.values, dpct)]
else:
    met["Avg_signal_2m"] = np.nan; met["Avg_base_4m"] = np.nan; met["Señal"] = "SIN DATOS"

qs_rec = set(qs_por_mes[qs_por_mes["AnoMes"].isin(m6m)]["Codigo"].unique())
met["QS_ultimos_6m"] = met.index.isin(qs_rec)

# ── 8) ANÁLISIS CONCENTRACIÓN DE PICO ─────────────────────────────────
if cliente_col and c6:
    df_ventas_6m = df_g[(df_g["Tipo"]=="venta") & (df_g["AnoMes"].isin(m6m))].copy()
    vc = (df_ventas_6m.groupby(["Codigo","AnoMes",cliente_col], as_index=False)
                       .agg(Cant_cliente=("Cant_neta","sum")))
    cant_mes_total = (df_ventas_6m.groupby(["Codigo","AnoMes"], as_index=False)
                                   .agg(Cant_total=("Cant_neta","sum")))
    idx_pico    = cant_mes_total.groupby("Codigo")["Cant_total"].idxmax()
    pico_mes_df = (cant_mes_total.loc[idx_pico, ["Codigo","AnoMes","Cant_total"]]
                                 .copy().rename(columns={"AnoMes":"Pico_mes","Cant_total":"Pico_cant"}))

    def _analizar_pico(row):
        cod = row["Codigo"]; pico_m = row["Pico_mes"]
        sub = vc[(vc["Codigo"]==cod) & (vc["AnoMes"]==pico_m)]
        n_cli   = sub[cliente_col].nunique()
        cli_nom = sub.loc[sub["Cant_cliente"].idxmax(), cliente_col] if n_cli==1 else None
        recompro = False
        if cli_nom is not None:
            meses_post = [m for m in m6m if m > pico_m]
            if meses_post:
                mask = ((df_ventas_6m["Codigo"]==cod) & (df_ventas_6m[cliente_col]==cli_nom) &
                        (df_ventas_6m["AnoMes"].isin(meses_post)))
                recompro = df_ventas_6m[mask]["Cant_neta"].sum() > 0
            if not recompro:
                pico_fecha = df_g[df_g["AnoMes"]==pico_m]["Fecha_de_Carga"].max()
                if pd.notna(pico_fecha):
                    hist_mask = ((df_g["Codigo"]==cod) & (df_g[cliente_col]==cli_nom) &
                                 (df_g["Tipo"]=="venta") & (df_g["Fecha_de_Carga"] > pico_fecha))
                    recompro = df_g[hist_mask]["Cant_neta"].sum() > 0
        return pd.Series({"Pico_clientes_unicos":n_cli,"Pico_concentrado":n_cli==1,
                          "Pico_cliente_nombre":cli_nom,"Pico_cliente_recompro":recompro})

    pico_df = pd.concat([pico_mes_df, pico_mes_df.apply(_analizar_pico, axis=1)], axis=1)

    def _prom_sin_pico(row):
        cod = row["Codigo"]; pico_m = row["Pico_mes"]
        otros = [str(m) for m in m6m if m != pico_m and str(m) in pn.columns]
        if not otros or cod not in pn.index: return np.nan
        return round(pd.to_numeric(pn.loc[cod, otros], errors="coerce").fillna(0).mean(), 2)

    pico_df["Prom6m_sin_pico"] = pico_df.apply(_prom_sin_pico, axis=1)
    pico_df["Pico_mes"]        = pico_df["Pico_mes"].astype(str)
    met = met.reset_index().merge(pico_df, on="Codigo", how="left").set_index("Codigo")
else:
    for col in ["Pico_mes","Pico_cant","Pico_clientes_unicos","Pico_concentrado",
                "Pico_cliente_nombre","Pico_cliente_recompro","Prom6m_sin_pico"]:
        met[col] = np.nan
    print("Análisis de concentración desactivado (sin columna cliente).")

# ── 9) STOCK ACTUAL ────────────────────────────────────────────────────
EXCLUIR_STOCK = {"ENVIOS-PACK","151","152","75","78","2GB-SO-DDR3-UNK","850065783635","ART-VAR"}

_stk_base = (df_g[df_g["Stock_Actual"].notna()].sort_values("Fecha_de_Carga")
              .drop_duplicates("Codigo", keep="last"))
_stk_base["Stock_Actual"]        = pd.to_numeric(_stk_base["Stock_Actual"], errors="coerce").fillna(0).clip(lower=0)
_stk_base["costo_neto_unitario"] = pd.to_numeric(
    _stk_base["costo_neto_unitario"] if "costo_neto_unitario" in _stk_base.columns
    else pd.Series(dtype=float), errors="coerce")
_stk_base = _stk_base[~_stk_base["Codigo"].isin(EXCLUIR_STOCK)].set_index("Codigo")

met["Stock_Total"]    = _stk_base["Stock_Actual"]       .reindex(met.index).fillna(0)
met["Costo_unit_USD"] = _stk_base["costo_neto_unitario"].reindex(met.index)
met["Valor_stock_USD"]= (met["Stock_Total"] * met["Costo_unit_USD"]).fillna(0).round(2)

# ── 10) MERGE ÚLTIMA COMPRA ────────────────────────────────────────────
if not duc.empty:
    mc = [c for c in ["Codigo","Ultima_compra_fecha","Ultima_compra_cant","Ultima_compra_comp",
                       "Lote_duro_meses","Lote_termino_en_QS","Consumo_implicito_lote"] if c in duc.columns]
    met = met.reset_index().merge(duc[mc], on="Codigo", how="left").set_index("Codigo")
else:
    for col in ["Ultima_compra_fecha","Ultima_compra_cant","Ultima_compra_comp",
                "Lote_duro_meses","Lote_termino_en_QS","Consumo_implicito_lote"]:
        met[col] = np.nan

# ── 11) CONSUMO PROYECTADO ─────────────────────────────────────────────
def consumo_proyectado(row):
    mu_act  = max(float(row.get("mu_activo")  or 0), 0)
    señal   = row["Señal"]
    qs      = bool(row["QS_ultimos_6m"])
    avg_s   = max(float(row.get("Avg_signal_2m") or 0), 0)
    c_imp   = row.get("Consumo_implicito_lote")
    qs_lote = bool(row.get("Lote_termino_en_QS", False))
    dur     = float(row.get("Lote_duro_meses") or 0)
    p_conc  = bool(row.get("Pico_concentrado", False))
    p_reco  = bool(row.get("Pico_cliente_recompro", False))
    p_sinp  = row.get("Prom6m_sin_pico")
    mu_sinp = max(float(p_sinp or 0), 0) if pd.notna(p_sinp) else mu_act

    # Nivel 1: lote confiable
    if pd.notna(c_imp) and float(c_imp) > 0 and dur >= LOTE_MIN_MESES:
        ci   = float(c_imp)
        base = ci * 1.15 if qs_lote else ci
        if   señal == "SUBE": base *= 1.20
        elif señal == "BAJA": base = max(base * 0.85, 0.5)
        return round(base, 1)

    # Nivel 2: sin lote — base = mu_activo
    # Pico espurio (1 cliente, no recompró): usar tasa sin ese mes
    base = mu_sinp if (p_conc and not p_reco) else mu_act

    if señal == "SUBE":
        base = max(base, avg_s) * 1.15
    elif señal == "BAJA":
        base = base * 0.85

    # QS: +10% por demanda suprimida. NO se usa max12 para no inflar con picos.
    if qs and not (p_conc and not p_reco):
        base *= 1.10

    return round(max(base, 0), 1)

met["Consumo_proy_mes"] = met.apply(consumo_proyectado, axis=1)

# ── 12) CANTIDAD A PEDIR — order-up-to ────────────────────────────────
# SS_cob = Z × σ_activo × √(LT + Cobertura)  → cubre variabilidad en todo el ciclo
# Target = μ × (LT + Cob) + SS_cob
# Q      = max(0, ceil(Target − Stock))
#
# Sin SS en cobertura: con CV=1.0 hay ~50% chance de quebrar en algún mes del ciclo.
# Con Z=1.28 (90% NS) el buffer es justo y evita sobrestock sistemático.

met["Cobertura_actual_meses"] = np.where(
    met["Consumo_proy_mes"] > 0,
    (met["Stock_Total"] / met["Consumo_proy_mes"]).round(1), np.inf)

met["Consumo_en_leadtime"] = (met["Consumo_proy_mes"] * LEAD_TIME).round(1)
met["Stock_al_llegar"]     = (met["Stock_Total"] - met["Consumo_en_leadtime"]).round(1)
met["Ruptura_en_transit"]  = met["Stock_al_llegar"] < 0

met["SS_cobertura"] = np.where(
    met["sigma_activo"] > 0,
    (Z_COB * met["sigma_activo"] * np.sqrt(MESES_TOTAL)).round(1),
    0.0
)

met["Target_inv"] = (met["Consumo_proy_mes"] * MESES_TOTAL + met["SS_cobertura"]).round(1)

met["Q_pedir_sugerido"] = np.clip(
    np.ceil(met["Target_inv"] - met["Stock_Total"]), 0, None).astype(int)

met["Q_pedir_conservador"] = np.clip(
    np.ceil(met["Consumo_proy_mes"] * MESES_TOTAL) - met["Stock_Total"], 0, None).astype(int)

met["Valor_Q_USD"] = (met["Q_pedir_sugerido"] * met["Costo_unit_USD"]).fillna(0).round(2)

# ── 13) ALERTA OVERSIZING ─────────────────────────────────────────────
def _alerta_os(row):
    q    = int(row["Q_pedir_sugerido"])
    cc   = row.get("Ultima_compra_cant")
    qs_l = bool(row.get("Lote_termino_en_QS", False))
    if pd.isna(cc) or float(cc) <= 0: return False
    if qs_l: return False
    return q > (FACTOR_OVERSIZING * float(cc))

met["Alerta_oversizing"] = met.apply(_alerta_os, axis=1)

# ── 14) PRIORIDAD ─────────────────────────────────────────────────────
def prioridad(row):
    cob   = float(row["Cobertura_actual_meses"]) if np.isfinite(row["Cobertura_actual_meses"]) else 999
    pedir = int(row["Q_pedir_sugerido"])
    rupt  = bool(row["Ruptura_en_transit"])
    qs    = bool(row["QS_ultimos_6m"])
    if pedir <= 0 and not qs: return "OK"
    if cob <= LEAD_TIME or rupt: return "URGENTE"
    if cob <= 3.0:              return "PRONTO"
    if cob <= COBERTURA_OBJ:   return "INCLUIR"
    if qs and row["Señal"] == "SUBE": return "PRONTO"
    return "OK"

met["Prioridad"] = met.apply(prioridad, axis=1)

# ── 15) TABLA FINAL ───────────────────────────────────────────────────
met = met.reset_index()
met = met[(met["Total12m"] > 0) | (met["Stock_Total"] > 0)].copy()

prio_ord = {"URGENTE":0,"PRONTO":1,"INCLUIR":2,"OK":3}
met["_o"] = met["Prioridad"].map(prio_ord).fillna(4)
met = met.sort_values(["_o","Q_pedir_sugerido"], ascending=[True,False]).drop(columns=["_o"])

cols_base = [
    "Codigo","Articulo","Marca","Prioridad",
    "Stock_Total","Cobertura_actual_meses","Consumo_proy_mes",
    "mu_activo","sigma_activo","SS_cobertura","Target_inv",
    "Prom6m","Max6m","Max12m","Avg_signal_2m","Avg_base_4m","Señal","QS_ultimos_6m",
    "Consumo_en_leadtime","Stock_al_llegar","Ruptura_en_transit",
    "Q_pedir_sugerido","Q_pedir_conservador",
    "Costo_unit_USD","Valor_Q_USD","Valor_stock_USD",
    "Ultima_compra_comp","Ultima_compra_fecha","Ultima_compra_cant",
    "Lote_duro_meses","Lote_termino_en_QS","Consumo_implicito_lote",
    "Pico_mes","Pico_cant","Pico_clientes_unicos","Pico_concentrado",
    "Pico_cliente_nombre","Pico_cliente_recompro","Prom6m_sin_pico",
    "Alerta_oversizing","Meses_venta","Total12m",
]
cols_out   = [c for c in cols_base if c in met.columns]
plan_out   = met[cols_out].copy()
a_pedir    = plan_out[plan_out["Q_pedir_sugerido"] > 0].copy()
urgentes   = plan_out[plan_out["Prioridad"]=="URGENTE"].copy()
a_revisar  = plan_out[plan_out["Alerta_oversizing"]==True].copy()
picos_conc = plan_out[
    (plan_out.get("Pico_concentrado", pd.Series(False))==True) &
    (plan_out["Q_pedir_sugerido"] > 0)
].copy()

# ── 16) RESUMEN EJECUTIVO ─────────────────────────────────────────────
resumen = (plan_out.groupby("Prioridad", as_index=False)
                   .agg(Productos        =("Codigo","count"),
                        Q_total_sugerido =("Q_pedir_sugerido","sum"),
                        Q_total_conserv  =("Q_pedir_conservador","sum"),
                        Valor_Q_USD      =("Valor_Q_USD","sum"),
                        Stock_total      =("Stock_Total","sum"),
                        Valor_stock_USD  =("Valor_stock_USD","sum"),
                        Alertas_OS       =("Alerta_oversizing","sum")))
resumen["_o"] = resumen["Prioridad"].map(prio_ord).fillna(4)
resumen = resumen.sort_values("_o").drop(columns=["_o"])

# ── 17) PARÁMETROS ────────────────────────────────────────────────────
params = pd.DataFrame({
    "Parametro": [
        "Fecha referencia Stock","Lead time","Cobertura obj","Total meses",
        "Z cobertura (NS)","Meses promedio","Meses pico largo","Meses señal","Meses base señal",
        "Lote min meses","Factor oversizing",
        "Ajuste QS_lote","Ajuste SUBE (lote)","Ajuste BAJA (lote)",
        "QS sin lote","Pico 1 cliente sin recompra","Pico 1 cliente con recompra","Formula Q",
    ],
    "Valor": [
        str(FECHA_STOCK.date()), LEAD_TIME, COBERTURA_OBJ, MESES_TOTAL,
        f"{Z_COB} (90% NS)", N_PROM, N_PICO_LARGO, K_SIGNAL, N_BASE_SIGNAL,
        LOTE_MIN_MESES, f">{FACTOR_OVERSIZING}x ultima_compra",
        "c_impl × 1.15","base × 1.20","base × 0.85",
        "mu_activo × 1.10 (NO max12)",
        "usar mu_sinpico (ignorar mes espurio)","usar lógica normal",
        "Target = μ×8 + Z×σ×√8 | Q = ceil(Target − Stock)",
    ],
})

# ── 18) EXPORT ────────────────────────────────────────────────────────
with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
    plan_out.to_excel(writer,   sheet_name="Plan_10GTeK_Completo", index=False)
    a_pedir.to_excel(writer,    sheet_name="A_Pedir",              index=False)
    urgentes.to_excel(writer,   sheet_name="URGENTES",             index=False)
    if not a_revisar.empty:
        a_revisar.to_excel(writer,  sheet_name="Revisar_Oversizing", index=False)
    if not picos_conc.empty:
        picos_conc.to_excel(writer, sheet_name="Picos_1_Cliente",    index=False)
    resumen.to_excel(writer,    sheet_name="Resumen_Ejecutivo",    index=False)
    pivot.to_excel(writer,      sheet_name="Ventas_Mensuales",     index=False)
    params.to_excel(writer,     sheet_name="Parametros",           index=False)

    try:
        from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
        from openpyxl.utils import get_column_letter

        CP  = {"URGENTE":"FFB3B3","PRONTO":"FFD9A0","INCLUIR":"FFF4A0","OK":"D4F5D4"}
        NF  = PatternFill("solid", fgColor="121C3A")
        NX  = Font(bold=True, color="FFFFFF", size=10)
        OF  = PatternFill("solid", fgColor="FF6B35")
        PC  = PatternFill("solid", fgColor="E8D5FF")
        ts  = Side(border_style="thin", color="CCCCCC")
        tb  = Border(left=ts, right=ts, top=ts, bottom=ts)

        def _fmt(ws, pc=None, ac=None, pico_col=None):
            hdr = [c.value for c in ws[1]]
            for cc in ws.columns:
                ml = max((len(str(c.value)) if c.value else 0) for c in cc)
                ws.column_dimensions[get_column_letter(cc[0].column)].width = min(ml+3, 52)
            for cell in ws[1]:
                cell.fill = NF; cell.font = NX
                cell.alignment = Alignment(horizontal="center", wrap_text=True)
            ws.row_dimensions[1].height = 30
            ws.freeze_panes = "D2"
            pi   = hdr.index(pc)       if pc       and pc       in hdr else None
            ai   = hdr.index(ac)       if ac       and ac       in hdr else None
            pici = hdr.index(pico_col) if pico_col and pico_col in hdr else None
            for row in ws.iter_rows(min_row=2):
                is_alert = ai   is not None and row[ai].value is True
                is_pico  = pici is not None and row[pici].value is True
                if is_alert:
                    for cell in row: cell.fill = OF; cell.border = tb
                elif is_pico:
                    for cell in row: cell.fill = PC; cell.border = tb
                elif pi is not None:
                    pv  = row[pi].value
                    fgc = CP.get(str(pv) if pv else "", None)
                    if fgc:
                        for cell in row: cell.fill = PatternFill("solid",fgColor=fgc); cell.border = tb

        for sh in ["Plan_10GTeK_Completo","A_Pedir","URGENTES","Revisar_Oversizing","Picos_1_Cliente"]:
            if sh in writer.sheets:
                _fmt(writer.sheets[sh], "Prioridad", "Alerta_oversizing", "Pico_concentrado")

        ws_r = writer.sheets["Resumen_Ejecutivo"]
        for cell in ws_r[1]: cell.fill = NF; cell.font = NX
        for cc in ws_r.columns:
            ml = max((len(str(c.value)) if c.value else 0) for c in cc)
            ws_r.column_dimensions[get_column_letter(cc[0].column)].width = min(ml+3, 35)

        ws_p = writer.sheets["Ventas_Mensuales"]
        SF = PatternFill("solid", fgColor="E0E0E0")
        QF = PatternFill("solid", fgColor="FFF0C0")
        for cell in ws_p[1]: cell.fill = NF; cell.font = NX
        for row in ws_p.iter_rows(min_row=2):
            for cell in row:
                if cell.value == "SRA": cell.fill = SF
                elif cell.value == "QS": cell.fill = QF

        print("Formato Excel OK.")
    except Exception as ef:
        print(f"Formato parcial: {ef}")

# ── 19) CONSOLA ───────────────────────────────────────────────────────
SEP = "=" * 72
print(f"\n{SEP}")
print(f"  PLAN REPOSICIÓN 10GTeK v2  |  Stock: {FECHA_STOCK.date()}")
print(f"  Lead time {LEAD_TIME}m  +  Cobertura {COBERTURA_OBJ}m  =  {MESES_TOTAL}m totales")
print(f"  Z={Z_COB} (90% NS)  |  Último mes datos: {ultimo_mes}")
print(SEP)

print(f"\n{'PRIORIDAD':<10} {'PRODS':>6} {'Q SUGER':>10} {'Q CONSERV':>10} "
      f"{'VALOR US$':>12} {'STOCK':>10} {'OS':>5}")
print("-"*65)
for _, r in resumen.iterrows():
    print(f"{r['Prioridad']:<10} {int(r['Productos']):>6} "
          f"{int(r['Q_total_sugerido']):>10,} {int(r['Q_total_conserv']):>10,} "
          f"US$ {float(r['Valor_Q_USD'] or 0):>9,.0f} "
          f"{int(r['Stock_total']):>10,} {int(r.get('Alertas_OS',0)):>5}")
print("-"*65)
tots = int(resumen["Q_total_sugerido"].sum())
totc = int(resumen["Q_total_conserv"].sum())
totv = float(resumen["Valor_Q_USD"].sum())
tost = int(resumen["Stock_total"].sum())
tota = int(plan_out["Alerta_oversizing"].sum())
print(f"{'TOTAL':<10} {int(resumen['Productos'].sum()):>6} {tots:>10,} "
      f"{totc:>10,} US$ {totv:>9,.0f} {tost:>10,} {tota:>5}")

print(f"\nCapital en stock 10GTeK: US$ {float(plan_out['Valor_stock_USD'].sum()):,.0f}")

print(f"\n── TOP URGENTES ─────────────────────────────────────────────────")
for _, r in urgentes.head(15).iterrows():
    print(f"  {str(r['Articulo'])[:50]:<50} | "
          f"Stock={int(r['Stock_Total']):>4} | "
          f"μact={float(r.get('mu_activo') or 0):.1f}/m | "
          f"σ={float(r.get('sigma_activo') or 0):.1f} | "
          f"SS={float(r.get('SS_cobertura') or 0):.0f} | "
          f"Q={int(r['Q_pedir_sugerido']):>5} | "
          f"US$ {float(r['Valor_Q_USD'] or 0):>7,.0f}")

n_pc = int((plan_out.get("Pico_concentrado", pd.Series(False))==True).sum())
if n_pc > 0:
    print(f"\n── PICOS 1 CLIENTE ({n_pc}) — lila en Excel ────────────────────")
    pc_rows = plan_out[plan_out["Pico_concentrado"]==True][
        ["Articulo","Pico_mes","Pico_cant","Pico_cliente_nombre",
         "Pico_cliente_recompro","mu_activo","Prom6m_sin_pico","Q_pedir_sugerido"]
    ].head(10)
    for _, r in pc_rows.iterrows():
        rs = "recompró" if r["Pico_cliente_recompro"] else "NO recompró"
        print(f"  {str(r['Articulo'])[:45]:<45} | "
              f"pico {r['Pico_mes']} ({int(r['Pico_cant'] or 0)} uds) | "
              f"{rs} | μact: {float(r.get('mu_activo') or 0):.1f} "
              f"→ sin pico: {float(r['Prom6m_sin_pico'] or 0):.1f} | "
              f"Q: {int(r['Q_pedir_sugerido'])}")

n_os = int(plan_out["Alerta_oversizing"].sum())
if n_os > 0:
    print(f"\n── OVERSIZING ({n_os}) — naranja en Excel ─────────────────────")
    for _, r in plan_out[plan_out["Alerta_oversizing"]==True].head(8).iterrows():
        print(f"  {str(r['Articulo'])[:45]:<45} | "
              f"ult.compra: {int(r['Ultima_compra_cant'] or 0):>4} | "
              f"Q_suger: {int(r['Q_pedir_sugerido']):>5} | "
              f"μact: {float(r.get('mu_activo') or 0):.1f}/m | "
              f"c.impl: {float(r['Consumo_implicito_lote'] or 0):.1f}/m")

print(f"""
COLORES EN EXCEL:
  Rojo      URGENTE  — sin stock o no cubre el lead time
  Naranja   PRONTO   — menos de 3 meses de cobertura
  Amarillo  INCLUIR  — entre 3 y 6 meses
  Verde     OK       — stock suficiente
  Naranja oscuro — Oversizing (Q > {FACTOR_OVERSIZING}x última compra)
  Lila      — Pico concentrado en 1 cliente

COLUMNAS NUEVAS EN EXCEL (v2):
  mu_activo      tasa real = Total12m / Meses_venta
  sigma_activo   desv.std solo en meses activos
  SS_cobertura   Z × σ × √8 — buffer para todo el ciclo
  Target_inv     μ×8 + SS — inventario objetivo

HOJAS:
  A_Pedir / URGENTES / Picos_1_Cliente / Revisar_Oversizing
  Plan_10GTeK_Completo / Ventas_Mensuales / Parametros

Archivo: {out_file}
""")

Filas 10GTeK: 6,702  |  2023-08-01 → 2026-03-31


/tmp/ipykernel_36231/2859300542.py:171: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dfm["tuvo_QS"]  = dfm["tuvo_QS"].fillna(False).infer_objects(copy=False)


Formato Excel OK.

  PLAN REPOSICIÓN 10GTeK v2  |  Stock: 2026-04-06
  Lead time 2.0m  +  Cobertura 6.0m  =  8.0m totales
  Z=1.28 (90% NS)  |  Último mes datos: 2026-03

PRIORIDAD   PRODS    Q SUGER  Q CONSERV    VALOR US$      STOCK    OS
-----------------------------------------------------------------
URGENTE        34      4,056      2,778 US$    60,441        162     6
PRONTO          8      1,004        579 US$     9,323        358     4
INCLUIR        21      1,709      1,161 US$    32,987      1,333     0
OK             38        581        200 US$     5,545      4,595     0
-----------------------------------------------------------------
TOTAL         101      7,350      4,718 US$   108,296      6,448    10

Capital en stock 10GTeK: US$ 58,680

── TOP URGENTES ─────────────────────────────────────────────────
  Modulo SFP+ 10G 20KM 10GBase-LR SMF Dual-LC OEM CI | Stock=  66 | μact=42.6/m | σ=19.6 | SS=71 | Q=  506 | US$   8,374
  Cable 1 Metro Fibra OM3 LC-LC MMF 10Gb Duplex

#ORDEN QNAP

In [51]:
"""
# ============================================================
# PLAN DE REPOSICION QNAP v4 — con analisis de concentracion de pico
# Objetivo: 6 meses + 1.5 lead time = 7.5 meses
#
# Jerarquia consumo proyectado:
#   1. Lote confiable (>= 3m duracion) -> Consumo_implicito_lote como base
#      QS_lote: +15% | SUBE: +20% | BAJA: -15%
#   2. Sin lote confiable:
#      Pico concentrado (1 cliente) -> ignorar pico, usar Prom6m_sin_pico
#      Pico distribuido (2+ clientes) -> usar logica señal normal
#      QS reciente -> max(Max12m, Prom6m*1.2)
#      SUBE -> max(Max6m, Avg_signal*1.1)
#      BAJA -> max(Avg_signal, 0.5)
#      ESTABLE -> Prom6m
# ============================================================

import os, itertools
import numpy as np
import pandas as pd

FECHA_STOCK       = pd.Timestamp("2026-03-02")
LEAD_TIME         = 1.5
COBERTURA_OBJ     = 6.0
MESES_TOTAL       = LEAD_TIME + COBERTURA_OBJ  # 7.5
N_PROM            = 6
N_PICO_LARGO      = 12
K_SIGNAL          = 2
N_BASE_SIGNAL     = 4
THR_HIGH_BASE     = 20
THR_MID_BASE      = 5
THR_HIGH_UP       = 0.20
THR_MID_UP        = 0.35
THR_LOW_ABS       = 2
FACTOR_OVERSIZING = 2.0
LOTE_MIN_MESES    = 3.0

out_dir  = output_path_base if "output_path_base" in globals() else "."
os.makedirs(out_dir, exist_ok=True)
out_file = os.path.join(out_dir, "plan_reposicion_qnap.xlsx")

# ── 1) NORMALIZAR ──────────────────────────────────────────────────────
if "df_merged_usd" not in globals():
    raise NameError("df_merged_usd no existe.")

req = ["Fecha_de_Carga","Marca","Comprobante","Codigo","Articulo",
       "Cantidad","Stock_Actual","Deposito","Clase"]
miss = [c for c in req if c not in df_merged_usd.columns]
if miss:
    raise KeyError(f"Faltan columnas: {miss}")

# Detectar columna cliente
cliente_col = None
for cand in ["Cliente_Proveedor","cliente_proveedor","Cliente","cliente"]:
    if cand in df_merged_usd.columns:
        cliente_col = cand
        break
if cliente_col is None:
    print("AVISO: no se encontro columna de cliente. Analisis de concentracion desactivado.")

df0 = df_merged_usd.copy()
df0["Fecha_de_Carga"] = pd.to_datetime(df0["Fecha_de_Carga"], errors="coerce", dayfirst=True)
for col in ["Comprobante","Codigo","Articulo","Marca","Deposito","Clase"]:
    df0[col] = df0[col].astype(str).str.strip()
df0["Codigo"] = (df0["Codigo"]
                 .str.replace(r"\s+","-",regex=True)
                 .str.replace(r"-{2,}","-",regex=True))
df0["Comp_norm"]  = df0["Comprobante"].str.lower().str.replace(r"\s+"," ",regex=True).str.strip()
df0["Clase_norm"] = df0["Clase"].str.lower().str.strip()
df0["Marca_norm"] = df0["Marca"].str.lower().str.strip()
df0["Cantidad"]     = pd.to_numeric(df0["Cantidad"],     errors="coerce").fillna(0)
df0["Stock_Actual"] = pd.to_numeric(df0["Stock_Actual"], errors="coerce")
if "Stock_Anterior" in df0.columns:
    df0["Stock_Anterior"] = pd.to_numeric(df0["Stock_Anterior"], errors="coerce")
if cliente_col:
    df0[cliente_col] = df0[cliente_col].astype(str).str.strip().str.lower()

# ── 2) FILTRAR QNAP ────────────────────────────────────────────────────
df_q = df0[df0["Marca_norm"].str.contains("qnap", na=False) &
           df0["Fecha_de_Carga"].notna()].copy()
if df_q.empty:
    raise ValueError("No hay filas QNAP en df_merged_usd.")
ultima_fecha = df_q["Fecha_de_Carga"].max()
print(f"Filas QNAP: {len(df_q):,}  |  {df_q['Fecha_de_Carga'].min().date()} -> {ultima_fecha.date()}")

# ── 3) CLASIFICAR ──────────────────────────────────────────────────────
is_compra = df_q["Clase_norm"].eq("01.factura")
pat_pres  = r"\b(?:pre|pres|presup|cot|cotiz)\b"
is_pres   = (df_q["Comp_norm"].str.contains(pat_pres, na=False, regex=True) |
             df_q["Clase_norm"].str.contains("presup", na=False))
is_venta  = (df_q["Comp_norm"].str.startswith(("fc a","fc b","fc c"), na=False)
             & ~is_compra & ~is_pres)
is_nc     = (df_q["Comp_norm"].str.startswith(("nc a","nc b","nc c","nc d"), na=False)
             & ~is_compra)
df_q["Tipo"] = np.select([is_venta, is_nc, is_compra],
                         ["venta","devolucion","compra"], default="otro")
df_q["Cant_neta"] = np.where(df_q["Tipo"]=="venta", df_q["Cantidad"].abs(),
                    np.where(df_q["Tipo"]=="devolucion", -df_q["Cantidad"].abs(), 0))
df_q["AnoMes"] = df_q["Fecha_de_Carga"].dt.to_period("M")

# ── 4) QS POR MES ──────────────────────────────────────────────────────
if "Stock_Anterior" in df_q.columns:
    sa = df_q["Stock_Anterior"].fillna(-999)
    cq = df_q["Cantidad"].abs().fillna(-998)
    df_q["QS_flag"] = (df_q["Tipo"]=="venta") & (sa == cq)
else:
    df_q["QS_flag"] = False

qs_por_mes = (df_q[df_q["QS_flag"]][["Codigo","AnoMes","Fecha_de_Carga"]]
              .drop_duplicates(subset=["Codigo","AnoMes"])
              .rename(columns={"Fecha_de_Carga":"Fecha_QS"})
              .assign(tuvo_QS=True))

# ── 5) ANALISIS ULTIMA COMPRA ──────────────────────────────────────────
df_compras = df_q[df_q["Tipo"]=="compra"].copy()
df_compras["Cant_compra"] = df_compras["Cantidad"].abs()

if df_compras.empty:
    print("AVISO: no se encontraron compras (01.factura) para QNAP.")
    duc = pd.DataFrame(columns=["Codigo","Ultima_compra_fecha","Ultima_compra_cant",
                                 "Ultima_compra_comp","Lote_duro_meses",
                                 "Lote_termino_en_QS","Consumo_implicito_lote"])
else:
    cmp_agg = (df_compras.groupby(["Codigo","Comprobante","Fecha_de_Carga"], as_index=False)
                          .agg(Cant_compra=("Cant_compra","sum")))
    idx_ult = cmp_agg.groupby("Codigo")["Fecha_de_Carga"].idxmax()
    duc = (cmp_agg.loc[idx_ult, ["Codigo","Fecha_de_Carga","Cant_compra","Comprobante"]]
                  .copy()
                  .rename(columns={"Fecha_de_Carga":"Ultima_compra_fecha",
                                   "Cant_compra":   "Ultima_compra_cant",
                                   "Comprobante":   "Ultima_compra_comp"}))
    primer_qs = (df_q[df_q["QS_flag"]]
                 .groupby("Codigo")["Fecha_de_Carga"]
                 .min().rename("Primer_QS_fecha"))
    duc = duc.merge(primer_qs.reset_index(), on="Codigo", how="left")

    def _calc_lote(row):
        fc = row["Ultima_compra_fecha"]
        qt = float(row["Ultima_compra_cant"] or 0)
        pq = row.get("Primer_QS_fecha")
        if pd.isna(fc):
            return pd.Series({"Lote_duro_meses":np.nan,
                               "Lote_termino_en_QS":False,
                               "Consumo_implicito_lote":np.nan})
        if pd.notna(pq) and pq > fc:
            fecha_fin = pq; termino_qs = True
        else:
            fecha_fin = FECHA_STOCK; termino_qs = False
        dur    = max((fecha_fin - fc).days / 30.44, 0.5)
        c_impl = round(qt / dur, 2) if dur > 0 else np.nan
        return pd.Series({"Lote_duro_meses":round(dur,1),
                          "Lote_termino_en_QS":termino_qs,
                          "Consumo_implicito_lote":c_impl})

    extras = duc.apply(_calc_lote, axis=1)
    duc = pd.concat([duc, extras], axis=1)
    duc.drop(columns=["Primer_QS_fecha"], errors="ignore", inplace=True)

# ── 6) PIVOT VENTAS MENSUALES ──────────────────────────────────────────
df_ventas   = df_q[df_q["Cant_neta"] != 0].copy()
mensual     = (df_ventas.groupby(["Codigo","AnoMes"], as_index=False)
                        .agg(Cant_mes=("Cant_neta","sum")))
todos_meses = pd.period_range(df_q["AnoMes"].min(), df_q["AnoMes"].max(), freq="M")
productos   = df_q[["Codigo","Articulo","Marca"]].drop_duplicates("Codigo", keep="last")

malla = (pd.DataFrame(itertools.product(productos["Codigo"], todos_meses),
                      columns=["Codigo","AnoMes"])
           .merge(productos, on="Codigo", how="left"))
dfm = (malla.merge(mensual, on=["Codigo","AnoMes"], how="left")
            .merge(qs_por_mes[["Codigo","AnoMes","tuvo_QS"]], on=["Codigo","AnoMes"], how="left"))
dfm["Cant_mes"] = dfm["Cant_mes"].fillna(0)
dfm["tuvo_QS"]  = dfm["tuvo_QS"].fillna(False)

fp_dict = (df_q[df_q["Stock_Actual"].fillna(0) > 0]
           .groupby("Codigo")["Fecha_de_Carga"]
           .min().dt.to_period("M").to_dict())

def celda_mensual(row):
    cod, mes, cant, qs = row["Codigo"], row["AnoMes"], row["Cant_mes"], row["tuvo_QS"]
    if cant != 0: return cant
    fp = fp_dict.get(cod)
    if fp is not None and mes < fp: return "SRA"
    return "QS" if qs else 0

dfm["Valor_Mes"] = dfm.apply(celda_mensual, axis=1)

pivot = (dfm.pivot_table(index=["Codigo","Articulo","Marca"],
                         columns="AnoMes", values="Valor_Mes", aggfunc="first")
            .reset_index())
mes_cols_p = sorted([c for c in pivot.columns if isinstance(c, pd.Period)],
                    key=lambda x: x.to_timestamp())
pivot = pivot[["Codigo","Articulo","Marca"] + mes_cols_p]
pivot.columns = ["Codigo","Articulo","Marca"] + [str(m) for m in mes_cols_p]

# ── 7) METRICAS DEMANDA ────────────────────────────────────────────────
ultimo_mes = df_q["AnoMes"].max()
m6m   = [m for m in sorted(todos_meses,reverse=True) if m<=ultimo_mes][:N_PROM]
m12m  = [m for m in sorted(todos_meses,reverse=True) if m<=ultimo_mes][:N_PICO_LARGO]
msen  = [m for m in sorted(todos_meses,reverse=True) if m<=ultimo_mes][:K_SIGNAL]
mbas  = [m for m in sorted(todos_meses,reverse=True) if m<=ultimo_mes][K_SIGNAL:K_SIGNAL+N_BASE_SIGNAL]

def cs(meses):
    return [str(m) for m in meses if str(m) in pivot.columns]

c6 = cs(m6m); c12 = cs(m12m); csen = cs(msen); cbas = cs(mbas)

pn = (pivot.set_index("Codigo")
      [[c for c in pivot.columns if c not in ["Codigo","Articulo","Marca"]]]
      .apply(pd.to_numeric, errors="coerce").fillna(0))

met = productos.set_index("Codigo").copy()
met["Prom6m"]      = pn[c6].mean(axis=1).round(2)  if c6  else 0.0
met["Max6m"]       = pn[c6].max(axis=1).round(1)   if c6  else 0.0
met["Max12m"]      = pn[c12].max(axis=1).round(1)  if c12 else 0.0
met["Total12m"]    = pn[c12].sum(axis=1).round(0)  if c12 else 0.0
met["Meses_venta"] = (pn[c12] > 0).sum(axis=1)     if c12 else 0

if csen and cbas:
    avg_s = pn[csen].mean(axis=1)
    avg_b = pn[cbas].mean(axis=1)
    met["Avg_signal_2m"] = avg_s.round(2)
    met["Avg_base_4m"]   = avg_b.round(2)
    dabs = avg_s - avg_b
    dpct = np.where(avg_b > 0, dabs / avg_b, np.nan)

    def _señal(bavg, da, dp):
        if np.isnan(dp):
            return "NUEVO" if bavg==0 and da>0 else "SIN DATOS"
        if bavg < 1:
            return "SUBE" if da>=THR_LOW_ABS else ("BAJA" if da<=-THR_LOW_ABS else "ESTABLE")
        if bavg >= THR_HIGH_BASE:
            return "SUBE" if dp>=THR_HIGH_UP else ("BAJA" if dp<=-THR_HIGH_UP else "ESTABLE")
        if bavg >= THR_MID_BASE:
            return "SUBE" if dp>=THR_MID_UP else ("BAJA" if dp<=-THR_MID_UP else "ESTABLE")
        return "SUBE" if da>=THR_LOW_ABS else ("BAJA" if da<=-THR_LOW_ABS else "ESTABLE")

    met["Señal"] = [_señal(b,d,p) for b,d,p in zip(avg_b.values,dabs.values,dpct)]
else:
    met["Avg_signal_2m"] = np.nan
    met["Avg_base_4m"]   = np.nan
    met["Señal"]         = "SIN DATOS"

qs_rec = set(qs_por_mes[qs_por_mes["AnoMes"].isin(m6m)]["Codigo"].unique())
met["QS_ultimos_6m"] = met.index.isin(qs_rec)

# ── 8) ANALISIS CONCENTRACION DE PICO ─────────────────────────────────
# Para cada producto, miramos el mes con maxima venta en los ultimos 6 meses.
# Si ese pico vino de 1 solo cliente → pico concentrado, no es tendencia real.
# Ademas chequeamos si ese cliente siguio comprando en meses posteriores.
#
# Columnas que agrega:
#   Pico_mes                mes donde ocurrio el maximo
#   Pico_cant               unidades vendidas ese mes (el Max6m)
#   Pico_clientes_unicos    cuantos clientes distintos compraron ese mes
#   Pico_concentrado        TRUE si fue 1 solo cliente
#   Pico_cliente_nombre     nombre del cliente del pico (si fue 1)
#   Pico_cliente_recompro   TRUE si ese cliente volvio a comprar despues del pico
#   Prom6m_sin_pico         promedio de los otros 5 meses excluyendo el mes pico

if cliente_col and c6:
    # Solo ventas dentro de los ultimos 6 meses
    df_ventas_6m = df_q[
        (df_q["Tipo"] == "venta") &
        (df_q["AnoMes"].isin(m6m))
    ].copy()

    # Ventas mensuales por producto y cliente
    vc = (df_ventas_6m.groupby(["Codigo","AnoMes",cliente_col], as_index=False)
                       .agg(Cant_cliente=("Cant_neta","sum")))

    # Mes pico por producto (mes con mayor venta total)
    cant_mes_total = (df_ventas_6m.groupby(["Codigo","AnoMes"], as_index=False)
                                   .agg(Cant_total=("Cant_neta","sum")))
    idx_pico = cant_mes_total.groupby("Codigo")["Cant_total"].idxmax()
    pico_mes_df = cant_mes_total.loc[idx_pico, ["Codigo","AnoMes","Cant_total"]].copy()
    pico_mes_df = pico_mes_df.rename(columns={"AnoMes":"Pico_mes","Cant_total":"Pico_cant"})

    # Para cada producto en su mes pico: contar clientes unicos y guardar el nombre si es 1
    def _analizar_pico(row):
        cod      = row["Codigo"]
        pico_m   = row["Pico_mes"]
        sub = vc[(vc["Codigo"]==cod) & (vc["AnoMes"]==pico_m)]
        n_cli    = sub[cliente_col].nunique()
        cli_nom  = sub.loc[sub["Cant_cliente"].idxmax(), cliente_col] if n_cli == 1 else None

        # Verificar recompra: compro ese cliente en meses POSTERIORES al pico?
        recompro = False
        if cli_nom is not None:
            meses_post = [m for m in m6m if m > pico_m]
            if meses_post:
                recompra_mask = (
                    (df_ventas_6m["Codigo"] == cod) &
                    (df_ventas_6m[cliente_col] == cli_nom) &
                    (df_ventas_6m["AnoMes"].isin(meses_post))
                )
                recompro = df_ventas_6m[recompra_mask]["Cant_neta"].sum() > 0

            # Buscar tambien fuera de los 6m (en todo el historial QNAP)
            if not recompro:
                pico_fecha = df_q[df_q["AnoMes"]==pico_m]["Fecha_de_Carga"].max()
                if pd.notna(pico_fecha):
                    hist_mask = (
                        (df_q["Codigo"] == cod) &
                        (df_q[cliente_col] == cli_nom) &
                        (df_q["Tipo"] == "venta") &
                        (df_q["Fecha_de_Carga"] > pico_fecha)
                    )
                    recompro = df_q[hist_mask]["Cant_neta"].sum() > 0

        return pd.Series({
            "Pico_clientes_unicos": n_cli,
            "Pico_concentrado":     n_cli == 1,
            "Pico_cliente_nombre":  cli_nom,
            "Pico_cliente_recompro": recompro,
        })

    pico_extra = pico_mes_df.apply(_analizar_pico, axis=1)
    pico_df    = pd.concat([pico_mes_df, pico_extra], axis=1)

    # Prom6m_sin_pico: promedio de los meses que NO son el pico
    def _prom_sin_pico(row):
        cod    = row["Codigo"]
        pico_m = row["Pico_mes"]
        otros  = [str(m) for m in m6m if m != pico_m and str(m) in pn.columns]
        if not otros or cod not in pn.index:
            return np.nan
        vals = pn.loc[cod, otros]
        nums = pd.to_numeric(vals, errors="coerce").fillna(0)
        return round(nums.mean(), 2)

    pico_df["Prom6m_sin_pico"] = pico_df.apply(_prom_sin_pico, axis=1)

    # Convertir Pico_mes a string para merge
    pico_df["Pico_mes"] = pico_df["Pico_mes"].astype(str)

    # Merge a met
    met = met.reset_index().merge(pico_df, on="Codigo", how="left").set_index("Codigo")

else:
    # Sin columna cliente: agregar columnas vacias para no romper el resto
    for col in ["Pico_mes","Pico_cant","Pico_clientes_unicos","Pico_concentrado",
                "Pico_cliente_nombre","Pico_cliente_recompro","Prom6m_sin_pico"]:
        met[col] = np.nan
    print("Analisis de concentracion desactivado (sin columna cliente).")

# ── 9) STOCK POR DEPOSITO ──────────────────────────────────────────────
stk = df_q[df_q["Stock_Actual"].notna()].sort_values("Fecha_de_Carga")
if not stk.empty:
    idx_last = stk.groupby(["Codigo","Deposito"])["Fecha_de_Carga"].idxmax()
    ld = stk.loc[idx_last, ["Codigo","Deposito","Stock_Actual"]].copy()
    ld["Stock_Actual"] = pd.to_numeric(ld["Stock_Actual"],errors="coerce").fillna(0).clip(lower=0)
    stock_total = ld.groupby("Codigo")["Stock_Actual"].sum().rename("Stock_Total")
    sdep = ld.pivot_table(index="Codigo",columns="Deposito",values="Stock_Actual",aggfunc="sum").fillna(0)
    sdep.columns = [f"Stock_{d.replace(' ','_')}" for d in sdep.columns]
else:
    stock_total = pd.Series(dtype=float, name="Stock_Total")
    sdep = pd.DataFrame()

met["Stock_Total"] = stock_total.reindex(met.index).fillna(0)
for col in (sdep.columns if not sdep.empty else []):
    met[col] = sdep[col].reindex(met.index).fillna(0)

# ── 10) MERGE ULTIMA COMPRA ────────────────────────────────────────────
if not duc.empty:
    mc = ["Codigo","Ultima_compra_fecha","Ultima_compra_cant",
          "Ultima_compra_comp","Lote_duro_meses",
          "Lote_termino_en_QS","Consumo_implicito_lote"]
    mc = [c for c in mc if c in duc.columns]
    met = met.reset_index().merge(duc[mc], on="Codigo", how="left").set_index("Codigo")
else:
    for col in ["Ultima_compra_fecha","Ultima_compra_cant","Ultima_compra_comp",
                "Lote_duro_meses","Lote_termino_en_QS","Consumo_implicito_lote"]:
        met[col] = np.nan

# ── 11) CONSUMO PROYECTADO ─────────────────────────────────────────────
# Jerarquia completa:
#
# NIVEL 1 — Lote confiable (>= 3 meses):
#   base = Consumo_implicito_lote  <- lo que realmente se consumio
#   QS_lote:  base * 1.15  (demanda suprimida, algo mayor)
#   SUBE:     base * 1.20  (tendencia creciente encima del implicito)
#   BAJA:     base * 0.85  (tendencia decreciente)
#
# NIVEL 2 — Sin lote confiable, caemos a señal:
#   Pico concentrado Y no recompro -> usar Prom6m_sin_pico (ignorar pico espurio)
#   Pico concentrado Y SI recompro -> confiar en el pico (cliente fidelizado)
#   QS reciente    -> max(Max12m, Prom6m*1.2)
#   SUBE           -> max(Max6m, Avg_signal*1.1)
#   BAJA           -> max(Avg_signal, 0.5)
#   ESTABLE/NUEVO  -> Prom6m

def consumo_proyectado(row):
    prom6    = max(float(row["Prom6m"]  or 0), 0)
    max6     = max(float(row["Max6m"]   or 0), 0)
    max12    = max(float(row["Max12m"]  or 0), 0)
    señal    = row["Señal"]
    qs       = bool(row["QS_ultimos_6m"])
    avg_s    = max(float(row.get("Avg_signal_2m") or 0), 0)
    c_imp    = row.get("Consumo_implicito_lote")
    qs_lote  = bool(row.get("Lote_termino_en_QS", False))
    dur      = float(row.get("Lote_duro_meses") or 0)
    p_conc   = bool(row.get("Pico_concentrado", False))
    p_recomp = bool(row.get("Pico_cliente_recompro", False))
    p_sinp   = row.get("Prom6m_sin_pico")
    p_sinp   = max(float(p_sinp or 0), 0) if pd.notna(p_sinp) else prom6

    # ── NIVEL 1: lote confiable ──────────────────────────────────────
    if pd.notna(c_imp) and float(c_imp) > 0 and dur >= LOTE_MIN_MESES:
        ci   = float(c_imp)
        base = ci * 1.15 if qs_lote else ci
        if   señal == "SUBE": base = base * 1.20
        elif señal == "BAJA": base = max(base * 0.85, 0.5)
        return round(base, 1)

    # ── NIVEL 2: sin lote confiable ──────────────────────────────────

    # Pico concentrado en 1 cliente que NO volvio a comprar: ignorar ese pico
    if p_conc and not p_recomp:
        # Usar promedio sin el mes del pico como base mas realista
        base = p_sinp
        if qs:
            base = round(max(base * 1.2, p_sinp), 1)
        elif señal == "SUBE":
            # Sube pero el pico fue un cliente puntual: subir moderado
            base = round(base * 1.15, 1)
        elif señal == "BAJA":
            base = round(max(base * 0.85, 0.5 if prom6 > 0 else 0), 1)
        return round(max(base, 0), 1)

    # Pico concentrado pero el cliente SI recompro: confiar en el pico
    # (cae al flujo normal de abajo)

    # Flujo normal
    if qs:
        return round(max(max12, prom6 * 1.2), 1)
    if señal == "SUBE":
        return round(max(max6, avg_s * 1.1), 1)
    if señal == "BAJA":
        return round(max(avg_s, 0.5 if prom6 > 0 else 0), 1)
    if señal in ("NUEVO","SIN DATOS"):
        return round(max(prom6, max12), 1)
    return round(prom6, 1)

met["Consumo_proy_mes"] = met.apply(consumo_proyectado, axis=1)

# ── 12) CANTIDAD A PEDIR ───────────────────────────────────────────────
met["Cobertura_actual_meses"] = np.where(
    met["Consumo_proy_mes"] > 0,
    (met["Stock_Total"] / met["Consumo_proy_mes"]).round(1), np.inf)

met["Consumo_en_leadtime"] = (met["Consumo_proy_mes"] * LEAD_TIME).round(1)
met["Stock_al_llegar"]     = (met["Stock_Total"] - met["Consumo_en_leadtime"]).round(1)
met["Ruptura_en_transit"]  = met["Stock_al_llegar"] < 0

met["Unidades_necesarias"] = np.ceil(met["Consumo_proy_mes"] * MESES_TOTAL).clip(0).astype(int)
met["Q_pedir_sugerido"]    = np.clip(
    met["Unidades_necesarias"] - met["Stock_Total"], 0, None).astype(int)
met["Q_pedir_conservador"] = np.clip(
    np.ceil(met["Prom6m"] * MESES_TOTAL) - met["Stock_Total"], 0, None).astype(int)

# ── 13) ALERTA OVERSIZING ─────────────────────────────────────────────
def _alerta_os(row):
    q    = int(row["Q_pedir_sugerido"])
    cc   = row.get("Ultima_compra_cant")
    qs_l = bool(row.get("Lote_termino_en_QS", False))
    if pd.isna(cc) or float(cc) <= 0: return False
    if qs_l: return False
    return q > (FACTOR_OVERSIZING * float(cc))

met["Alerta_oversizing"] = met.apply(_alerta_os, axis=1)

# ── 14) PRIORIDAD ─────────────────────────────────────────────────────
def prioridad(row):
    cob   = float(row["Cobertura_actual_meses"]) if np.isfinite(row["Cobertura_actual_meses"]) else 999
    pedir = int(row["Q_pedir_sugerido"])
    rupt  = bool(row["Ruptura_en_transit"])
    qs    = bool(row["QS_ultimos_6m"])
    if pedir <= 0 and not qs: return "OK"
    if cob <= LEAD_TIME or rupt: return "URGENTE"
    if cob <= 3.0: return "PRONTO"
    if cob <= COBERTURA_OBJ: return "INCLUIR"
    if qs and row["Señal"] == "SUBE": return "PRONTO"
    return "OK"

met["Prioridad"] = met.apply(prioridad, axis=1)

# ── 15) TABLA FINAL ───────────────────────────────────────────────────
met = met.reset_index()
met = met[met["Total12m"] > 0].copy()

prio_ord = {"URGENTE":0,"PRONTO":1,"INCLUIR":2,"OK":3}
met["_o"] = met["Prioridad"].map(prio_ord).fillna(4)
met = met.sort_values(["_o","Q_pedir_sugerido"], ascending=[True,False]).drop(columns=["_o"])

cols_base = [
    "Codigo","Articulo","Marca","Prioridad",
    "Stock_Total","Cobertura_actual_meses","Consumo_proy_mes",
    "Prom6m","Max6m","Max12m","Avg_signal_2m","Avg_base_4m","Señal","QS_ultimos_6m",
    "Consumo_en_leadtime","Stock_al_llegar","Ruptura_en_transit",
    "Unidades_necesarias","Q_pedir_sugerido","Q_pedir_conservador",
    # ultima compra
    "Ultima_compra_comp","Ultima_compra_fecha","Ultima_compra_cant",
    "Lote_duro_meses","Lote_termino_en_QS","Consumo_implicito_lote",
    # concentracion pico
    "Pico_mes","Pico_cant","Pico_clientes_unicos","Pico_concentrado",
    "Pico_cliente_nombre","Pico_cliente_recompro","Prom6m_sin_pico",
    # alertas
    "Alerta_oversizing","Meses_venta","Total12m",
]
dep_cols = [c for c in met.columns if c.startswith("Stock_") and c != "Stock_Total"]
cols_out = [c for c in cols_base + dep_cols if c in met.columns]
plan_out  = met[cols_out].copy()
a_pedir   = plan_out[plan_out["Q_pedir_sugerido"] > 0].copy()
a_revisar = plan_out[plan_out["Alerta_oversizing"] == True].copy()
picos_conc = plan_out[
    (plan_out["Pico_concentrado"] == True) &
    (plan_out["Q_pedir_sugerido"] > 0)
].copy()

# ── 16) RESUMEN EJECUTIVO ─────────────────────────────────────────────
resumen = (plan_out.groupby("Prioridad", as_index=False)
                   .agg(Productos        =("Codigo","count"),
                        Q_total_sugerido =("Q_pedir_sugerido","sum"),
                        Q_total_conserv  =("Q_pedir_conservador","sum"),
                        Stock_total      =("Stock_Total","sum"),
                        Alertas_OS       =("Alerta_oversizing","sum"),
                        Picos_conc       =("Pico_concentrado","sum")))
resumen["_o"] = resumen["Prioridad"].map(prio_ord).fillna(4)
resumen = resumen.sort_values("_o").drop(columns=["_o"])

# ── 17) PARAMETROS ────────────────────────────────────────────────────
params = pd.DataFrame({
    "Parametro": [
        "Fecha referencia Stock","Lead time","Cobertura obj","Total meses",
        "Meses promedio","Meses pico","Meses señal","Meses base señal",
        "Lote min meses","Factor oversizing",
        "Ajuste QS_lote","Ajuste SUBE (lote)","Ajuste BAJA (lote)",
        "Ajuste QS sin lote","Pico 1 cliente sin recompra",
        "Pico 1 cliente con recompra",
    ],
    "Valor": [
        str(FECHA_STOCK.date()),LEAD_TIME,COBERTURA_OBJ,MESES_TOTAL,
        N_PROM,N_PICO_LARGO,K_SIGNAL,N_BASE_SIGNAL,
        LOTE_MIN_MESES,f">{FACTOR_OVERSIZING}x ultima_compra",
        "c_impl * 1.15","base * 1.20","base * 0.85",
        "max(Max12m, Prom6m*1.2)",
        "usar Prom6m_sin_pico (ignorar mes pico)",
        "usar logica normal (confiar en el pico)",
    ],
})

# ── 18) EXPORT EXCEL ──────────────────────────────────────────────────
with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
    plan_out.to_excel(writer,  sheet_name="Plan_QNAP_Completo",  index=False)
    a_pedir.to_excel(writer,   sheet_name="A_Pedir",             index=False)
    if not a_revisar.empty:
        a_revisar.to_excel(writer, sheet_name="Revisar_Oversizing", index=False)
    if not picos_conc.empty:
        picos_conc.to_excel(writer, sheet_name="Picos_1_Cliente",   index=False)
    resumen.to_excel(writer,   sheet_name="Resumen_Ejecutivo",   index=False)
    pivot.to_excel(writer,     sheet_name="Ventas_Mensuales",    index=False)
    params.to_excel(writer,    sheet_name="Parametros",          index=False)

    try:
        from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
        from openpyxl.utils import get_column_letter

        CP = {"URGENTE":"FFB3B3","PRONTO":"FFD9A0","INCLUIR":"FFF4A0","OK":"D4F5D4"}
        NF = PatternFill("solid", fgColor="121C3A")
        NX = Font(bold=True, color="FFFFFF", size=10)
        OF = PatternFill("solid", fgColor="FF6B35")  # naranja = oversizing
        PC = PatternFill("solid", fgColor="E8D5FF")  # lila = pico concentrado
        ts = Side(border_style="thin", color="CCCCCC")
        tb = Border(left=ts, right=ts, top=ts, bottom=ts)

        def _fmt(ws, pc=None, ac=None, pico_col=None):
            hdr = [c.value for c in ws[1]]
            for cc in ws.columns:
                ml = max((len(str(c.value)) if c.value else 0) for c in cc)
                ws.column_dimensions[get_column_letter(cc[0].column)].width = min(ml+3,52)
            for cell in ws[1]:
                cell.fill = NF; cell.font = NX
                cell.alignment = Alignment(horizontal="center", wrap_text=True)
            ws.row_dimensions[1].height = 30
            ws.freeze_panes = "D2"

            pi   = hdr.index(pc)       if pc       and pc       in hdr else None
            ai   = hdr.index(ac)       if ac       and ac       in hdr else None
            pici = hdr.index(pico_col) if pico_col and pico_col in hdr else None

            for row in ws.iter_rows(min_row=2):
                is_alert = ai   is not None and row[ai].value   is True
                is_pico  = pici is not None and row[pici].value is True
                if is_alert:
                    for cell in row: cell.fill = OF; cell.border = tb
                elif is_pico:
                    for cell in row: cell.fill = PC; cell.border = tb
                elif pi is not None:
                    pv  = row[pi].value
                    fgc = CP.get(str(pv) if pv else "", None)
                    if fgc:
                        fill = PatternFill("solid", fgColor=fgc)
                        for cell in row: cell.fill = fill; cell.border = tb

        for sh in ["Plan_QNAP_Completo","A_Pedir"]:
            _fmt(writer.sheets[sh], "Prioridad", "Alerta_oversizing", "Pico_concentrado")
        for sh in ["Revisar_Oversizing","Picos_1_Cliente"]:
            if sh in writer.sheets:
                _fmt(writer.sheets[sh], "Prioridad", "Alerta_oversizing", "Pico_concentrado")

        ws_r = writer.sheets["Resumen_Ejecutivo"]
        for cell in ws_r[1]: cell.fill = NF; cell.font = NX
        for cc in ws_r.columns:
            ml = max((len(str(c.value)) if c.value else 0) for c in cc)
            ws_r.column_dimensions[get_column_letter(cc[0].column)].width = min(ml+3,35)

        ws_p = writer.sheets["Ventas_Mensuales"]
        SF = PatternFill("solid", fgColor="E0E0E0")
        QF = PatternFill("solid", fgColor="FFF0C0")
        for cell in ws_p[1]: cell.fill = NF; cell.font = NX
        for row in ws_p.iter_rows(min_row=2):
            for cell in row:
                if cell.value == "SRA": cell.fill = SF
                elif cell.value == "QS": cell.fill = QF

        print("Formato Excel OK.")
    except Exception as ef:
        print(f"Formato parcial: {ef}")

# ── 19) CONSOLA ───────────────────────────────────────────────────────
SEP = "=" * 72
print(f"\n{SEP}")
print(f"  PLAN REPOSICION QNAP v4  |  Stock: {FECHA_STOCK.date()}")
print(f"  Lead time {LEAD_TIME}m  +  Cobertura {COBERTURA_OBJ}m  =  {MESES_TOTAL}m totales")
print(f"  Ultimo mes datos: {ultimo_mes}")
print(SEP)
print(f"\n{'PRIORIDAD':<10} {'PRODS':>6} {'Q SUGER':>10} {'Q CONSERV':>10} "
      f"{'STOCK':>10} {'OS':>5} {'PICO1':>6}")
print("-"*60)
for _, r in resumen.iterrows():
    print(f"{r['Prioridad']:<10} {int(r['Productos']):>6} "
          f"{int(r['Q_total_sugerido']):>10,} {int(r['Q_total_conserv']):>10,} "
          f"{int(r['Stock_total']):>10,} {int(r.get('Alertas_OS',0)):>5} "
          f"{int(r.get('Picos_conc',0)):>6}")
print("-"*60)
tots = int(resumen["Q_total_sugerido"].sum())
totc = int(resumen["Q_total_conserv"].sum())
tost = int(resumen["Stock_total"].sum())
tota = int(plan_out["Alerta_oversizing"].sum())
totp = int(plan_out.get("Pico_concentrado", pd.Series(False)).sum())
print(f"{'TOTAL':<10} {int(resumen['Productos'].sum()):>6} {tots:>10,} "
      f"{totc:>10,} {tost:>10,} {tota:>5} {totp:>6}")

# Detallar picos concentrados
n_pc = int(plan_out.get("Pico_concentrado", pd.Series(False)).sum()) if "Pico_concentrado" in plan_out else 0
if n_pc > 0:
    print(f"\nPICOS DE 1 CLIENTE ({n_pc} producto/s) — lila en Excel:")
    pc_rows = plan_out[plan_out["Pico_concentrado"]==True][
        ["Articulo","Pico_mes","Pico_cant","Pico_cliente_nombre",
         "Pico_cliente_recompro","Prom6m","Prom6m_sin_pico",
         "Q_pedir_sugerido","Q_pedir_conservador"]
    ].head(15)
    for _, r in pc_rows.iterrows():
        recomp_str = "recompro" if r["Pico_cliente_recompro"] else "NO recompro"
        print(f"  {str(r['Articulo'])[:45]:<45} | "
              f"pico {r['Pico_mes']} ({int(r['Pico_cant'] or 0)} uds) | "
              f"cliente: {str(r['Pico_cliente_nombre'] or '')[:25]:<25} {recomp_str} | "
              f"Prom6m: {float(r['Prom6m'] or 0):.1f} -> sin pico: {float(r['Prom6m_sin_pico'] or 0):.1f} | "
              f"Q: {int(r['Q_pedir_sugerido'])}")

n_os = int(plan_out["Alerta_oversizing"].sum())
if n_os > 0:
    print(f"\nOVERSIZING ({n_os} producto/s) — naranja en Excel:")
    os_rows = plan_out[plan_out["Alerta_oversizing"]==True][
        ["Articulo","Ultima_compra_cant","Q_pedir_sugerido",
         "Consumo_implicito_lote","Lote_termino_en_QS"]].head(10)
    for _, r in os_rows.iterrows():
        print(f"  {str(r['Articulo'])[:45]:<45} | "
              f"ult.compra: {int(r['Ultima_compra_cant'] or 0):>4} | "
              f"Q_suger: {int(r['Q_pedir_sugerido']):>4} | "
              f"c.impl: {float(r['Consumo_implicito_lote'] or 0):.1f}/m")

print(f
COLORES EN EXCEL:
  Rojo    URGENTE  — pedir ya, stock no cubre el leadtime
  Naranja PRONTO   — menos de 3 meses de cobertura
  Amarillo INCLUIR — entre 3 y 6 meses
  Verde   OK       — stock suficiente
  Naranja oscuro   — Alerta oversizing (Q > {FACTOR_OVERSIZING}x ultima compra sin QS)
  Lila    — Pico concentrado en 1 cliente (ver columnas Pico_*)

COLUMNAS CLAVE NUEVA — CONCENTRACION DE PICO:
  Pico_mes              mes con mayor venta en los ultimos 6 meses
  Pico_cant             cuanto se vendio ese mes
  Pico_clientes_unicos  cuantos clientes distintos compraron ese mes
  Pico_concentrado      TRUE = fue 1 solo cliente
  Pico_cliente_nombre   quien fue ese cliente
  Pico_cliente_recompro TRUE = ese cliente volvio a comprar despues
  Prom6m_sin_pico       promedio excluyendo el mes pico
                        -> si Pico_concentrado=TRUE y no recompro,
                           el modelo usa este numero en vez de Prom6m

LOGICA CONSUMO PROYECTADO:
  Nivel 1 (lote >= 3m):  Consumo_implicito_lote es la base real
  Nivel 2 (sin lote):
    Pico 1 cliente sin recompra -> Prom6m_sin_pico (pico espurio ignorado)
    Pico 1 cliente con recompra -> logica normal (cliente fidelizado)
    QS reciente                 -> max(Max12m, Prom6m*1.2)
    SUBE / BAJA / ESTABLE       -> ajuste por señal sobre Prom6m o Max6m

HOJAS:
  A_Pedir            lo que hay que comprar
  Picos_1_Cliente    productos con pico de 1 solo cliente (revisar)
  Revisar_Oversizing productos con Q > {FACTOR_OVERSIZING}x ultima compra
  Plan_QNAP_Completo todos los productos
  Ventas_Mensuales   historial: SRA=gris, QS=amarillo
  Parametros         logica usada

Archivo: {out_file}
"""

<>:66: SyntaxWarning: invalid escape sequence '\s'
<>:66: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_36231/3269371034.py:66: SyntaxWarning: invalid escape sequence '\s'
  .str.replace(r"\s+","-",regex=True)


'\n# ============================================================\n# PLAN DE REPOSICION QNAP v4 — con analisis de concentracion de pico\n# Objetivo: 6 meses + 1.5 lead time = 7.5 meses\n#\n# Jerarquia consumo proyectado:\n#   1. Lote confiable (>= 3m duracion) -> Consumo_implicito_lote como base\n#      QS_lote: +15% | SUBE: +20% | BAJA: -15%\n#   2. Sin lote confiable:\n#      Pico concentrado (1 cliente) -> ignorar pico, usar Prom6m_sin_pico\n#      Pico distribuido (2+ clientes) -> usar logica señal normal\n#      QS reciente -> max(Max12m, Prom6m*1.2)\n#      SUBE -> max(Max6m, Avg_signal*1.1)\n#      BAJA -> max(Avg_signal, 0.5)\n#      ESTABLE -> Prom6m\n# ============================================================\n\nimport os, itertools\nimport numpy as np\nimport pandas as pd\n\nFECHA_STOCK       = pd.Timestamp("2026-03-02")\nLEAD_TIME         = 1.5\nCOBERTURA_OBJ     = 6.0\nMESES_TOTAL       = LEAD_TIME + COBERTURA_OBJ  # 7.5\nN_PROM            = 6\nN_PICO_LARGO      = 12\nK

#ROTACION

In [52]:
# ==========================================================
# ALERTA TEMPRANA — DEMANDA ÚLTIMOS K MESES vs BASE N MESES
# + Cruce con Stock (si está disponible)
#
# - Señal: últimos K=2 meses
# - Base: N=4 meses previos
# - Umbrales adaptativos por volumen
# ==========================================================

import os, re
import numpy as np
import pandas as pd

# -------------------------
# CONFIG (ajustable)
# -------------------------
K_SIGNAL = 2         # últimos K meses (señal)
N_BASE   = 4         # meses previos para baseline
MIN_MONTHS_REQUIRED = K_SIGNAL + N_BASE

THR = {
    "high": {"base_min": 20, "up_pct": 0.20, "down_pct": -0.20, "min_abs": 5},
    "mid":  {"base_min": 5,  "up_pct": 0.35, "down_pct": -0.35, "min_abs": 2},
    "low":  {"base_min": 0,  "up_pct": np.nan, "down_pct": np.nan, "min_abs": 2},  # solo abs
}

# Stock coverage thresholds (meses)
COV_OVERSTOCK = 4.0   # si cae la demanda y tenés >4 meses de cobertura
COV_STOCKOUT  = 1.5   # si sube la demanda y tenés <1.5 meses

# -------------------------
# SALIDA (sin subcarpetas)
# -------------------------
out_dir = output_path_base if "output_path_base" in globals() else "."
os.makedirs(out_dir, exist_ok=True)

# -------------------------
# 1) Detectar columnas mes
# -------------------------
if "df_pivot" not in globals():
    raise NameError("No existe df_pivot. Ejecutá antes la celda que arma el pivot mensual.")

month_col_pattern = re.compile(r"^\d{4}-\d{2}$")
month_cols = [c for c in df_pivot.columns if isinstance(c, str) and month_col_pattern.match(c)]
if not month_cols:
    raise ValueError("No se detectaron columnas mensuales 'YYYY-MM' en df_pivot.")

month_cols_sorted = sorted(month_cols, key=lambda s: pd.to_datetime(s, format="%Y-%m"))

if len(month_cols_sorted) < MIN_MONTHS_REQUIRED:
    raise ValueError(
        f"Hay {len(month_cols_sorted)} meses y se requieren al menos {MIN_MONTHS_REQUIRED} "
        f"(K_SIGNAL={K_SIGNAL} + N_BASE={N_BASE})."
    )

last_month = month_cols_sorted[-1]
signal_months = month_cols_sorted[-K_SIGNAL:]
base_months   = month_cols_sorted[-(K_SIGNAL + N_BASE):-K_SIGNAL]

# -------------------------
# 2) Matriz numérica demanda
# -------------------------
meta_cols = [c for c in ["Codigo", "Articulo", "Marca"] if c in df_pivot.columns]
work = df_pivot[meta_cols + month_cols_sorted].copy()

mat = work[month_cols_sorted].apply(pd.to_numeric, errors="coerce")  # SRA/QS quedan NaN
work["_non_numeric_months_cnt"] = work[month_cols_sorted].isna().sum(axis=1)

# Para demanda: NaN lo tomamos 0 para sumas/promedios
mat0 = mat.fillna(0)

# -------------------------
# 3) Métricas señal vs base
# -------------------------
base_vals   = mat0[base_months]
signal_vals = mat0[signal_months]

base_avg   = base_vals.mean(axis=1)
base_std   = base_vals.std(axis=1, ddof=0)  # por si lo querés usar más adelante
signal_avg = signal_vals.mean(axis=1)

delta_abs = signal_avg - base_avg
delta_pct = np.where(base_avg > 0, delta_abs / base_avg, np.nan)

is_new = (base_vals.sum(axis=1) == 0) & (signal_vals.sum(axis=1) > 0)

# -------------------------
# 4) Stock (si existe)
# -------------------------
stock_series = None

for cand in ["Stock_ActualUnid", "Stock_Actual_Total", "Stock_Actual", "Stock", "stock"]:
    if cand in df_pivot.columns:
        stock_series = pd.to_numeric(df_pivot[cand], errors="coerce").fillna(0)
        break

if stock_series is None and "metricas" in globals():
    if isinstance(metricas, pd.DataFrame) and "Stock_ActualUnid" in metricas.columns:
        m = metricas["Stock_ActualUnid"].copy()
        m.index = m.index.astype(str).str.strip()
        stock_series = work["Codigo"].astype(str).str.strip().map(m).fillna(0).astype(float)

if stock_series is None:
    stock_series = pd.Series(0.0, index=work.index)

coverage_m = np.where(base_avg > 0, stock_series.values / base_avg.values, np.nan)

# -------------------------
# 5) Clasificación (semáforo)
# -------------------------
def classify_row(bavg, dabs, dpct, new_flag):
    if new_flag:
        return "NUEVO"

    # baja rotación: base promedio < 1 y señal también baja
    if bavg < 1 and abs(dabs) < THR["low"]["min_abs"]:
        return "BAJA_ROTACION"

    if bavg >= THR["high"]["base_min"]:
        up_ok   = (dpct >= THR["high"]["up_pct"]) and (dabs >= THR["high"]["min_abs"])
        down_ok = (dpct <= THR["high"]["down_pct"]) and (abs(dabs) >= THR["high"]["min_abs"])
    elif bavg >= THR["mid"]["base_min"]:
        up_ok   = (dpct >= THR["mid"]["up_pct"]) and (dabs >= THR["mid"]["min_abs"])
        down_ok = (dpct <= THR["mid"]["down_pct"]) and (abs(dabs) >= THR["mid"]["min_abs"])
    else:
        up_ok   = (dabs >= THR["low"]["min_abs"])
        down_ok = (dabs <= -THR["low"]["min_abs"])

    if up_ok:
        return "SUBE"
    if down_ok:
        return "CAE"
    return "NORMAL"

estado = [
    classify_row(bavg, dabs, dpct, nf)
    for bavg, dabs, dpct, nf in zip(base_avg.values, delta_abs.values, delta_pct, is_new.values)
]

accion = []
for est, cov in zip(estado, coverage_m):
    if est == "CAE" and (pd.notna(cov) and cov >= COV_OVERSTOCK):
        accion.append("SOBRESTOCK_RIESGO")
    elif est == "SUBE" and (pd.notna(cov) and cov <= COV_STOCKOUT):
        accion.append("RUPTURA_RIESGO")
    elif est == "NUEVO":
        accion.append("SEGUIR_DE_CERCA")
    else:
        accion.append("OK")

# -------------------------
# 6) Output tabla (Excel en out_dir)
# -------------------------
res = work[meta_cols].copy()
res["Mes_ultimo"] = last_month
res["Signal_meses"] = " / ".join(signal_months)
res["Base_meses"]   = " / ".join(base_months)

res["Base_avg_mensual_unid"]   = base_avg.round(2)
res["Signal_avg_mensual_unid"] = signal_avg.round(2)
res["Delta_abs_unid"]          = delta_abs.round(2)
res["Delta_pct"]               = (delta_pct * 100).round(1)

res["Stock_unid"] = pd.to_numeric(stock_series, errors="coerce").fillna(0).round(0).astype(int)
res["Cobertura_meses_vs_base"] = pd.Series(coverage_m).round(2)

res["Estado"] = estado
res["Accion"] = accion
res["Meses_no_numericos_cnt"] = work["_non_numeric_months_cnt"].astype(int)

def _prio(row):
    if row["Accion"] == "SOBRESTOCK_RIESGO":
        return 1
    if row["Accion"] == "RUPTURA_RIESGO":
        return 2
    if row["Estado"] == "CAE":
        return 3
    if row["Estado"] == "SUBE":
        return 4
    return 5

res["Prioridad"] = res.apply(_prio, axis=1)
res = res.sort_values(["Prioridad", "Stock_unid", "Delta_abs_unid"], ascending=[True, False, True])

out_xlsx = os.path.join(out_dir, f"alerta_demanda_{last_month.replace('-','')}.xlsx")
res.to_excel(out_xlsx, index=False)

print("✅ Exportado:", out_xlsx)
print("Ventana:", f"BASE {base_months[0]}→{base_months[-1]} | SIGNAL {signal_months[0]}→{signal_months[-1]} | Último mes: {last_month}")


/tmp/ipykernel_36231/751031943.py:106: RuntimeWarning: divide by zero encountered in divide
  coverage_m = np.where(base_avg > 0, stock_series.values / base_avg.values, np.nan)
/tmp/ipykernel_36231/751031943.py:106: RuntimeWarning: invalid value encountered in divide
  coverage_m = np.where(base_avg > 0, stock_series.values / base_avg.values, np.nan)


✅ Exportado: /content/gdrive/My Drive/Grapes/reporte_grapes/resultados/alerta_demanda_202603.xlsx
Ventana: BASE 2025-10→2026-01 | SIGNAL 2026-02→2026-03 | Último mes: 2026-03


In [53]:
# ==========================================================
# ROTACION / INMOVILIZACION — RECONSTRUCCION POR MOVIMIENTOS (v6.1)
# ----------------------------------------------------------
# Métrica principal:
#   dias_no_rota = "hace cuánto no rota con stock disponible",
#   medido por ciclo de stock (entre quiebres).
#
# Reglas operativas:
# - Si Stock_actual_est <= 0  => dias_no_rota = 0
# - "quiebre" = cualquier momento donde stock_est_post <= 0
#   (VENTA o AJUSTE negativo u otro mov que deje <=0)
# - "restock efectivo" = primer movimiento posterior al quiebre que deja stock > 0
#   (COMPRA o AJUSTE positivo)
# - Ciclo actual = desde el último restock efectivo luego del último quiebre
#   (si nunca quebró: desde la primera vez que stock pasa a >0)
# - Ancla de no rotación:
#   - si hubo ventas en el ciclo: última VENTA del ciclo
#   - si no hubo ventas en el ciclo: inicio del ciclo (restock)
#   => dias_no_rota = now - fecha_ancla
#
# FIX principal:
# - AJUSTE (Aj.Stock) usa delta = Stock_Actual - Stock_Anterior (si ambos existen),
#   porque Cantidad en ajustes puede ser inconsistente.
#
# Salida:
# - Excel en output_path_base (sin subcarpetas)
#   Sheets: RESUMEN, LEYENDA, MOV_AUDITORIA
# ==========================================================

import os
import numpy as np
import pandas as pd

# -----------------------------
# PRECONDICIONES
# -----------------------------
if "df_merged_usd" not in globals():
    raise NameError("df_merged_usd no existe. Ejecutá antes la celda que arma df_merged_usd.")

if "output_path_base" not in globals():
    output_path_base = "."

os.makedirs(output_path_base, exist_ok=True)

NEEDED = ["Codigo", "Fecha_de_Carga", "Comprobante", "Cantidad"]
for c in NEEDED:
    if c not in df_merged_usd.columns:
        raise KeyError(f"Falta columna requerida en df_merged_usd: {c}")

# -----------------------------
# CONFIG (ajustable)
# -----------------------------
MIN_DEMAND_FOR_COVER = 0.50   # u/mes mínima para que cobertura tenga sentido
TH_COVER_LOW  = 1.5           # meses -> riesgo ruptura
TH_COVER_HIGH = 6.0           # meses -> sobrestock (ajustable)
GRACE_DAYS_NEW = 60           # no penalizar "recién comprado"
DAYS_CLAW_180 = 180
DAYS_CLAW_365 = 365

# -----------------------------
# Resolver columnas opcionales
# -----------------------------
col_art     = "Articulo" if "Articulo" in df_merged_usd.columns else None
col_marca   = "Marca" if "Marca" in df_merged_usd.columns else None
col_clase   = "Clase" if "Clase" in df_merged_usd.columns else None
col_stk_pre = "Stock_Anterior" if "Stock_Anterior" in df_merged_usd.columns else None
col_stk_act = "Stock_Actual" if "Stock_Actual" in df_merged_usd.columns else None

# (opcional) costo unitario si existe en tu histórico
col_cost = None
for cand in ["costo_neto_unitario", "Costo_Unitario", "costo_unitario", "Costo_usd", "CostoUSD"]:
    if cand in df_merged_usd.columns:
        col_cost = cand
        break

# -----------------------------
# 1) Limpieza / normalización
# -----------------------------
mov = df_merged_usd.copy()

mov["Codigo"] = mov["Codigo"].astype(str).str.strip().str.upper()
mov["Fecha_de_Carga"] = pd.to_datetime(mov["Fecha_de_Carga"], errors="coerce", dayfirst=True)
mov = mov[mov["Fecha_de_Carga"].notna()].copy()

mov["Comprobante_norm"] = (
    mov["Comprobante"].astype(str)
      .str.strip().str.lower()
      .str.replace(r"\s+", " ", regex=True)
)

mov["Clase_norm"] = (mov[col_clase].astype(str).str.strip().str.lower() if col_clase else "")
# Normalización extra para clase: tolera "01.factura" / "01. factura" / "01 factura"
mov["Clase_norm2"] = (
    mov["Clase_norm"]
      .str.replace(r"\s+", " ", regex=True)
      .str.replace(r"\.", " ", regex=True)
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)

mov["Marca_norm"] = (mov[col_marca].astype(str).str.strip() if col_marca else "")
mov["Articulo_norm"] = (mov[col_art].astype(str).str.strip() if col_art else "")

mov["Cantidad_raw"] = pd.to_numeric(mov["Cantidad"], errors="coerce").fillna(0.0)

mov["Stock_Anterior_raw"] = (pd.to_numeric(mov[col_stk_pre], errors="coerce") if col_stk_pre else np.nan)
mov["Stock_Actual_raw"]   = (pd.to_numeric(mov[col_stk_act], errors="coerce") if col_stk_act else np.nan)

if col_cost:
    mov["costo_unit_raw"] = pd.to_numeric(mov[col_cost], errors="coerce")
else:
    mov["costo_unit_raw"] = np.nan

# -----------------------------
# 2) Clasificación de movimiento (CRÍTICO)
# -----------------------------
is_fc = mov["Comprobante_norm"].str.startswith(("fc a", "fc b", "fc c", "fc"), na=False)
is_nc = mov["Comprobante_norm"].str.startswith(("nc a", "nc b", "nc c", "nc d", "nc"), na=False)
is_di = mov["Comprobante_norm"].str.startswith(("di",), na=False)

# Compra por clase (robusto)
is_compra_by_clase = (
    mov["Clase_norm2"].str.match(r"^01\s*factura\b", na=False)
    | mov["Clase_norm2"].str.contains(r"\bcompra\b", na=False, regex=True)
)

# Ajuste de stock
is_ajuste = (
    mov["Comprobante_norm"].str.contains("aj.stock", na=False)
    | mov["Clase_norm2"].str.contains(r"ajuste|ajustes de stock|stock - ajustes", na=False, regex=True)
)

# Ventas cliente: FC que no es compra y no es ajuste
is_venta = is_fc & (~is_compra_by_clase) & (~is_ajuste)
# Compras: DI o FC compra por clase, no ajuste
is_compra = (is_di | (is_fc & is_compra_by_clase)) & (~is_ajuste)
# NC cliente
is_nc_cliente = is_nc & (~is_compra_by_clase) & (~is_ajuste)

mov["tipo_mov"] = np.select(
    [is_ajuste, is_compra, is_venta, is_nc_cliente],
    ["AJUSTE", "COMPRA", "VENTA", "NC_CLIENTE"],
    default="OTRO"
)

# -----------------------------
# 2b) delta_stock (FIX AJUSTE)
# -----------------------------
# delta base por reglas
mov["delta_stock"] = np.select(
    [
        mov["tipo_mov"].eq("VENTA"),
        mov["tipo_mov"].eq("NC_CLIENTE"),
        mov["tipo_mov"].eq("COMPRA"),
        mov["tipo_mov"].eq("AJUSTE"),
    ],
    [
        -mov["Cantidad_raw"].abs(),
        +mov["Cantidad_raw"].abs(),
        +mov["Cantidad_raw"].abs(),
        mov["Cantidad_raw"],  # AJUSTE respeta signo... pero lo vamos a corregir si hay stocks
    ],
    default=0.0
).astype(float)

# FIX: AJUSTE usa Stock_Actual - Stock_Anterior cuando existan
mask_aj = mov["tipo_mov"].eq("AJUSTE") & mov["Stock_Anterior_raw"].notna() & mov["Stock_Actual_raw"].notna()
mov.loc[mask_aj, "delta_stock"] = (mov.loc[mask_aj, "Stock_Actual_raw"] - mov.loc[mask_aj, "Stock_Anterior_raw"]).astype(float)
mov["delta_ajuste_por_stock"] = mask_aj

# ventas netas (demanda): VENTA suma, NC resta
mov["ventas_netas_u"] = np.select(
    [mov["tipo_mov"].eq("VENTA"), mov["tipo_mov"].eq("NC_CLIENTE")],
    [mov["Cantidad_raw"].abs(), -mov["Cantidad_raw"].abs()],
    default=0.0
).astype(float)

mov["ventas_brutas_u"] = np.select(
    [mov["tipo_mov"].eq("VENTA")],
    [mov["Cantidad_raw"].abs()],
    default=0.0
).astype(float)

# -----------------------------
# 3) Reconstrucción de stock estimado por Código
#   - Stock_Anterior se toma como "justo antes del movimiento" (tu definición)
# -----------------------------
mov = mov.sort_values(["Codigo", "Fecha_de_Carga"]).reset_index(drop=True)

def reconstruir_stock_por_codigo(g: pd.DataFrame) -> pd.DataFrame:
    g = g.copy()
    pre_list, post_list = [], []
    last_post = None

    for _, r in g.iterrows():
        stk_pre = r["Stock_Anterior_raw"]
        d = r["delta_stock"]

        if pd.notna(stk_pre):
            pre = float(stk_pre)
        else:
            pre = 0.0 if last_post is None else float(last_post)

        post = pre + float(d)
        pre_list.append(pre)
        post_list.append(post)
        last_post = post

    g["stock_est_pre"] = pre_list
    g["stock_est_post"] = post_list

    if g["Stock_Actual_raw"].notna().any():
        g["diff_vs_stock_actual"] = g["Stock_Actual_raw"] - g["stock_est_post"]
    else:
        g["diff_vs_stock_actual"] = np.nan

    return g

mov2 = mov.groupby("Codigo", group_keys=False).apply(reconstruir_stock_por_codigo)
now_dt = mov2["Fecha_de_Carga"].max()

# Stock actual estimado
last_by_code = mov2.sort_values(["Codigo", "Fecha_de_Carga"]).groupby("Codigo", as_index=False).tail(1)
stock_actual_est = last_by_code.set_index("Codigo")["stock_est_post"].fillna(0.0)

# -----------------------------
# 4) Fechas clave globales (informativas)
# -----------------------------
last_sale_all = mov2[mov2["tipo_mov"].eq("VENTA")].groupby("Codigo")["Fecha_de_Carga"].max()
last_buy_all  = mov2[mov2["tipo_mov"].eq("COMPRA")].groupby("Codigo")["Fecha_de_Carga"].max()

dias_desde_ultima_venta  = (now_dt - last_sale_all).dt.days
dias_desde_ultima_compra = (now_dt - last_buy_all).dt.days
flag_nuevo_reciente = dias_desde_ultima_compra.notna() & (dias_desde_ultima_compra <= GRACE_DAYS_NEW)

# -----------------------------
# 5) Ciclo de stock + dias_no_rota (LA PARTE CLAVE)
#    Esto NO usa "última venta global": usa ciclo actual entre quiebres.
# -----------------------------
def ciclo_y_no_rota_por_codigo(g: pd.DataFrame) -> pd.Series:
    g = g.sort_values("Fecha_de_Carga").copy()

    stock_final = float(g["stock_est_post"].iloc[-1])

    # Si hoy no hay stock, no tiene sentido "no rota con stock"
    if stock_final <= 0:
        q = g[g["stock_est_post"] <= 0]
        t_quiebre = q["Fecha_de_Carga"].max() if not q.empty else pd.NaT
        return pd.Series({
            "fecha_ultimo_quiebre_stock": t_quiebre,
            "fecha_inicio_ciclo_stock": pd.NaT,
            "fecha_ultima_venta_ciclo": pd.NaT,
            "fecha_ancla_no_rota": pd.NaT,
            "dias_no_rota": 0,
            "ciclo_desde_quiebre": np.nan,
        })

    # último quiebre (si existió)
    q = g[g["stock_est_post"] <= 0]
    t_quiebre = q["Fecha_de_Carga"].max() if not q.empty else pd.NaT

    # inicio de ciclo:
    # - si hubo quiebre: primer movimiento posterior que lleva de <=0 a >0 (COMPRA o AJUSTE positivo)
    # - si no hubo quiebre: primera vez que stock pasa a >0
    ciclo_desde_quiebre = False

    if pd.notna(t_quiebre):
        post = g[g["Fecha_de_Carga"] > t_quiebre].copy()
        rest = post[
            (post["tipo_mov"].isin(["COMPRA", "AJUSTE"])) &
            (post["stock_est_pre"] <= 0) &
            (post["stock_est_post"] > 0)
        ]
        if not rest.empty:
            t_inicio = rest["Fecha_de_Carga"].min()
            ciclo_desde_quiebre = True
        else:
            # fallback raro
            pos = g[g["stock_est_post"] > 0]
            t_inicio = pos["Fecha_de_Carga"].min() if not pos.empty else pd.NaT
    else:
        pos = g[g["stock_est_post"] > 0]
        t_inicio = pos["Fecha_de_Carga"].min() if not pos.empty else pd.NaT

    # ventas dentro del ciclo
    if pd.notna(t_inicio):
        v = g[(g["tipo_mov"].eq("VENTA")) & (g["Fecha_de_Carga"] >= t_inicio)]
    else:
        v = g[g["tipo_mov"].eq("VENTA")]

    t_venta_ciclo = v["Fecha_de_Carga"].max() if not v.empty else pd.NaT

    # ancla: última venta del ciclo si existe, si no el inicio del ciclo
    t_ancla = t_venta_ciclo if pd.notna(t_venta_ciclo) else t_inicio
    dias = int((now_dt - t_ancla).days) if pd.notna(t_ancla) else np.nan

    return pd.Series({
        "fecha_ultimo_quiebre_stock": t_quiebre,
        "fecha_inicio_ciclo_stock": t_inicio,
        "fecha_ultima_venta_ciclo": t_venta_ciclo,
        "fecha_ancla_no_rota": t_ancla,
        "dias_no_rota": dias,
        "ciclo_desde_quiebre": ciclo_desde_quiebre,
    })

ciclo = mov2.groupby("Codigo", group_keys=False).apply(ciclo_y_no_rota_por_codigo)

# -----------------------------
# 6) Ventanas / demanda / cobertura
# -----------------------------
def ventas_netas_window(days: int):
    dt_cut = now_dt - pd.Timedelta(days=days)
    w = mov2[mov2["Fecha_de_Carga"] >= dt_cut]
    return w.groupby("Codigo")["ventas_netas_u"].sum()

v90  = ventas_netas_window(90)
v180 = ventas_netas_window(180)
v365 = ventas_netas_window(365)

dem_6m  = (v180 / 6.0).replace([np.inf, -np.inf], np.nan)
dem_12m = (v365 / 12.0).replace([np.inf, -np.inf], np.nan)

coverage_6m = pd.Series(np.nan, index=stock_actual_est.index, dtype=float)
dem6 = dem_6m.reindex(stock_actual_est.index).fillna(0)
mask_cover = (dem6 >= MIN_DEMAND_FOR_COVER) & (stock_actual_est > 0)
coverage_6m.loc[mask_cover] = (stock_actual_est.loc[mask_cover] / dem6.loc[mask_cover]).replace([np.inf, -np.inf], np.nan)

# -----------------------------
# 7) Costos / valorización (opcional)
# -----------------------------
last_cost_buy = None
if col_cost:
    last_cost_buy = (
        mov2[mov2["tipo_mov"].eq("COMPRA")]
        .dropna(subset=["costo_unit_raw"])
        .sort_values(["Codigo", "Fecha_de_Carga"])
        .groupby("Codigo")["costo_unit_raw"].last()
    )

# -----------------------------
# 8) RESUMEN
# -----------------------------
res = pd.DataFrame(index=stock_actual_est.index)
res.index.name = "Codigo"

res["Articulo"] = mov2.groupby("Codigo")["Articulo_norm"].last() if col_art else ""
res["Marca"]    = mov2.groupby("Codigo")["Marca_norm"].last() if col_marca else ""
res["Stock_actual_est"] = stock_actual_est.round(0).fillna(0).astype(int)

# métrica principal
res = res.join(ciclo, how="left")

# informativas globales
res["Fecha_ultima_venta"] = last_sale_all
res["dias_desde_ultima_venta"] = dias_desde_ultima_venta

res["Fecha_ultima_compra"] = last_buy_all
res["dias_desde_ultima_compra"] = dias_desde_ultima_compra
res["flag_nuevo_reciente"] = flag_nuevo_reciente.reindex(res.index).fillna(False)

# ventanas
res["ventas_netas_u_90d"]  = v90.reindex(res.index).fillna(0).round(0).astype(int)
res["ventas_netas_u_180d"] = v180.reindex(res.index).fillna(0).round(0).astype(int)
res["ventas_netas_u_365d"] = v365.reindex(res.index).fillna(0).round(0).astype(int)

res["demanda_prom_u_mes_6m"]  = dem_6m.reindex(res.index).fillna(0).round(2)
res["demanda_prom_u_mes_12m"] = dem_12m.reindex(res.index).fillna(0).round(2)
res["cobertura_meses_6m"] = coverage_6m.reindex(res.index).round(2)

# costos
if last_cost_buy is not None:
    res["ult_costo_compra_unit"] = last_cost_buy.reindex(res.index)
    res["valor_stock_est"] = (res["Stock_actual_est"] * res["ult_costo_compra_unit"]).replace([np.inf, -np.inf], np.nan)
else:
    res["ult_costo_compra_unit"] = np.nan
    res["valor_stock_est"] = np.nan

res["flag_stock_cero"] = res["Stock_actual_est"].le(0)

# -----------------------------
# 9) Acciones sugeridas
# -----------------------------
def accion_sugerida(r):
    stk = int(r["Stock_actual_est"])
    if stk <= 0:
        return "SIN_STOCK"

    dias = r["dias_no_rota"]
    if pd.isna(dias):
        dias = 0

    d6 = float(r["demanda_prom_u_mes_6m"]) if pd.notna(r["demanda_prom_u_mes_6m"]) else 0.0
    cov = r["cobertura_meses_6m"]
    nuevo = bool(r.get("flag_nuevo_reciente", False))

    if nuevo and (dias <= GRACE_DAYS_NEW):
        return "RECIENTE (monitorear)"

    if (d6 >= MIN_DEMAND_FOR_COVER) and (pd.notna(cov) and cov <= TH_COVER_LOW):
        return "RUPTURA_RIESGO (reponer)"

    if (d6 >= MIN_DEMAND_FOR_COVER) and (pd.notna(cov) and cov >= TH_COVER_HIGH) and (dias >= DAYS_CLAW_180):
        return "SOBRESTOCK (frenar compras / promo)"

    if dias >= DAYS_CLAW_365 and (r["ventas_netas_u_365d"] <= 0):
        return "CLAVO (liquidar / bundle / descuento)"

    if dias >= DAYS_CLAW_180:
        return "LENTO (empujar venta / revisar precio)"

    return "OK"

res["accion"] = res.apply(accion_sugerida, axis=1)

def prio_row(r):
    stk = int(r["Stock_actual_est"])
    if stk <= 0:
        return 99
    a = str(r["accion"])
    if a.startswith("CLAVO"): return 1
    if a.startswith("SOBRESTOCK"): return 2
    if a.startswith("RUPTURA_RIESGO"): return 3
    if a.startswith("LENTO"): return 4
    return 5

res["prio"] = res.apply(prio_row, axis=1)

res = res.sort_values(
    ["prio", "dias_no_rota", "valor_stock_est", "Stock_actual_est"],
    ascending=[True, False, False, False]
)

# -----------------------------
# 10) Reorden columnas (dias_no_rota ENTRE LAS PRIMERAS + Codigo primero)
# -----------------------------
res_out = res.reset_index()

preferred = [
    "Codigo", "Articulo", "Marca", "Stock_actual_est", "dias_no_rota",
    "fecha_ancla_no_rota", "fecha_inicio_ciclo_stock",
    "fecha_ultima_venta_ciclo", "fecha_ultimo_quiebre_stock",
    "accion", "prio"
]
preferred = [c for c in preferred if c in res_out.columns]
res_out = res_out[preferred + [c for c in res_out.columns if c not in preferred]]

# -----------------------------
# 11) LEYENDA
# -----------------------------
leyenda = pd.DataFrame([
    ["dias_no_rota", "Métrica principal. Con stock>0: días desde última VENTA del ciclo; si no hubo ventas, desde el RESTOCK del ciclo. Con stock=0: 0."],
    ["fecha_inicio_ciclo_stock", "Inicio del ciclo actual: primer restock efectivo luego del último quiebre (stock<=0). Si nunca quebró: primera fecha donde stock>0."],
    ["fecha_ultimo_quiebre_stock", "Última fecha en que el stock estimado quedó <=0 (por venta o ajuste)."],
    ["AJUSTE (Aj.Stock)", "Puede quebrar o sumar stock. NO cuenta como venta. Para delta se usa Stock_Actual-Stock_Anterior si existe."],
    ["delta_ajuste_por_stock", "True si el delta del AJUSTE se calculó por Stock_Actual-Stock_Anterior (recomendado)."],
    ["accion", "Sugerencia operativa: CLAVO / SOBRESTOCK / RUPTURA / LENTO / OK."],
], columns=["Campo", "Interpretación"])

# -----------------------------
# 12) Export Excel
# -----------------------------
out_xlsx = os.path.join(output_path_base, "rotacion_inventario_reconstruida.xlsx")

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    res_out.to_excel(writer, index=False, sheet_name="RESUMEN")
    leyenda.to_excel(writer, index=False, sheet_name="LEYENDA")

    # Auditoría
    audit_cols = [
        "Codigo","Fecha_de_Carga","Comprobante","tipo_mov",
        "Cantidad_raw","delta_stock","delta_ajuste_por_stock",
        "Stock_Anterior_raw","Stock_Actual_raw",
        "stock_est_pre","stock_est_post","diff_vs_stock_actual",
        "Clase_norm","Clase_norm2"
    ]
    audit_cols = [c for c in audit_cols if c in mov2.columns]
    mov2[audit_cols].to_excel(writer, index=False, sheet_name="MOV_AUDITORIA")

print("✅ Exportado:", out_xlsx)
print("📌 Corte temporal (now):", now_dt)


/tmp/ipykernel_36231/3671917721.py:219: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mov2 = mov.groupby("Codigo", group_keys=False).apply(reconstruir_stock_por_codigo)
/tmp/ipykernel_36231/3671917721.py:306: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ciclo = mov2.groupby("Codigo", group_keys=False).apply(ciclo_y_no_rota_por_codigo)
/tmp/ipykernel_36231/3671917721.py:359: FutureWarning: Downcasting objec

✅ Exportado: /content/gdrive/My Drive/Grapes/reporte_grapes/resultados/rotacion_inventario_reconstruida.xlsx
📌 Corte temporal (now): 2026-03-31 18:50:54.007000


In [54]:
# ==========================================================
# CLIENTES (WEB) — NUEVOS POR MES, CANAL (INFO), TOPS ANUALES,
#                  Y DETECCIÓN DE HIATOS (CHURN/REACTIVACIÓN)
# Requiere: df_merged_usd en memoria (nivel línea)
# Salidas:
#   - Excel: clientes_analitica.xlsx (output_path_base)
#   - PNGs : gráficos (output_path_base)
# ==========================================================

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# PRECONDICIONES
# -----------------------------
if "df_merged_usd" not in globals():
    raise NameError("df_merged_usd no existe. Ejecutá antes la celda que arma df_merged_usd.")

if "output_path_base" not in globals():
    output_path_base = "."

os.makedirs(output_path_base, exist_ok=True)

df = df_merged_usd.copy()

# -----------------------------
# 0) Columnas mínimas / normalización
# -----------------------------
needed = ["Fecha_de_Carga", "Comprobante", "Cantidad"]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise KeyError(f"Faltan columnas en df_merged_usd: {missing}")

df["Fecha_de_Carga"] = pd.to_datetime(df["Fecha_de_Carga"], errors="coerce", dayfirst=True)
df = df[df["Fecha_de_Carga"].notna()].copy()

# Cliente
col_cliente = "Cliente_Proveedor" if "Cliente_Proveedor" in df.columns else None
if col_cliente is None:
    cand = [c for c in df.columns if c.lower() in ["cliente", "cliente_proveedor", "razon_social", "razon social"]]
    if not cand:
        raise KeyError("No encuentro columna de cliente. Esperaba 'Cliente_Proveedor' u otra equivalente.")
    col_cliente = cand[0]

df[col_cliente] = df[col_cliente].astype(str).fillna("").str.strip()
df["cliente_norm"] = (
    df[col_cliente]
      .astype(str)
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)

df["Comp_norm"] = (
    df["Comprobante"].astype(str)
      .str.strip().str.lower()
      .str.replace(r"\s+", " ", regex=True)
)

df["Clase_norm"] = df["Clase"].astype(str).str.strip().str.lower() if "Clase" in df.columns else ""

# USD columns
col_profit_usd = "profit_ii_articulo_venta_usd" if "profit_ii_articulo_venta_usd" in df.columns else None
col_neto_usd   = "total_neto_articulo_venta_usd" if "total_neto_articulo_venta_usd" in df.columns else None
if col_profit_usd is None or col_neto_usd is None:
    raise KeyError("No encuentro columnas USD esperadas: 'profit_ii_articulo_venta_usd' y/o 'total_neto_articulo_venta_usd'.")

df[col_profit_usd] = pd.to_numeric(df[col_profit_usd], errors="coerce").fillna(0.0)
df[col_neto_usd]   = pd.to_numeric(df[col_neto_usd], errors="coerce").fillna(0.0)
df["Cantidad"]     = pd.to_numeric(df["Cantidad"], errors="coerce").fillna(0.0)

# -----------------------------
# 0.b) WEB FILTER por Condicion_de_Venta (TU REGLA)
#   - "contado" => WEB
#   - comienza con "only-web" => WEB (only-webvisa, only-web mastercard, etc.)
# -----------------------------
if "Condicion_de_Venta" not in df.columns:
    raise KeyError("Falta columna 'Condicion_de_Venta' (necesaria para filtrar WEB).")

df["cond_venta_norm"] = (
    df["Condicion_de_Venta"].astype(str).fillna("")
      .str.strip().str.lower()
      .str.replace(r"\s+", "", regex=True)   # saco espacios para matchear only-web mastercard
)

mask_web = (
    df["cond_venta_norm"].eq("contado")
    | df["cond_venta_norm"].str.startswith("only-web", na=False)
)

# -----------------------------
# 1) Definir qué es VENTA (FC y NC cliente) y armar sales_lines
# -----------------------------
prefixes_venta = ("fc a", "fc b", "fc c", "fc")
prefixes_nc    = ("nc a", "nc b", "nc c", "nc d", "nc")
prefixes_docs  = prefixes_venta + prefixes_nc

mask_docs = df["Comp_norm"].str.startswith(prefixes_docs, na=False)

# excluir compras por clase
mask_compra = (
    df["Clase_norm"].str.startswith("01.factura", na=False)
    | df["Clase_norm"].str.contains(r"\bcompra\b", regex=True, na=False)
)

# excluir presupuestos/cotizaciones
pat_pres = r"\b(?:pre|pres|presup|presupuesto|cot|cotiz|cotización)\b"
mask_pres = df["Comp_norm"].str.contains(pat_pres, na=False, regex=True) | df["Clase_norm"].str.contains("presup", na=False)

sales_lines = df[mask_docs & ~mask_compra & ~mask_pres].copy()

# signo (FC suma, NC resta)
sales_lines["signo"] = np.select(
    [
        sales_lines["Comp_norm"].str.startswith(prefixes_venta, na=False),
        sales_lines["Comp_norm"].str.startswith(prefixes_nc, na=False),
    ],
    [1, -1],
    default=0
)

sales_lines["neto_usd_signed"]   = sales_lines[col_neto_usd]   * sales_lines["signo"]
sales_lines["profit_usd_signed"] = sales_lines[col_profit_usd] * sales_lines["signo"]
sales_lines["unidades_netas"]    = sales_lines["Cantidad"].abs() * sales_lines["signo"]

# Mantengo flag web a nivel línea (para luego bajar a comprobante)
sales_lines["is_web_line"] = mask_web.reindex(sales_lines.index).fillna(False)

# -----------------------------
# 2) Tabla a nivel comprobante (df_comp)
# -----------------------------
if "k_comp" in sales_lines.columns:
    comp_key = "k_comp"
    sales_lines[comp_key] = sales_lines[comp_key].astype(str)
else:
    sales_lines["Fecha_dia"] = sales_lines["Fecha_de_Carga"].dt.floor("D")
    comp_key = "k_comp_fallback"
    sales_lines[comp_key] = (
        sales_lines["Comprobante"].astype(str).str.strip().str.lower()
        + "||" + sales_lines["Fecha_dia"].astype(str)
        + "||" + sales_lines["cliente_norm"].astype(str).str.lower()
    )

agg_dict = {
    "Fecha_de_Carga": "min",
    "Comprobante": "first",
    "cliente_norm": "first",
    "neto_usd_signed": "sum",
    "profit_usd_signed": "sum",
    "unidades_netas": "sum",
    # si una línea del comprobante es web, el comprobante lo tomo web
    "is_web_line": "max",
}

df_comp = sales_lines.groupby(comp_key, as_index=False).agg(agg_dict)
df_comp["Fecha_de_Carga"] = pd.to_datetime(df_comp["Fecha_de_Carga"], errors="coerce")
df_comp["anio"] = df_comp["Fecha_de_Carga"].dt.year.astype("Int64")
df_comp["mes_p"] = df_comp["Fecha_de_Carga"].dt.to_period("M")
df_comp["mes"] = df_comp["mes_p"].astype(str)

# -----------------------------
# 3) Canal (informativo): Web / MercadoLibre / Otro
#   OJO: "cliente nuevo" lo define Condicion_de_Venta (is_web_line), no esto.
# -----------------------------
canal_col = None
for cand in ["Canal", "canal", "Origen", "origen", "Plataforma", "plataforma", "Marketplace", "marketplace"]:
    if cand in df.columns:
        canal_col = cand
        break

if canal_col:
    canal_map = df[[comp_key, canal_col]].dropna().drop_duplicates(subset=[comp_key], keep="last")
    df_comp = df_comp.merge(canal_map, on=comp_key, how="left")
    df_comp["canal_norm"] = df_comp[canal_col].astype(str).str.strip().str.lower()
else:
    s = (df_comp["cliente_norm"].astype(str).str.lower() + " " + df_comp["Comprobante"].astype(str).str.lower())
    is_ml  = s.str.contains(r"\b(?:mercado\s*libre|mercadolibre|meli|ml)\b", regex=True, na=False)
    is_web_txt = s.str.contains(r"\b(?:web|tienda\s*online|e-?commerce|shop|carrito)\b", regex=True, na=False)
    df_comp["canal_norm"] = np.select([is_ml, is_web_txt], ["mercadolibre", "web"], default="otro")

# -----------------------------
# 4) CLIENTES NUEVOS POR MES — SOLO WEB (según Condicion_de_Venta)
#   - usamos SOLO FACTURAS (FC) y SOLO WEB
# -----------------------------
df_fc = df_comp[df_comp["Comprobante"].astype(str).str.lower().str.startswith(prefixes_venta, na=False)].copy()
df_fc_web = df_fc[df_fc["is_web_line"].fillna(False)].copy()

# primera compra WEB por cliente
first_buy_web = df_fc_web.groupby("cliente_norm", as_index=False)["Fecha_de_Carga"].min()
first_buy_web["mes_primera_compra_web_p"] = first_buy_web["Fecha_de_Carga"].dt.to_period("M")
first_buy_web["mes_primera_compra_web"] = first_buy_web["mes_primera_compra_web_p"].astype(str)

nuevos_mes_web = (
    first_buy_web.groupby("mes_primera_compra_web", as_index=False)
                .agg(clientes_nuevos_web=("cliente_norm", "nunique"))
                .sort_values("mes_primera_compra_web")
)

# (opcional) canal informativo de esa primera compra (por canal_norm)
first_buy_web2 = first_buy_web.merge(
    df_fc_web[["cliente_norm", "Fecha_de_Carga", "canal_norm"]],
    on=["cliente_norm", "Fecha_de_Carga"],
    how="left"
)

nuevos_mes_web_canal = (
    first_buy_web2.groupby(["mes_primera_compra_web", "canal_norm"], as_index=False)
                .agg(clientes_nuevos_web=("cliente_norm", "nunique"))
                .sort_values(["mes_primera_compra_web", "clientes_nuevos_web"], ascending=[True, False])
)

# -----------------------------
# 5) TOP clientes por año (puede ser sobre todo o solo WEB)
#   - Te lo dejo en DOS versiones:
#     a) ALL: todos los comprobantes de venta (df_fc)
#     b) WEB: solo df_fc_web
# -----------------------------
cli_year_all = (
    df_fc.groupby(["anio", "cliente_norm"], as_index=False)
        .agg(
            neto_usd=("neto_usd_signed", "sum"),
            profit_usd=("profit_usd_signed", "sum"),
            comprobantes=(comp_key, "nunique"),
            unidades_netas=("unidades_netas", "sum"),
            canal_top=("canal_norm", lambda x: x.value_counts().index[0] if len(x.dropna()) else "na")
        )
)

cli_year_web = (
    df_fc_web.groupby(["anio", "cliente_norm"], as_index=False)
            .agg(
                neto_usd=("neto_usd_signed", "sum"),
                profit_usd=("profit_usd_signed", "sum"),
                comprobantes=(comp_key, "nunique"),
                unidades_netas=("unidades_netas", "sum"),
                canal_top=("canal_norm", lambda x: x.value_counts().index[0] if len(x.dropna()) else "na")
            )
)

def top_n(df_in, year, by, n=30):
    w = df_in[df_in["anio"].eq(year)].copy()
    return w.sort_values(by, ascending=False).head(n)

years_all = sorted([int(y) for y in cli_year_all["anio"].dropna().unique().tolist()])
years_web = sorted([int(y) for y in cli_year_web["anio"].dropna().unique().tolist()])

tops_all_profit = {y: top_n(cli_year_all, y, "profit_usd", 30) for y in years_all}
tops_all_neto   = {y: top_n(cli_year_all, y, "neto_usd", 30) for y in years_all}
tops_all_comp   = {y: top_n(cli_year_all, y, "comprobantes", 30) for y in years_all}

tops_web_profit = {y: top_n(cli_year_web, y, "profit_usd", 30) for y in years_web}
tops_web_neto   = {y: top_n(cli_year_web, y, "neto_usd", 30) for y in years_web}
tops_web_comp   = {y: top_n(cli_year_web, y, "comprobantes", 30) for y in years_web}

# Top global (ALL y WEB)
cli_all_all = (
    df_fc.groupby("cliente_norm", as_index=False)
        .agg(
            neto_usd=("neto_usd_signed", "sum"),
            profit_usd=("profit_usd_signed", "sum"),
            comprobantes=(comp_key, "nunique"),
            unidades_netas=("unidades_netas", "sum"),
            canal_top=("canal_norm", lambda x: x.value_counts().index[0] if len(x.dropna()) else "na"),
            primera_compra=("Fecha_de_Carga","min"),
            ultima_compra=("Fecha_de_Carga","max"),
        )
)
cli_all_all["dias_desde_ultima_compra"] = (df_fc["Fecha_de_Carga"].max() - cli_all_all["ultima_compra"]).dt.days

cli_all_web = (
    df_fc_web.groupby("cliente_norm", as_index=False)
            .agg(
                neto_usd=("neto_usd_signed", "sum"),
                profit_usd=("profit_usd_signed", "sum"),
                comprobantes=(comp_key, "nunique"),
                unidades_netas=("unidades_netas", "sum"),
                canal_top=("canal_norm", lambda x: x.value_counts().index[0] if len(x.dropna()) else "na"),
                primera_compra=("Fecha_de_Carga","min"),
                ultima_compra=("Fecha_de_Carga","max"),
            )
)
cli_all_web["dias_desde_ultima_compra"] = (df_fc_web["Fecha_de_Carga"].max() - cli_all_web["ultima_compra"]).dt.days

# -----------------------------
# 6) HIATOS (sobre WEB para coherencia con "cliente web")
# -----------------------------
GAP_MIN_MESES = 2  # ajustable

cli_mes = (
    df_fc_web.groupby(["cliente_norm", "mes_p"], as_index=False)
            .agg(
                neto_usd=("neto_usd_signed", "sum"),
                profit_usd=("profit_usd_signed", "sum"),
                compras=(comp_key, "nunique"),
                canal_top=("canal_norm", lambda x: x.value_counts().index[0] if len(x.dropna()) else "na"),
            )
)

cli_mes = cli_mes.sort_values(["cliente_norm", "mes_p"]).reset_index(drop=True)
cli_mes["mes_prev"] = cli_mes.groupby("cliente_norm")["mes_p"].shift(1)

cli_mes["mes_idx"] = cli_mes["mes_p"].apply(lambda p: p.year * 12 + p.month).astype("Int64")
cli_mes["mes_prev_idx"] = cli_mes["mes_prev"].apply(lambda p: p.year * 12 + p.month if pd.notna(p) else pd.NA).astype("Int64")

cli_mes["gap_meses"] = pd.Series(pd.NA, index=cli_mes.index, dtype="Int64")
mask_prev = cli_mes["mes_prev_idx"].notna()
cli_mes.loc[mask_prev, "gap_meses"] = (cli_mes.loc[mask_prev, "mes_idx"] - cli_mes.loc[mask_prev, "mes_prev_idx"]).astype("Int64")

hiatos = cli_mes[cli_mes["gap_meses"].fillna(0) >= GAP_MIN_MESES].copy()
hiatos["desde_mes"] = hiatos["mes_prev"].astype(str)
hiatos["hasta_mes"] = hiatos["mes_p"].astype(str)

hiatos_res = (
    hiatos.groupby("cliente_norm", as_index=False)
        .agg(
            hiato_max_meses=("gap_meses", "max"),
            cant_hiatos=("gap_meses", "size"),
            ultimo_hiato_desde=("desde_mes", "last"),
            ultimo_hiato_hasta=("hasta_mes", "last"),
        )
        .sort_values(["hiato_max_meses", "cant_hiatos"], ascending=False)
)

# -----------------------------
# 7) GRAFICOS (PNGs)
# -----------------------------
png_paths = []

def save_fig(name):
    path = os.path.join(output_path_base, name)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    png_paths.append(path)

# 7.1 Clientes nuevos WEB por mes
if not nuevos_mes_web.empty:
    x = pd.PeriodIndex(nuevos_mes_web["mes_primera_compra_web"], freq="M").to_timestamp()
    y = nuevos_mes_web["clientes_nuevos_web"].astype(float)

    plt.figure(figsize=(11, 4))
    plt.plot(x, y, marker="o")
    plt.title("Clientes nuevos WEB por mes (Condicion_de_Venta = contado / only-web*)")
    plt.xlabel("Mes")
    plt.ylabel("Clientes nuevos WEB")
    plt.grid(True, alpha=0.3)
    save_fig("clientes_nuevos_web_por_mes.png")

# 7.2 Nuevos WEB por mes por canal_norm (informativo)
if not nuevos_mes_web_canal.empty:
    tmp = nuevos_mes_web_canal.copy()
    tmp["mes_p"] = pd.PeriodIndex(tmp["mes_primera_compra_web"], freq="M")
    piv = tmp.pivot_table(index="mes_p", columns="canal_norm", values="clientes_nuevos_web", aggfunc="sum").fillna(0)
    piv = piv.sort_index()

    plt.figure(figsize=(11, 4.8))
    piv.plot(kind="bar", stacked=True, figsize=(11, 4.8))
    plt.title("Clientes nuevos WEB por mes — desagregado por canal (informativo)")
    plt.xlabel("Mes")
    plt.ylabel("Clientes nuevos WEB")
    save_fig("clientes_nuevos_web_mes_canal_stacked.png")

# 7.3 TOP WEB global por monto
topN = 20
top_web_monto = cli_all_web.sort_values("neto_usd", ascending=False).head(topN).copy()
if not top_web_monto.empty:
    top_web_monto = top_web_monto.sort_values("neto_usd", ascending=True)
    plt.figure(figsize=(11, 6))
    plt.barh(top_web_monto["cliente_norm"], top_web_monto["neto_usd"])
    plt.title(f"Top {topN} clientes WEB — Ventas netas USD (global)")
    plt.xlabel("USD neto (FC - NC)")
    plt.ylabel("Cliente")
    save_fig("top_clientes_web_monto_global.png")

# 7.4 TOP WEB por comprobantes
top_web_comp = cli_all_web.sort_values("comprobantes", ascending=False).head(topN).copy()
if not top_web_comp.empty:
    top_web_comp = top_web_comp.sort_values("comprobantes", ascending=True)
    plt.figure(figsize=(11, 6))
    plt.barh(top_web_comp["cliente_norm"], top_web_comp["comprobantes"])
    plt.title(f"Top {topN} clientes WEB — Cantidad de comprobantes (global)")
    plt.xlabel("Cantidad de comprobantes")
    plt.ylabel("Cliente")
    save_fig("top_clientes_web_comprobantes_global.png")

# -----------------------------
# 8) Export a Excel
# -----------------------------
out_xlsx = os.path.join(output_path_base, "clientes_analitica.xlsx")

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    df_comp.sort_values("Fecha_de_Carga").to_excel(writer, index=False, sheet_name="FACTURAS_BASE")

    nuevos_mes_web.to_excel(writer, index=False, sheet_name="NUEVOS_WEB_MES")
    nuevos_mes_web_canal.to_excel(writer, index=False, sheet_name="NUEVOS_WEB_MES_CANAL")

    cli_year_all.sort_values(["anio", "profit_usd"], ascending=[True, False]).to_excel(writer, index=False, sheet_name="CLIENTE_ANIO_ALL")
    cli_year_web.sort_values(["anio", "profit_usd"], ascending=[True, False]).to_excel(writer, index=False, sheet_name="CLIENTE_ANIO_WEB")

    # TOPS por año (ALL)
    for y in years_all:
        tops_all_profit[y].to_excel(writer, index=False, sheet_name=f"TOP_ALL_PROFIT_{y}")
        tops_all_neto[y].to_excel(writer, index=False, sheet_name=f"TOP_ALL_VENTAS_{y}")
        tops_all_comp[y].to_excel(writer, index=False, sheet_name=f"TOP_ALL_COMPROB_{y}")

    # TOPS por año (WEB)
    for y in years_web:
        tops_web_profit[y].to_excel(writer, index=False, sheet_name=f"TOP_WEB_PROFIT_{y}")
        tops_web_neto[y].to_excel(writer, index=False, sheet_name=f"TOP_WEB_VENTAS_{y}")
        tops_web_comp[y].to_excel(writer, index=False, sheet_name=f"TOP_WEB_COMPROB_{y}")

    # TOPS globales
    cli_all_all.sort_values("neto_usd", ascending=False).to_excel(writer, index=False, sheet_name="TOPS_GLOBAL_ALL")
    cli_all_web.sort_values("neto_usd", ascending=False).to_excel(writer, index=False, sheet_name="TOPS_GLOBAL_WEB")

    # Hiatos WEB
    hiatos.sort_values(["gap_meses", "cliente_norm"], ascending=[False, True]).to_excel(writer, index=False, sheet_name="HIATOS_WEB_DETALLE")
    hiatos_res.to_excel(writer, index=False, sheet_name="HIATOS_WEB_RESUMEN")

print("✅ Exportado:", out_xlsx)
print("📈 PNGs guardados:")
for p in png_paths:
    print(" -", p)
print("Años ALL:", years_all)
print("Años WEB:", years_web)
print("Clientes nuevos WEB (mes):", len(nuevos_mes_web))
print("Hiatos WEB detectados:", len(hiatos))


✅ Exportado: /content/gdrive/My Drive/Grapes/reporte_grapes/resultados/clientes_analitica.xlsx
📈 PNGs guardados:
 - /content/gdrive/My Drive/Grapes/reporte_grapes/resultados/clientes_nuevos_web_por_mes.png
 - /content/gdrive/My Drive/Grapes/reporte_grapes/resultados/clientes_nuevos_web_mes_canal_stacked.png
 - /content/gdrive/My Drive/Grapes/reporte_grapes/resultados/top_clientes_web_monto_global.png
 - /content/gdrive/My Drive/Grapes/reporte_grapes/resultados/top_clientes_web_comprobantes_global.png
Años ALL: [2023, 2024, 2025, 2026]
Años WEB: [2023, 2024, 2025, 2026]
Clientes nuevos WEB (mes): 32
Hiatos WEB detectados: 271


<Figure size 1100x480 with 0 Axes>

In [55]:
df_merged_usd.columns

Index(['Fecha_de_Carga', 'Deposito', 'Comprobante', 'Cantidad', 'Stock_Anterior', 'Stock_Actual', 'Codigo', 'Articulo', 'Marca', 'Condicion_de_Venta', 'Fecha', 'Cliente_Proveedor', 'provincia_estado_region', 'clase_fiscal',
       'tipos_de_documentos', 'numero_de_documento', 'id_detalle_cliente', 'Nro_cliente_proveedor', 'Precio_sinIVA', 'Total_sinIVA', 'Moneda', 'Tipo_de_Cambio', 'Costo_Ultima_Compra', 'Compr_Asoc', 'Tipo_Cambio_Compr', 'ID', 'Tipo ID',
       'ID_Articulo', 'Clase', 'k_comp', 'k_code', 'es_proxy_remito', 'k_comp_origen_remito', 'Comprobante_origen_remito', 'match_en_ven', 'cantidad_articulo_venta', 'total_neto_articulo_venta', 'total_final_articulo_venta',
       'costo_total_articulo_venta', 'interes_sin_imp_articulo_venta', 'impuestos_articulo_venta', 'profit_articulo_venta', 'profit_ii_articulo_venta', 'total_neto_articulo_venta_usd', 'costo_total_articulo_venta_usd',
       'interes_sin_imp_articulo_venta_usd', 'profit_ii_articulo_venta_usd', 'Codigo_costo', 'co

In [56]:
df_valorizacion_stock.columns

Index(['Codigo', 'Articulo', 'Cantidad', 'Costo_usd', 'valor_stock_final', 'Moneda', 'Deposito', 'Categoria', 'Sub_Categoria', 'Precio', 'Precio_Valorizado', 'coslis_curr_desc', 'Costo_PPP_Valorizado', 'Impuestos_Articulo', 'Costo_PPP',
       'Permite_Cant_Decimales', 'Codigo_key'],
      dtype='object')

#Anging STOCK + DASHBOARD


In [57]:
# ============================================================
# INVENTARIO UNIFICADO — DASHBOARD + REPOSICIÓN + ROTACIÓN + PERMANENCIA
# GRAPES MARKET
#
# AJUSTES:
# - Rotación solo con stock > 0
# - Rotación real 12m = Ventas_12m_pos / Stock_Total
# - Meses activos = 12 - meses_QS
# - mu_activo_12m
# - Días sin rotación con stock:
#     arranca en la última venta o última compra, la más reciente
# - FECHA_STOCK automática = última fecha real del archivo
# - Sin prints; exporta Excel
# ============================================================

import os
import itertools
import warnings
import math
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ─── PARÁMETROS ────────────────────────────────────────────
FECHA_STOCK               = None
Z_SERVICE                 = 2.05
N_MESES_BASE              = 12
CICLO_COMPRA              = 6.0
DIAS_SIN_ROTAR            = 90
DIAS_NUEVO_ATENCION       = 60
UMBRAL_CAIDA              = 0.50
K_SIGNAL                  = 2
N_BASE_SIGNAL             = 4
ABC_CORTE_A               = 0.80
ABC_CORTE_B               = 0.95
CLAVO_MESES               = {"A": 18, "B": 18, "C": 12}

PERM_NUEVO_DIAS           = 60
PERM_RECIENTE_DIAS        = 180
PERM_ENVEJECIENDO_DIAS    = 360
PERM_VIEJO_DIAS           = 540

LEAD_TIME_POR_MARCA = {
    "QNAP": 2.0,
    "LaCie": 2.0,
    "Seagate": 2.0,
    "Kingston": 2.0,
    "Western Digital": 2.0,
    "OWC": 2.0,
    "10GTeK": 2.0,
    "UGREEN": 2.5,
    "GMKtec": 2.0,
    "Terramaster": 2.0,
    "Adata": 2.0,
    "Asus": 2.0,
    "Gigabyte": 2.0,
    "Synology": 2.0,
}
LEAD_TIME_DEFAULT = 2.0

GRUPO_LOGISTICO = {
    "10GTEK": "CHINA_CONSOLIDADO",
    "UGREEN": "CHINA_CONSOLIDADO",
    "QNAP": "PROVEEDOR_SEPARADO",
    "LACIE": "PROVEEDOR_SEPARADO",
    "SEAGATE": "PROVEEDOR_SEPARADO",
    "OWC": "PROVEEDOR_SEPARADO",
}

CODIGOS_INTERNOS = {
    "151", "78", "152", "ENVIOS-PACK", "ART-VAR", "75", "850065783635",
    "76", "74", "77"
}

aging_ord = {
    "CLAVO": 0,
    "MUY_LENTO": 1,
    "LENTO": 2,
    "ATENCION": 3,
    "OK": 4,
    "SIN STOCK": 5,
    "SIN DEMANDA": 6,
    "NUEVO": 7
}
repo_ord = {
    "SIN STOCK — pedir ya": 0,
    "BAJO SS — zona de riesgo": 1,
    "PEDIR YA": 2,
    "PEDIR PRONTO — producto A": 3,
    "PEDIR PRONTO": 4,
    "PLANIFICAR": 5,
    "OK": 6,
    "SIN DEMANDA": 7
}
abc_ord = {"A": 0, "B": 1, "C": 2}

ACCIONES_AGING = {
    "CLAVO":     "Liquidar / bundle / evaluar baja",
    "MUY_LENTO": "Descuento >20% / bundle / promo ML",
    "LENTO":     "Revisar precio / destacar listing",
    "ATENCION":  "Monitorear / acciones preventivas",
    "OK":        "Normal",
    "NUEVO":     "Dar tiempo / monitorear",
    "SIN STOCK": "—",
    "SIN DEMANDA": "Revisar caso",
}

out_dir = output_path_base if "output_path_base" in globals() else "."
os.makedirs(out_dir, exist_ok=True)
out_file = os.path.join(out_dir, "inventario_unificado.xlsx")

# ─── HELPERS ───────────────────────────────────────────────
def fmt_n(n):
    try:
        return f"{int(round(float(n))):,}".replace(",", ".")
    except:
        return str(n)

def fmt_usd(v, cero="—"):
    try:
        v = float(v)
        if pd.isna(v):
            return cero
        return "US$ " + f"{v:,.0f}".replace(",", ".")
    except:
        return cero

def fmt_pct(v, total, decimals=1):
    try:
        r = float(v) / float(total) * 100
        return f"{r:.{decimals}f}%".replace(".", ",")
    except:
        return "—"

def norm_code(s):
    s = str(s).strip().upper()
    s = s.replace("–", "-").replace("—", "-").replace("‐", "-").replace("‒", "-")
    s = pd.Series([s]).str.replace(r"\s+", "-", regex=True).str.replace(r"-{2,}", "-", regex=True).iloc[0]
    return s

def group_logistico(marca):
    return GRUPO_LOGISTICO.get(str(marca).strip().upper(), "OTROS")

# ─── 1) NORMALIZAR HISTÓRICO ───────────────────────────────
if "df_merged_usd" not in globals():
    raise NameError("df_merged_usd no existe.")

df0 = df_merged_usd.copy()

requeridas = ["Fecha_de_Carga", "Comprobante", "Codigo", "Articulo", "Marca", "Clase", "Cantidad"]
faltantes = [c for c in requeridas if c not in df0.columns]
if faltantes:
    raise KeyError(f"Faltan columnas en df_merged_usd: {faltantes}")

df0["Fecha_de_Carga"] = pd.to_datetime(df0["Fecha_de_Carga"], errors="coerce", dayfirst=True)
df0 = df0[df0["Fecha_de_Carga"].notna()].copy()

for col in ["Comprobante", "Codigo", "Articulo", "Marca", "Clase"]:
    df0[col] = df0[col].astype(str).str.strip()

df0["Codigo"] = df0["Codigo"].map(norm_code)
df0["Comp_norm"] = df0["Comprobante"].astype(str).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
df0["Clase_norm"] = df0["Clase"].astype(str).str.lower().str.strip()
df0["Cantidad"] = pd.to_numeric(df0["Cantidad"], errors="coerce").fillna(0)

if "Stock_Anterior" in df0.columns:
    df0["Stock_Anterior"] = pd.to_numeric(df0["Stock_Anterior"], errors="coerce")
else:
    df0["Stock_Anterior"] = np.nan

if "costo_neto_unitario" in df0.columns:
    df0["costo_neto_unitario"] = pd.to_numeric(df0["costo_neto_unitario"], errors="coerce")
else:
    df0["costo_neto_unitario"] = np.nan

if "profit_ii_articulo_venta_usd" in df0.columns:
    df0["profit_ii_articulo_venta_usd"] = pd.to_numeric(df0["profit_ii_articulo_venta_usd"], errors="coerce")
else:
    df0["profit_ii_articulo_venta_usd"] = np.nan

df0 = df0[~df0["Codigo"].isin(CODIGOS_INTERNOS)].copy()

ultima_fecha = df0["Fecha_de_Carga"].max()
df0["AnoMes"] = df0["Fecha_de_Carga"].dt.to_period("M")
ultimo_mes = df0["AnoMes"].max()

if FECHA_STOCK is None:
    FECHA_STOCK = ultima_fecha.normalize()

# ─── 2) CLASIFICAR MOVIMIENTOS ─────────────────────────────
is_compra = df0["Clase_norm"].eq("01.factura")
pat_pres  = r"\b(?:pre|pres|presup|cot|cotiz)\b"
is_pres   = (
    df0["Comp_norm"].str.contains(pat_pres, na=False, regex=True) |
    df0["Clase_norm"].str.contains("presup", na=False)
)
is_venta  = (
    df0["Comp_norm"].str.startswith(("fc a", "fc b", "fc c"), na=False) &
    ~is_compra & ~is_pres
)
is_nc     = (
    df0["Comp_norm"].str.startswith(("nc a", "nc b", "nc c", "nc d"), na=False) &
    ~is_compra
)

df0["Tipo"] = np.select(
    [is_venta, is_nc, is_compra],
    ["venta", "devolucion", "compra"],
    default="otro"
)

df0["Cant_neta"] = np.where(
    df0["Tipo"] == "venta", df0["Cantidad"].abs(),
    np.where(df0["Tipo"] == "devolucion", -df0["Cantidad"].abs(), 0)
)

df0["Cant_compra"] = np.where(df0["Tipo"] == "compra", df0["Cantidad"].abs(), 0)
df0["Cant_venta_pos"] = np.where(df0["Tipo"] == "venta", df0["Cantidad"].abs(), 0)

# ─── 3) QUIEBRE DE STOCK ───────────────────────────────────
sa = df0["Stock_Anterior"].fillna(-999999)
cq = df0["Cantidad"].abs().fillna(-888888)
df0["QS_flag"] = (df0["Tipo"] == "venta") & (sa == cq)

qs_por_mes_classic = (
    df0[df0["QS_flag"]][["Codigo", "AnoMes"]]
    .drop_duplicates()
    .assign(tuvo_QS=True)
)

stock_min_mes = (
    df0.groupby(["Codigo", "AnoMes"], as_index=False)["Stock_Anterior"]
    .min()
    .rename(columns={"Stock_Anterior": "stock_min_mes"})
)

codigos_con_ventas = set(df0.loc[df0["Tipo"] == "venta", "Codigo"].unique())

qs_por_mes_sinstock = (
    stock_min_mes[
        (stock_min_mes["stock_min_mes"] <= 0) &
        (stock_min_mes["Codigo"].isin(codigos_con_ventas))
    ][["Codigo", "AnoMes"]]
    .drop_duplicates()
    .assign(tuvo_QS=True)
)

qs_por_mes = (
    pd.concat([qs_por_mes_classic, qs_por_mes_sinstock], ignore_index=True)
    .drop_duplicates(["Codigo", "AnoMes"])
    .reset_index(drop=True)
)

# ─── 4) CATÁLOGO ───────────────────────────────────────────
productos = (
    df0[["Codigo", "Articulo", "Marca"]]
    .drop_duplicates("Codigo", keep="last")
    .copy()
)
productos["Grupo_logistico"] = productos["Marca"].map(group_logistico)

# ─── 5) PIVOT MENSUAL ──────────────────────────────────────
m_inicio   = ultimo_mes - (N_MESES_BASE - 1)
meses_base = pd.period_range(m_inicio, ultimo_mes, freq="M")
m_signal   = sorted(pd.period_range(end=ultimo_mes, periods=K_SIGNAL, freq="M").tolist())
m_base     = sorted(pd.period_range(end=ultimo_mes, periods=K_SIGNAL + N_BASE_SIGNAL, freq="M").tolist())[:-K_SIGNAL]

mensual = (
    df0[df0["Cant_neta"] != 0]
    .groupby(["Codigo", "AnoMes"], as_index=False)
    .agg(Cant_mes=("Cant_neta", "sum"))
)

malla = pd.DataFrame(
    itertools.product(productos["Codigo"].unique(), meses_base),
    columns=["Codigo", "AnoMes"]
)

dfm = (
    malla
    .merge(mensual, on=["Codigo", "AnoMes"], how="left")
    .merge(qs_por_mes, on=["Codigo", "AnoMes"], how="left")
)

dfm["Cant_mes"] = dfm["Cant_mes"].fillna(0)
dfm["tuvo_QS"]  = dfm["tuvo_QS"].fillna(False).infer_objects(copy=False)

pivot = dfm.pivot_table(index="Codigo", columns="AnoMes", values="Cant_mes", aggfunc="sum", fill_value=0)

# ─── 6) μ, σ, CV ───────────────────────────────────────────
def _stats(grupo):
    sin_qs = grupo.loc[~grupo["tuvo_QS"], "Cant_mes"]
    mu = float(grupo["Cant_mes"].mean())
    sigma = float(sin_qs.std(ddof=0)) if len(sin_qs) >= 2 else float(grupo["Cant_mes"].std(ddof=0))
    sigma = max(sigma, 0.0)
    return pd.Series({
        "mu": round(mu, 2),
        "sigma": round(sigma, 2),
        "n_meses_venta": int((grupo["Cant_mes"] > 0).sum()),
        "n_meses_qs": int(grupo["tuvo_QS"].sum()),
    })

stats = (
    dfm.groupby("Codigo", group_keys=False)
    .apply(_stats, include_groups=False)
    .reset_index()
)

mediana_mes = dfm.groupby("Codigo")["Cant_mes"].median().rename("mediana")

stats["CV"] = np.where(stats["mu"] > 0, (stats["sigma"] / stats["mu"]).round(2), np.nan)

def _patron(cv):
    if pd.isna(cv):
        return "SIN DEMANDA"
    if cv < 0.5:
        return "ESTABLE"
    if cv <= 1.0:
        return "VARIABLE"
    return "ERRATICO"

stats["Patron_demanda"] = stats["CV"].apply(_patron)

def cols_ok(meses):
    return [m for m in meses if m in pivot.columns]

csig  = cols_ok(m_signal)
cbase = cols_ok(m_base)

# ─── 7) DEMANDA 12M / COMPRAS 12M / ABC ────────────────────
ventas_12m = (
    df0[(df0["AnoMes"] >= m_inicio)]
    .groupby("Codigo", as_index=False)
    .agg(
        Ventas_12m_netas=("Cant_neta", "sum"),
        Ventas_12m_pos=("Cant_venta_pos", "sum"),
        Compras_12m=("Cant_compra", "sum")
    )
)

abc_base = (
    df0[(df0["Tipo"] == "venta") & (df0["AnoMes"] >= m_inicio)]
    .groupby("Codigo")["profit_ii_articulo_venta_usd"]
    .sum()
    .clip(lower=0)
    .rename("Margen_12m")
)

abc_df = abc_base.reset_index().sort_values("Margen_12m", ascending=False)
total_m = float(abc_df["Margen_12m"].sum())
abc_df["Pct_acum"] = abc_df["Margen_12m"].cumsum() / (total_m if total_m > 0 else np.nan)

def _abc(p):
    if pd.isna(p):
        return "C"
    if p <= ABC_CORTE_A:
        return "A"
    if p <= ABC_CORTE_B:
        return "B"
    return "C"

abc_df["ABC"] = abc_df["Pct_acum"].apply(_abc)
abc_map = abc_df.set_index("Codigo")["ABC"].to_dict()
margen_map = abc_df.set_index("Codigo")["Margen_12m"].to_dict()

n_a = int((abc_df["ABC"] == "A").sum())
n_b = int((abc_df["ABC"] == "B").sum())
n_c = int((abc_df["ABC"] == "C").sum())

# ─── 8) TABLA BASE ─────────────────────────────────────────
met = productos.set_index("Codigo").copy()
met = met.join(stats.set_index("Codigo")[["mu", "sigma", "CV", "Patron_demanda", "n_meses_venta", "n_meses_qs"]])
met = met.join(ventas_12m.set_index("Codigo")[["Ventas_12m_netas", "Ventas_12m_pos", "Compras_12m"]], how="left")

met["Ventas_12m_netas"] = met["Ventas_12m_netas"].fillna(0)
met["Ventas_12m_pos"] = met["Ventas_12m_pos"].fillna(0)
met["Compras_12m"] = met["Compras_12m"].fillna(0)
met["mediana"] = mediana_mes.reindex(met.index).fillna(0)
met["Margen_12m"] = met.index.map(margen_map).fillna(0)
met["ABC"] = met.index.map(abc_map).fillna("C")
met["Lead_time"] = met["Marca"].map(LEAD_TIME_POR_MARCA).fillna(LEAD_TIME_DEFAULT)

met["mu_ef"] = np.where(
    met["Patron_demanda"] == "ERRATICO",
    met["mediana"],
    met["mu"].fillna(0)
)

met["Meses_activos_12m"] = (N_MESES_BASE - met["n_meses_qs"].fillna(0)).clip(lower=1)

met["mu_activo_12m"] = np.where(
    met["Meses_activos_12m"] > 0,
    (met["Ventas_12m_pos"] / met["Meses_activos_12m"]).round(2),
    0
)

if csig and cbase:
    avg_sig  = pivot[csig].mean(axis=1)
    avg_base = pivot[cbase].mean(axis=1)
    met["Avg_signal_2m"] = avg_sig.reindex(met.index).round(2)
    met["Avg_base_4m"] = avg_base.reindex(met.index).round(2)
    met["Delta_demanda_pct"] = np.where(
        avg_base.reindex(met.index) > 0,
        ((avg_sig.reindex(met.index) - avg_base.reindex(met.index)) / avg_base.reindex(met.index) * 100).round(1),
        np.nan
    )
else:
    met["Avg_signal_2m"] = np.nan
    met["Avg_base_4m"] = np.nan
    met["Delta_demanda_pct"] = np.nan

# ─── 9) STOCK Y VALUACIÓN DESDE df_valorizacion_stock ──────
if "df_valorizacion_stock" not in globals():
    raise NameError("df_valorizacion_stock no existe. Ejecutá la celda de inicialización primero.")

_val = df_valorizacion_stock.copy()

if "Codigo_key" in _val.columns:
    _val["_key"] = _val["Codigo_key"].astype(str).map(norm_code)
elif "Codigo" in _val.columns:
    _val["_key"] = _val["Codigo"].astype(str).map(norm_code)
else:
    raise KeyError("df_valorizacion_stock no tiene ni 'Codigo_key' ni 'Codigo'.")

if "Moneda" in _val.columns:
    _val["_moneda"] = _val["Moneda"].astype(str).str.strip().str.upper()
elif "coslis_curr_desc" in _val.columns:
    _val["_moneda"] = _val["coslis_curr_desc"].astype(str).str.strip().str.upper()
else:
    raise KeyError("df_valorizacion_stock no tiene ni 'Moneda' ni 'coslis_curr_desc'.")

for c in ["Cantidad", "Costo_usd", "Deposito"]:
    if c not in _val.columns:
        raise KeyError(f"Falta columna crítica en df_valorizacion_stock: {c}")

_val["Cantidad"] = pd.to_numeric(_val["Cantidad"], errors="coerce").fillna(0)
_val["Costo_usd"] = pd.to_numeric(_val["Costo_usd"], errors="coerce").fillna(0)
_val["Deposito"] = _val["Deposito"].astype(str).str.strip()

_val = _val[~_val["_key"].isin(CODIGOS_INTERNOS)].copy()
_val_usd = _val[_val["_moneda"].isin(["USD", "US$"])].copy()
_val_usd["Valor_stock_USD_calc"] = _val_usd["Cantidad"] * _val_usd["Costo_usd"]

_by_dep = (
    _val_usd
    .groupby(["_key", "Deposito"], as_index=False)
    .agg(
        Stock_dep=("Cantidad", "sum"),
        Valor_dep=("Valor_stock_USD_calc", "sum"),
    )
)
_by_dep["Stock_dep"] = _by_dep["Stock_dep"].clip(lower=0)

_full = (
    _by_dep[_by_dep["Deposito"].str.upper() == "FULL"]
    .set_index("_key")[["Stock_dep", "Valor_dep"]]
    .rename(columns={"Stock_dep": "Stock_FULL", "Valor_dep": "Valor_FULL"})
)

_cerr = (
    _by_dep[_by_dep["Deposito"].str.upper() == "CERRITO"]
    .set_index("_key")[["Stock_dep", "Valor_dep"]]
    .rename(columns={"Stock_dep": "Stock_Cerrito", "Valor_dep": "Valor_Cerrito"})
)

_stock_ref = (
    _val_usd
    .groupby("_key", as_index=False)
    .agg(
        Stock_Total=("Cantidad", "sum"),
        Costo_unit_USD=("Costo_usd", "mean"),
        Valor_stock_USD=("Valor_stock_USD_calc", "sum"),
    )
)

_stock_ref["Stock_Total"] = _stock_ref["Stock_Total"].clip(lower=0)

_stock_ref = (
    _stock_ref.set_index("_key")
    .join(_full, how="left")
    .join(_cerr, how="left")
)

for c in ["Stock_FULL", "Valor_FULL", "Stock_Cerrito", "Valor_Cerrito"]:
    _stock_ref[c] = _stock_ref[c].fillna(0)

met["Stock_Total"] = _stock_ref["Stock_Total"].reindex(met.index).fillna(0)
met["Costo_unit_USD"] = _stock_ref["Costo_unit_USD"].reindex(met.index)
met["Valor_stock_USD"] = _stock_ref["Valor_stock_USD"].reindex(met.index).fillna(0)
met["Stock_FULL"] = _stock_ref["Stock_FULL"].reindex(met.index).fillna(0)
met["Valor_FULL"] = _stock_ref["Valor_FULL"].reindex(met.index).fillna(0)
met["Stock_Cerrito"] = _stock_ref["Stock_Cerrito"].reindex(met.index).fillna(0)
met["Valor_Cerrito"] = _stock_ref["Valor_Cerrito"].reindex(met.index).fillna(0)

# ─── 10) FECHAS DE ÚLTIMA VENTA Y ÚLTIMA COMPRA ────────────
ult_venta = (
    df0[df0["Tipo"] == "venta"]
    .groupby("Codigo")["Fecha_de_Carga"]
    .max()
    .rename("Ultima_venta")
)
ult_compra = (
    df0[df0["Tipo"] == "compra"]
    .groupby("Codigo")["Fecha_de_Carga"]
    .max()
    .rename("Ultima_compra")
)

met["Ultima_venta"] = ult_venta.reindex(met.index)
met["Ultima_compra"] = ult_compra.reindex(met.index)

# comercial puro
met["Dias_sin_venta"] = np.where(
    met["Ultima_venta"].notna(),
    np.maximum((FECHA_STOCK - met["Ultima_venta"]).dt.days, 0),
    np.nan
)

# permanencia compra
met["Dias_desde_ultima_compra"] = np.where(
    met["Ultima_compra"].notna(),
    np.maximum((FECHA_STOCK - met["Ultima_compra"]).dt.days, 0),
    np.nan
)

# rotación con stock: si hubo recompra después de la última venta, reinicia reloj
met["Fecha_inicio_rotacion_stock"] = met["Ultima_venta"]

mask_recompra_mas_reciente = (
    met["Ultima_compra"].notna() &
    (
        met["Ultima_venta"].isna() |
        (met["Ultima_compra"] > met["Ultima_venta"])
    )
)

met.loc[mask_recompra_mas_reciente, "Fecha_inicio_rotacion_stock"] = met.loc[
    mask_recompra_mas_reciente, "Ultima_compra"
]

met["Dias_sin_rotacion_stock"] = np.where(
    (met["Stock_Total"] > 0) & met["Fecha_inicio_rotacion_stock"].notna(),
    np.maximum((FECHA_STOCK - met["Fecha_inicio_rotacion_stock"]).dt.days, 0),
    np.nan
)

# ─── 11) REPOSICIÓN ────────────────────────────────────────
met["SS"] = np.where(
    met["mu_ef"] > 0,
    (Z_SERVICE * met["sigma"].fillna(0) * np.sqrt(met["Lead_time"])).round(1),
    0.0
)

met["ROP"] = np.where(
    met["mu_ef"] > 0,
    (met["mu_ef"] * met["Lead_time"] + met["SS"]).round(1),
    0.0
)

met["Stock_objetivo"] = np.where(
    met["mu_ef"] > 0,
    (met["mu_ef"] * (met["Lead_time"] + CICLO_COMPRA) + met["SS"]).round(1),
    0.0
)

met["Stock_al_llegar"] = (met["Stock_Total"] - met["mu_ef"] * met["Lead_time"]).round(1)
met["Ruptura_en_transit"] = met["Stock_al_llegar"] < met["SS"]

met["Cobertura_bruta_meses"] = np.where(
    met["mu_ef"] > 0,
    (met["Stock_Total"] / met["mu_ef"]).round(1),
    np.inf
)

met["Cobertura_sobre_SS"] = np.where(
    met["mu_ef"] > 0,
    ((met["Stock_Total"] - met["SS"]) / met["mu_ef"]).round(1),
    np.inf
)

met["Q_pedir"] = np.where(
    met["mu_ef"] > 0,
    np.clip(np.ceil(met["Stock_objetivo"]) - met["Stock_Total"], 0, None).astype(int),
    0
)

met["Valor_Q_USD"] = (met["Q_pedir"] * met["Costo_unit_USD"]).round(2)

# ─── 12) ROTACIÓN ──────────────────────────────────────────
met["Rotacion_12m_unid"] = np.where(
    met["Stock_Total"] > 0,
    (met["Ventas_12m_pos"] / met["Stock_Total"]).round(2),
    np.nan
)

met["Cobertura_activa_meses"] = np.where(
    met["mu_activo_12m"] > 0,
    (met["Stock_Total"] / met["mu_activo_12m"]).round(1),
    np.inf
)

def clasif_rotacion(row):
    stock = float(row["Stock_Total"] or 0)
    rot = row["Rotacion_12m_unid"]
    ventas = float(row["Ventas_12m_pos"] or 0)

    if stock <= 0:
        return "SIN STOCK"
    if ventas <= 0:
        return "SIN DEMANDA"
    if pd.isna(rot):
        return "SIN DEMANDA"
    if rot >= 4:
        return "ALTA"
    elif rot >= 2:
        return "MEDIA"
    elif rot >= 1:
        return "BAJA"
    return "MUY_BAJA"

met["Clasif_rotacion"] = met.apply(clasif_rotacion, axis=1)

# ─── 13) PERMANENCIA POR COMPRA ────────────────────────────
def clasif_permanencia(row):
    stock = float(row["Stock_Total"] or 0)
    dcomp = row["Dias_desde_ultima_compra"]

    if stock <= 0:
        return "SIN STOCK"
    if pd.isna(dcomp):
        return "SIN COMPRA"
    if dcomp <= PERM_NUEVO_DIAS:
        return "NUEVO"
    if dcomp <= PERM_RECIENTE_DIAS:
        return "RECIENTE"
    if dcomp <= PERM_ENVEJECIENDO_DIAS:
        return "ENVEJECIENDO"
    if dcomp <= PERM_VIEJO_DIAS:
        return "VIEJO"
    return "CRITICO"

met["Permanencia_compra"] = met.apply(clasif_permanencia, axis=1)

# ─── 14) AGING DE COBERTURA ────────────────────────────────
def clasificar_aging(row):
    stk = float(row["Stock_Total"])
    mu = float(row["mu_ef"] or 0)
    cob = float(row["Cobertura_sobre_SS"]) if np.isfinite(row["Cobertura_sobre_SS"]) else 999
    dias_rot = float(row["Dias_sin_rotacion_stock"] or 0) if pd.notna(row["Dias_sin_rotacion_stock"]) else 0
    abc = str(row.get("ABC", "C"))
    clavo = CLAVO_MESES.get(abc, 18)

    if stk <= 0:
        return "SIN STOCK"
    if mu <= 0:
        if dias_rot >= 365:
            return "CLAVO"
        if dias_rot >= 270:
            return "MUY_LENTO"
        if dias_rot >= 180:
            return "LENTO"
        if dias_rot >= DIAS_NUEVO_ATENCION:
            return "ATENCION"
        return "NUEVO"

    if cob > clavo:
        return "CLAVO"
    elif cob > clavo * 0.67:
        return "MUY_LENTO"
    elif cob > 9:
        return "LENTO"
    elif cob > 6:
        return "ATENCION"
    else:
        return "OK"

met["Tramo_aging"] = met.apply(clasificar_aging, axis=1)
met["Accion_aging"] = met["Tramo_aging"].map(ACCIONES_AGING).fillna("—")

# ─── 15) ESTADO REPOSICIÓN ─────────────────────────────────
def _estado(row):
    mu = float(row["mu_ef"] or 0)
    stk = float(row["Stock_Total"])
    rop = float(row["ROP"] or 0)
    ss  = float(row["SS"] or 0)
    cob = float(row["Cobertura_sobre_SS"]) if np.isfinite(row["Cobertura_sobre_SS"]) else 999
    abc = str(row.get("ABC", "C"))

    if mu <= 0:
        return "SIN DEMANDA"
    if stk <= 0:
        return "SIN STOCK — pedir ya"
    if stk <= ss:
        return "BAJO SS — zona de riesgo"
    if stk <= rop or bool(row["Ruptura_en_transit"]):
        return "PEDIR YA"
    if abc == "A" and cob <= 3.0:
        return "PEDIR PRONTO — producto A"
    if cob <= 2.0:
        return "PEDIR PRONTO"
    if cob <= CICLO_COMPRA:
        return "PLANIFICAR"
    return "OK"

met["Estado_repo"] = met.apply(_estado, axis=1)

met.loc[met["Tramo_aging"].isin(["CLAVO", "MUY_LENTO"]), "Q_pedir"] = 0
met.loc[met["Tramo_aging"].isin(["CLAVO", "MUY_LENTO"]), "Valor_Q_USD"] = 0

met["Accion_sugerida"] = met["Accion_aging"]
met.loc[
    met["Estado_repo"].isin(["SIN STOCK — pedir ya", "BAJO SS — zona de riesgo", "PEDIR YA"]),
    "Accion_sugerida"
] = "REPONER YA — riesgo de ruptura"

# ─── 16) FLAGS ─────────────────────────────────────────────
qs_ult = set(qs_por_mes[qs_por_mes["AnoMes"].isin(m_signal)]["Codigo"].unique())
met["QS_en_signal"] = met.index.isin(qs_ult)

qs_6m = set(
    qs_por_mes[
        qs_por_mes["AnoMes"].isin(sorted(pd.period_range(end=ultimo_mes, periods=6, freq="M").tolist()))
    ]["Codigo"].unique()
)
met["QS_ultimos_6m"] = met.index.isin(qs_6m)

met["Flag_sin_rotacion"] = (
    (met["Stock_Total"] > 0) &
    (met["Dias_sin_rotacion_stock"].fillna(999) >= DIAS_SIN_ROTAR) &
    (~met["QS_ultimos_6m"])
)

met["Flag_caida_demanda"] = (
    (met["Avg_base_4m"].fillna(0) > 0) &
    (met["Delta_demanda_pct"].fillna(0) <= -(UMBRAL_CAIDA * 100)) &
    (~met["QS_en_signal"])
)

met["Flag_cayo_a_cero"] = (
    (met["Avg_base_4m"].fillna(0) > 0) &
    (met["Avg_signal_2m"].fillna(1) == 0) &
    (~met["QS_en_signal"])
)

def _alerta(row):
    if row["Flag_cayo_a_cero"]:
        return f"CAYÓ A CERO (venía {row['Avg_base_4m']:.1f}/m)"
    if row["Flag_caida_demanda"]:
        pct = abs(float(row["Delta_demanda_pct"] or 0))
        return f"CAÍDA {pct:.0f}% ({row['Avg_base_4m']:.1f}→{row['Avg_signal_2m']:.1f}/m)"
    return ""

met["Alerta_demanda"] = met.apply(_alerta, axis=1)

# ─── 17) TABLA FINAL ───────────────────────────────────────
met = met.reset_index()
met = met[(met["Stock_Total"] > 0) | (met["mu"].fillna(0) > 0)].copy()

met["_ao"]  = met["Tramo_aging"].map(aging_ord).fillna(9)
met["_eo"]  = met["Estado_repo"].map(repo_ord).fillna(9)
met["_abc"] = met["ABC"].map(abc_ord).fillna(9)

met = met.sort_values(
    ["_eo", "_ao", "_abc", "Valor_stock_USD"],
    ascending=[True, True, True, False]
).drop(columns=["_ao", "_eo", "_abc"])

cols_out = [
    "Codigo", "Articulo", "Marca", "Grupo_logistico", "ABC",
    "Patron_demanda", "Clasif_rotacion", "Permanencia_compra",
    "Tramo_aging", "Estado_repo", "Accion_sugerida",
    "Stock_Total", "Stock_FULL", "Stock_Cerrito",
    "Ventas_12m_netas", "Ventas_12m_pos", "Compras_12m",
    "Meses_activos_12m", "mu_activo_12m",
    "SS", "ROP", "Stock_objetivo", "Cobertura_bruta_meses", "Cobertura_activa_meses", "Cobertura_sobre_SS",
    "mu_ef", "sigma", "CV", "mediana", "mu",
    "Avg_signal_2m", "Avg_base_4m", "Delta_demanda_pct",
    "n_meses_venta", "n_meses_qs",
    "Costo_unit_USD", "Valor_stock_USD", "Valor_FULL", "Valor_Cerrito", "Margen_12m",
    "Lead_time", "Stock_al_llegar", "Ruptura_en_transit", "Q_pedir", "Valor_Q_USD",
    "Rotacion_12m_unid",
    "Ultima_venta", "Dias_sin_venta",
    "Ultima_compra", "Dias_desde_ultima_compra",
    "Fecha_inicio_rotacion_stock", "Dias_sin_rotacion_stock",
    "QS_ultimos_6m", "QS_en_signal",
    "Flag_sin_rotacion", "Flag_caida_demanda", "Flag_cayo_a_cero", "Alerta_demanda",
]
cols_out = [c for c in cols_out if c in met.columns]
plan_out = met[cols_out].copy()

# ─── 18) SOLAPAS OPERATIVAS ────────────────────────────────
a_liquidar = plan_out[
    plan_out["Tramo_aging"].isin(["CLAVO", "MUY_LENTO"]) &
    (plan_out["Stock_Total"] > 0)
].copy()

a_atender = plan_out[
    plan_out["Tramo_aging"].isin(["LENTO", "ATENCION"]) &
    (plan_out["Stock_Total"] > 0)
].copy()

pedir_ya = plan_out[
    plan_out["Estado_repo"].isin(["SIN STOCK — pedir ya", "BAJO SS — zona de riesgo", "PEDIR YA"])
].copy()

pedir_pron = plan_out[
    plan_out["Estado_repo"].isin(["PEDIR PRONTO — producto A", "PEDIR PRONTO"])
].copy()

sin_rotacion = plan_out[
    (plan_out["Flag_sin_rotacion"] == True) &
    (plan_out["Stock_Total"] > 0)
].sort_values(["Dias_sin_rotacion_stock", "Valor_stock_USD"], ascending=[False, False]).copy()

caida = plan_out[
    (plan_out["Stock_Total"] > 0) &
    ((plan_out["Flag_caida_demanda"] == True) | (plan_out["Flag_cayo_a_cero"] == True))
].sort_values("Delta_demanda_pct").copy()

rotacion = plan_out[
    plan_out["Stock_Total"] > 0
][
    [
        "Codigo", "Articulo", "Marca", "Grupo_logistico", "ABC",
        "Stock_Total", "Ventas_12m_pos", "Meses_activos_12m", "mu_activo_12m", "mu_ef",
        "Cobertura_bruta_meses", "Cobertura_activa_meses",
        "Rotacion_12m_unid", "Clasif_rotacion",
        "n_meses_qs", "QS_ultimos_6m",
        "Valor_stock_USD",
        "Ultima_venta", "Dias_sin_venta",
        "Ultima_compra", "Fecha_inicio_rotacion_stock", "Dias_sin_rotacion_stock"
    ]
].copy().sort_values(["Clasif_rotacion", "Valor_stock_USD"], ascending=[True, False])

permanencia = plan_out[
    plan_out["Stock_Total"] > 0
][
    [
        "Codigo", "Articulo", "Marca", "Grupo_logistico", "ABC",
        "Stock_Total", "Valor_stock_USD",
        "Ultima_compra", "Dias_desde_ultima_compra",
        "Ultima_venta", "Dias_sin_venta",
        "Fecha_inicio_rotacion_stock", "Dias_sin_rotacion_stock",
        "Permanencia_compra",
        "Compras_12m", "Ventas_12m_pos"
    ]
].copy().sort_values(["Dias_desde_ultima_compra", "Valor_stock_USD"], ascending=[False, False])

sin_venta_larga = plan_out[
    (plan_out["Stock_Total"] > 0) &
    (plan_out["Dias_sin_rotacion_stock"].fillna(999999) >= DIAS_SIN_ROTAR)
][
    [
        "Codigo", "Articulo", "Marca", "Grupo_logistico", "ABC",
        "Stock_Total", "Valor_stock_USD",
        "Ultima_venta", "Dias_sin_venta",
        "Ultima_compra", "Fecha_inicio_rotacion_stock", "Dias_sin_rotacion_stock",
        "Dias_desde_ultima_compra", "Permanencia_compra",
        "Tramo_aging", "Estado_repo"
    ]
].copy().sort_values(["Dias_sin_rotacion_stock", "Valor_stock_USD"], ascending=[False, False])

# ─── 19) RESÚMENES ─────────────────────────────────────────
res_tramo = (
    plan_out[plan_out["Stock_Total"] > 0]
    .groupby("Tramo_aging", as_index=False)
    .agg(
        Productos=("Codigo", "count"),
        Stock_uds=("Stock_Total", "sum"),
        Valor_USD=("Valor_stock_USD", lambda x: round(x.sum(), 0))
    )
)
res_tramo["_o"] = res_tramo["Tramo_aging"].map(aging_ord).fillna(9)
res_tramo = res_tramo.sort_values("_o").drop(columns=["_o"])

res_abc = (
    plan_out.groupby(["ABC", "Tramo_aging"], as_index=False)
    .agg(
        Productos=("Codigo", "count"),
        Margen_12m=("Margen_12m", "sum"),
        Stock_uds=("Stock_Total", "sum"),
        Valor_USD=("Valor_stock_USD", lambda x: round(x.sum(), 0))
    )
)
res_abc["_ao"] = res_abc["Tramo_aging"].map(aging_ord).fillna(9)
res_abc["_abc"] = res_abc["ABC"].map(abc_ord).fillna(9)
res_abc = res_abc.sort_values(["_abc", "_ao"]).drop(columns=["_ao", "_abc"])

res_marca = (
    plan_out[plan_out["Stock_Total"] > 0]
    .groupby(["Marca", "Tramo_aging"], as_index=False)
    .agg(
        Productos=("Codigo", "count"),
        Stock_uds=("Stock_Total", "sum"),
        Valor_USD=("Valor_stock_USD", lambda x: round(x.sum(), 0))
    )
)
res_marca["_o"] = res_marca["Tramo_aging"].map(aging_ord).fillna(9)
res_marca = res_marca.sort_values(["Marca", "_o"]).drop(columns=["_o"])

res_repo_marca = (
    plan_out[(plan_out["mu_ef"].fillna(0) > 0) & (plan_out["Stock_Total"] > 0)]
    .groupby(["Marca", "Estado_repo"], as_index=False)
    .agg(
        Productos=("Codigo", "count"),
        Q_total=("Q_pedir", "sum"),
        Valor_Q_USD=("Valor_Q_USD", "sum")
    )
)
res_repo_marca["_o"] = res_repo_marca["Estado_repo"].map(repo_ord).fillna(9)
res_repo_marca = res_repo_marca.sort_values(["Marca", "_o"]).drop(columns=["_o"])

stock_x_marca = (
    plan_out[plan_out["Stock_Total"] > 0]
    .groupby("Marca", as_index=False)
    .agg(
        SKUs=("Codigo", "count"),
        Stock_uds=("Stock_Total", "sum"),
        Valor_USD=("Valor_stock_USD", lambda x: round(x.sum(), 0)),
        Valor_FULL=("Valor_FULL", lambda x: round(x.sum(), 0)),
        Valor_Cerrito=("Valor_Cerrito", lambda x: round(x.sum(), 0))
    )
    .sort_values("Valor_USD", ascending=False)
)

rot_x_grupo = (
    plan_out[plan_out["Stock_Total"] > 0]
    .groupby("Grupo_logistico", as_index=False)
    .agg(
        SKUs=("Codigo", "count"),
        Valor_stock=("Valor_stock_USD", "sum"),
        Ventas_12m_pos=("Ventas_12m_pos", "sum"),
        Compras_12m=("Compras_12m", "sum"),
        Valor_Q_USD=("Valor_Q_USD", "sum")
    )
)

val_total_marcas = float(stock_x_marca["Valor_USD"].sum() or 1)

val_total = float(_stock_ref["Valor_stock_USD"].sum() or 0)
val_total_full = float(_stock_ref["Valor_FULL"].sum() or 0)
val_total_cerrito = float(_stock_ref["Valor_Cerrito"].sum() or 0)
val_clavo = float(a_liquidar["Valor_stock_USD"].sum() or 0)
pct_riesgo = round(val_clavo / val_total * 100, 1) if val_total > 0 else 0
pct_ok = round(
    plan_out[(plan_out["Tramo_aging"] == "OK") & (plan_out["Stock_Total"] > 0)].shape[0] /
    max(len(plan_out[plan_out["Stock_Total"] > 0]), 1) * 100,
    0
)
n_pedir_ya = len(pedir_ya)
n_pedir_pron = len(pedir_pron)
n_sin_rot = len(sin_rotacion)
n_caida = len(caida)
val_q_ya = float(pedir_ya["Valor_Q_USD"].sum() or 0)
val_q_total = float(plan_out["Valor_Q_USD"].sum() or 0)
n_a_riesgo = len(
    plan_out[
        (plan_out["ABC"] == "A") &
        (plan_out["Stock_Total"] > 0) &
        plan_out["Estado_repo"].isin(["SIN STOCK — pedir ya", "BAJO SS — zona de riesgo", "PEDIR YA"])
    ]
)
n_qs = int(plan_out["QS_ultimos_6m"].sum())

# ─── 20) EXPORT ────────────────────────────────────────────
aging_fill = {
    "CLAVO": "C0392B", "MUY_LENTO": "F1948A", "LENTO": "FFD9A0",
    "ATENCION": "FFF4A0", "OK": "D4F5D4", "NUEVO": "D6EAF8",
    "SIN STOCK": "F0F0F0", "SIN DEMANDA": "F0F0F0",
}
repo_fill = {
    "SIN STOCK — pedir ya": "C0392B", "BAJO SS — zona de riesgo": "F1948A",
    "PEDIR YA": "FF6B35", "PEDIR PRONTO — producto A": "F39C12",
    "PEDIR PRONTO": "FFD9A0", "PLANIFICAR": "FFF4A0",
    "OK": "D4F5D4", "SIN DEMANDA": "F0F0F0",
}
abc_fill = {"A": "D5F5E3", "B": "FEF9E7", "C": "FDFEFE"}
patron_fill = {
    "ESTABLE": "D4F5D4", "VARIABLE": "FFF4A0",
    "ERRATICO": "FFD9A0", "SIN DEMANDA": "F0F0F0"
}

with pd.ExcelWriter(out_file, engine="openpyxl") as writer:

    plan_out.to_excel(writer,        sheet_name="Completo",         index=False)
    rotacion.to_excel(writer,        sheet_name="Rotacion",         index=False)
    permanencia.to_excel(writer,     sheet_name="Permanencia",      index=False)
    sin_venta_larga.to_excel(writer, sheet_name="Sin_Venta_Larga",  index=False)
    a_liquidar.to_excel(writer,      sheet_name="A_Liquidar",       index=False)
    a_atender.to_excel(writer,       sheet_name="A_Atender",        index=False)
    pedir_ya.to_excel(writer,        sheet_name="PEDIR_YA",         index=False)
    pedir_pron.to_excel(writer,      sheet_name="Pedir_Pronto",     index=False)
    if not sin_rotacion.empty:
        sin_rotacion.to_excel(writer, sheet_name="Sin_Rotacion",    index=False)
    if not caida.empty:
        caida.to_excel(writer,       sheet_name="Caida_Demanda",    index=False)
    res_abc.to_excel(writer,         sheet_name="ABC_x_Tramo",      index=False)
    res_marca.to_excel(writer,       sheet_name="Aging_x_Marca",    index=False)
    res_repo_marca.to_excel(writer,  sheet_name="Repo_x_Marca",     index=False)
    rot_x_grupo.to_excel(writer,     sheet_name="Grupo_Logistico",  index=False)

    try:
        from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
        from openpyxl.utils import get_column_letter
        from openpyxl.chart import BarChart, PieChart, Reference

        NF = PatternFill("solid", fgColor="121C3A")
        NX = Font(bold=True, color="FFFFFF", size=10)
        ts = Side(border_style="thin", color="CCCCCC")
        tb = Border(left=ts, right=ts, top=ts, bottom=ts)

        def _fmt(ws, aging_col=None, repo_col=None, abc_col=None, patron_col=None,
                 flag_caida=None, flag_sinrot=None, perm_col=None):
            hdr = [c.value for c in ws[1]]
            for cc in ws.columns:
                ml = max((len(str(c.value)) if c.value else 0) for c in cc)
                ws.column_dimensions[get_column_letter(cc[0].column)].width = min(ml + 3, 45)
            for cell in ws[1]:
                cell.fill = NF
                cell.font = NX
                cell.alignment = Alignment(horizontal="center", wrap_text=True)
            ws.row_dimensions[1].height = 28
            ws.freeze_panes = "C2"

            ai   = hdr.index(aging_col)  if aging_col and aging_col in hdr else None
            ri   = hdr.index(repo_col)   if repo_col and repo_col in hdr else None
            abci = hdr.index(abc_col)    if abc_col and abc_col in hdr else None
            pi   = hdr.index(patron_col) if patron_col and patron_col in hdr else None
            fci  = hdr.index(flag_caida) if flag_caida and flag_caida in hdr else None
            sri  = hdr.index(flag_sinrot) if flag_sinrot and flag_sinrot in hdr else None
            pei  = hdr.index(perm_col) if perm_col and perm_col in hdr else None

            perm_fill = {
                "NUEVO": "D6EAF8",
                "RECIENTE": "D4F5D4",
                "ENVEJECIENDO": "FFF4A0",
                "VIEJO": "FFD9A0",
                "CRITICO": "F1948A",
                "SIN COMPRA": "F0F0F0",
            }

            for row in ws.iter_rows(min_row=2):
                is_caida = fci is not None and row[fci].value is True
                is_sinrot = sri is not None and row[sri].value is True

                if is_caida:
                    for c in row:
                        c.fill = PatternFill("solid", fgColor="FF6B35")
                        c.border = tb
                elif is_sinrot:
                    for c in row:
                        c.fill = PatternFill("solid", fgColor="D7BDE2")
                        c.border = tb
                elif ri is not None:
                    fgc = repo_fill.get(str(row[ri].value) if row[ri].value else "", "")
                    if fgc:
                        for c in row:
                            c.fill = PatternFill("solid", fgColor=fgc)
                            c.border = tb
                elif ai is not None:
                    fgc = aging_fill.get(str(row[ai].value) if row[ai].value else "", "")
                    if fgc:
                        for c in row:
                            c.fill = PatternFill("solid", fgColor=fgc)
                            c.border = tb
                elif pei is not None:
                    fgc = perm_fill.get(str(row[pei].value) if row[pei].value else "", "")
                    if fgc:
                        for c in row:
                            c.fill = PatternFill("solid", fgColor=fgc)
                            c.border = tb

                if abci is not None:
                    av = row[abci].value
                    row[abci].fill = PatternFill("solid", fgColor=abc_fill.get(str(av) if av else "", "FFFFFF"))
                    row[abci].font = Font(
                        bold=True, size=9,
                        color="1A6630" if av == "A" else "7D6608" if av == "B" else "7B241C"
                    )

                if pi is not None:
                    pv = row[pi].value
                    row[pi].fill = PatternFill("solid", fgColor=patron_fill.get(str(pv) if pv else "", "FFFFFF"))

        for sh, ac, rc, abcc, pc, fc, sc2, pe in [
            ("Completo", "Tramo_aging", "Estado_repo", "ABC", "Patron_demanda", "Flag_caida_demanda", "Flag_sin_rotacion", "Permanencia_compra"),
            ("Rotacion", None, None, "ABC", None, None, None, None),
            ("Permanencia", None, None, "ABC", None, None, None, "Permanencia_compra"),
            ("Sin_Venta_Larga", "Tramo_aging", "Estado_repo", "ABC", None, None, None, "Permanencia_compra"),
            ("A_Liquidar", "Tramo_aging", None, "ABC", "Patron_demanda", None, None, None),
            ("A_Atender", "Tramo_aging", None, "ABC", "Patron_demanda", None, None, None),
            ("PEDIR_YA", None, "Estado_repo", "ABC", "Patron_demanda", None, None, None),
            ("Pedir_Pronto", None, "Estado_repo", "ABC", "Patron_demanda", None, None, None),
            ("Sin_Rotacion", "Tramo_aging", None, "ABC", "Patron_demanda", None, "Flag_sin_rotacion", None),
            ("Caida_Demanda", "Tramo_aging", None, "ABC", "Patron_demanda", "Flag_caida_demanda", None, None),
            ("ABC_x_Tramo", "Tramo_aging", None, None, None, None, None, None),
            ("Aging_x_Marca", "Tramo_aging", None, None, None, None, None, None),
            ("Repo_x_Marca", None, "Estado_repo", None, None, None, None, None),
        ]:
            if sh in writer.sheets:
                _fmt(writer.sheets[sh], ac, rc, abcc, pc, fc, sc2, pe)

        wb = writer.book
        ws_d = wb.create_sheet("DASHBOARD", 0)
        ws_d.sheet_view.showGridLines = False

        C_NAVY = "121C3A"; C_WHITE = "FFFFFF"; C_GRAY = "F5F7FA"
        C_RED = "C0392B"; C_GREEN = "27AE60"; C_BLUE = "2980B9"
        C_ORANGE = "E67E22"; C_PURPLE = "8E44AD"; C_TEAL = "16A085"

        def ms(ws, r1, c1, r2, c2, v="", bold=False, size=11, color="000000", bg=None, align="center"):
            ws.merge_cells(start_row=r1, start_column=c1, end_row=r2, end_column=c2)
            cell = ws.cell(row=r1, column=c1, value=v)
            cell.font = Font(bold=bold, size=size, color=color)
            cell.alignment = Alignment(horizontal=align, vertical="center", wrap_text=True)
            if bg:
                cell.fill = PatternFill("solid", fgColor=bg)
            return cell

        def sc(ws, r, c, v, bold=False, size=9, color="000000", bg=None, align="center"):
            cell = ws.cell(row=r, column=c, value=v)
            cell.font = Font(bold=bold, size=size, color=color)
            cell.alignment = Alignment(horizontal=align, vertical="center", wrap_text=True)
            if bg:
                cell.fill = PatternFill("solid", fgColor=bg)
            return cell

        col_widths = [0.5, 22, 14, 14, 14, 14, 14, 14, 14, 0.5]
        for i, w in enumerate(col_widths, 1):
            ws_d.column_dimensions[get_column_letter(i)].width = w
        for col_idx in range(len(col_widths) + 1, 80):
            ws_d.column_dimensions[get_column_letter(col_idx)].hidden = True

        for r, h in {
            1: 6, 2: 45, 3: 8,
            4: 20, 5: 28, 6: 28, 7: 20, 8: 8,
            9: 20, 10: 28, 11: 28, 12: 20, 13: 8,
            14: 20, 15: 28, 16: 28, 17: 20, 18: 8
        }.items():
            ws_d.row_dimensions[r].height = h

        ms(ws_d, 2, 1, 2, 9, "GRAPES MARKET — Inventario & Reposición",
           bold=True, size=22, color=C_WHITE, bg=C_NAVY)

        for r in range(4, 8):
            for c in range(1, 10):
                ws_d.cell(r, c).fill = PatternFill("solid", fgColor=C_GRAY)

        ms(ws_d, 4, 2, 4, 3, "CAPITAL EN STOCK", bold=True, size=9, color="777777", bg=C_GRAY)
        ms(ws_d, 5, 2, 6, 3, fmt_usd(val_total, "Sin datos"), bold=True, size=15, color=C_NAVY, bg=C_GRAY)
        ms(ws_d, 7, 2, 7, 3, "Cerrito + FULL (a costo)", bold=False, size=8, color="999999", bg=C_GRAY)

        ms(ws_d, 4, 4, 4, 5, "CAPITAL EN RIESGO", bold=True, size=9, color="777777", bg=C_GRAY)
        ms(ws_d, 5, 4, 6, 5, fmt_usd(val_clavo, "US$ 0"), bold=True, size=15, color=C_RED, bg=C_GRAY)
        ms(ws_d, 7, 4, 7, 5, f"CLAVO + MUY_LENTO ({str(pct_riesgo).replace('.', ',')}%)",
           bold=False, size=8, color="999999", bg=C_GRAY)

        ms(ws_d, 4, 6, 4, 7, "% STOCK OK", bold=True, size=9, color="777777", bg=C_GRAY)
        ms(ws_d, 5, 6, 6, 7, f"{int(pct_ok)}%", bold=True, size=22,
           color=C_GREEN if pct_ok >= 60 else C_ORANGE, bg=C_GRAY)
        ms(ws_d, 7, 6, 7, 7, "productos dentro del ciclo", bold=False, size=8, color="999999", bg=C_GRAY)

        ms(ws_d, 4, 8, 4, 8, "QS ÚLTIMOS 6M", bold=True, size=9, color="777777", bg=C_GRAY)
        ms(ws_d, 5, 8, 6, 8, str(n_qs), bold=True, size=22,
           color=C_RED if n_qs > 0 else C_GREEN, bg=C_GRAY)
        ms(ws_d, 7, 8, 7, 8, "productos con quiebre de stock", bold=False, size=8, color="999999", bg=C_GRAY)

        for r in range(9, 13):
            for c in range(1, 10):
                ws_d.cell(r, c).fill = PatternFill("solid", fgColor="EBF5FB")

        ms(ws_d, 9, 2, 9, 3, "PEDIR YA", bold=True, size=9, color="777777", bg="EBF5FB")
        ms(ws_d, 10, 2, 11, 3, str(n_pedir_ya), bold=True, size=22,
           color=C_RED if n_pedir_ya > 0 else C_GREEN, bg="EBF5FB")
        ms(ws_d, 12, 2, 12, 3, f"{fmt_usd(val_q_ya)} inversión", bold=False, size=8, color="999999", bg="EBF5FB")

        ms(ws_d, 9, 4, 9, 5, "PEDIR PRONTO", bold=True, size=9, color="777777", bg="EBF5FB")
        ms(ws_d, 10, 4, 11, 5, str(n_pedir_pron), bold=True, size=22,
           color=C_ORANGE if n_pedir_pron > 0 else C_GREEN, bg="EBF5FB")
        ms(ws_d, 12, 4, 12, 5, "próximo pedido", bold=False, size=8, color="999999", bg="EBF5FB")

        ms(ws_d, 9, 6, 9, 7, "SIN ROTACIÓN", bold=True, size=9, color="777777", bg="EBF5FB")
        ms(ws_d, 10, 6, 11, 7, str(n_sin_rot), bold=True, size=22,
           color=C_PURPLE if n_sin_rot > 0 else C_GREEN, bg="EBF5FB")
        ms(ws_d, 12, 6, 12, 7, f"con stock >{DIAS_SIN_ROTAR} días", bold=False, size=8, color="999999", bg="EBF5FB")

        ms(ws_d, 9, 8, 9, 8, "CAÍDA DEMANDA", bold=True, size=9, color="777777", bg="EBF5FB")
        ms(ws_d, 10, 8, 11, 8, str(n_caida), bold=True, size=22,
           color=C_ORANGE if n_caida > 0 else C_GREEN, bg="EBF5FB")
        ms(ws_d, 12, 8, 12, 8, f"≥{int(UMBRAL_CAIDA * 100)}% sin QS", bold=False, size=8, color="999999", bg="EBF5FB")

        C_DEP_BG = "EAF2FF"
        for r in range(14, 18):
            for c in range(1, 10):
                ws_d.cell(r, c).fill = PatternFill("solid", fgColor=C_DEP_BG)

        ms(ws_d, 14, 2, 14, 3, "TOTAL DEPÓSITOS", bold=True, size=9, color="777777", bg=C_DEP_BG)
        ms(ws_d, 15, 2, 16, 3, fmt_usd(val_total, "—"), bold=True, size=14, color=C_NAVY, bg=C_DEP_BG)
        ms(ws_d, 17, 2, 17, 3, "Cerrito + FULL", bold=False, size=8, color="999999", bg=C_DEP_BG)

        pct_cerr = round(val_total_cerrito / val_total * 100, 1) if val_total > 0 else 0
        ms(ws_d, 14, 4, 14, 5, "CERRITO", bold=True, size=9, color="777777", bg=C_DEP_BG)
        ms(ws_d, 15, 4, 16, 5, fmt_usd(val_total_cerrito, "—"), bold=True, size=14, color=C_BLUE, bg=C_DEP_BG)
        ms(ws_d, 17, 4, 17, 5, f"{str(pct_cerr).replace('.', ',')}% del total", bold=False, size=8, color="999999", bg=C_DEP_BG)

        pct_full = round(val_total_full / val_total * 100, 1) if val_total > 0 else 0
        ms(ws_d, 14, 6, 14, 7, "FULL (ML)", bold=True, size=9, color="777777", bg=C_DEP_BG)
        ms(ws_d, 15, 6, 16, 7, fmt_usd(val_total_full, "—"), bold=True, size=14, color=C_TEAL, bg=C_DEP_BG)
        ms(ws_d, 17, 6, 17, 7, f"{str(pct_full).replace('.', ',')}% del total", bold=False, size=8, color="999999", bg=C_DEP_BG)

        n_sku_cerr = int((plan_out["Stock_Cerrito"] > 0).sum())
        n_sku_full = int((plan_out["Stock_FULL"] > 0).sum())
        ms(ws_d, 14, 8, 14, 8, "SKUs con stock", bold=True, size=9, color="777777", bg=C_DEP_BG)
        ms(ws_d, 15, 8, 16, 8, f"Cerr: {n_sku_cerr}  /  FULL: {n_sku_full}", bold=True, size=11, color=C_NAVY, bg=C_DEP_BG)
        ms(ws_d, 17, 8, 17, 8, "productos con cantidad > 0", bold=False, size=8, color="999999", bg=C_DEP_BG)

        R = 19

        ms(ws_d, R, 2, R, 8, "CAPITAL EN DEPÓSITO POR MARCA (a costo unitario)",
          bold=True, size=11, color=C_WHITE, bg=C_BLUE, align="left")
        R += 1
        for j, h in enumerate(["Marca", "SKUs", "Stock (uds)", "Total USD", "Cerrito", "FULL", "% total"], 2):
            sc(ws_d, R, j, h, bold=True, color=C_WHITE, bg="1A5276")
        R += 1

        row_colors_marca = ["F0F4FA", "FFFFFF"]
        for idx, (_, rw) in enumerate(stock_x_marca.iterrows()):
            v = float(rw["Valor_USD"]) if pd.notna(rw["Valor_USD"]) else 0
            v_cerr = float(rw["Valor_Cerrito"]) if pd.notna(rw["Valor_Cerrito"]) else 0
            v_full = float(rw["Valor_FULL"]) if pd.notna(rw["Valor_FULL"]) else 0
            fgc = row_colors_marca[idx % 2]
            peso_alto = (v / val_total_marcas) > 0.20

            for j, val in enumerate([
                str(rw["Marca"]), int(rw["SKUs"]),
                fmt_n(rw["Stock_uds"]),
                fmt_usd(v), fmt_usd(v_cerr) if v_cerr > 0 else "—",
                fmt_usd(v_full) if v_full > 0 else "—",
                fmt_pct(v, val_total_marcas),
            ], 2):
                cell = ws_d.cell(R, j, val)
                cell.fill = PatternFill("solid", fgColor=fgc)
                cell.font = Font(size=9, bold=(j == 2 or (j == 5 and peso_alto)),
                                color=C_NAVY if peso_alto and j == 5 else "000000")
                cell.alignment = Alignment(horizontal="center", vertical="center")
                cell.border = tb
            R += 1
        R += 1

        ms(ws_d, R, 2, R, 8, "CLASIFICACIÓN ABC (margen 12 meses)",
          bold=True, size=11, color=C_WHITE, bg="16A085", align="left")
        R += 1
        for j, h in enumerate(["Cat.", "Descripción", "SKUs", "Valor Stock", "% Margen", "Tramo más crítico"], 2):
            sc(ws_d, R, j, h, bold=True, color=C_WHITE, bg="0E6655")
        R += 1

        for cat, desc, cfill in [
            ("A", "Top 80% margen — prioridad máxima", "D5F5E3"),
            ("B", "Siguiente 15% — según demanda", "FEF9E7"),
            ("C", "Resto — bajo pedido / evaluar baja", "FDFEFE"),
        ]:
            sub = plan_out[plan_out["ABC"] == cat]
            v_cat = float(sub["Valor_stock_USD"].sum() or 0)
            m_cat = float(sub["Margen_12m"].sum() or 0)
            m_tot = float(plan_out["Margen_12m"].sum() or 1)
            worst = sub["Tramo_aging"].map(aging_ord).min()
            wname = [k for k, v in aging_ord.items() if v == worst]

            for j, val in enumerate([
                cat, desc, len(sub),
                fmt_usd(v_cat), fmt_pct(m_cat, m_tot),
                wname[0] if wname else "—"
            ], 2):
                cell = ws_d.cell(R, j, val)
                cell.fill = PatternFill("solid", fgColor=cfill)
                cell.font = Font(bold=(j == 2), size=9)
                cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
                cell.border = tb
            R += 1
        R += 1

        ms(ws_d, R, 2, R, 8, "AGING POR TRAMO (cobertura neta de stock de seguridad)",
          bold=True, size=11, color=C_WHITE, bg=C_NAVY, align="left")
        R += 1
        for j, h in enumerate(["Tramo", "Descripción", "SKUs", "Stock (uds)", "Valor USD", "% Valor", "Acción"], 2):
            sc(ws_d, R, j, h, bold=True, color=C_WHITE, bg="3B5BA5")
        R += 1

        dm = {
            "OK": "Stock dentro del ciclo",
            "ATENCION": "Sobrante al próximo pedido",
            "LENTO": "No limpia antes del próximo ciclo",
            "MUY_LENTO": "Cruza dos ciclos de compra",
            "CLAVO": "Tres ciclos y sigue — liquidar",
            "SIN STOCK": "Sin stock",
            "NUEVO": "Sin demanda histórica (<60 días)",
            "SIN DEMANDA": "Sin demanda",
        }

        for _, row in res_tramo.iterrows():
            tr = str(row["Tramo_aging"])
            fgc = aging_fill.get(tr, "FFFFFF")
            txt = C_WHITE if tr == "CLAVO" else "000000"
            v = float(row["Valor_USD"]) if pd.notna(row["Valor_USD"]) else 0

            for j, val in enumerate([
                tr, dm.get(tr, "—"), int(row["Productos"]),
                fmt_n(row["Stock_uds"]), fmt_usd(v),
                fmt_pct(v, val_total),
                ACCIONES_AGING.get(tr, "—")[:35]
            ], 2):
                cell = ws_d.cell(R, j, val)
                cell.fill = PatternFill("solid", fgColor=fgc)
                cell.font = Font(bold=(j == 2), size=9, color=txt)
                cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
                cell.border = tb
            R += 1
        R += 1

        ms(ws_d, R, 2, R, 8, "PEDIR YA — en o bajo el punto de reorden",
          bold=True, size=11, color=C_WHITE, bg=C_RED, align="left")
        R += 1
        for j, h in enumerate(["Código", "Artículo", "Marca", "ABC", "Stock", "ROP", "SS", "Q Pedir", "US$ Inv."], 2):
            sc(ws_d, R, j, h, bold=True, color=C_WHITE, bg="922B21")
        R += 1

        for _, (_, rw) in enumerate(pedir_ya.head(7).iterrows()):
            fgc = repo_fill.get(str(rw["Estado_repo"]), "FDECEA")
            av = str(rw.get("ABC", "C"))
            vi = float(rw.get("Valor_Q_USD", 0) or 0)

            for j, val in enumerate([
                str(rw["Codigo"])[:15], str(rw["Articulo"])[:28],
                str(rw["Marca"]), av,
                fmt_n(rw["Stock_Total"]),
                f"{rw['ROP']:.1f}".replace(".", ","),
                f"{rw['SS']:.1f}".replace(".", ","),
                fmt_n(rw["Q_pedir"]),
                fmt_usd(vi)
            ], 2):
                cell = ws_d.cell(R, j, val)
                cell.fill = PatternFill("solid", fgColor=fgc)
                cell.font = Font(size=9, bold=(av == "A"))
                cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
                cell.border = tb
            R += 1
        R += 1

        ms(ws_d, R, 2, R, 8, "TOP A LIQUIDAR (CLAVO + MUY_LENTO)",
          bold=True, size=11, color=C_WHITE, bg="922B21", align="left")
        R += 1
        for j, h in enumerate(["Código", "Artículo", "Marca", "ABC", "Stock", "Cob. SS", "Valor USD"], 2):
            sc(ws_d, R, j, h, bold=True, color=C_WHITE, bg="7B241C")
        R += 1

        for _, (_, rw) in enumerate(a_liquidar.sort_values("Valor_stock_USD", ascending=False).head(5).iterrows()):
            fgc = aging_fill.get(str(rw["Tramo_aging"]), "FFFFFF")
            cob = rw["Cobertura_sobre_SS"]
            v = float(rw.get("Valor_stock_USD", 0) or 0)
            av = str(rw.get("ABC", "C"))

            for j, val in enumerate([
                str(rw["Codigo"])[:15], str(rw["Articulo"])[:28],
                str(rw["Marca"]), av,
                fmt_n(rw["Stock_Total"]),
                f"{cob:.1f}m".replace(".", ",") if np.isfinite(cob) else "∞",
                fmt_usd(v)
            ], 2):
                cell = ws_d.cell(R, j, val)
                cell.fill = PatternFill("solid", fgColor=fgc)
                cell.font = Font(size=9, bold=(av == "A"))
                cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
                cell.border = tb
            R += 1
        R += 1

        if not caida.empty:
            ms(ws_d, R, 2, R, 8,
              f"ALERTAS DEMANDA — caída ≥{int(UMBRAL_CAIDA * 100)}% o a cero (sin QS)",
              bold=True, size=11, color=C_WHITE, bg=C_ORANGE, align="left")
            R += 1
            for j, h in enumerate(["Código", "Artículo", "Marca", "ABC", "Base 4m", "Señal 2m", "Alerta"], 2):
                sc(ws_d, R, j, h, bold=True, color=C_WHITE, bg="A04000")
            R += 1

            for _, (_, rw) in enumerate(caida.head(5).iterrows()):
                av = str(rw.get("ABC", "C"))
                for j, val in enumerate([
                    str(rw["Codigo"])[:15], str(rw["Articulo"])[:28],
                    str(rw["Marca"]), av,
                    f"{float(rw.get('Avg_base_4m', 0) or 0):.1f}/m".replace(".", ","),
                    f"{float(rw.get('Avg_signal_2m', 0) or 0):.1f}/m".replace(".", ","),
                    str(rw.get("Alerta_demanda", ""))[:35]
                ], 2):
                    cell = ws_d.cell(R, j, val)
                    cell.fill = PatternFill("solid", fgColor="FFE8D6")
                    cell.font = Font(size=9, bold=(av == "A"))
                    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
                    cell.border = tb
                R += 1
            R += 1

        ms(ws_d, R, 2, R, 8, "GRUPOS LOGÍSTICOS — compra consolidada / separada",
          bold=True, size=11, color=C_WHITE, bg=C_TEAL, align="left")
        R += 1
        for j, h in enumerate(["Grupo", "SKUs", "Valor Stock", "Ventas 12m", "Compras 12m", "Valor a pedir"], 2):
            sc(ws_d, R, j, h, bold=True, color=C_WHITE, bg="0E6655")
        R += 1

        for _, rw in rot_x_grupo.iterrows():
            vals = [
                str(rw["Grupo_logistico"]),
                int(rw["SKUs"]),
                fmt_usd(rw["Valor_stock"]),
                fmt_n(rw["Ventas_12m_pos"]),
                fmt_n(rw["Compras_12m"]),
                fmt_usd(rw["Valor_Q_USD"]),
            ]
            for j, val in enumerate(vals, 2):
                cell = ws_d.cell(R, j, val)
                cell.fill = PatternFill("solid", fgColor="F7FCF9")
                cell.font = Font(size=9, bold=(j == 2))
                cell.alignment = Alignment(horizontal="center", vertical="center")
                cell.border = tb
            R += 1

        ms(ws_d, R + 2, 2, R + 2, 8,
          f"Datos al {FECHA_STOCK.date()} | "
          f"Compras = Clase_norm '01.factura' | "
          f"Ventas = FC A/B/C | "
          f"Reposición base semestral ({int(CICLO_COMPRA)}m) | "
          f"Rotación real 12m = Ventas_12m_pos / Stock_Total | "
          f"Meses activos = 12 - meses_QS | "
          f"Valuación = Cantidad × Costo_usd | "
          f"Solo USD/US$ | "
          f"No se clipea Valor_stock_USD",
          bold=False, size=8, color="999999", bg=C_GRAY)

        aux = R + 6
        ws_d.cell(aux, 1, "Tramo"); ws_d.cell(aux, 2, "Valor USD")
        for i, (_, rw) in enumerate(res_tramo.iterrows()):
            ws_d.cell(aux + 1 + i, 1, str(rw["Tramo_aging"]))
            ws_d.cell(aux + 1 + i, 2, float(rw["Valor_USD"]) if pd.notna(rw["Valor_USD"]) else 0)
        n_tr = len(res_tramo)

        c1 = BarChart()
        c1.type = "bar"
        c1.grouping = "clustered"
        c1.title = "Capital por tramo (USD)"
        c1.style = 10
        c1.width = 14
        c1.height = 10
        c1.add_data(Reference(ws_d, min_col=2, min_row=aux, max_row=aux + n_tr), titles_from_data=True)
        c1.set_categories(Reference(ws_d, min_col=1, min_row=aux + 1, max_row=aux + n_tr))
        ws_d.add_chart(c1, "B" + str(aux + n_tr + 3))

        aux2 = aux + n_tr + 20
        ws_d.cell(aux2, 1, "Tramo"); ws_d.cell(aux2, 2, "SKUs")
        for i, (_, rw) in enumerate(res_tramo.iterrows()):
            ws_d.cell(aux2 + 1 + i, 1, str(rw["Tramo_aging"]))
            ws_d.cell(aux2 + 1 + i, 2, int(rw["Productos"]))

        c2 = PieChart()
        c2.title = "SKUs por tramo"
        c2.style = 10
        c2.width = 12
        c2.height = 10
        c2.add_data(Reference(ws_d, min_col=2, min_row=aux2, max_row=aux2 + n_tr), titles_from_data=True)
        c2.set_categories(Reference(ws_d, min_col=1, min_row=aux2 + 1, max_row=aux2 + n_tr))
        ws_d.add_chart(c2, "F" + str(aux + n_tr + 3))

        for rh in range(aux, aux2 + n_tr + 15):
            ws_d.row_dimensions[rh].hidden = True

    except Exception as e:
        import traceback
        traceback.print_exc()

# ─── 21) FIN ───────────────────────────────────────────────

#REPORTE GERENCIAL GRAPES MARKET PPT

In [58]:
# -*- coding: utf-8 -*-
# ============================================================
# REPORTE GERENCIAL — GRAPES MARKET  v4
# Genera: PPT ejecutivo + Excel anexo  |  Google Colab
#
# Inputs requeridos en memoria: df_merged_usd, df_valorizacion_stock
# Input opcional:               plan_out  (del script inventario_unificado)
#
# v4 — Cambios vs v3:
#  - Marca en stock: cruza df_valorizacion_stock con df_merged_usd por Codigo
#    (df_valorizacion_stock no tiene columna Marca)
#  - Inventario foto: un solo gráfico horizontal de capital por marca
#  - Gráfico histórico: 3 líneas en panel 1 (Ventas, Profit, Costo)
#  - Clientes nuevos: líneas (no barras apiladas ilegibles)
#  - Riesgo/reposición: sin columnas Bloque ni Código — solo Artículo + resto
#  - Sin siglas "YTD": usa "ene-mar 2026" o lo que corresponda
# ============================================================

from __future__ import annotations
import os, re, textwrap, calendar
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.ticker import FuncFormatter
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.shapes import MSO_SHAPE
from pptx.enum.text import PP_ALIGN
from pptx.dml.color import RGBColor

# ─────────────────────────────────────────────────────────────
# 0) DRIVE + OUTPUT DIR
# ─────────────────────────────────────────────────────────────
def _try_mount_drive():
    try:
        from google.colab import drive  # type: ignore
        if not os.path.ismount("/content/gdrive"):
            drive.mount("/content/gdrive", force_remount=False)
    except Exception:
        pass

_try_mount_drive()
_DRIVE_OUT = "/content/gdrive/My Drive/Grapes/reporte_grapes/resultados"
_LOCAL_OUT  = "./salida_reporte_gerencial"
OUTPUT_DIR  = _DRIVE_OUT if os.path.exists("/content/gdrive/My Drive") else _LOCAL_OUT
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────
# 1) CONFIG
# ─────────────────────────────────────────────────────────────
REPORT_DATE        = pd.Timestamp.today().normalize()
START_HISTORY_DATE = pd.Timestamp("2024-01-01")
REPORT_TITLE       = "Reporte Gerencial — Grapes Market"
REPORT_SUBTITLE    = "Comercial + Inventario"
MAIN_PPT_NAME      = f"reporte_gerencial_grapes_{REPORT_DATE:%Y%m%d}.pptx"
ANNEX_XLSX_NAME    = f"anexo_reporte_gerencial_grapes_{REPORT_DATE:%Y%m%d}.xlsx"
IMG_DPI            = 150
FIGSIZE            = (16, 9)

PATH_DF_MERGED_USD         = None
PATH_DF_VALORIZACION_STOCK = None
PATH_PLAN_OUT              = None
LOGO_PATH                  = None

SYSTEM_ARS = {
    2023: {"ventas": 196_736_324.77,    "profit": 121_932_548.96},
    2024: {"ventas": 470_387_053.93,    "profit": 210_521_737.52},
    2025: {"ventas": 1_041_313_842.16,  "profit": 344_332_453.50},
}
CAUSA_DIFFS = (
    "2025: ND A 00006-00000004 (ajuste financiero sin mov de stock) fuera del universo FC/NC. "
    "2023-2024: remitos derivados de FC A con importes distintos registrados en mov_stock "
    "pero excluidos del reporte (solo FC/NC)."
)

EXCLUDED_CODES = {"151","78","152","ENVIOS-PACK","ART-VAR","75","850065783635","76","74","77"}
VALID_USD      = {"USD","US$","U$S","DOLAR","DÓLAR","DOLARES","DÓLARES"}
KEY_BRANDS     = ["QNAP","10GTEK","LACIE","SEAGATE","OWC"]

MESES_ES = {1:"ene",2:"feb",3:"mar",4:"abr",5:"may",6:"jun",
            7:"jul",8:"ago",9:"sep",10:"oct",11:"nov",12:"dic"}

NAVY   = "#12213f"; BLUE   = "#355caa"; GREEN  = "#1f9d75"; ORANGE = "#d97a2b"
PURPLE = "#7a57a8"; RED    = "#c44e52"; TEAL   = "#2b8cbe"; GRAY   = "#6b7280"
GRAY_L = "#e5e7eb"; GRAY_XL= "#f5f6f8"; GOLD   = "#caa64b"
BANNER_RGB = (18,33,63); WHITE_RGB = (255,255,255); TEXT_DARK = (38,38,38)

BRAND_COLORS = [NAVY, BLUE, GREEN, TEAL, ORANGE, PURPLE, RED, "#20b2aa",
                "#e07b39", "#5b9bd5", "#ADB5BD", "#6C757D"]

# ─────────────────────────────────────────────────────────────
# 2) HELPERS
# ─────────────────────────────────────────────────────────────
def ensure_dir(p): os.makedirs(p, exist_ok=True)

def norm_text(x):
    if pd.isna(x): return ""
    return re.sub(r"\s+", " ", str(x).strip())

def norm_key(x):
    s = norm_text(x).upper()
    for a,b in [("Á","A"),("É","E"),("Í","I"),("Ó","O"),("Ú","U"),
                ("–","-"),("—","-"),("_"," ")]:
        s = s.replace(a,b)
    s = re.sub(r"[^A-Z0-9 /.\-]","",s)
    return re.sub(r"\s+"," ",s).strip()

def to_num(s):
    if s is None: return pd.Series(dtype="float64")
    if isinstance(s, pd.Series):
        if pd.api.types.is_numeric_dtype(s): return pd.to_numeric(s, errors="coerce")
        x = (s.astype(str).str.strip()
              .str.replace(r"\s+","",regex=True)
              .str.replace(".","",regex=False)
              .str.replace(",",".",regex=False)
              .replace({"":"nan","nan":"nan","None":"nan"}))
        return pd.to_numeric(x, errors="coerce")
    return pd.to_numeric(s, errors="coerce")

def to_dt(s):
    if isinstance(s, pd.Series):
        r = pd.to_datetime(s, errors="coerce", dayfirst=True)
        bad = r.isna() & s.notna() & (s.astype(str).str.strip()!="")
        if bad.any():
            r = r.copy(); r.loc[bad] = pd.to_datetime(s[bad], errors="coerce", dayfirst=False)
        return r
    return pd.to_datetime(s, errors="coerce", dayfirst=True)

def safe_div(a,b):
    try:
        a,b = float(a),float(b)
        return np.nan if (b==0 or np.isnan(b)) else a/b
    except: return np.nan

def pct_change(n,o):
    if pd.isna(n) or pd.isna(o): return np.nan
    if o==0: return np.nan if n==0 else np.inf
    return (n/o-1)*100

def pct_str(n,o,dec=1):
    d = pct_change(n,o)
    if pd.isna(d): return "—"
    if np.isinf(d): return "∞"
    return f"{'+' if d>0 else ''}{fmt_num(d,dec)}%"

def pp_str(n,o,dec=1):
    if pd.isna(n) or pd.isna(o): return "—"
    d = float(n)-float(o)
    return f"{'+' if d>0 else ''}{fmt_num(d,dec)} pp"

def fmt_num(v,dec=0):
    try:
        s = f"{float(v):,.{dec}f}"
        return s.replace(",","X").replace(".",",").replace("X",".")
    except: return str(v)

def fmt_money(v,cur="",dec=0):
    if pd.isna(v): return "—"
    pre = {"ARS":"$","USD":"US$ "}.get(cur,"")
    return pre+fmt_num(v,dec)

def fmt_pct(v,dec=1):
    if pd.isna(v): return "—"
    return f"{fmt_num(float(v),dec)}%"

def shorten(x,w=40):
    return textwrap.shorten(norm_text(x),width=w,placeholder="…")

def money_fmt(values, cur="USD"):
    s = pd.Series(values).replace([np.inf,-np.inf],np.nan).dropna()
    vmax = 0 if s.empty else float(s.abs().max())
    if vmax>=1e9:   div,lbl,dec = 1e9, f"miles de millones {cur}", 1
    elif vmax>=1e6: div,lbl,dec = 1e6, f"millones {cur}", 1
    elif vmax>=1e3: div,lbl,dec = 1e3, f"miles {cur}", 0
    else:           div,lbl,dec = 1, cur, 0
    return FuncFormatter(lambda x,_: fmt_num(x/div,dec)), lbl

def add_other_brands(df,bc,vc,tops,other="Otras"):
    if df.empty: return df.copy(),""
    key_set = {norm_key(x) for x in tops}
    b = df.copy()
    b["_bn"] = b[bc].map(norm_key)
    b["_bo"] = np.where(b["_bn"].isin(key_set),b[bc],other)
    out = (b.groupby("_bo",dropna=False)[vc].sum().reset_index()
            .rename(columns={"_bo":bc}).sort_values(vc,ascending=False))
    others = (b.loc[~b["_bn"].isin(key_set),bc].astype(str).str.strip()
               .replace("",np.nan).dropna().sort_values().unique().tolist())
    return out,", ".join(others)

def save_fig(fig,path):
    fig.savefig(path,dpi=IMG_DPI,facecolor="white",bbox_inches="tight",format="png")
    plt.close(fig)

def add_footer(fig):
    fig.text(0.99,0.005,f"Generado el {REPORT_DATE:%d/%m/%Y}",ha="right",va="bottom",
             fontsize=8,color=GRAY)

def _banner(ax,title,sub=""):
    ax.add_patch(patches.Rectangle((0,0.89),1,0.11,transform=ax.transAxes,color=NAVY,zorder=0))
    ax.text(0.03,0.945,title,color="white",fontsize=21,fontweight="bold",
            transform=ax.transAxes,va="center",zorder=1)
    if sub:
        ax.text(0.03,0.906,sub,color="#c3d0ea",fontsize=10,transform=ax.transAxes,zorder=1)

# ─────────────────────────────────────────────────────────────
# 3) DETECCIÓN DE COLUMNAS
# ─────────────────────────────────────────────────────────────
@dataclass
class ColSpec:
    name: str; candidates: List[str]; required: bool = False

class ColumnResolver:
    def __init__(self,df):
        self.cols = list(df.columns)
        self.nm   = {c: norm_key(c) for c in self.cols}

    def _score(self,col,tgt):
        rc,tg = self.nm[col],norm_key(tgt)
        if rc==tg: return 100
        if rc.replace(" ","")==tg.replace(" ",""): return 95
        if tg in rc: return 75
        if rc in tg: return 60
        inter = len(set(tg.split())&set(rc.split()))
        return 40+inter*5 if inter else 0

    def find(self,candidates):
        best,score = None,-1
        for c in candidates:
            for col in self.cols:
                s = self._score(col,c)
                if s>score: best,score = col,s
        return best if score>=60 else None

    def resolve(self,specs):
        mapping,rows = {},[]
        for spec in specs:
            col = self.find(spec.candidates)
            mapping[spec.name] = col
            rows.append({"campo":spec.name,"req":spec.required,
                         "detectada":col or "",
                         "status":"ok" if col else ("faltante" if spec.required else "—")})
        return mapping,pd.DataFrame(rows)

COMM_SPECS = [
    ColSpec("fecha",      ["Fecha_de_Carga","Fecha de Carga","Fecha"],         required=True),
    ColSpec("comprobante",["Comprobante","Tipo Comprobante"],                   required=True),
    ColSpec("clase",      ["Clase","clase"]),
    ColSpec("codigo",     ["Codigo","Código","SKU","Part Number","P/N"],        required=True),
    ColSpec("articulo",   ["Articulo","Artículo","Producto","Descripcion"],     required=True),
    ColSpec("marca",      ["Marca","marca"]),
    ColSpec("cliente",    ["Cliente_Proveedor","Cliente","Razon Social","Razón Social"]),
    ColSpec("cantidad",   ["Cantidad","cant","Unidades"]),
    ColSpec("neto_ars",   ["total_neto_articulo_venta","Total Neto","Neto"],    required=True),
    ColSpec("profit_ars", ["profit_ii_articulo_venta","Profit","Ganancia"]),
    ColSpec("neto_usd",   ["total_neto_articulo_venta_usd","Total Neto USD"]),
    ColSpec("profit_usd", ["profit_ii_articulo_venta_usd","Profit USD"]),
    ColSpec("tipo_cambio",["Tipo_de_Cambio","Tipo de Cambio","TC"]),
    ColSpec("canal",      ["Canal","canal","Marketplace"]),
]

# df_valorizacion_stock NO tiene Marca → se enriquece desde df_merged_usd en build_report
STOCK_SPECS = [
    ColSpec("codigo",    ["Codigo","Código","Codigo_key","SKU"],        ),
    ColSpec("articulo",  ["Articulo","Artículo","Producto"],            required=True),
    ColSpec("cantidad",  ["Cantidad","Stock","cant"],                   required=True),
    ColSpec("costo_usd", ["Costo_usd","Costo_PPP","valor_stock_final",
                          "Costo_PPP_Valorizado"],                     required=True),
    ColSpec("moneda",    ["coslis_curr_desc","Moneda","moneda"]),
    ColSpec("deposito",  ["Deposito","Depósito","Sucursal"]),
]

PLAN_SPECS = [
    ColSpec("codigo",       ["Codigo","Código","SKU"]),
    ColSpec("articulo",     ["Articulo","Artículo","Producto"]),
    ColSpec("marca",        ["Marca","marca"]),
    ColSpec("abc",          ["ABC","abc"]),
    ColSpec("stock_total",  ["Stock_Total","Stock Total"]),
    ColSpec("valor_usd",    ["Valor_stock_USD","Valor Stock USD"]),
    ColSpec("tramo_aging",  ["Tramo_aging","Tramo Aging","Aging"]),
    ColSpec("estado_repo",  ["Estado_repo","Estado Repo","Estado"]),
    ColSpec("dias_sv",      ["Dias_sin_venta","Días sin venta"]),
    ColSpec("clasif_rot",   ["Clasif_rotacion","Clasif Rotacion"]),
    ColSpec("qs_6m",        ["QS_ultimos_6m","QS últimos 6m"]),
    ColSpec("cobertura",    ["Cobertura_bruta_meses","Cobertura"]),
    ColSpec("q_sugerida",   ["Q_pedir","Q_pedir_sugerido","Q sugerida"]),
    ColSpec("valor_q_usd",  ["Valor_Q_USD","Valor Q USD"]),
    ColSpec("accion",       ["Estado_repo","Accion","Acción","Estado"]),
    ColSpec("ultima_venta", ["Ultima_venta","Última venta"]),
]

# ─────────────────────────────────────────────────────────────
# 4) CARGA
# ─────────────────────────────────────────────────────────────
def load_df(name,path=None):
    if name in globals() and isinstance(globals()[name],pd.DataFrame):
        return globals()[name].copy()
    if path and os.path.exists(path):
        ext = os.path.splitext(path)[1].lower()
        if ext in [".xlsx",".xls"]: return pd.read_excel(path)
        if ext in [".csv",".txt"]:
            try: return pd.read_csv(path)
            except: return pd.read_csv(path,sep=";")
        if ext==".parquet": return pd.read_parquet(path)
    return None

# ─────────────────────────────────────────────────────────────
# 5) NORMALIZACIÓN COMERCIAL
# ─────────────────────────────────────────────────────────────
def _doc_type(comp):
    c = norm_key(comp)
    if c.startswith(("FC A","FC B","FC C")): return "VENTA"
    if c.startswith(("NC A","NC B","NC C","NC D")): return "DEVOLUCION"
    return "OTRO"

def _is_compra(clase):
    c = norm_key(clase)
    return c.startswith("01.FACTURA") or "COMPRA" in c

def _is_presup(comp,clase):
    c1,c2 = norm_key(comp),norm_key(clase)
    return any(t in c1 or t in c2 for t in ("PRESUP","COTIZ"))

def normalize_commercial(df_raw):
    res = ColumnResolver(df_raw)
    m,diag = res.resolve(COMM_SPECS)
    miss = diag.loc[(diag["req"])&(diag["detectada"]==""),"campo"].tolist()
    if miss: raise ValueError(f"Columnas requeridas faltantes en df_merged_usd: {miss}")

    df = df_raw.copy()
    out = pd.DataFrame(index=df.index)
    out["fecha"]      = to_dt(df[m["fecha"]])
    out["comprobante"]= df[m["comprobante"]].map(norm_text)
    out["clase"]      = df[m["clase"]].map(norm_text) if m["clase"] else ""
    out["codigo"]     = df[m["codigo"]].map(norm_text)
    out["codigo_norm"]= out["codigo"].map(norm_key)
    out["articulo"]   = df[m["articulo"]].map(norm_text)
    out["marca"]      = df[m["marca"]].map(norm_text) if m["marca"] else "Sin marca"
    out["cliente"]    = df[m["cliente"]].map(norm_text) if m["cliente"] else "Sin cliente"
    out["canal"]      = df[m["canal"]].map(norm_text) if m["canal"] else ""
    out["cantidad"]   = to_num(df[m["cantidad"]]) if m["cantidad"] else np.nan
    out["neto_ars"]   = to_num(df[m["neto_ars"]])
    out["profit_ars"] = to_num(df[m["profit_ars"]]) if m["profit_ars"] else np.nan
    out["neto_usd"]   = to_num(df[m["neto_usd"]])   if m["neto_usd"]   else np.nan
    out["profit_usd"] = to_num(df[m["profit_usd"]]) if m["profit_usd"] else np.nan
    out["tc"]         = to_num(df[m["tipo_cambio"]]) if m["tipo_cambio"] else np.nan
    out.loc[out["tc"]==0,"tc"] = np.nan

    out["doc_type"]  = out["comprobante"].map(_doc_type)
    out["es_presup"] = [_is_presup(c,cl) for c,cl in zip(out["comprobante"],out["clase"])]
    out["es_compra"] = out["clase"].map(_is_compra)
    out["excl_cod"]  = out["codigo_norm"].isin({norm_key(x) for x in EXCLUDED_CODES})

    base = out.loc[
        (~out["es_presup"])&(~out["es_compra"])&
        (~out["excl_cod"])&(out["doc_type"].isin(["VENTA","DEVOLUCION"]))
    ].copy()

    dev = base["doc_type"].eq("DEVOLUCION")
    base["ventas_ars"] = np.where(dev&(base["neto_ars"]>0),  -base["neto_ars"],  base["neto_ars"])
    base["profit_ars"] = np.where(dev&(base["profit_ars"].fillna(0)>0),-base["profit_ars"],base["profit_ars"])

    if base["neto_usd"].notna().any():
        base["ventas_usd"] = np.where(dev&(base["neto_usd"].fillna(0)>0),-base["neto_usd"],base["neto_usd"])
    else:
        base["ventas_usd"] = base["neto_ars"]/base["tc"]

    if base["profit_usd"].notna().any():
        base["profit_usd"] = np.where(dev&(base["profit_usd"].fillna(0)>0),-base["profit_usd"],base["profit_usd"])
    else:
        base["profit_usd"] = base["profit_ars"]/base["tc"]

    base["unidades"] = (np.where(dev,-base["cantidad"].abs(),base["cantidad"].abs())
                        if base["cantidad"].notna().any() else np.nan)
    base = base.loc[base["fecha"].notna()].copy()
    base["year"]  = base["fecha"].dt.year
    base["month"] = base["fecha"].dt.month
    base["canal_simple"] = np.select(
        [base["canal"].map(norm_key).str.contains("MERCADO|ML|FULL",na=False)],
        ["MercadoLibre"],
        default=np.where(base["canal"].str.strip()=="","No informado","Directa / otros")
    )
    return base,diag,m

# ─────────────────────────────────────────────────────────────
# 6) NORMALIZACIÓN STOCK
# df_valorizacion_stock NO tiene Marca.
# La marca se añade en build_report cruzando con df_merged_usd por Codigo.
# ─────────────────────────────────────────────────────────────
def normalize_stock(df_raw):
    res = ColumnResolver(df_raw)
    m,diag = res.resolve(STOCK_SPECS)
    miss = diag.loc[(diag["req"])&(diag["detectada"]==""),"campo"].tolist()
    if miss: raise ValueError(f"Columnas requeridas faltantes en df_valorizacion_stock: {miss}")

    df = df_raw.copy()
    out = pd.DataFrame(index=df.index)
    out["codigo"]     = df[m["codigo"]].map(norm_text) if m["codigo"] else ""
    out["codigo_norm"]= out["codigo"].map(norm_key)
    out["articulo"]   = df[m["articulo"]].map(norm_text)
    out["marca"]      = "Sin marca"     # se enriquece después desde comm
    out["cantidad"]   = to_num(df[m["cantidad"]])
    out["costo_usd"]  = to_num(df[m["costo_usd"]])
    out["deposito"]   = df[m["deposito"]].map(norm_text) if m["deposito"] else "No informado"

    if m["moneda"]:
        moneda_norm = df[m["moneda"]].map(norm_key)
        valid_norm  = {norm_key(x) for x in VALID_USD}
        out = out.loc[moneda_norm.isin(valid_norm)].copy()

    out["cantidad"]  = out["cantidad"].fillna(0)
    out["costo_usd"] = out["costo_usd"].fillna(0)

    # Una fila por (codigo_norm, deposito) — keep mayor costo
    out = out.sort_values("costo_usd",ascending=True)
    out = out.drop_duplicates(subset=["codigo_norm","deposito"],keep="last")

    out["valor_stock_usd"] = out["cantidad"]*out["costo_usd"]
    out = out.loc[(out["cantidad"]!=0)|(out["valor_stock_usd"]!=0)].copy()
    return out,diag,m

# ─────────────────────────────────────────────────────────────
# 7) ENRIQUECER STOCK CON MARCA (desde df_merged_usd)
# ─────────────────────────────────────────────────────────────
def enrich_stock_marca(stock, comm, plan_raw=None):
    """
    Asigna Marca a cada fila de stock (df_valorizacion_stock no tiene Marca).
    Orden de prioridad:
      1. plan_out (inventario_unificado) — columna Marca asignada directamente
         desde df_merged_usd. Es la fuente más completa y precisa.
      2. df_merged_usd (historial ventas FC) — marca más frecuente por codigo_norm.
      3. "Sin marca" — código sin historial en ninguna de las dos fuentes.
    """
    stock = stock.copy()

    # Fuente 1: plan_out tiene Marca por codigo_norm para TODOS los productos
    if plan_raw is not None and not plan_raw.empty:
        # Buscar columna Marca en plan_out original (antes de normalizar)
        marca_cols = [c for c in plan_raw.columns
                      if norm_key(c) in ("MARCA","BRAND","FABRICANTE")]
        cod_cols   = [c for c in plan_raw.columns
                      if norm_key(c) in ("CODIGO","CODIGO NORM","SKU","COD")]
        if marca_cols and cod_cols:
            pm = (plan_raw[[cod_cols[0], marca_cols[0]]]
                  .copy()
                  .rename(columns={cod_cols[0]:"_cod", marca_cols[0]:"_marca"}))
            pm["_cod"] = pm["_cod"].map(norm_key)
            pm = (pm.replace("",np.nan).dropna(subset=["_marca"])
                    .drop_duplicates("_cod",keep="last")
                    .set_index("_cod")["_marca"])
            stock["marca"] = stock["codigo_norm"].map(pm).fillna("Sin marca")

    # Fuente 2: para los que siguen "Sin marca", intentar con historial de ventas
    sin_marca = stock["marca"].eq("Sin marca")
    if sin_marca.any():
        marca_map = (
            comm.loc[comm["doc_type"]=="VENTA",["codigo_norm","marca"]]
            .replace("",np.nan).dropna(subset=["marca"])
            .groupby("codigo_norm")["marca"]
            .agg(lambda x: x.mode().iloc[0] if len(x)>0 else np.nan)
        )
        stock.loc[sin_marca,"marca"] = (
            stock.loc[sin_marca,"codigo_norm"].map(marca_map).fillna("Sin marca")
        )

    return stock

# ─────────────────────────────────────────────────────────────
# 8) NORMALIZACIÓN PLAN_OUT
# ─────────────────────────────────────────────────────────────
def _plan_accion(x):
    s = norm_key(x)
    if not s: return ""
    if "SIN STOCK" in s or "PEDIR YA" in s or "URGENTE" in s: return "Pedir ya"
    if "BAJO SS" in s or "ZONA DE RIESGO" in s: return "Pedir ya"
    if "PRONTO" in s: return "Pedir pronto"
    if "NO PEDIR" in s or "SIN DEMANDA" in s: return "No pedir"
    if "CAIDA" in s or "CAYÓ" in s: return "Caída demanda"
    if "CLAVO" in s or "MUY LENTO" in s or "MUY_LENTO" in s: return "No pedir"
    return norm_text(x)

def _aging_bucket_from_dias(x):
    if pd.isna(x): return "Sin dato"
    x = float(x)
    if x<=90:  return "0-90 días"
    if x<=180: return "91-180 días"
    if x<=360: return "181-360 días"
    return ">360 días"

def normalize_planout(df_raw):
    _ed = pd.DataFrame(columns=["campo","req","detectada","status"])
    if df_raw is None or df_raw.empty: return None,_ed,{}
    res = ColumnResolver(df_raw)
    m,diag = res.resolve(PLAN_SPECS)
    df = df_raw.copy()
    out = pd.DataFrame(index=df.index)
    g = lambda c,fn=norm_text: df[m[c]].map(fn) if m.get(c) else ("" if fn==norm_text else np.nan)
    out["codigo"]      = g("codigo")
    out["codigo_norm"] = out["codigo"].map(norm_key)
    out["articulo"]    = g("articulo")
    out["marca"]       = g("marca") if m.get("marca") else "Sin marca"
    out["abc"]         = g("abc")
    out["stock_total"] = to_num(df[m["stock_total"]]) if m.get("stock_total") else np.nan
    out["valor_usd"]   = to_num(df[m["valor_usd"]])   if m.get("valor_usd")   else np.nan
    out["tramo_aging"] = g("tramo_aging")
    out["estado_repo"] = g("estado_repo")
    out["dias_sv"]     = to_num(df[m["dias_sv"]])     if m.get("dias_sv")     else np.nan
    out["clasif_rot"]  = g("clasif_rot")
    out["qs_6m"]       = to_num(df[m["qs_6m"]])       if m.get("qs_6m")       else np.nan
    out["cobertura"]   = to_num(df[m["cobertura"]])   if m.get("cobertura")   else np.nan
    out["q_sugerida"]  = to_num(df[m["q_sugerida"]])  if m.get("q_sugerida")  else np.nan
    out["valor_q_usd"] = to_num(df[m["valor_q_usd"]]) if m.get("valor_q_usd") else np.nan
    out["ultima_venta"]= to_dt(df[m["ultima_venta"]])  if m.get("ultima_venta") else pd.NaT
    out["accion_raw"]  = g("accion")
    out["accion"]      = out["accion_raw"].map(_plan_accion)
    out["aging_bucket"]= np.where(
        out["tramo_aging"].astype(str).str.strip().ne(""),
        out["tramo_aging"],
        out["dias_sv"].map(_aging_bucket_from_dias)
    )
    return out,diag,m

# ─────────────────────────────────────────────────────────────
# 9) PERÍODOS Y SCOPES
# ─────────────────────────────────────────────────────────────
def compute_periods(comm):
    max_d = comm["fecha"].max().normalize()
    cy    = int(max_d.year)
    py    = cy-1
    ytd_s = pd.Timestamp(cy,1,1)
    py_s  = pd.Timestamp(py,1,1)
    day   = min(max_d.day, calendar.monthrange(py,max_d.month)[1])
    py_cut= pd.Timestamp(py,max_d.month,day)

    m_from = MESES_ES[1]
    m_to   = MESES_ES[max_d.month]
    ytd_label = f"{m_from}-{m_to} {cy}" if max_d.month > 1 else f"{m_to} {cy}"

    return {
        "max_date":    max_d, "current_year":cy, "prior_year":py,
        "ytd_start":   ytd_s, "py_start":py_s,   "py_cut":py_cut,
        "ytd_label":   ytd_label,
        "y2024_start": pd.Timestamp(2024,1,1), "y2024_end": pd.Timestamp(2024,12,31),
        "y2025_start": pd.Timestamp(2025,1,1), "y2025_end": pd.Timestamp(2025,12,31),
        "y2026_start": pd.Timestamp(2026,1,1), "y2026_end": max_d,
    }

def build_scopes(comm,p):
    f = comm["fecha"]
    return {
        "historico": comm.loc[f>=START_HISTORY_DATE].copy(),
        "ytd":       comm.loc[(f>=p["ytd_start"])&(f<=p["max_date"])].copy(),
        "ytd_py":    comm.loc[(f>=p["py_start"])&(f<=p["py_cut"])].copy(),
        "y2024":     comm.loc[(f>=p["y2024_start"])&(f<=p["y2024_end"])].copy(),
        "y2025":     comm.loc[(f>=p["y2025_start"])&(f<=p["y2025_end"])].copy(),
        "y2026":     comm.loc[(f>=p["y2026_start"])&(f<=p["y2026_end"])].copy(),
    }

# ─────────────────────────────────────────────────────────────
# 10) TABLAS DE NEGOCIO
# ─────────────────────────────────────────────────────────────
def _markup(profit, ventas):
    costo = ventas - profit
    return safe_div(profit, costo) * 100

def kpis(df):
    v = df["ventas_usd"].sum(skipna=True); p = df["profit_usd"].sum(skipna=True)
    va= df["ventas_ars"].sum(skipna=True); pa= df["profit_ars"].sum(skipna=True)
    ventas_fc = df.loc[df["doc_type"].eq("VENTA")] if "doc_type" in df.columns else df
    cli = ventas_fc["cliente"].replace("",np.nan).dropna().nunique()
    ml  = df.loc[df["canal_simple"].eq("MercadoLibre"),"ventas_usd"].sum(skipna=True)
    return {"ventas_usd":v,"profit_usd":p,"costo_usd":v-p,
            "margen_usd":safe_div(p,v)*100,
            "markup_usd":_markup(p,v),
            "ventas_ars":va,"profit_ars":pa,
            "margen_ars":safe_div(pa,va)*100,
            "clientes":cli,"unidades":df["unidades"].sum(skipna=True),
            "ml_share":safe_div(ml,v)*100}

def monthly_table(df):
    df_fc = df.loc[df["doc_type"].eq("VENTA")] if "doc_type" in df.columns else df
    base = (df.assign(mes=df["fecha"].dt.to_period("M").dt.to_timestamp())
              .groupby("mes",as_index=False)
              .agg(ventas_usd=("ventas_usd","sum"),profit_usd=("profit_usd","sum"),
                   ventas_ars=("ventas_ars","sum"),profit_ars=("profit_ars","sum"),
                   unidades=("unidades","sum"))
              .sort_values("mes"))
    cl_mes = (df_fc.assign(mes=df_fc["fecha"].dt.to_period("M").dt.to_timestamp())
                .groupby("mes")["cliente"]
                .apply(lambda s: s.replace("",np.nan).dropna().nunique())
                .rename("clientes").reset_index())
    base = base.merge(cl_mes, on="mes", how="left")
    base["clientes"]   = base["clientes"].fillna(0).astype(int)
    base["costo_usd"]  = base["ventas_usd"] - base["profit_usd"]
    base["margen_usd"] = base["profit_usd"]/base["ventas_usd"].replace(0,np.nan)*100
    base["markup_usd"] = base.apply(lambda r: _markup(r["profit_usd"],r["ventas_usd"]),axis=1)
    return base

def new_clients_monthly(df, meses_churn=6):
    """
    Clasifica clientes con compra cada mes en 3 categorías:
      - nuevos:      primera FC del cliente en el dataset fue ese mes
      - recurrentes: compraron ese mes Y también en los últimos `meses_churn`
                     (excluyendo el propio mes)
      - reactivados: compraron ese mes, ya habían comprado antes,
                     pero no en los últimos `meses_churn` meses
    """
    df_fc = df.loc[
        (df["doc_type"].eq("VENTA")) &
        (df["cliente"].replace("",np.nan).notna())
    ].copy()
    if df_fc.empty:
        return pd.DataFrame(columns=["mes","nuevos","recurrentes","reactivados","total"])

    df_fc["mes"] = df_fc["fecha"].dt.to_period("M").dt.to_timestamp()

    # Fecha de primera compra de cada cliente
    primera = df_fc.groupby("cliente")["fecha"].min().rename("primera_compra")
    df_fc = df_fc.merge(primera, on="cliente", how="left")
    df_fc["primer_mes"] = df_fc["primera_compra"].dt.to_period("M").dt.to_timestamp()

    # Por cada (cliente, mes) único vamos a decidir la categoría
    um = df_fc.drop_duplicates(subset=["mes","cliente"])[
        ["mes","cliente","primer_mes","fecha"]].copy()

    # Nuevo: el mes coincide con su primer_mes
    um["es_nuevo"] = um["mes"] == um["primer_mes"]

    # Para los que NO son nuevos, determinar si son recurrentes o reactivados
    # → buscar si el cliente tuvo otra compra en los `meses_churn` anteriores al mes actual
    compras_previas = df_fc[["cliente","mes"]].drop_duplicates()

    # Construir set de (cliente, mes) para lookup rápido
    compras_set = set(zip(compras_previas["cliente"], compras_previas["mes"]))

    def _tiene_compra_reciente(row):
        if row["es_nuevo"]:
            return True  # no aplica pero evita ruido
        mes_actual = row["mes"]
        cliente = row["cliente"]
        for k in range(1, meses_churn+1):
            mes_anterior = (mes_actual - pd.DateOffset(months=k)).normalize()
            if (cliente, mes_anterior) in compras_set:
                return True
        return False

    um["compra_reciente"] = um.apply(_tiene_compra_reciente, axis=1)

    def _cat(row):
        if row["es_nuevo"]: return "nuevos"
        if row["compra_reciente"]: return "recurrentes"
        return "reactivados"

    um["categoria"] = um.apply(_cat, axis=1)

    out = (um.groupby(["mes","categoria"])["cliente"].nunique()
             .reset_index().rename(columns={"cliente":"n"}))
    pivot = out.pivot(index="mes", columns="categoria", values="n").fillna(0).reset_index()
    pivot.columns.name = None
    for c in ["nuevos","recurrentes","reactivados"]:
        if c not in pivot.columns: pivot[c] = 0
    pivot["total"] = pivot["nuevos"] + pivot["recurrentes"] + pivot["reactivados"]
    return pivot.sort_values("mes")

def three_year_table(y2024,y2025,y2026,ytd_label):
    k4,k5,k6 = kpis(y2024),kpis(y2025),kpis(y2026)
    rows = [
        ("Ventas netas USD",  k4["ventas_usd"], k5["ventas_usd"], k6["ventas_usd"],
         pct_str(k5["ventas_usd"],k4["ventas_usd"]),pct_str(k6["ventas_usd"],k5["ventas_usd"])),
        ("Profit USD",        k4["profit_usd"], k5["profit_usd"], k6["profit_usd"],
         pct_str(k5["profit_usd"],k4["profit_usd"]),pct_str(k6["profit_usd"],k5["profit_usd"])),
        ("Markup % (profit/costo)", k4["markup_usd"],k5["markup_usd"],k6["markup_usd"],
         pp_str(k5["markup_usd"],k4["markup_usd"]),pp_str(k6["markup_usd"],k5["markup_usd"])),
        ("Margen % (profit/ventas)",k4["margen_usd"],k5["margen_usd"],k6["margen_usd"],
         pp_str(k5["margen_usd"],k4["margen_usd"]),pp_str(k6["margen_usd"],k5["margen_usd"])),
        ("Ventas netas ARS",  k4["ventas_ars"], k5["ventas_ars"], k6["ventas_ars"],
         pct_str(k5["ventas_ars"],k4["ventas_ars"]),pct_str(k6["ventas_ars"],k5["ventas_ars"])),
        ("Profit ARS",        k4["profit_ars"], k5["profit_ars"], k6["profit_ars"],
         pct_str(k5["profit_ars"],k4["profit_ars"]),pct_str(k6["profit_ars"],k5["profit_ars"])),
        ("Clientes c/ compra",k4["clientes"],   k5["clientes"],   k6["clientes"],
         pct_str(k5["clientes"],k4["clientes"]),pct_str(k6["clientes"],k5["clientes"])),
        ("Unidades netas",    k4["unidades"],   k5["unidades"],   k6["unidades"],
         pct_str(k5["unidades"],k4["unidades"]),pct_str(k6["unidades"],k5["unidades"])),
        ("ML share %",        k4["ml_share"],   k5["ml_share"],   k6["ml_share"],
         pp_str(k5["ml_share"],k4["ml_share"]),pp_str(k6["ml_share"],k5["ml_share"])),
    ]
    return pd.DataFrame(rows,columns=["indicador","2024","2025",ytd_label,"Δ25vs24","Δ26vs25"])

def brand_mix_table(y2024,y2025,y2026):
    g4,detail = add_other_brands(y2024.groupby("marca",as_index=False)["ventas_usd"].sum(),"marca","ventas_usd",KEY_BRANDS)
    g5,_      = add_other_brands(y2025.groupby("marca",as_index=False)["ventas_usd"].sum(),"marca","ventas_usd",KEY_BRANDS)
    g6,_      = add_other_brands(y2026.groupby("marca",as_index=False)["ventas_usd"].sum(),"marca","ventas_usd",KEY_BRANDS)
    g4=g4.rename(columns={"ventas_usd":"v2024"})
    g5=g5.rename(columns={"ventas_usd":"v2025"})
    g6=g6.rename(columns={"ventas_usd":"v2026"})
    out = g4.merge(g5,on="marca",how="outer").merge(g6,on="marca",how="outer").fillna(0)
    for yr in [2024,2025,2026]:
        col=f"v{yr}"; out[f"s{yr}"]=out[col]/out[col].sum()*100
    pm5 = y2025.groupby("marca",as_index=False)["profit_usd"].sum().rename(columns={"profit_usd":"p2025"})
    out = out.merge(pm5,on="marca",how="left").fillna(0)
    out["m2025"] = out["p2025"]/out["v2025"].replace(0,np.nan)*100
    return out.sort_values("v2025",ascending=False),detail

def top_clients_table(df,n=10):
    """
    Top clientes por ventas USD. Incluye:
    - ventas_usd, profit_usd (ganancia absoluta — lo que realmente dejó)
    - share_pct (peso sobre el total del período)
    - margen_pct (profit / ventas)
    - markup_pct (profit / costo — más útil para importador)
    - n_comprobantes (frecuencia: cuántas FC distintas tuvo)
    """
    df_fc = df.loc[df["doc_type"].eq("VENTA")] if "doc_type" in df.columns else df
    df_fc = df_fc[df_fc["cliente"].replace("",np.nan).notna()]

    # Agregados principales
    g = (df_fc.groupby("cliente",as_index=False)
           .agg(ventas_usd=("ventas_usd","sum"),
                profit_usd=("profit_usd","sum"),
                ventas_ars=("ventas_ars","sum"),
                unidades=("unidades","sum")))

    # Cantidad de comprobantes únicos (FC distintas) por cliente
    # Si existe columna 'comprobante', contamos pares (fecha, comprobante) únicos
    if "comprobante" in df_fc.columns:
        n_fc = (df_fc.groupby("cliente")
                 .apply(lambda d: d[["fecha","comprobante"]].drop_duplicates().shape[0])
                 .rename("n_comprobantes").reset_index())
        g = g.merge(n_fc, on="cliente", how="left")
    else:
        g["n_comprobantes"] = np.nan

    g = g.sort_values("ventas_usd",ascending=False).head(n)
    total = df_fc["ventas_usd"].sum()
    g["share_pct"]  = g["ventas_usd"]/total*100
    g["margen_pct"] = g["profit_usd"]/g["ventas_usd"].replace(0,np.nan)*100
    g["markup_pct"] = g.apply(lambda r: _markup(r["profit_usd"],r["ventas_usd"]), axis=1)
    return g

def _best_articulo_per_code(df):
    def _pick(arts):
        arts = [str(a).strip() for a in arts if str(a).strip()]
        if not arts: return ""
        non_numeric = [a for a in arts if not re.match(r"^\d+$",a)]
        pool = non_numeric if non_numeric else arts
        return max(pool, key=len)
    return (df.groupby("codigo_norm",as_index=False)
              .agg(articulo_best=("articulo",_pick))
              .set_index("codigo_norm")["articulo_best"])

def top_products_table(df, n=15, sort_by="ventas_usd"):
    """
    Top productos. Parámetro sort_by define el criterio:
      - "ventas_usd":  ranking clásico por volumen de ventas
      - "profit_usd":  los que más ganancia absoluta generaron
      - "markup_pct":  los que más dejan proporcionalmente (filtra productos con >=3 uds vendidas)
    """
    best_art = _best_articulo_per_code(df)
    df2 = df.copy()
    df2["articulo_display"] = df2["codigo_norm"].map(best_art).fillna(df2["articulo"])
    g = (df2.groupby(["codigo_norm","articulo_display","marca"],as_index=False)
            .agg(ventas_usd=("ventas_usd","sum"),
                 profit_usd=("profit_usd","sum"),
                 ventas_ars=("ventas_ars","sum"),
                 unidades=("unidades","sum")))
    total = df["ventas_usd"].sum()
    g["share_pct"]  = g["ventas_usd"]/total*100
    g["margen_pct"] = g["profit_usd"]/g["ventas_usd"].replace(0,np.nan)*100
    g["markup_pct"] = g.apply(lambda r: _markup(r["profit_usd"],r["ventas_usd"]), axis=1)

    if sort_by == "markup_pct":
        # Filtrar productos con volumen mínimo para evitar ruido de casos raros
        # (ventas > US$ 300 y unidades >= 3 es un filtro razonable)
        g = g.loc[(g["ventas_usd"] > 300) & (g["unidades"].fillna(0) >= 3)]

    g = g.sort_values(sort_by, ascending=False).head(n)
    return g

def inventory_summary(stock,plan):
    total      = stock["valor_stock_usd"].sum()
    stock_full = stock.loc[stock["deposito"].str.upper().str.contains("FULL",na=False),"valor_stock_usd"].sum()
    stock_cerr = stock.loc[stock["deposito"].str.upper().str.contains("CERRIT",na=False),"valor_stock_usd"].sum()
    dep_mix    = stock.groupby("deposito",as_index=False)["valor_stock_usd"].sum().sort_values("valor_stock_usd",ascending=False)
    brand_stock= stock.groupby("marca",as_index=False)["valor_stock_usd"].sum().sort_values("valor_stock_usd",ascending=False)

    if plan is not None and not plan.empty:
        p = plan.copy()
        if p["valor_usd"].isna().all() or (p["valor_usd"]==0).all():
            sb = stock.groupby("codigo_norm",as_index=False).agg(
                _val=("valor_stock_usd","sum"),_qty=("cantidad","sum"))
            p = p.merge(sb,on="codigo_norm",how="left",suffixes=("","_s"))
            p["valor_usd"]   = p["valor_usd"].fillna(p["_val"])
            p["stock_total"] = p["stock_total"].fillna(p["_qty"])
            p.drop(columns=[c for c in p.columns if c.endswith("_s") or c in ("_val","_qty")],inplace=True,errors="ignore")

        # Fallback si aging sigue vacío
        aging_vals = p["aging_bucket"].value_counts()
        n_sin_dato = aging_vals.get("Sin dato",0)
        if n_sin_dato/max(len(p),1) > 0.8:
            tramo_cols = [c for c in plan.columns if any(k in c.lower() for k in ["tramo","aging"])]
            if tramo_cols:
                p["aging_bucket"] = plan[tramo_cols[0]].fillna("Sin dato").astype(str)

        aging = p.groupby("aging_bucket",as_index=False)["valor_usd"].sum().sort_values("valor_usd",ascending=False)

        risk_mask = (
            p["aging_bucket"].isin(["CLAVO","MUY_LENTO"]) |
            p["accion"].isin(["No pedir","Caída demanda"]) |
            p["clasif_rot"].str.contains("MUY.BAJA",case=False,na=False)
        )
        capital_riesgo = p.loc[risk_mask,"valor_usd"].sum()

        pedir_ya_mask = (
            p["accion"].eq("Pedir ya") |
            p["estado_repo"].str.contains("SIN STOCK|PEDIR YA|BAJO SS",case=False,na=False) |
            (p["stock_total"].fillna(0)<=0)
        )
        pedir_pr_mask = (
            p["accion"].eq("Pedir pronto") |
            p["estado_repo"].str.contains("PEDIR PRONTO",case=False,na=False)
        )
        quiebres_mask = pedir_ya_mask | pedir_pr_mask | (p["qs_6m"].fillna(0)>0)

        risk_table = pd.concat([
            p.loc[pedir_ya_mask].assign(bloque="Pedir ya"),
            p.loc[pedir_pr_mask].assign(bloque="Pedir pronto"),
            p.loc[quiebres_mask].assign(bloque="Quiebres / riesgo"),
        ],ignore_index=True).drop_duplicates(subset=["codigo_norm","articulo","bloque"])
    else:
        aging       = pd.DataFrame({"aging_bucket":["Sin plan_out"],"valor_usd":[total]})
        capital_riesgo = np.nan
        risk_table  = pd.DataFrame()

    inv = {
        "capital_total":   total,
        "capital_riesgo":  capital_riesgo,
        "capital_full":    stock_full,
        "capital_cerrito": stock_cerr,
        "skus_activos":    stock["codigo_norm"].nunique(),
    }
    return inv,aging,dep_mix,brand_stock,risk_table

def build_recon(hist,inv):
    tmp = hist.copy()
    tmp["anio"] = tmp["fecha"].dt.year
    rep = tmp.groupby("anio").agg(v_ars=("ventas_ars","sum"),p_ars=("profit_ars","sum")).to_dict("index")
    rows = []
    for yr in sorted(SYSTEM_ARS.keys()):
        sv=SYSTEM_ARS[yr]["ventas"]; sp=SYSTEM_ARS[yr]["profit"]
        rv=rep.get(yr,{}).get("v_ars",0); rp=rep.get(yr,{}).get("p_ars",0)
        rows.append({"año":yr,"ventas_sistema":sv,"ventas_reporte":rv,"delta_ventas":sv-rv,
                     "profit_sistema":sp,"profit_reporte":rp,"delta_profit":sp-rp})
    return pd.DataFrame(rows)

def exec_conclusions(k_ytd,k_py,bm,tc,inv,risk,p,k2024,k2025,k2026):
    cy = p["current_year"]; yl = p["ytd_label"]
    bullets = []
    bullets.append(
        f"Ventas USD: {fmt_money(k2024['ventas_usd'],'USD',0)} (2024) → "
        f"{fmt_money(k2025['ventas_usd'],'USD',0)} (2025, {pct_str(k2025['ventas_usd'],k2024['ventas_usd'])}) → "
        f"{fmt_money(k2026['ventas_usd'],'USD',0)} ({yl})."
    )
    bullets.append(
        f"Markup: {fmt_pct(k2024['markup_usd'],1)} (2024) → "
        f"{fmt_pct(k2025['markup_usd'],1)} (2025) → "
        f"{fmt_pct(k2026['markup_usd'],1)} ({yl}).  "
        f"Margen: {fmt_pct(k2024['margen_usd'],1)} → {fmt_pct(k2025['margen_usd'],1)} → {fmt_pct(k2026['margen_usd'],1)}."
    )
    if not bm.empty and "v2025" in bm.columns:
        top = bm.iloc[0]
        bullets.append(
            f"Marca líder 2025: {top['marca']} ({fmt_pct(top.get('s2025',0),1)} del mix, "
            f"margen {fmt_pct(top.get('m2025',np.nan),1)})."
        )
    if not tc.empty:
        conc = tc.head(3)["share_pct"].sum()
        bullets.append(
            f"Top 3 clientes {yl}: {fmt_pct(conc,1)} de ventas. Líder: {shorten(tc.iloc[0]['cliente'],32)}."
        )
    cap=inv.get("capital_total",np.nan); rie=inv.get("capital_riesgo",np.nan)
    if pd.notna(cap):
        r_txt = f" | En riesgo (CLAVO/MUY_LENTO): {fmt_money(rie,'USD',0)}" if pd.notna(rie) and rie>0 else ""
        bullets.append(
            f"Stock: {fmt_money(cap,'USD',0)} (Cerrito {fmt_money(inv.get('capital_cerrito',np.nan),'USD',0)}"
            f" + FULL/ML {fmt_money(inv.get('capital_full',np.nan),'USD',0)}){r_txt}."
        )
    if risk is not None and not risk.empty:
        n_ya = int(risk.loc[risk["bloque"]=="Pedir ya","articulo"].nunique())
        n_pr = int(risk.loc[risk["bloque"]=="Pedir pronto","articulo"].nunique())
        bullets.append(f"Reposición: {n_ya} SKUs urgentes, {n_pr} próximos.")
    return bullets[:7]

# ─────────────────────────────────────────────────────────────
# 11) GRÁFICOS
# ─────────────────────────────────────────────────────────────
def chart_exec_summary(path,k2024,k2025,k2026,inv,conclusions,p):
    yl = p["ytd_label"]
    fig,ax = plt.subplots(figsize=FIGSIZE)
    ax.axis("off"); ax.set_facecolor("white")
    _banner(ax,"Resumen ejecutivo",
            f"2024 completo | 2025 completo | {yl}")
    cards = [
        ("Ventas 2024",     fmt_money(k2024["ventas_usd"],"USD",0), "",                                    TEAL),
        ("Ventas 2025",     fmt_money(k2025["ventas_usd"],"USD",0), pct_str(k2025["ventas_usd"],k2024["ventas_usd"]), BLUE),
        (f"Ventas {yl}",    fmt_money(k2026["ventas_usd"],"USD",0), pct_str(k2026["ventas_usd"],k2025["ventas_usd"]), GREEN),
        ("Capital stock",   fmt_money(inv.get("capital_total",np.nan),"USD",0), "", ORANGE),
        ("Capital riesgo",  fmt_money(inv.get("capital_riesgo",np.nan),"USD",0), "", RED),
    ]
    x0,y0,w,h,gap = 0.03,0.59,0.18,0.21,0.012
    for i,(lbl,val,dlt,col) in enumerate(cards):
        x = x0+i*(w+gap)
        ax.add_patch(patches.FancyBboxPatch((x,y0),w,h,boxstyle="round,pad=0.01",
            transform=ax.transAxes,facecolor=GRAY_XL,edgecolor=col,linewidth=2))
        ax.add_patch(patches.Rectangle((x,y0+h-0.032),w,0.032,transform=ax.transAxes,color=col))
        ax.text(x+w/2,y0+h-0.07,lbl,ha="center",va="center",fontsize=9.5,color=NAVY,transform=ax.transAxes)
        ax.text(x+w/2,y0+0.088,val,ha="center",va="center",fontsize=14,fontweight="bold",color=col,transform=ax.transAxes)
        if dlt:
            clr = "#c0392b" if str(dlt).startswith("-") else "#1b7a3e"
            ax.text(x+w/2,y0+0.022,dlt,ha="center",va="center",fontsize=10,color=clr,transform=ax.transAxes)

    ax.add_patch(patches.FancyBboxPatch((0.03,0.08),0.94,0.43,boxstyle="round,pad=0.01",
        transform=ax.transAxes,facecolor="#f3f6fb",edgecolor=GRAY_L))
    ax.text(0.05,0.495,"Conclusiones clave del período",transform=ax.transAxes,
            fontsize=12,fontweight="bold",color=NAVY,va="top")
    yy = 0.43
    for b in conclusions:
        wrapped = textwrap.fill(b,width=118)
        ax.text(0.05,yy,f"• {wrapped}",transform=ax.transAxes,fontsize=10.5,color="#243244",va="top")
        yy -= max(0.070,0.050+wrapped.count("\n")*0.042)
    add_footer(fig); save_fig(fig,path)


def chart_historical_trend(path,hist_m):
    """
    Panel 1: 3 líneas — Ventas USD, Costo USD, Profit USD
    Panel 2: Markup % + Clientes con compra (eje derecho)
    """
    if "markup_usd" not in hist_m.columns:
        hist_m = hist_m.copy()
        hist_m["markup_usd"] = hist_m.apply(lambda r: _markup(r["profit_usd"],r["ventas_usd"]),axis=1)
    if "costo_usd" not in hist_m.columns:
        hist_m = hist_m.copy()
        hist_m["costo_usd"] = hist_m["ventas_usd"] - hist_m["profit_usd"]

    fig,axes = plt.subplots(2,1,figsize=FIGSIZE,sharex=True)
    for ax in axes:
        ax.set_facecolor("white")
        ax.grid(axis="y",color=GRAY_L,linestyle="--",alpha=0.7)
        ax.spines[["right","top"]].set_visible(False)

    x = hist_m["mes"]
    colors_yr = {2024:TEAL,2025:BLUE,2026:GREEN}
    for yr,col in colors_yr.items():
        mask = hist_m["mes"].dt.year==yr
        if hist_m.loc[mask].empty: continue
        xs = hist_m.loc[mask,"mes"]
        for ax in axes: ax.axvspan(xs.min(),xs.max(),alpha=0.06,color=col,zorder=0)

    # Panel 1: Ventas + Costo + Profit (3 líneas)
    axes[0].plot(x,hist_m["ventas_usd"],color=BLUE,  marker="o",ms=5,linewidth=2.5,label="Ventas USD")
    axes[0].plot(x,hist_m["costo_usd"], color=ORANGE, marker="s",ms=4,linewidth=1.8,linestyle="--",alpha=0.85,label="Costo USD")
    axes[0].plot(x,hist_m["profit_usd"],color=GREEN,  marker="o",ms=4,linewidth=2.0,label="Profit USD")
    fmt,lbl = money_fmt(pd.concat([hist_m["ventas_usd"],hist_m["costo_usd"],hist_m["profit_usd"]]),"USD")
    axes[0].yaxis.set_major_formatter(fmt); axes[0].set_ylabel(lbl,fontsize=11)
    axes[0].set_title("Ventas / Costo / Profit USD mensuales — histórico desde 2024",
                      fontsize=14,fontweight="bold",color=NAVY)

    # Anotar máximo anual de ventas
    for yr in [2024,2025,2026]:
        mask = hist_m["mes"].dt.year==yr
        if hist_m.loc[mask].empty: continue
        idx_max = hist_m.loc[mask,"ventas_usd"].idxmax()
        xv=hist_m.loc[idx_max,"mes"]; yv=hist_m.loc[idx_max,"ventas_usd"]
        mu=hist_m.loc[idx_max,"markup_usd"]
        lbl_txt = fmt_money(yv,"USD",0)
        if pd.notna(mu): lbl_txt += f"\nmkup {fmt_pct(mu,1)}"
        axes[0].annotate(lbl_txt,(xv,yv),xytext=(0,10),textcoords="offset points",
                         ha="center",fontsize=7.5,color=BLUE)

    from matplotlib.patches import Patch
    legend_els = [Patch(facecolor=c,alpha=0.35,label=str(yr)) for yr,c in colors_yr.items()]
    axes[0].legend(handles=axes[0].get_lines()[:3]+legend_els,fontsize=10,ncol=6)

    # Panel 2: Markup % + Margen % + Clientes
    axes[1].plot(x,hist_m["markup_usd"],color=PURPLE,marker="o",ms=5,linewidth=2.5,
                 label="Markup % (profit/costo)")
    if "margen_usd" in hist_m.columns:
        axes[1].plot(x,hist_m["margen_usd"],color=GRAY,marker="",linewidth=1.2,
                     linestyle="--",alpha=0.6,label="Margen % (profit/ventas)")
    axes[1].yaxis.set_major_formatter(FuncFormatter(lambda v,_:f"{v:.0f}%"))
    axes[1].set_ylabel("Markup / Margen %",fontsize=11)
    ax2 = axes[1].twinx()
    ax2.plot(x,hist_m["clientes"],color=ORANGE,marker="s",ms=4,linewidth=1.8,
             linestyle="-",alpha=0.85,label="Clientes c/ compra")
    ax2.set_ylabel("Clientes con compra",color=ORANGE,fontsize=10)
    ax2.tick_params(axis="y",labelcolor=ORANGE,labelsize=9)
    axes[1].set_title("Evolución Markup % y clientes con compra mensual",
                      fontsize=13,fontweight="bold",color=NAVY)
    axes[1].legend(loc="upper left",fontsize=9)
    ax2.legend(loc="upper right",fontsize=9)
    fig.autofmt_xdate(rotation=40); add_footer(fig); save_fig(fig,path)


def _fmt_kv(v,ind):
    if pd.isna(v): return "—"
    il = ind.lower()
    if "usd" in il and "%" not in il and "markup" not in il and "margen" not in il: return fmt_money(v,"USD",0)
    if "ars" in il and "%" not in il: return fmt_money(v,"ARS",0)
    if any(k in il for k in ["%","margen","markup","share","ml"]): return fmt_pct(v,1)
    return fmt_num(v,0)

def _delta_clr(s):
    if not s or s in ("—","∞"): return GRAY
    return "#c0392b" if s.startswith("-") else "#1b7a3e"

def chart_three_years(path,table,p):
    yl = p["ytd_label"]
    n_rows = len(table)
    fig,ax = plt.subplots(figsize=FIGSIZE)
    ax.axis("off"); ax.set_facecolor("white")
    _banner(ax,f"Análisis comparativo — 2024 / 2025 / {yl}",
            "Markup = profit / costo × 100  |  Margen = profit / ventas × 100")

    xpos = [0.03,0.30,0.47,0.64,0.79,0.91]
    hdrs = ["Indicador","2024","2025",yl,"Δ 25 vs 24","Δ 26 vs 25"]
    y0 = 0.825; row_h = min(0.075, 0.78/max(n_rows,1))
    ax.add_patch(patches.Rectangle((0.02,y0),0.96,0.050,transform=ax.transAxes,color=NAVY))
    for x,h in zip(xpos,hdrs):
        ax.text(x,y0+0.025,h,transform=ax.transAxes,fontsize=11,fontweight="bold",color="white",va="center")

    HIGHLIGHT_ROWS = {"markup % (profit/costo)","margen % (profit/ventas)"}
    y = y0-row_h*0.5
    col_yl = yl  # nombre de la columna dinámica
    for i,(_,row) in enumerate(table.iterrows()):
        ind = str(row["indicador"])
        is_metric = ind.lower() in HIGHLIGHT_ROWS
        bg = "#FFF8E1" if is_metric else ("white" if i%2==0 else "#eef2f7")
        ax.add_patch(patches.Rectangle((0.02,y-row_h*0.45),0.96,row_h*0.88,transform=ax.transAxes,color=bg))
        v_yl = row.get(col_yl, np.nan)
        vals = [ind, _fmt_kv(row["2024"],ind), _fmt_kv(row["2025"],ind), _fmt_kv(v_yl,ind),
                str(row["Δ25vs24"]) if not pd.isna(row["Δ25vs24"]) else "—",
                str(row["Δ26vs25"]) if not pd.isna(row["Δ26vs25"]) else "—"]
        clrs = [NAVY,TEAL,BLUE,GREEN,_delta_clr(str(row["Δ25vs24"])),_delta_clr(str(row["Δ26vs25"]))]
        for x,v,c in zip(xpos,vals,clrs):
            ax.text(x,y,v,transform=ax.transAxes,fontsize=11.5,color=c,va="center",
                    fontweight="bold" if is_metric and x>0.28 else "normal")
        y -= row_h

    ax.text(0.03,0.025,
            "Markup % = profit / costo × 100  |  Ejemplo: markup 40% → ganás $40 por cada $100 de costo.",
            transform=ax.transAxes,fontsize=9,color=GRAY,va="bottom",style="italic")
    add_footer(fig); save_fig(fig,path)


def chart_reconciliation(path,recon):
    fig,ax = plt.subplots(figsize=FIGSIZE)
    ax.axis("off"); ax.set_facecolor("white")
    _banner(ax,"Conciliación vs sistema (ARS)",
            "FC/NC — excluye compras, presupuestos, ajustes ND, remitos")
    hdrs = ["Año","Ventas Sistema","Ventas Reporte","Δ Ventas","Profit Sistema","Profit Reporte","Δ Profit"]
    xpos = [0.03,0.16,0.31,0.46,0.59,0.74,0.89]; y0=0.78
    ax.add_patch(patches.Rectangle((0.02,y0),0.96,0.055,transform=ax.transAxes,color=BLUE))
    for x,h in zip(xpos,hdrs):
        ax.text(x,y0+0.027,h,transform=ax.transAxes,fontsize=9.5,fontweight="bold",color="white",va="center")
    y=y0-0.065
    for i,(_,row) in enumerate(recon.iterrows()):
        bg="white" if i%2==0 else "#eef2f7"
        ax.add_patch(patches.Rectangle((0.02,y-0.028),0.96,0.056,transform=ax.transAxes,color=bg))
        vals=[str(int(row["año"])),
              fmt_money(row["ventas_sistema"],"ARS",0),fmt_money(row["ventas_reporte"],"ARS",0),fmt_money(row["delta_ventas"],"ARS",0),
              fmt_money(row["profit_sistema"],"ARS",0),fmt_money(row["profit_reporte"],"ARS",0),fmt_money(row["delta_profit"],"ARS",0)]
        for x,v in zip(xpos,vals): ax.text(x,y,v,transform=ax.transAxes,fontsize=10.5,color=NAVY,va="center")
        y -= 0.065
    ax.add_patch(patches.FancyBboxPatch((0.02,0.07),0.96,0.30,boxstyle="round,pad=0.01",
        transform=ax.transAxes,facecolor="#fffbea",edgecolor=GOLD,linewidth=2))
    ax.text(0.04,0.355,"Causas de las diferencias:",transform=ax.transAxes,fontsize=11,fontweight="bold",color="#7a5800",va="top")
    ax.text(0.04,0.300,textwrap.fill(CAUSA_DIFFS,width=138),transform=ax.transAxes,fontsize=10,color="#5a4000",va="top")
    add_footer(fig); save_fig(fig,path)


def chart_brand_mix(path,bm,detail,p):
    yl = p["ytd_label"]
    fig,ax = plt.subplots(figsize=FIGSIZE)
    ax.set_facecolor("white"); ax.grid(axis="x",color=GRAY_L,linestyle="--",alpha=0.7)
    ax.spines[["right","top"]].set_visible(False)
    df = bm.copy()
    for c in ["v2024","v2025","v2026"]:
        if c not in df.columns: df[c]=0
    df = df.sort_values("v2025",ascending=True)
    y = np.arange(len(df)); bh=0.25
    ax.barh(y-bh,df["v2024"],height=bh-0.02,color=TEAL,label="2024")
    ax.barh(y,   df["v2025"],height=bh-0.02,color=BLUE,label="2025")
    ax.barh(y+bh,df["v2026"],height=bh-0.02,color=GREEN,label=yl)
    ax.set_yticks(y); ax.set_yticklabels(df["marca"],fontsize=12)
    fmtr,lbl = money_fmt(pd.concat([df["v2024"],df["v2025"],df["v2026"]]),"USD")
    ax.xaxis.set_major_formatter(fmtr); ax.set_xlabel(f"({lbl})",fontsize=11)
    if "m2025" in df.columns:
        vmax = float(df["v2025"].max()) if not df.empty else 1
        for i,(_,row) in enumerate(df.iterrows()):
            if row["v2025"]>0:
                ax.text(row["v2025"]+vmax*0.01,i,f" margen 2025: {fmt_pct(row['m2025'],1)}",va="center",fontsize=9,color=NAVY)
    ax.set_title(f"Mix ventas por marca — 2024 / 2025 / {yl} (USD)",fontsize=15,fontweight="bold",color=NAVY)
    ax.legend(fontsize=11)
    if detail: fig.text(0.01,0.01,f"Otras incluye: {detail}",ha="left",va="bottom",fontsize=9,color=GRAY)
    add_footer(fig); save_fig(fig,path)


def chart_top_clients(path,tc,p):
    """
    Gráfico de top clientes con información enriquecida a la derecha de cada barra:
      profit USD (lo que dejó) | markup % (proporcional) | share total | comprobantes (recurrencia)
    """
    yl = p["ytd_label"]
    df = tc.sort_values("ventas_usd",ascending=True)
    df = df[df["cliente"].replace("",np.nan).notna()]
    fig,ax = plt.subplots(figsize=FIGSIZE)
    ax.set_facecolor("white")
    ax.barh(df["cliente"].map(lambda x:shorten(x,34)),df["ventas_usd"],
            color=BLUE,edgecolor="white",height=0.65)
    ax.grid(axis="x",color=GRAY_L,linestyle="--",alpha=0.7)
    ax.spines[["right","top"]].set_visible(False)
    fmtr,lbl = money_fmt(df["ventas_usd"],"USD")
    ax.xaxis.set_major_formatter(fmtr); ax.set_xlabel(f"Ventas USD ({lbl})",fontsize=11)
    vmax = float(df["ventas_usd"].max()) if not df.empty else 1

    for i,(_,row) in enumerate(df.iterrows()):
        profit_txt = fmt_money(row.get("profit_usd",np.nan),"USD",0)
        markup_txt = fmt_pct(row.get("markup_pct",np.nan),1)
        share_txt  = fmt_pct(row.get("share_pct",np.nan),1)
        n_fc       = row.get("n_comprobantes",np.nan)
        fc_txt = f"{int(n_fc)} FC" if pd.notna(n_fc) else ""
        parts = [f"profit {profit_txt}", f"markup {markup_txt}", f"{share_txt} del total"]
        if fc_txt: parts.append(fc_txt)
        ax.text(row["ventas_usd"]+vmax*0.008, i,
                "  " + "  |  ".join(parts),
                va="center", fontsize=9.5, color="#243244")

    ax.set_title(f"Top clientes {yl} — ventas USD\n"
                 "Anotado: profit absoluto · markup (profit/costo) · share del total · cantidad de FC",
                 fontsize=14,fontweight="bold",color=NAVY)
    add_footer(fig); save_fig(fig,path)


def chart_new_clients(path,comm,p):
    """
    Panel 1: líneas Total + Recurrentes (últ. 6m) + Nuevos + Reactivados.
    Panel 2: barra apilada de composición mensual (% de cada categoría).
    """
    nc = new_clients_monthly(comm, meses_churn=6)
    if nc.empty:
        fig,ax = plt.subplots(figsize=FIGSIZE); ax.axis("off")
        ax.text(0.5,0.5,"Sin datos",ha="center",va="center",fontsize=16,color=GRAY,transform=ax.transAxes)
        add_footer(fig); save_fig(fig,path); return

    nc = nc.loc[nc["mes"]>=pd.Timestamp(START_HISTORY_DATE)].copy()

    fig,axes = plt.subplots(2,1,figsize=FIGSIZE,sharex=True,
                             gridspec_kw={"height_ratios":[2,1.2]})
    for ax in axes:
        ax.set_facecolor("white")
        ax.grid(axis="y",color=GRAY_L,linestyle="--",alpha=0.6)
        ax.spines[["right","top"]].set_visible(False)

    x = nc["mes"]
    colors_yr = {2024:TEAL,2025:BLUE,2026:GREEN}
    for yr,col in colors_yr.items():
        mask = nc["mes"].dt.year==yr
        if nc.loc[mask].empty: continue
        xs = nc.loc[mask,"mes"]
        for ax in axes: ax.axvspan(xs.min(),xs.max(),alpha=0.07,color=col,zorder=0)

    # Panel 1 — líneas con área sombreada
    axes[0].fill_between(x, nc["recurrentes"], alpha=0.20, color=BLUE)
    axes[0].plot(x, nc["total"],       color=NAVY,  linewidth=2.5, marker="o", ms=4, label="Total con compra")
    axes[0].plot(x, nc["recurrentes"], color=BLUE,  linewidth=1.8, marker="s", ms=3, label="Recurrentes (últ. 6m)")
    axes[0].plot(x, nc["nuevos"],      color=GREEN, linewidth=1.8, marker="^", ms=4, label="Nuevos (primera compra)")
    axes[0].plot(x, nc["reactivados"], color=ORANGE, linewidth=1.8, marker="D", ms=3, label="Reactivados (>6m sin comprar)")
    axes[0].set_ylabel("Clientes con compra mensual",fontsize=11)
    axes[0].set_title("Clientes con compra — Nuevos / Recurrentes / Reactivados\n"
                      "Recurrente = compró en alguno de los 6 meses previos  |  "
                      "Reactivado = volvió después de >6 meses sin comprar",
                      fontsize=13,fontweight="bold",color=NAVY)
    axes[0].legend(fontsize=10,loc="upper left",ncol=2)

    # Anotar máximo total de cada año
    for yr in [2024,2025,2026]:
        mask = nc["mes"].dt.year==yr
        if nc.loc[mask].empty: continue
        idx = nc.loc[mask,"total"].idxmax()
        xv=nc.loc[idx,"mes"]; yv=nc.loc[idx,"total"]
        axes[0].annotate(f"{int(yv)}",(xv,yv),xytext=(0,10),textcoords="offset points",
                         ha="center",fontsize=9,color=NAVY,fontweight="bold")

    # Panel 2 — composición % apilada
    total_safe = nc["total"].replace(0,np.nan)
    pct_rec = nc["recurrentes"] / total_safe * 100
    pct_new = nc["nuevos"]      / total_safe * 100
    pct_rea = nc["reactivados"] / total_safe * 100

    # Ancho de barra: elegir algo razonable para que se vean bien (~25 días)
    bw_days = 24
    axes[1].bar(x, pct_rec, bw_days, color=BLUE,   alpha=0.85, label="Recurrentes")
    axes[1].bar(x, pct_rea, bw_days, bottom=pct_rec, color=ORANGE, alpha=0.85, label="Reactivados")
    axes[1].bar(x, pct_new, bw_days, bottom=pct_rec+pct_rea, color=GREEN, alpha=0.85, label="Nuevos")
    axes[1].yaxis.set_major_formatter(FuncFormatter(lambda v,_:f"{v:.0f}%"))
    axes[1].set_ylim(0,105); axes[1].set_ylabel("Composición %",fontsize=11)
    axes[1].set_title(f"Composición mensual de clientes — "
                      f"promedios: {pct_new.mean():.0f}% nuevos, "
                      f"{pct_rec.mean():.0f}% recurrentes, "
                      f"{pct_rea.mean():.0f}% reactivados",
                      fontsize=11,fontweight="bold",color=NAVY)
    axes[1].legend(fontsize=10, loc="upper right", ncol=3)
    fig.autofmt_xdate(rotation=35)
    plt.tight_layout(); add_footer(fig); save_fig(fig,path)


def chart_top_products(path, tp, p, criterio="ventas"):
    """
    Top productos. `criterio` define qué se grafica y cómo se anota:
      - "ventas": barra = ventas USD, anotado: profit · markup · margen
      - "markup": barra = markup %, anotado: ventas · profit · unidades
      - "profit": barra = profit USD, anotado: ventas · markup · unidades
    """
    yl = p["ytd_label"]
    label_col = "articulo_display" if "articulo_display" in tp.columns else "articulo"

    if criterio == "markup":
        df = tp.sort_values("markup_pct", ascending=True)
        labels = df[label_col].map(lambda x: shorten(x, 48))
        fig,ax = plt.subplots(figsize=FIGSIZE)
        ax.set_facecolor("white")
        ax.barh(labels, df["markup_pct"], color=PURPLE, edgecolor="white", height=0.65)
        ax.grid(axis="x", color=GRAY_L, linestyle="--", alpha=0.7)
        ax.spines[["right","top"]].set_visible(False)
        ax.xaxis.set_major_formatter(FuncFormatter(lambda v,_: f"{v:.0f}%"))
        ax.set_xlabel("Markup % (profit / costo)", fontsize=11)
        vmax = float(df["markup_pct"].max()) if not df.empty else 100
        for i,(_,row) in enumerate(df.iterrows()):
            un = int(row.get("unidades",0) or 0)
            ax.text(row["markup_pct"]+vmax*0.01, i,
                    f"  ventas {fmt_money(row['ventas_usd'],'USD',0)}  |  "
                    f"profit {fmt_money(row['profit_usd'],'USD',0)}  |  "
                    f"{un} u.",
                    va="center", fontsize=9.5, color="#243244")
        ax.set_title(f"Top productos {yl} — por markup (los que más dejan proporcionalmente)\n"
                     "Filtrado: mínimo US$ 300 de ventas y 3 unidades",
                     fontsize=14, fontweight="bold", color=NAVY)

    elif criterio == "profit":
        df = tp.sort_values("profit_usd", ascending=True)
        labels = df[label_col].map(lambda x: shorten(x, 48))
        fig,ax = plt.subplots(figsize=FIGSIZE)
        ax.set_facecolor("white")
        ax.barh(labels, df["profit_usd"], color=ORANGE, edgecolor="white", height=0.65)
        ax.grid(axis="x", color=GRAY_L, linestyle="--", alpha=0.7)
        ax.spines[["right","top"]].set_visible(False)
        fmtr,lbl = money_fmt(df["profit_usd"], "USD")
        ax.xaxis.set_major_formatter(fmtr); ax.set_xlabel(f"Profit USD ({lbl})", fontsize=11)
        vmax = float(df["profit_usd"].max()) if not df.empty else 1
        for i,(_,row) in enumerate(df.iterrows()):
            un = int(row.get("unidades",0) or 0)
            ax.text(row["profit_usd"]+vmax*0.01, i,
                    f"  ventas {fmt_money(row['ventas_usd'],'USD',0)}  |  "
                    f"markup {fmt_pct(row['markup_pct'],1)}  |  "
                    f"{un} u.",
                    va="center", fontsize=9.5, color="#243244")
        ax.set_title(f"Top productos {yl} — por profit absoluto (los que más ganancia generaron)",
                     fontsize=14, fontweight="bold", color=NAVY)

    else:  # criterio == "ventas" (default)
        df = tp.sort_values("ventas_usd", ascending=True)
        labels = df[label_col].map(lambda x: shorten(x, 48))
        fig,ax = plt.subplots(figsize=FIGSIZE)
        ax.set_facecolor("white")
        ax.barh(labels, df["ventas_usd"], color=GREEN, edgecolor="white", height=0.65)
        ax.grid(axis="x", color=GRAY_L, linestyle="--", alpha=0.7)
        ax.spines[["right","top"]].set_visible(False)
        fmtr,lbl = money_fmt(df["ventas_usd"], "USD")
        ax.xaxis.set_major_formatter(fmtr); ax.set_xlabel(f"Ventas USD ({lbl})", fontsize=11)
        vmax = float(df["ventas_usd"].max()) if not df.empty else 1
        for i,(_,row) in enumerate(df.iterrows()):
            ax.text(row["ventas_usd"]+vmax*0.008, i,
                    f"  profit {fmt_money(row['profit_usd'],'USD',0)}  |  "
                    f"markup {fmt_pct(row['markup_pct'],1)}  |  "
                    f"{fmt_pct(row['share_pct'],1)} del total",
                    va="center", fontsize=9.5, color="#243244")
        ax.set_title(f"Top productos {yl} — por ventas USD\n"
                     "Anotado: profit absoluto · markup (profit/costo) · share del total",
                     fontsize=14, fontweight="bold", color=NAVY)

    ax.tick_params(axis="y", labelsize=10)
    plt.tight_layout(); add_footer(fig); save_fig(fig,path)


def chart_inventory_photo(path,inv,brand_stock):
    """
    Barras horizontales: capital en stock por marca.
    Marca obtenida cruzando df_valorizacion_stock con df_merged_usd por Codigo
    (realizado en build_report antes de llamar inventory_summary).
    """
    bs = brand_stock.copy()
    bs = bs[bs["valor_stock_usd"]>0].sort_values("valor_stock_usd",ascending=True)

    if bs.empty:
        fig,ax = plt.subplots(figsize=FIGSIZE); ax.axis("off"); ax.set_facecolor("white")
        ax.text(0.5,0.5,"Sin datos de stock",ha="center",va="center",fontsize=16,color=GRAY,transform=ax.transAxes)
        add_footer(fig); save_fig(fig,path); return

    total = bs["valor_stock_usd"].sum()
    bar_colors = [BRAND_COLORS[i%len(BRAND_COLORS)] for i in range(len(bs))]

    fig,ax = plt.subplots(figsize=FIGSIZE)
    ax.set_facecolor("white")
    ax.grid(axis="x",color=GRAY_L,linestyle="--",alpha=0.7)
    ax.spines[["right","top"]].set_visible(False)

    bars = ax.barh(bs["marca"],bs["valor_stock_usd"],color=bar_colors,edgecolor="white",height=0.65)

    fmtr,lbl = money_fmt(bs["valor_stock_usd"],"USD")
    ax.xaxis.set_major_formatter(fmtr)
    ax.set_xlabel(f"Capital en stock ({lbl})",fontsize=11)

    vmax = float(bs["valor_stock_usd"].max())
    for i,(_,row) in enumerate(bs.iterrows()):
        v = float(row["valor_stock_usd"])
        pct = v/total*100 if total>0 else 0
        ax.text(v+vmax*0.01,i,f"  {fmt_money(v,'USD',0)}  ({pct:.1f}%)",
                va="center",fontsize=10,color="#243244")

    total_txt = fmt_money(inv.get("capital_total",np.nan),"USD",0)
    cerrito   = fmt_money(inv.get("capital_cerrito",np.nan),"USD",0)
    full      = fmt_money(inv.get("capital_full",np.nan),"USD",0)
    riesgo    = fmt_money(inv.get("capital_riesgo",np.nan),"USD",0)

    ax.set_title(
        f"Capital en stock por marca\n"
        f"Total {total_txt}  |  Cerrito {cerrito}  |  FULL/ML {full}  |  En riesgo {riesgo}",
        fontsize=14,fontweight="bold",color=NAVY)
    ax.tick_params(axis="y",labelsize=11)
    plt.tight_layout(); add_footer(fig); save_fig(fig,path)


def chart_inventory_health(path, plan, inv):
    """
    Dashboard ejecutivo de salud del inventario.
    Clasifica el capital en 3 zonas (semáforo) según Tramo_aging del plan_out:
      🟢 Saludable  = OK + NUEVO            → rota dentro del ciclo
      🟡 Atención   = ATENCION + LENTO      → a monitorear / mejorar precio
      🔴 Riesgo     = MUY_LENTO + CLAVO     → liquidar / no volver a pedir
    Además muestra productos clase A en riesgo operativo y capital crítico.
    """
    fig = plt.figure(figsize=FIGSIZE)
    fig.patch.set_facecolor("white")

    if plan is None or plan.empty:
        ax = fig.add_subplot(111); ax.axis("off")
        _banner(ax,"Inventario — salud del stock","")
        ax.text(0.05,0.55,"plan_out no disponible. Corré el inventario unificado primero.",
                transform=ax.transAxes,fontsize=14,color=GRAY)
        add_footer(fig); save_fig(fig,path); return

    p = plan.copy()

    # Mapa tramo → zona semáforo
    zona_map = {
        "OK":"Saludable","NUEVO":"Saludable",
        "ATENCION":"Atención","LENTO":"Atención",
        "MUY_LENTO":"Riesgo","CLAVO":"Riesgo",
        "SIN STOCK":"Sin stock","SIN DEMANDA":"Sin stock",
    }
    p["zona"] = p["tramo_aging"].map(lambda x: zona_map.get(str(x).strip().upper(),"Sin dato"))
    p_con_stock = p.loc[p["stock_total"].fillna(0)>0].copy()
    total_cap = p_con_stock["valor_usd"].sum()

    # Agrupados por zona (solo con stock > 0)
    grp_zona = (p_con_stock.groupby("zona",as_index=False)
                   .agg(skus=("codigo_norm","nunique"),capital=("valor_usd","sum")))
    def _pick(z):
        r = grp_zona.loc[grp_zona["zona"]==z]
        return (int(r["skus"].iloc[0]) if not r.empty else 0,
                float(r["capital"].iloc[0]) if not r.empty else 0.0)
    n_sa,c_sa = _pick("Saludable")
    n_at,c_at = _pick("Atención")
    n_ri,c_ri = _pick("Riesgo")
    total_skus = n_sa+n_at+n_ri if (n_sa+n_at+n_ri)>0 else 1

    # ═══ BANNER ═══
    ax_ban = fig.add_axes([0,0.90,1,0.10]); ax_ban.axis("off")
    ax_ban.add_patch(patches.Rectangle((0,0),1,1,transform=ax_ban.transAxes,color=NAVY))
    ax_ban.text(0.03,0.55,"Inventario — salud del stock",color="white",
                fontsize=21,fontweight="bold",transform=ax_ban.transAxes,va="center")
    ax_ban.text(0.03,0.15,"Clasificación del capital por velocidad de rotación (tramo aging)",
                color="#c3d0ea",fontsize=10,transform=ax_ban.transAxes)

    # ═══ 3 TARJETAS SEMÁFORO ═══
    ax_cards = fig.add_axes([0,0.66,1,0.22]); ax_cards.axis("off")
    GREEN_SOFT = "#d4edda"; YELLOW_SOFT = "#fff3cd"; RED_SOFT = "#f8d7da"
    cards = [
        ("🟢 Saludable","OK + NUEVO",      n_sa,c_sa, GREEN, GREEN_SOFT,  "rota dentro del ciclo"),
        ("🟡 Atención", "ATENCIÓN + LENTO",n_at,c_at, ORANGE,YELLOW_SOFT, "revisar precio / destacar"),
        ("🔴 Riesgo",   "MUY_LENTO + CLAVO",n_ri,c_ri,RED,   RED_SOFT,    "liquidar / no reponer"),
    ]
    x0,w,gap = 0.04,0.30,0.015
    for i,(t,sub,sk,cap,cbar,cbg,foot) in enumerate(cards):
        x = x0+i*(w+gap)
        ax_cards.add_patch(patches.FancyBboxPatch((x,0.05),w,0.92,boxstyle="round,pad=0.01",
            transform=ax_cards.transAxes,facecolor=cbg,edgecolor=cbar,linewidth=2.5))
        ax_cards.add_patch(patches.Rectangle((x,0.85),w,0.12,
            transform=ax_cards.transAxes,color=cbar))
        ax_cards.text(x+w/2,0.91,t,ha="center",va="center",color="white",
                      fontsize=13,fontweight="bold",transform=ax_cards.transAxes)
        ax_cards.text(x+w/2,0.72,sub,ha="center",va="center",color=NAVY,
                      fontsize=9.5,transform=ax_cards.transAxes)
        ax_cards.text(x+w/2,0.48,fmt_money(cap,"USD",0),ha="center",va="center",
                      fontsize=21,fontweight="bold",color=cbar,transform=ax_cards.transAxes)
        pct_cap = cap/total_cap*100 if total_cap>0 else 0
        pct_sku = sk/total_skus*100 if total_skus>0 else 0
        ax_cards.text(x+w/2,0.28,f"{sk} SKUs  ({pct_sku:.0f}% del total)",ha="center",va="center",
                      fontsize=10,color=NAVY,transform=ax_cards.transAxes)
        ax_cards.text(x+w/2,0.17,f"{pct_cap:.1f}% del capital",ha="center",va="center",
                      fontsize=10,color=NAVY,fontweight="bold",transform=ax_cards.transAxes)
        ax_cards.text(x+w/2,0.07,foot,ha="center",va="center",
                      fontsize=9,color=GRAY,style="italic",transform=ax_cards.transAxes)

    # ═══ PANEL IZQUIERDO: barra apilada horizontal del capital por zona ═══
    ax_bar = fig.add_axes([0.04,0.40,0.54,0.22])
    ax_bar.set_facecolor("white")
    ax_bar.spines[["right","top","left"]].set_visible(False)
    ax_bar.set_yticks([])
    caps = [c_sa,c_at,c_ri]; colors_z = [GREEN,ORANGE,RED]; labels_z = ["Saludable","Atención","Riesgo"]
    left = 0
    for cap,col,lb in zip(caps,colors_z,labels_z):
        if cap<=0: continue
        ax_bar.barh(0,cap,left=left,height=0.55,color=col,edgecolor="white",linewidth=3)
        pct = cap/total_cap*100 if total_cap>0 else 0
        if pct >= 4:
            ax_bar.text(left+cap/2,0,f"{lb}\n{fmt_money(cap,'USD',0)}\n{pct:.1f}%",
                        ha="center",va="center",fontsize=10,color="white",fontweight="bold")
        left += cap
    ax_bar.set_xlim(0,total_cap*1.02 if total_cap>0 else 1)
    ax_bar.set_ylim(-0.5,0.5)
    fmtr,lbl = money_fmt(pd.Series(caps),"USD")
    ax_bar.xaxis.set_major_formatter(fmtr)
    ax_bar.set_xlabel(f"Capital en stock ({lbl})",fontsize=10)
    ax_bar.set_title("Distribución del capital por zona de salud",
                     fontsize=12,fontweight="bold",color=NAVY,pad=8,loc="left")

    # ═══ PANEL DERECHO: tabla detalle por tramo ═══
    ax_tab = fig.add_axes([0.60,0.36,0.37,0.28]); ax_tab.axis("off")

    # Orden de tramos para la tabla
    tramo_order = ["CLAVO","MUY_LENTO","LENTO","ATENCION","OK","NUEVO"]
    tramo_colors = {"CLAVO":RED,"MUY_LENTO":"#e07b5f","LENTO":ORANGE,
                    "ATENCION":GOLD,"OK":GREEN,"NUEVO":"#88d9b0"}

    grp_tramo = (p_con_stock.groupby("tramo_aging",as_index=False)
                    .agg(skus=("codigo_norm","nunique"),capital=("valor_usd","sum")))
    grp_tramo["tramo_aging"] = grp_tramo["tramo_aging"].astype(str).str.upper()
    grp_tramo = grp_tramo.set_index("tramo_aging")

    hdrs = ["Tramo","SKUs","Capital","% cap."]
    xp   = [0.02,0.40,0.56,0.86]
    ax_tab.add_patch(patches.Rectangle((0,0.92),1,0.08,transform=ax_tab.transAxes,color=NAVY))
    for x,h in zip(xp,hdrs):
        ax_tab.text(x,0.96,h,transform=ax_tab.transAxes,fontsize=10,
                    fontweight="bold",color="white",va="center")
    yy = 0.85; rh = 0.13
    for tr in tramo_order:
        if tr not in grp_tramo.index: continue
        row = grp_tramo.loc[tr]
        c_dot = tramo_colors.get(tr,GRAY)
        ax_tab.add_patch(patches.Circle((0.035,yy),0.025,transform=ax_tab.transAxes,
                                         color=c_dot,clip_on=False))
        pct = row["capital"]/total_cap*100 if total_cap>0 else 0
        vals = [tr,str(int(row["skus"])),fmt_money(row["capital"],"USD",0),f"{pct:.1f}%"]
        for x,v in zip(xp,vals):
            ax_tab.text(x+(0.04 if x==xp[0] else 0),yy,v,transform=ax_tab.transAxes,
                        fontsize=10,color=NAVY,va="center",
                        fontweight="bold" if tr in ("CLAVO","MUY_LENTO") else "normal")
        yy -= rh

    # ═══ ZONA INFERIOR: alertas críticas ═══
    ax_al = fig.add_axes([0.04,0.04,0.93,0.27]); ax_al.axis("off")
    ax_al.add_patch(patches.FancyBboxPatch((0,0),1,1,boxstyle="round,pad=0.01",
        transform=ax_al.transAxes,facecolor="#fff6e8",edgecolor=GOLD,linewidth=1.8))
    ax_al.text(0.02,0.90,"⚠  Alertas críticas",transform=ax_al.transAxes,
               fontsize=13,fontweight="bold",color="#7a5800",va="top")

    # Producto A en stockout / bajo SS
    a_riesgo = p.loc[
        (p["abc"].astype(str).str.upper()=="A") &
        (p["estado_repo"].astype(str).str.contains("SIN STOCK|PEDIR YA|BAJO SS",case=False,na=False))
    ]
    a_riesgo_n = int(a_riesgo["codigo_norm"].nunique())

    # Producto A con baja cobertura (si existe columna cobertura)
    a_baja_cob_n = 0
    if "cobertura" in p.columns:
        a_baja_cob = p.loc[
            (p["abc"].astype(str).str.upper()=="A") &
            (p["stock_total"].fillna(0)>0) &
            (p["cobertura"].fillna(999)<3)
        ]
        a_baja_cob_n = int(a_baja_cob["codigo_norm"].nunique())

    # Capital en CLAVO específicamente
    cap_clavo = float(p_con_stock.loc[
        p_con_stock["tramo_aging"].astype(str).str.upper()=="CLAVO","valor_usd"].sum())
    skus_clavo = int(p_con_stock.loc[
        p_con_stock["tramo_aging"].astype(str).str.upper()=="CLAVO","codigo_norm"].nunique())

    # Cantidad de productos sin rotación (>90 días con stock)
    sin_rot_n = 0
    if "dias_sv" in p.columns:
        sin_rot = p.loc[(p["stock_total"].fillna(0)>0) & (p["dias_sv"].fillna(0)>=90)]
        sin_rot_n = int(sin_rot["codigo_norm"].nunique())

    bullets = []
    bullets.append((RED,   f"Productos clase A en stockout o bajo SS: {a_riesgo_n}  "
                           f"(pedido urgente — impacto directo en ventas)"))
    if a_baja_cob_n > 0:
        bullets.append((ORANGE,f"Productos A con cobertura <3 meses: {a_baja_cob_n}  (planificar pedido próximo ciclo)"))
    bullets.append((RED,   f"Capital en CLAVO (liquidar): {fmt_money(cap_clavo,'USD',0)} en {skus_clavo} SKUs"))
    if sin_rot_n > 0:
        bullets.append((PURPLE,f"SKUs sin rotación hace más de 90 días (con stock): {sin_rot_n}"))
    bullets.append((NAVY,  f"Total SKUs con stock: {total_skus}  |  "
                           f"Capital total inmovilizado: {fmt_money(total_cap,'USD',0)}"))

    ty = 0.74
    for color,text in bullets[:5]:
        ax_al.text(0.04,ty,"●",color=color,fontsize=14,transform=ax_al.transAxes,
                   va="center",fontweight="bold")
        ax_al.text(0.08,ty,text,color="#3a3a3a",fontsize=11,transform=ax_al.transAxes,va="center")
        ty -= 0.16

    add_footer(fig); save_fig(fig,path)


def chart_inventory_risk(path,risk_table):
    """
    Tabla simplificada: sin columnas Bloque ni Código.
    Columnas: Artículo | Marca | Stock | Q sug. | Valor USD
    Artículo ocupa la mayor parte del ancho para que sea legible.
    """
    fig,ax = plt.subplots(figsize=FIGSIZE)
    ax.axis("off"); ax.set_facecolor("white")
    _banner(ax,"Inventario — riesgo y reposición","")

    if risk_table is None or risk_table.empty:
        ax.text(0.05,0.60,"plan_out no disponible.",transform=ax.transAxes,fontsize=14,color=GRAY)
        add_footer(fig); save_fig(fig,path); return

    vcol = next((c for c in ["valor_usd","Valor_stock_USD","valor_stock_usd"]
                 if c in risk_table.columns),None)

    # Cards resumen por bloque
    summary = (risk_table.groupby("bloque",as_index=False)
               .agg(items=("articulo","nunique"),
                    capital=(vcol,"sum") if vcol else ("articulo","count"))
               .sort_values("capital",ascending=False))

    x0,bw,bh = 0.04,0.22,0.17
    for i,(_,row) in enumerate(summary.head(4).iterrows()):
        x = x0+i*(bw+0.02)
        ax.add_patch(patches.FancyBboxPatch((x,0.66),bw,bh,boxstyle="round,pad=0.01",
            transform=ax.transAxes,facecolor="#f0f4ff",edgecolor=BLUE,linewidth=2))
        ax.text(x+bw/2,0.80,row["bloque"],ha="center",va="center",transform=ax.transAxes,
                fontsize=13,fontweight="bold",color=NAVY)
        ax.text(x+bw/2,0.73,f"{int(row['items'])} SKUs",ha="center",va="center",
                transform=ax.transAxes,fontsize=12,color=BLUE)
        ax.text(x+bw/2,0.67,fmt_money(row["capital"],"USD",0),ha="center",va="center",
                transform=ax.transAxes,fontsize=12,color=ORANGE,fontweight="bold")

    ax.text(0.04,0.625,"SKUs prioritarios",transform=ax.transAxes,
            fontsize=13,fontweight="bold",color=NAVY)

    top = (risk_table.sort_values(vcol,ascending=False) if vcol else risk_table
           ).drop_duplicates(subset=["articulo"]).head(7)

    # Columnas: Artículo (ancho) | Marca | Stock | Q sug. | Valor USD
    hdrs = ["Artículo","Marca","Stock","Q sug.","Valor USD"]
    xp   = [0.04, 0.64, 0.75, 0.84, 0.92]

    ax.add_patch(patches.Rectangle((0.03,0.565),0.94,0.048,transform=ax.transAxes,color=NAVY))
    for x,h in zip(xp,hdrs):
        ax.text(x,0.589,h,transform=ax.transAxes,fontsize=10.5,fontweight="bold",color="white",va="center")

    y=0.505; rh=0.062
    for i,(_,row) in enumerate(top.iterrows()):
        bg = "white" if i%2==0 else "#eef2f7"
        ax.add_patch(patches.Rectangle((0.03,y-rh*0.46),0.94,rh*0.90,transform=ax.transAxes,color=bg))

        art   = shorten(row.get("articulo",""),58)
        marca = shorten(row.get("marca",""),14)
        stock = fmt_num(row.get("stock_total",np.nan),0)
        q_val = row.get("q_sugerida",np.nan)
        q_txt = fmt_num(q_val,0) if pd.notna(q_val) and float(q_val if pd.notna(q_val) else 0)>0 else "—"
        val_v = float(row[vcol]) if vcol and pd.notna(row.get(vcol)) else np.nan

        for x,v in zip(xp,[art,marca,stock,q_txt,fmt_money(val_v,"USD",0)]):
            ax.text(x,y,v,transform=ax.transAxes,fontsize=10,color=NAVY,va="center")
        y -= rh

    add_footer(fig); save_fig(fig,path)


def chart_conclusions(path,conclusions,p):
    yl = p["ytd_label"]
    fig,ax = plt.subplots(figsize=FIGSIZE)
    ax.axis("off"); ax.set_facecolor("white")
    _banner(ax,"Conclusiones ejecutivas y foco de acción",
            f"Período {yl} al {p['max_date'].date()}")
    ax.add_patch(patches.FancyBboxPatch((0.04,0.08),0.92,0.73,boxstyle="round,pad=0.01",
        transform=ax.transAxes,facecolor="#f3f6fb",edgecolor=GRAY_L))
    yy=0.76
    for b in conclusions:
        wrapped = textwrap.fill(b,width=110)
        ax.text(0.07,yy,f"▶  {wrapped}",transform=ax.transAxes,fontsize=13,color="#243244",va="top")
        yy -= max(0.10,0.075+wrapped.count("\n")*0.052)
    add_footer(fig); save_fig(fig,path)

# ─────────────────────────────────────────────────────────────
# 12) PPT
# ─────────────────────────────────────────────────────────────
def _ppt_logo(slide,prs):
    if not LOGO_PATH or not os.path.exists(LOGO_PATH): return
    try: slide.shapes.add_picture(LOGO_PATH,prs.slide_width-Inches(1.5),Inches(0.08),width=Inches(1.2))
    except: pass

def _ppt_pg(slide,prs,n):
    tx = slide.shapes.add_textbox(prs.slide_width-Inches(0.6),prs.slide_height-Inches(0.35),Inches(0.4),Inches(0.2))
    pp = tx.text_frame.paragraphs[0]
    pp.text=str(n); pp.font.size=Pt(9); pp.font.color.rgb=RGBColor(160,160,160); pp.alignment=PP_ALIGN.RIGHT

def ppt_cover(prs,p):
    yl = p["ytd_label"]
    slide=prs.slides.add_slide(prs.slide_layouts[6]); w=prs.slide_width
    ban=slide.shapes.add_shape(MSO_SHAPE.RECTANGLE,0,0,w,Inches(1.65))
    ban.fill.solid(); ban.fill.fore_color.rgb=RGBColor(*BANNER_RGB); ban.line.fill.background()
    def _tx(l,t,ww,hh,txt,sz,bold=False,rgb=WHITE_RGB):
        tb=slide.shapes.add_textbox(Inches(l),Inches(t),Inches(ww),Inches(hh))
        tb.line.fill.background()
        pp=tb.text_frame.paragraphs[0]
        pp.text=txt; pp.font.size=Pt(sz); pp.font.bold=bold; pp.font.color.rgb=RGBColor(*rgb)
    _tx(0.7,0.28,11,0.8,REPORT_TITLE,34,bold=True)
    _tx(0.7,1.20,11,0.6,REPORT_SUBTITLE,22,bold=True,rgb=TEXT_DARK)
    _tx(0.7,1.90,11,0.5,f"2024 completo | 2025 completo | {yl}",11,rgb=(90,90,90))
    _tx(0.7,2.45,11,0.4,f"Generado el {REPORT_DATE:%d/%m/%Y}",10,rgb=(140,140,140))
    lin=slide.shapes.add_shape(MSO_SHAPE.RECTANGLE,Inches(0.7),Inches(1.85),Inches(4.5),Inches(0.04))
    lin.fill.solid(); lin.fill.fore_color.rgb=RGBColor(59,91,165); lin.line.fill.background()
    _ppt_logo(slide,prs)

def ppt_img_slide(prs,title,img,n):
    slide=prs.slides.add_slide(prs.slide_layouts[6]); w,h=prs.slide_width,prs.slide_height
    ban=slide.shapes.add_shape(MSO_SHAPE.RECTANGLE,0,0,w,Inches(0.90))
    ban.fill.solid(); ban.fill.fore_color.rgb=RGBColor(*BANNER_RGB); ban.line.fill.background()
    tx=slide.shapes.add_textbox(Inches(0.45),Inches(0.12),w-Inches(2.2),Inches(0.55))
    tx.line.fill.background()
    pp=tx.text_frame.paragraphs[0]; pp.text=REPORT_TITLE; pp.font.size=Pt(18); pp.font.bold=True
    pp.font.color.rgb=RGBColor(*WHITE_RGB)
    tx2=slide.shapes.add_textbox(Inches(0.45),Inches(0.94),w-Inches(1.0),Inches(0.32))
    tx2.line.fill.background()
    pp2=tx2.text_frame.paragraphs[0]; pp2.text=title; pp2.font.size=Pt(11); pp2.font.bold=True
    pp2.font.color.rgb=RGBColor(*TEXT_DARK)
    slide.shapes.add_picture(img,Inches(0.45),Inches(1.28),width=w-Inches(0.90),height=h-Inches(1.65))
    _ppt_logo(slide,prs); _ppt_pg(slide,prs,n)

# ─────────────────────────────────────────────────────────────
# 13) EXCEL ANEXO
# ─────────────────────────────────────────────────────────────
def write_excel(path,comm,stock,plan,hist_m,three_yr,recon,bm,tc,tp,
                aging,dep_mix,brand_stock,risk,conclusions,p,diag_c,diag_s,diag_p):
    with pd.ExcelWriter(path,engine="openpyxl") as w:
        meta = pd.DataFrame({"campo":["fecha_generacion","hist_desde","max_date","año","periodo"],
                              "valor":[REPORT_DATE.date(),START_HISTORY_DATE.date(),
                                       p["max_date"].date(),p["current_year"],p["ytd_label"]]})
        meta.to_excel(w,sheet_name="00_meta",index=False)
        pd.DataFrame({"conclusion":conclusions}).to_excel(w,sheet_name="00_meta",index=False,startrow=len(meta)+3)
        for df,sn in [(diag_c,"01_diag_comercial"),(diag_s,"01_diag_stock"),(diag_p,"01_diag_planout"),
                      (hist_m,"02_historico_mensual"),(three_yr,"03_tres_anios"),(recon,"04_conciliacion"),
                      (bm,"05_mix_marcas"),(tc,"06_top_clientes"),(tp,"07_top_productos"),
                      (stock,"08_stock_valorizado"),(aging,"09_aging"),
                      (dep_mix,"10_stock_deposito"),(brand_stock,"10b_stock_marca")]:
            df.to_excel(w,sheet_name=sn,index=False)
        (plan if plan is not None else pd.DataFrame({"info":["plan_out no disponible"]})).to_excel(
            w,sheet_name="11_plan_out",index=False)
        (risk if (risk is not None and not risk.empty) else pd.DataFrame({"info":["Sin datos"]})).to_excel(
            w,sheet_name="12_riesgo_reposicion",index=False)
        comm.to_excel(w,sheet_name="13_base_comercial",index=False)

# ─────────────────────────────────────────────────────────────
# 14) MAIN
# ─────────────────────────────────────────────────────────────
def build_report():
    ensure_dir(OUTPUT_DIR)
    cd = os.path.join(OUTPUT_DIR,"charts"); ensure_dir(cd)
    c  = lambda n: os.path.join(cd,n)

    print("▶ Cargando inputs...")
    df_m = load_df("df_merged_usd",        PATH_DF_MERGED_USD)
    df_s = load_df("df_valorizacion_stock", PATH_DF_VALORIZACION_STOCK)
    df_p = load_df("plan_out",             PATH_PLAN_OUT)
    if df_m is None: raise ValueError("df_merged_usd no encontrado.")
    if df_s is None: raise ValueError("df_valorizacion_stock no encontrado.")

    print("▶ Normalizando comercial...")
    comm,diag_c,_ = normalize_commercial(df_m)
    print("▶ Normalizando stock...")
    stock,diag_s,_ = normalize_stock(df_s)

    # Normalizar plan_out ANTES de enriquecer stock con marca
    # (plan_out del inventario_unificado tiene Marca ya asignada)
    print("▶ Normalizando plan_out...")
    plan,diag_p,_ = normalize_planout(df_p)
    if plan is not None:
        print(f"  aging_bucket muestra: {plan['aging_bucket'].value_counts().head(5).to_dict()}")

    # ENRIQUECER STOCK CON MARCA
    # Prioridad: 1) plan_out (Marca directa) → 2) historial ventas → 3) "Sin marca"
    print("▶ Enriqueciendo stock con Marca...")
    stock = enrich_stock_marca(stock, comm, plan_raw=df_p)
    marca_sin = (stock["marca"]=="Sin marca").sum()
    marca_total = len(stock)
    print(f"  Marcas asignadas: {marca_total-marca_sin}/{marca_total} SKUs")
    if marca_sin > 0:
        codigos_sin = stock.loc[stock["marca"]=="Sin marca","codigo"].tolist()[:5]
        print(f"  Sin marca ({marca_sin} SKUs): {codigos_sin}...")

    print("▶ Calculando períodos...")
    p = compute_periods(comm)
    s = build_scopes(comm,p)
    print(f"  Período activo: {p['ytd_label']}  (datos hasta {p['max_date'].date()})")

    print("▶ Calculando tablas...")
    k2024,k2025,k2026 = kpis(s["y2024"]),kpis(s["y2025"]),kpis(s["y2026"])
    hist_m    = monthly_table(s["historico"])
    three_yr  = three_year_table(s["y2024"],s["y2025"],s["y2026"],p["ytd_label"])
    bm,bmd    = brand_mix_table(s["y2024"],s["y2025"],s["y2026"])
    tc        = top_clients_table(s["ytd"])
    tp_ventas = top_products_table(s["ytd"], sort_by="ventas_usd")
    tp_markup = top_products_table(s["ytd"], sort_by="markup_pct")
    tp        = tp_ventas  # alias para compat con Excel anexo
    inv,aging,dep_mix,brand_stock,risk = inventory_summary(stock,plan)
    recon     = build_recon(s["historico"],inv)
    conc      = exec_conclusions(kpis(s["ytd"]),kpis(s["ytd_py"]),bm,tc,inv,risk,p,k2024,k2025,k2026)

    print(f"\n{'─'*52}")
    print(f"  Capital stock  : {fmt_money(inv['capital_total'],'USD',0)}")
    print(f"  Cerrito        : {fmt_money(inv['capital_cerrito'],'USD',0)}")
    print(f"  FULL/ML        : {fmt_money(inv['capital_full'],'USD',0)}")
    print(f"  Capital riesgo : {fmt_money(inv['capital_riesgo'],'USD',0)}")
    print(f"  Marcas en stock: {brand_stock['marca'].tolist()}")
    print(f"{'─'*52}\n")

    yl = p["ytd_label"]

    print("▶ Generando gráficos...")
    slides = []
    charts = [
        ("01_resumen_ejecutivo.png", "Resumen ejecutivo",
         chart_exec_summary,    (k2024,k2025,k2026,inv,conc,p)),
        ("02_historico_mensual.png", f"Evolución histórica mensual desde {START_HISTORY_DATE.year}",
         chart_historical_trend,(hist_m,)),
        ("03_tres_anios.png",        f"Comparativo 2024 / 2025 / {yl}",
         chart_three_years,     (three_yr,p)),
        ("04_conciliacion.png",      "Conciliación vs sistema",
         chart_reconciliation,  (recon,)),
        ("05_mix_marcas.png",        f"Mix de ventas por marca — 2024 / 2025 / {yl}",
         chart_brand_mix,       (bm,bmd,p)),
        ("06_top_clientes.png",      f"Top clientes {yl}",
         chart_top_clients,     (tc,p)),
        ("06b_nuevos_clientes.png",  "Adquisición — clientes nuevos vs recurrentes",
         chart_new_clients,     (s["historico"],p)),
        ("07_top_productos_ventas.png", f"Top productos {yl} — por ventas USD",
         chart_top_products,    (tp_ventas, p, "ventas")),
        ("07b_top_productos_markup.png", f"Top productos {yl} — por markup (los que más dejan)",
         chart_top_products,    (tp_markup, p, "markup")),
        ("08_inventario_foto.png",   "Inventario — capital por marca",
         chart_inventory_photo, (inv,brand_stock)),
        ("08b_inventario_salud.png", "Inventario — salud del stock (rotación vs clavo)",
         chart_inventory_health,(plan,inv)),
        ("09_inventario_riesgo.png", "Inventario — riesgo y reposición",
         chart_inventory_risk,  (risk,)),
        ("10_conclusiones.png",      "Conclusiones y foco de acción",
         chart_conclusions,     (conc,p)),
    ]
    for name,title,fn,args in charts:
        path_img = c(name)
        fn(path_img,*args)
        slides.append((title,path_img))
        print(f"  ✓ {name}")

    print(f"▶ Generando PPT ({len(slides)+1} slides)...")
    prs = Presentation()
    ppt_cover(prs,p)
    for i,(title,img) in enumerate(slides,start=2):
        ppt_img_slide(prs,title,img,i)
    ppt_path = os.path.join(OUTPUT_DIR,MAIN_PPT_NAME)
    prs.save(ppt_path)

    print("▶ Generando Excel...")
    xlsx_path = os.path.join(OUTPUT_DIR,ANNEX_XLSX_NAME)
    write_excel(xlsx_path,comm,stock,plan,hist_m,three_yr,recon,bm,tc,tp,
                aging,dep_mix,brand_stock,risk,conc,p,diag_c,diag_s,diag_p)

    print(f"\n{'='*62}")
    print(f"PPT   → {ppt_path}")
    print(f"Excel → {xlsx_path}")
    print(f"{'='*62}")
    return {"ppt":ppt_path,"xlsx":xlsx_path,"periods":p,"inv":inv}

# ─────────────────────────────────────────────────────────────
# 15) RUN
# ─────────────────────────────────────────────────────────────
result = build_report()


▶ Cargando inputs...
▶ Normalizando comercial...
▶ Normalizando stock...
▶ Normalizando plan_out...
  aging_bucket muestra: {'SIN STOCK': 80, 'OK': 36, 'NUEVO': 32, 'ATENCION': 20, 'CLAVO': 15}
▶ Enriqueciendo stock con Marca...
  Marcas asignadas: 120/134 SKUs
  Sin marca (14 SKUs): ['2GB-SO-DDR4-UNK', 'ABL35-24-20-D-Jun', 'S95N1X4S-0', 'ASF-GE2-T', 'OM4.LC.LC.D10M']...
▶ Calculando períodos...
  Período activo: ene-mar 2026  (datos hasta 2026-03-31)
▶ Calculando tablas...

────────────────────────────────────────────────────
  Capital stock  : US$ 218.145
  Cerrito        : US$ 217.631
  FULL/ML        : US$ 514
  Capital riesgo : US$ 167.036
  Marcas en stock: ['QNAP', '10GteK', 'Kingston', 'OWC', 'LaCie', 'Seagate', 'Western Digital', 'Sin marca', 'Terramaster', 'Synology', 'Asus', 'Generico']
────────────────────────────────────────────────────

▶ Generando gráficos...
  ✓ 01_resumen_ejecutivo.png
  ✓ 02_historico_mensual.png
  ✓ 03_tres_anios.png
  ✓ 04_conciliacion.png
  ✓ 05_mi

/tmp/ipykernel_36231/3596316273.py:195: UserWarning: Glyph 128994 (\N{LARGE GREEN CIRCLE}) missing from font(s) DejaVu Sans.
  fig.savefig(path,dpi=IMG_DPI,facecolor="white",bbox_inches="tight",format="png")
/tmp/ipykernel_36231/3596316273.py:195: UserWarning: Glyph 128993 (\N{LARGE YELLOW CIRCLE}) missing from font(s) DejaVu Sans.
  fig.savefig(path,dpi=IMG_DPI,facecolor="white",bbox_inches="tight",format="png")
/tmp/ipykernel_36231/3596316273.py:195: UserWarning: Glyph 128308 (\N{LARGE RED CIRCLE}) missing from font(s) DejaVu Sans.
  fig.savefig(path,dpi=IMG_DPI,facecolor="white",bbox_inches="tight",format="png")


  ✓ 08b_inventario_salud.png
  ✓ 09_inventario_riesgo.png
  ✓ 10_conclusiones.png
▶ Generando PPT (14 slides)...
▶ Generando Excel...

PPT   → /content/gdrive/My Drive/Grapes/reporte_grapes/resultados/reporte_gerencial_grapes_20260424.pptx
Excel → /content/gdrive/My Drive/Grapes/reporte_grapes/resultados/anexo_reporte_gerencial_grapes_20260424.xlsx


#GRAFICO DE VENTAS MENSUALES


In [59]:
df_merged_usd.columns

Index(['Fecha_de_Carga', 'Deposito', 'Comprobante', 'Cantidad', 'Stock_Anterior', 'Stock_Actual', 'Codigo', 'Articulo', 'Marca', 'Condicion_de_Venta', 'Fecha', 'Cliente_Proveedor', 'provincia_estado_region', 'clase_fiscal',
       'tipos_de_documentos', 'numero_de_documento', 'id_detalle_cliente', 'Nro_cliente_proveedor', 'Precio_sinIVA', 'Total_sinIVA', 'Moneda', 'Tipo_de_Cambio', 'Costo_Ultima_Compra', 'Compr_Asoc', 'Tipo_Cambio_Compr', 'ID', 'Tipo ID',
       'ID_Articulo', 'Clase', 'k_comp', 'k_code', 'es_proxy_remito', 'k_comp_origen_remito', 'Comprobante_origen_remito', 'match_en_ven', 'cantidad_articulo_venta', 'total_neto_articulo_venta', 'total_final_articulo_venta',
       'costo_total_articulo_venta', 'interes_sin_imp_articulo_venta', 'impuestos_articulo_venta', 'profit_articulo_venta', 'profit_ii_articulo_venta', 'total_neto_articulo_venta_usd', 'costo_total_articulo_venta_usd',
       'interes_sin_imp_articulo_venta_usd', 'profit_ii_articulo_venta_usd', 'Codigo_costo', 'co

In [61]:
# ——————————————————————————————————————————————
# Ventas netas mensuales (UI mejorada) — HISTÓRICO COMPLETO
# v5 — iteración sobre v4:
#   - Panel 1: líneas de unidades (antes barras) + líneas de markup USD
#     eje derecho. Más limpio visualmente, coherente con panel 2.
#   - Sin toggle de modo: siempre muestra los dos paneles
#     (panel 2 se desactiva solo si faltan columnas USD).
# ——————————————————————————————————————————————

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, FuncFormatter
import textwrap

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    _WIDGETS_OK = True
except Exception:
    _WIDGETS_OK = False

# ---------- 0) Precondiciones ----------
if "df_merged_usd" not in globals():
    raise NameError("df_merged_usd no existe.")
if "aplicar_ean" not in globals():
    raise NameError("aplicar_ean no existe. Ejecutá la celda de inicialización primero.")

req_cols = ["Fecha_de_Carga","Comprobante","Codigo","Articulo","Marca","Cantidad"]
missing  = [c for c in req_cols if c not in df_merged_usd.columns]
if missing:
    raise KeyError(f"Faltan columnas en df_merged_usd: {missing}")

HAS_USD = all(c in df_merged_usd.columns for c in [
    "total_neto_articulo_venta_usd",
    "profit_ii_articulo_venta_usd",
])

# ---------- 1) Normalización ----------
df0 = df_merged_usd.copy()

df0["Fecha_de_Carga"] = pd.to_datetime(df0["Fecha_de_Carga"], errors="coerce", dayfirst=True)
df0["Marca_norm"]     = df0["Marca"].astype(str).str.strip().str.lower()
df0["Comp_norm"]      = (df0["Comprobante"].astype(str).str.lower()
                         .str.replace(r"\s+", " ", regex=True).str.strip())
df0["Clase_norm"]     = (df0["Clase"].astype(str).str.strip().str.lower()
                         if "Clase" in df0.columns else "")
df0["Articulo"]       = df0["Articulo"].astype(str).str.strip()
df0["Cantidad"]       = pd.to_numeric(df0["Cantidad"], errors="coerce").fillna(0)

if HAS_USD:
    df0["ventas_usd_raw"] = pd.to_numeric(df0["total_neto_articulo_venta_usd"], errors="coerce").fillna(0)
    df0["profit_usd_raw"] = pd.to_numeric(df0["profit_ii_articulo_venta_usd"], errors="coerce").fillna(0)

df0["Codigo_mapped"] = aplicar_ean(df0["Codigo"].astype(str).str.strip().str.upper())
df0["Codigo_norm"] = (
    df0["Codigo_mapped"]
        .str.replace("–", "-", regex=False)
        .str.replace("—", "-", regex=False)
        .str.replace(r"\s+", "-", regex=True)
        .str.replace(r"-{2,}", "-", regex=True)
        .str.lower().str.strip()
)

art_map = (
    df0[df0["Fecha_de_Carga"].notna()]
    .sort_values("Fecha_de_Carga")
    .drop_duplicates("Codigo_norm", keep="last")
    .set_index("Codigo_norm")["Articulo"]
    .to_dict()
)

# ---------- 2) Filtros base ----------
mask_fc     = df0["Comp_norm"].str.startswith(("fc a","fc b","fc c"), na=False)
mask_nc     = df0["Comp_norm"].str.startswith(("nc a","nc b","nc c","nc d"), na=False)
mask_doc    = mask_fc | mask_nc
pat_pres    = r"\b(?:pre|pres|presup|presupuesto|cot|cotiz|cotización)\b"
mask_pres   = df0["Comp_norm"].str.contains(pat_pres, na=False, regex=True)
mask_cls    = df0["Clase_norm"].str.contains("presup", na=False)
mask_compra = df0["Clase_norm"].str.startswith("01.factura", na=False)

base = df0[mask_doc & ~mask_compra & ~mask_pres & ~mask_cls].copy()
base = base[base["Fecha_de_Carga"].notna()].copy()

base["signo"] = np.select(
    [base["Comp_norm"].str.startswith(("fc a","fc b","fc c"), na=False),
     base["Comp_norm"].str.startswith(("nc a","nc b","nc c","nc d"), na=False)],
    [1, -1], default=0
)
base["Cantidad_neta"] = base["Cantidad"].abs() * base["signo"]

if HAS_USD:
    base["ventas_usd_neto"] = base["ventas_usd_raw"].abs() * base["signo"]
    base["profit_usd_neto"] = base["profit_usd_raw"].abs() * base["signo"]
    base["costo_usd_neto"]  = base["ventas_usd_neto"] - base["profit_usd_neto"]

base["Mes"] = base["Fecha_de_Carga"].dt.to_period("M")

# ---------- 3) Rango de meses ----------
if not base.empty:
    first_month = base["Fecha_de_Carga"].min().to_period("M")
    last_month  = base["Fecha_de_Carga"].max().to_period("M")
    meses       = pd.period_range(first_month, last_month, freq="M")
else:
    meses = pd.PeriodIndex([], freq="M")

# ---------- 4) Pivots por métrica ----------
def _build_pivot(col_valor):
    g = base.groupby(["Codigo_norm","Mes"], as_index=False)[col_valor].sum()
    piv = g.pivot_table(index="Codigo_norm", columns="Mes",
                         values=col_valor, aggfunc="sum")
    if len(meses) > 0:
        piv = piv.reindex(columns=meses, fill_value=0).fillna(0)
    return piv

pivot_uni = _build_pivot("Cantidad_neta")

if HAS_USD:
    pivot_ventas = _build_pivot("ventas_usd_neto")
    pivot_profit = _build_pivot("profit_usd_neto")
    pivot_costo  = _build_pivot("costo_usd_neto")
else:
    pivot_ventas = pivot_profit = pivot_costo = None

if not pivot_uni.empty:
    codes_index = pivot_uni.index.tolist()
    artic_index = [art_map.get(c, c) for c in codes_index]
    codigo2art  = {c: art_map.get(c, c) for c in codes_index}
    articulo2cods = {}
    for c, a in zip(codes_index, artic_index):
        articulo2cods.setdefault(a, []).append(c)
else:
    codes_index, artic_index = [], []
    codigo2art, articulo2cods = {}, {}

# ---------- 5) Helpers ----------
def abreviar(s, width=40):
    return textwrap.shorten(str(s), width=width, placeholder="…")

def _fmt_miles(x, _):
    try: return f"{int(round(x)):,}".replace(",", ".")
    except Exception: return str(x)

def _fmt_usd(x, _):
    try:
        if abs(x) >= 1000:
            return f"US$ {x/1000:,.1f}k".replace(",", ".")
        return f"US$ {x:,.0f}".replace(",", ".")
    except Exception:
        return str(x)

# ---------- 6) Plot ----------
def plot_multiple(codigos):
    """
    Siempre 2 paneles (si hay USD disponible):
      Panel 1: líneas de unidades + líneas de markup USD (eje derecho)
      Panel 2: áreas apiladas costo + profit, con ventas como línea
    Si no hay columnas USD, solo panel 1 con unidades.
    """
    if pivot_uni is None or pivot_uni.empty or len(meses) == 0:
        msg = "No hay datos para graficar (FC/NC, post-filtros)."
        if _WIDGETS_OK:
            with out_area:
                clear_output(wait=True); print(msg)
        else:
            print(msg)
        return

    if not codigos:
        msg = "Elegí al menos 1 código."
        if _WIDGETS_OK:
            with out_area:
                clear_output(wait=True); print(msg)
        else:
            print(msg)
        return

    # Límite legibilidad: el panel 2 suma costos+profit; con muchos productos
    # igual queda legible porque es la agregación del grupo, pero el panel 1
    # con muchas líneas se ensucia. Cap en 8.
    warn_msg = ""
    if len(codigos) > 8:
        codigos = codigos[:8]
        warn_msg = "Mostrando los primeros 8 productos (más líneas se superponen)."

    x = [m.to_timestamp() for m in meses]
    colors = plt.cm.tab10.colors

    # ═══════════════ SIN COLUMNAS USD — solo panel de unidades ═══════════════
    if not HAS_USD:
        fig, ax = plt.subplots(figsize=(16, 7))
        ax.set_facecolor("white")
        ax.grid(axis="y", color="#e5e7eb", linestyle="--", alpha=0.5)
        ax.spines[["right","top"]].set_visible(False)

        for i, cod in enumerate(codigos):
            if cod not in pivot_uni.index: continue
            art = codigo2art.get(cod, "")
            y = pivot_uni.loc[cod].values.astype(float)
            ax.plot(x, y, marker="o", linewidth=2, color=colors[i % 10],
                    label=f"{cod} — {abreviar(art, 38)}")
            for xi, yi in zip(x, y):
                if yi != 0:
                    ax.annotate(f"{int(round(yi))}", xy=(xi, yi), xytext=(0, 5),
                                textcoords="offset points", ha="center", va="bottom",
                                fontsize=8, color=colors[i % 10])

        ax.set_title("Ventas netas mensuales — Unidades (FC - NC)",
                     fontsize=15, fontweight="bold", color="#12213f")
        ax.set_ylabel("Unidades netas")
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))
        ax.yaxis.set_major_formatter(FuncFormatter(_fmt_miles))
        ax.legend(fontsize=9, loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0.)
        plt.tight_layout()

        if _WIDGETS_OK:
            with out_area:
                clear_output(wait=True)
                if warn_msg: print(warn_msg)
                display(fig)
            plt.close(fig)
        else:
            if warn_msg: print(warn_msg)
            plt.show()
        return

    # ═══════════════ MODO CON USD — 2 paneles ═══════════════
    fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True,
                              gridspec_kw={"height_ratios":[1, 1]})
    for ax in axes:
        ax.set_facecolor("white")
        ax.grid(axis="y", color="#e5e7eb", linestyle="--", alpha=0.5)
        ax.spines[["top"]].set_visible(False)

    # ── Panel 1: unidades (línea sólida) + markup USD (línea punteada eje der) ──
    for i, cod in enumerate(codigos):
        if cod not in pivot_uni.index: continue
        art  = codigo2art.get(cod, "")
        y_un = pivot_uni.loc[cod].values.astype(float)
        axes[0].plot(x, y_un, linewidth=2.2, marker="o", ms=4,
                     color=colors[i % 10],
                     label=f"{cod} — {abreviar(art, 30)}")

    axes[0].spines["right"].set_visible(True)
    axes[0].yaxis.set_major_locator(MaxNLocator(integer=True))
    axes[0].yaxis.set_major_formatter(FuncFormatter(_fmt_miles))
    axes[0].set_ylabel("Unidades netas", fontsize=11)

    ax0_r = axes[0].twinx()
    ax0_r.spines[["top"]].set_visible(False)
    ax0_r.yaxis.set_major_formatter(FuncFormatter(_fmt_usd))
    ax0_r.set_ylabel("Markup USD (profit absoluto)", color="#355caa", fontsize=11)
    ax0_r.tick_params(axis="y", labelcolor="#355caa", labelsize=9)

    for i, cod in enumerate(codigos):
        if cod not in pivot_profit.index: continue
        y_mk = pivot_profit.loc[cod].values.astype(float)
        ax0_r.plot(x, y_mk, color=colors[i % 10], linewidth=1.5,
                   marker="s", ms=3, linestyle="--", alpha=0.85)

    axes[0].set_title(
        "Unidades netas (línea sólida) + Markup USD (línea punteada, eje derecho)",
        fontsize=13, fontweight="bold", color="#12213f")

    # Leyenda combinada (solo uds + indicador de que markup usa mismo color punteado)
    h1, l1 = axes[0].get_legend_handles_labels()
    axes[0].legend(h1, l1, fontsize=9, loc="upper left",
                   bbox_to_anchor=(1.10, 1), borderaxespad=0.,
                   title="Unidades (mismo color → markup punteado)")

    # ── Panel 2: áreas apiladas costo + profit = ventas ──
    costo_total  = np.zeros(len(meses))
    profit_total = np.zeros(len(meses))
    for cod in codigos:
        if cod in pivot_costo.index:
            costo_total  += pivot_costo.loc[cod].values.astype(float)
        if cod in pivot_profit.index:
            profit_total += pivot_profit.loc[cod].values.astype(float)
    ventas_total = costo_total + profit_total

    costo_plot  = np.clip(costo_total,  0, None)
    profit_plot = np.clip(profit_total, 0, None)

    axes[1].fill_between(x, 0, costo_plot, color="#d97a2b", alpha=0.75,
                          label="Costo USD")
    axes[1].fill_between(x, costo_plot, costo_plot + profit_plot, color="#1f9d75",
                          alpha=0.75, label="Profit USD (markup)")
    axes[1].plot(x, ventas_total, color="#12213f", linewidth=2, marker="o", ms=4,
                  label="Ventas USD (total)")

    for xi, c, pr in zip(x, costo_total, profit_total):
        v = c + pr
        if v <= 0: continue
        mk_pct = (pr / c * 100) if c > 0 else 0
        axes[1].annotate(f"mkup {mk_pct:.0f}%",
                          xy=(xi, v), xytext=(0, 7),
                          textcoords="offset points", ha="center", va="bottom",
                          fontsize=8, color="#12213f")

    axes[1].yaxis.set_major_formatter(FuncFormatter(_fmt_usd))
    axes[1].set_ylabel("USD", fontsize=11)
    axes[1].spines[["right"]].set_visible(False)
    axes[1].set_title(
        f"Desglose financiero {'(agregado)' if len(codigos)>1 else ''} — Costo + Profit = Ventas USD",
        fontsize=13, fontweight="bold", color="#12213f")
    axes[1].legend(fontsize=10, loc="upper left", bbox_to_anchor=(1.02, 1),
                    borderaxespad=0.)

    fig.suptitle("Análisis financiero por producto — histórico mensual",
                 fontsize=15, fontweight="bold", color="#12213f", y=1.00)

    plt.tight_layout()

    if _WIDGETS_OK:
        with out_area:
            clear_output(wait=True)
            if warn_msg: print(warn_msg)
            display(fig)
        plt.close(fig)
    else:
        if warn_msg: print(warn_msg)
        plt.show()

# ---------- 7) Interfaz ----------
if _WIDGETS_OK and (pivot_uni is not None) and (not pivot_uni.empty):

    dd_art = widgets.SelectMultiple(
        options=sorted(list(set(artic_index))),
        description="Artículos",
        layout=widgets.Layout(width="520px", height="320px")
    )

    dd_cod = widgets.SelectMultiple(
        options=sorted(list(set(codes_index))),
        description="Códigos",
        layout=widgets.Layout(width="320px", height="320px")
    )

    txt_filter = widgets.Text(
        value="",
        placeholder="Filtrar códigos…",
        description="Buscar:",
        layout=widgets.Layout(width="320px")
    )

    chk_all   = widgets.Checkbox(value=False, description="Seleccionar TODOS", indent=False)
    btn_plot  = widgets.Button(description="Graficar", button_style="primary")
    btn_clear = widgets.Button(description="Limpiar selección")
    out_area  = widgets.Output()

    def _refresh_codes_options():
        arts_sel = list(dd_art.value)
        if arts_sel:
            cods = sorted({c for a in arts_sel for c in articulo2cods.get(a, [])})
        else:
            cods = sorted(list(set(codes_index)))
        q = txt_filter.value.strip().lower()
        if q:
            cods = [c for c in cods if q in c.lower()]
        dd_cod.options = cods

    def _on_art_change(change):
        if change["name"] == "value":
            chk_all.value = False
            _refresh_codes_options()

    def _on_filter_change(change):
        if change["name"] == "value":
            chk_all.value = False
            _refresh_codes_options()

    dd_art.observe(_on_art_change)
    txt_filter.observe(_on_filter_change)

    def _on_plot_clicked(_):
        cods_to_plot = list(dd_cod.options) if chk_all.value else list(dd_cod.value)
        plot_multiple(cods_to_plot)

    def _on_clear_clicked(_):
        dd_art.value      = ()
        dd_cod.value      = ()
        txt_filter.value  = ""
        chk_all.value     = False
        _refresh_codes_options()
        with out_area:
            clear_output(wait=True)

    btn_plot.on_click(_on_plot_clicked)
    btn_clear.on_click(_on_clear_clicked)

    col_left = widgets.VBox([dd_art])
    col_mid  = widgets.VBox([
        dd_cod, txt_filter, chk_all,
        widgets.HBox([btn_plot, btn_clear])
    ])

    display(widgets.HBox([
        col_left,
        widgets.HTML("<div style='width:20px'></div>"),
        col_mid
    ]))
    display(out_area)
    _refresh_codes_options()

else:
    if pivot_uni is None or pivot_uni.empty:
        print("No hay datos (FC/NC, post-filtros) para graficar.")
    else:
        print("ipywidgets no disponible. Ploteo los primeros 5 códigos.")
        plot_multiple(sorted(list(set(codes_index)))[:5])

Output()